## Environments

In [ ]:
# !pip install -q librosa easydict packaging \
#                 hear21passt timm torchcodec

In [ ]:
# !pip uninstall -y mamba-ssm causal-conv1d

# !pip install causal-conv1d mamba-ssm --no-build-isolation

# !pip install flash-attn

In [ ]:
from abc import ABC, abstractmethod
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime
from os.path import exists, join
from pathlib import Path
from typing import Any, Callable, Dict, List, Literal, Optional, Tuple, Union


import pprint
import copy
import gc
import json
import shutil
import logging
import math
import nltk
import numpy as np
import os
import pandas as pd
import pickle
import plotly.express as px
import random
import re
import seaborn as sns
import tarfile
import time
import warnings
import yaml
import zipfile
import concurrent.futures

import librosa
import matplotlib.pyplot as plt
import torch
import torchaudio
import wandb
import multiprocessing as mp

from easydict import EasyDict
from dotenv import load_dotenv
# from google.colab import userdata
from huggingface_hub import login as hf_login, snapshot_download
from IPython.display import Audio, display
from nltk.corpus import wordnet
from torch import Tensor
from torch.nn.functional import dropout, linear, pad, softmax
from torch.nn.init import constant_
from torch.nn.modules.linear import Linear
from torch.nn.modules.module import Module
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm, trange
from transformers import (
    AutoConfig,
    AutoModel,
    AutoProcessor,
    AutoTokenizer,
    ClapConfig,
    ClapFeatureExtractor,
    ClapModel,
    ClapProcessor,
    EarlyStoppingCallback,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    pipeline,
)

import torch.nn as nn
import torch.nn.functional as F

try:
    from torch.overrides import has_torch_function, handle_torch_function
except:
    from torch._overrides import has_torch_function, handle_torch_function

from sklearn.metrics import precision_recall_curve
from scipy.optimize import linear_sum_assignment

load_dotenv()


HF_TOKEN = os.getenv("HF_TOKEN")
# HF_TOKEN = userdata.get("HF_TOKEN")
PROJECT_NAME = ""
WANDB_PROJECT_NAME = "[DCASE2026] Task6"
# WANDB_API_KEY = userdata.get("WANDB_API_KEY")
WANDB_API_KEY = os.getenv("WANDB_API_KEY")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
wandb.login(key=WANDB_API_KEY)
hf_login(HF_TOKEN)

In [ ]:
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s.%(msecs)03d:%(levelname)s:%(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)

In [ ]:
# nltk.download('wordnet', quiet=True)
# nltk.download('omw-1.4', quiet=True)

In [ ]:
# LOCAL_DIR = Path("/content/drive/MyDrive/Dataset/DCASE2026").resolve()
# DATA_DIR = LOCAL_DIR / "CLOTHO-MOMENT"

# os.listdir(DATA_DIR)

In [ ]:
# TRAIN_DIR = DATA_DIR / "train"
# VAL_DIR = DATA_DIR / "valid"
# TEST_DIR = DATA_DIR / "test"
# PREPROCESSED_DIR = DATA_DIR / "preprocessed"
# FEATURES_DIR = DATA_DIR / "features"


# print(f"Examples from train dir: {os.listdir(TRAIN_DIR)[:5]}")
# print(f"Examples from validation dir: {os.listdir(VAL_DIR)[:5]}")
# print(f"Examples from test dir: {os.listdir(TEST_DIR)[:5]}")
# print(f"Examples from pre-processed dir: {os.listdir(PREPROCESSED_DIR)[:5]}")

## Experiments


In [ ]:
def get_run_name(prefix, lr: float = 2e-5, batch_size: int = 256):
    now = datetime.now().strftime("%m%d-%H%M")
    return f"{prefix}_lr{lr}_bs{batch_size}_{now}"


get_run_name(prefix="test-run-name")

In [ ]:
def clear_gpu_cache():
    print(f"[Before] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[Before] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

    # Clear GPU cache
    torch.cuda.empty_cache()
    # Run garbage collector
    gc.collect()

    # Verify memory is cleared
    print(f"[After] Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"[After] Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")


# Verify memory is cleared
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

In [ ]:
def set_seed(seed, use_cuda=True):
    """Sets the random seed.

    Args:
        seed (int): Seed.
        use_cuda (bool): Use cuda.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if use_cuda:
        torch.cuda.manual_seed_all(seed)

### Basic Utils

In [ ]:
class WandbLogger:
    def __init__(
        self, project_name: str, run_name: str, config: dict = None, entity: str = None
    ):
        """
        Initializes the W&B run.
        """
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=config,
            entity=entity,
            reinit=True,
        )
        self.best_accuracy = 0.0

    def log_metrics(self, metrics, step, prefix="eval"):
        """
        Logs a dictionary of metrics, converting AverageMeter to float.
        """
        log_dict = {}
        for k, v in metrics.items():
            # Extract .avg if it's an AverageMeter, otherwise keep as is
            val = v.avg if hasattr(v, "avg") else v
            log_dict[f"{prefix}/{k}"] = val

        self.run.log(log_dict, step=step)

    def log_artifact(self, model_path, name="model-checkpoint", aliases=["latest"]):
        artifact = wandb.Artifact(name, type="model")
        artifact.add_file(model_path)
        # Log with aliases like 'best' or 'production'
        self.run.log_artifact(artifact, aliases=aliases)

    def finish(self):
        """Closes the W&B run"""
        self.run.finish()

In [ ]:
def write_log(opt, epoch_i, loss_meters, metrics=None, mode="train", **kwargs):

    wandb_logger: WandbLogger = kwargs.get("wandb_logger", None)
    if wandb_logger is not None:
        wandb_logger.log_metrics(loss_meters, epoch_i, mode)
    if mode == "train":
        to_write = opt.train_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i + 1,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
        )
        filename = opt.train_log_filepath
    else:
        to_write = opt.eval_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
            eval_metrics_str=json.dumps(metrics),
        )
        filename = opt.eval_log_filepath

    with open(filename, "a") as f:
        f.write(to_write)


def save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt, **kwargs):
    wandb_logger = kwargs.get("wandb_logger", None)
    wandb_artifact_version = kwargs.get("wandb_artifact_version", [])
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch_i,
        "opt": opt,
    }
    torch.save(checkpoint, opt.ckpt_filepath)
    if wandb_logger is not None:
        wandb_logger.log_artifact(
            opt.ckpt_filepath,
            aliases=["latest"]
            if wandb_artifact_version is None
            else [wandb_artifact_version],
        )


def rename_latest_to_best(latest_file_paths):
    best_file_paths = [e.replace("latest", "best") for e in latest_file_paths]
    for src, tgt in zip(latest_file_paths, best_file_paths):
        os.renames(src, tgt)


def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)


def save_json(data, filename, save_pretty=False, sort_keys=False):
    with open(filename, "w") as f:
        if save_pretty:
            f.write(json.dumps(data, indent=4, sort_keys=sort_keys))
        else:
            json.dump(data, f)


def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(l.strip("\n")) for l in f.readlines()]


def save_jsonl(data, filename):
    """data is a list"""
    with open(filename, "w") as f:
        f.write("\n".join([json.dumps(e) for e in data]))


def save_lines(list_of_str, filepath):
    with open(filepath, "w") as f:
        f.write("\n".join(list_of_str))


def read_lines(filepath):
    with open(filepath, "r") as f:
        return [e.strip("\n") for e in f.readlines()]


def read_yaml(file_path: Union[str, Path]) -> Dict[str, Any]:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"YAML file not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = yaml.safe_load(f)
            return data if data is not None else {}
        except yaml.YAMLError as e:
            raise yaml.YAMLError(f"Error parsing YAML file {file_path}: {e}")


def mkdirp(p):
    if not os.path.exists(p):
        os.makedirs(p)


def flat_list_of_lists(l):
    """flatten a list of lists [[1,2], [3,4]] to [1,2,3,4]"""
    return [item for sublist in l for item in sublist]


def convert_to_seconds(hms_time):
    """convert '00:01:12' to 72 seconds.
    :hms_time (str): time in comma separated string, e.g. '00:01:12'
    :return (int): time in seconds, e.g. 72
    """
    times = [float(t) for t in hms_time.split(":")]
    return times[0] * 3600 + times[1] * 60 + times[2]


def get_video_name_from_url(url):
    return url.split("/")[-1][:-4]


def merge_dicts(list_dicts):
    merged_dict = list_dicts[0].copy()
    for i in range(1, len(list_dicts)):
        merged_dict.update(list_dicts[i])
    return merged_dict


def l2_normalize_np_array(np_array, eps=1e-5):
    """np_array: np.ndarray, (*, D), where the last dim will be normalized"""
    return np_array / (np.linalg.norm(np_array, axis=-1, keepdims=True) + eps)


def make_zipfile(
    src_dir,
    save_path,
    enclosing_dir="",
    exclude_dirs=None,
    exclude_extensions=None,
    exclude_dirs_substring=None,
):
    """make a zip file of root_dir, save it to save_path.
    exclude_paths will be excluded if it is a subdir of root_dir.
    An enclosing_dir is added is specified.
    """
    abs_src = os.path.abspath(src_dir)
    with zipfile.ZipFile(save_path, "w") as zf:
        for dirname, subdirs, files in os.walk(src_dir):
            if exclude_dirs is not None:
                for e_p in exclude_dirs:
                    if e_p in subdirs:
                        subdirs.remove(e_p)
            if exclude_dirs_substring is not None:
                to_rm = []
                for d in subdirs:
                    if exclude_dirs_substring in d:
                        to_rm.append(d)
                for e in to_rm:
                    subdirs.remove(e)
            arcname = os.path.join(enclosing_dir, dirname[len(abs_src) + 1 :])
            zf.write(dirname, arcname)
            for filename in files:
                if exclude_extensions is not None:
                    if os.path.splitext(filename)[1] in exclude_extensions:
                        continue  # do not zip it
                absname = os.path.join(dirname, filename)
                arcname = os.path.join(enclosing_dir, absname[len(abs_src) + 1 :])
                zf.write(absname, arcname)


class AverageMeter(object):
    """Computes and stores the average and current/max/min value"""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10

    def update(self, val, n=1):
        self.max = max(val, self.max)
        self.min = min(val, self.min)
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def dissect_by_lengths(np_array, lengths, dim=0, assert_equal=True):
    """Dissect an array (N, D) into a list a sub-array,
    np_array.shape[0] == sum(lengths), Output is a list of nd arrays, singlton dimention is kept"""
    if assert_equal:
        assert len(np_array) == sum(lengths)
    length_indices = [
        0,
    ]
    for i in range(len(lengths)):
        length_indices.append(length_indices[i] + lengths[i])
    if dim == 0:
        array_list = [
            np_array[length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 1:
        array_list = [
            np_array[:, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 2:
        array_list = [
            np_array[:, :, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    else:
        raise NotImplementedError
    return array_list


def get_ratio_from_counter(counter_obj, threshold=200):
    keys = counter_obj.keys()
    values = counter_obj.values()
    filtered_values = [counter_obj[k] for k in keys if k > threshold]
    return float(sum(filtered_values)) / sum(values)


def get_counter_dist(counter_object, sort_type="none"):
    _sum = sum(counter_object.values())
    dist = {k: float(f"{100 * v / _sum:.2f}") for k, v in counter_object.items()}
    if sort_type == "value":
        dist = OrderedDict(sorted(dist.items(), reverse=True))
    return dist


def get_show_name(vid_name):
    """
    get tvshow name from vid_name
    :param vid_name: video clip name
    :return: tvshow name
    """
    show_list = ["friends", "met", "castle", "house", "grey"]
    vid_name_prefix = vid_name.split("_")[0]
    show_name = vid_name_prefix if vid_name_prefix in show_list else "bbt"
    return show_name


def get_abspaths_by_ext(dir_path, ext=(".jpg",)):
    """Get absolute paths to files in dir_path with extensions specified by ext.
    Note this function does work recursively.
    """
    if isinstance(ext, list):
        ext = tuple(ext)
    if isinstance(ext, str):
        ext = tuple(
            [
                ext,
            ]
        )
    filepaths = [
        os.path.join(root, name)
        for root, dirs, files in os.walk(dir_path)
        for name in files
        if name.endswith(tuple(ext))
    ]
    return filepaths


def get_basename_no_ext(path):
    """'/data/movienet/240p_keyframe_feats/tt7672188.npz' --> 'tt7672188'"""
    return os.path.splitext(os.path.split(path)[1])[0]


def dict_to_markdown(d, max_str_len=120):
    # convert list into its str representation
    d = {k: v.__repr__() if isinstance(v, list) else v for k, v in d.items()}
    # truncate string that is longer than max_str_len
    if max_str_len is not None:
        d = {k: v[-max_str_len:] if isinstance(v, str) else v for k, v in d.items()}
    return pd.DataFrame(d, index=[0]).transpose().to_markdown()

In [ ]:
preprocessd_sample = load_jsonl(
    str(PREPROCESSED_DIR / "clotho_moment_train_release.jsonl")
)

preprocessd_sample[0]

In [ ]:
# y_sample, sr_sample = librosa.load(
#     str(TRAIN_DIR / "Venice_40_640.wav"),
#     sr=None
# )

# y_sample, sr_sample = torchaudio.load(str(TRAIN_DIR / "Venice_40_640.wav"))

In [ ]:
# display(Audio(data=y_sample,
#               rate=sr_sample))

In [ ]:
# plt.figure()
# librosa.display.waveshow(y_sample, sr=sr_sample, alpha=0.7)
# plt.title(f"Waveform – qid {preprocessd_sample[0]["qid"]}", fontsize=14, weight="bold")
# plt.xlabel("Time (s)")
# plt.ylabel("Amplitude")
# plt.tight_layout()
# plt.show()

In [ ]:
# n_fft = 1024
# hop_length = 256

# mel_spec = librosa.feature.melspectrogram(
#     y=y_sample,
#     sr=sr_sample,
#     n_fft=n_fft,
#     hop_length=hop_length,
#     n_mels=40,
#     fmax=sr_sample / 2,
# )
# log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)
# plt.figure()
# librosa.display.specshow(
#     log_mel_spec,
#     sr=sr_sample,
#     hop_length=hop_length,
#     x_axis="time",
#     y_axis="mel",
#     cmap="viridis",
# )
# plt.title("Log‑Mel Spectrogram", fontsize=14, weight="bold")
# plt.colorbar(format="%+2.0f dB")
# plt.tight_layout()
# plt.show()

### Span Utils

In [ ]:
def span_xx_to_cxw(xx_spans):
    """
    Args:
        xx_spans: tensor, (#windows, 2) or (..., 2), each row is a window of format (st, ed)

    Returns:
        cxw_spans: tensor, (#windows, 2), each row is a window of format (center=(st+ed)/2, width=(ed-st))
    >>> spans = torch.Tensor([[0, 1], [0.2, 0.4]])
    >>> span_xx_to_cxw(spans)
    tensor([[0.5000, 1.0000],
        [0.3000, 0.2000]])
    >>> spans = torch.Tensor([[[0, 1], [0.2, 0.4]]])
    >>> span_xx_to_cxw(spans)
    tensor([[[0.5000, 1.0000],
         [0.3000, 0.2000]]])
    """
    center = xx_spans.sum(-1) * 0.5
    width = xx_spans[..., 1] - xx_spans[..., 0]
    return torch.stack([center, width], dim=-1)


def span_cxw_to_xx(cxw_spans):
    """
    Args:
        cxw_spans: tensor, (#windows, 2) or (..., 2), the last dim is a row denoting a window of format (center, width)

    >>> spans = torch.Tensor([[0.5000, 1.0000], [0.3000, 0.2000]])
    >>> span_cxw_to_xx(spans)
    tensor([[0.0000, 1.0000],
        [0.2000, 0.4000]])
    >>> spans = torch.Tensor([[[0.5000, 1.0000], [0.3000, 0.2000]]])
    >>> span_cxw_to_xx(spans)
    tensor([[[0.0000, 1.0000],
        [0.2000, 0.4000]]])
    """
    x1 = cxw_spans[..., 0] - 0.5 * cxw_spans[..., 1]
    x2 = cxw_spans[..., 0] + 0.5 * cxw_spans[..., 1]
    return torch.stack([x1, x2], dim=-1)


def temporal_iou(spans1, spans2):
    """
    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        iou: (N, M) torch.Tensor
        union: (N, M) torch.Tensor
    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> temporal_iou(test_spans1, test_spans2)
    (tensor([[0.6667, 0.2000],
         [0.0000, 0.5000]]),
     tensor([[0.3000, 1.0000],
             [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = torch.max(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.min(spans1[:, None, 1], spans2[:, 1])  # (N, M)

    inter = (right - left).clamp(min=0)  # (N, M)
    union = areas1[:, None] + areas2 - inter  # (N, M)

    iou = inter / union
    return iou, union


def temporal_intersection_over_pred(gt_spans, pred_spans):
    """intersection over the second input spans
    Args:
        gt_spans: (N, 2),
        pred_spans: (M, 2)

    Returns:

    """
    left = torch.max(gt_spans[:, None, 0], pred_spans[:, 0])
    right = torch.min(gt_spans[:, None, 1], pred_spans[:, 1])

    inter = (right - left).clamp(min=0)  # (N, M)
    inter_over_pred = inter / (pred_spans[:, 1] - pred_spans[:, 0])
    return inter_over_pred


def generalized_temporal_iou(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    assert (spans1[:, 1] >= spans1[:, 0]).all()
    assert (spans2[:, 1] >= spans2[:, 0]).all()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area


def generalized_temporal_iou_(spans1, spans2):
    """
    Generalized IoU from https://giou.stanford.edu/
    Also reference to DETR implementation of generalized_box_iou
    https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40

    Args:
        spans1: (N, 2) torch.Tensor, each row defines a span in xx format [st, ed]
        spans2: (M, 2) torch.Tensor, ...

    Returns:
        giou: (N, M) torch.Tensor

    >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
    >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
    >>> generalized_temporal_iou(test_spans1, test_spans2)
    tensor([[ 0.6667,  0.2000],
        [-0.2000,  0.5000]])
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # (N, M)
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # (N, M)
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area

### Tensor Utils

In [ ]:
def pad_sequences_1d(
    sequences, dtype=torch.long, device=torch.device("cpu"), fixed_length=None
):
    """Pad a single-nested list or a sequence of n-d array (torch.tensor or np.ndarray)
    into a (n+1)-d array, only allow the first dim has variable lengths.
    Args:
        sequences: list(n-d tensor or list)
        dtype: np.dtype or torch.dtype
        device:
        fixed_length: pad all seq in sequences to fixed length. All seq should have a length <= fixed_length.
            return will be of shape [len(sequences), fixed_length, ...]
    Returns:
        padded_seqs: ((n+1)-d tensor) padded with zeros
        mask: (2d tensor) of the same shape as the first two dims of padded_seqs,
              1 indicate valid, 0 otherwise
    Examples:
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=torch.long)
        >>> test_data_3d = [torch.randn(2,3,4), torch.randn(4,3,4), torch.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=torch.float)
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=np.float32)
        >>> test_data_3d = [np.random.randn(2,3,4), np.random.randn(4,3,4), np.random.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=np.float32)
    """
    if isinstance(sequences[0], list):
        if "torch" in str(dtype):
            sequences = [torch.tensor(s, dtype=dtype, device=device) for s in sequences]
        else:
            sequences = [np.asarray(s, dtype=dtype) for s in sequences]

    extra_dims = sequences[0].shape[
        1:
    ]  # the extra dims should be the same for all elements
    lengths = [len(seq) for seq in sequences]
    if fixed_length is not None:
        max_length = fixed_length
    else:
        max_length = max(lengths)
    if isinstance(sequences[0], torch.Tensor):
        assert "torch" in str(dtype), "dtype and input type does not match"
        padded_seqs = torch.zeros(
            (len(sequences), max_length) + extra_dims, dtype=dtype, device=device
        )
        mask = torch.zeros(
            (len(sequences), max_length), dtype=torch.float32, device=device
        )
    else:  # np
        assert "numpy" in str(dtype), "dtype and input type does not match"
        padded_seqs = np.zeros((len(sequences), max_length) + extra_dims, dtype=dtype)
        mask = np.zeros((len(sequences), max_length), dtype=np.float32)

    for idx, seq in enumerate(sequences):
        end = lengths[idx]
        padded_seqs[idx, :end] = seq
        mask[idx, :end] = 1
    return padded_seqs, mask  # , lengths


def pad_sequences_2d(sequences, dtype=torch.long):
    """Pad a double-nested list or a sequence of n-d torch tensor into a (n+1)-d tensor,
        only allow the first two dims has variable lengths
    Args:
        sequences: list(n-d tensor or list)
        dtype: torch.long for word indices / torch.float (float32) for other cases
    Returns:
    Examples:
        >>> test_data_list = [[[1, 3, 5], [3, 7, 4, 1]], [[98, 34, 11, 89, 90], [22], [34, 56]],]
        >>> pad_sequences_2d(test_data_list, dtype=torch.long)  # torch.Size([2, 3, 5])
        >>> test_data_3d = [torch.randn(2,2,4), torch.randn(4,3,4), torch.randn(1,5,4)]
        >>> pad_sequences_2d(test_data_3d, dtype=torch.float)  # torch.Size([2, 3, 5])
        >>> test_data_3d2 = [[torch.randn(2,4), ], [torch.randn(3,4), torch.randn(5,4)]]
        >>> pad_sequences_2d(test_data_3d2, dtype=torch.float)  # torch.Size([2, 3, 5])
    # TODO add support for numpy array
    """
    bsz = len(sequences)
    para_lengths = [len(seq) for seq in sequences]
    max_para_len = max(para_lengths)
    sen_lengths = [[len(word_seq) for word_seq in seq] for seq in sequences]
    max_sen_len = max([max(e) for e in sen_lengths])

    if isinstance(sequences[0], torch.Tensor):
        extra_dims = sequences[0].shape[2:]
    elif isinstance(sequences[0][0], torch.Tensor):
        extra_dims = sequences[0][0].shape[1:]
    else:
        sequences = [
            [torch.Tensor(word_seq, dtype=dtype) for word_seq in seq]
            for seq in sequences
        ]
        extra_dims = ()

    padded_seqs = torch.zeros(
        (bsz, max_para_len, max_sen_len) + extra_dims, dtype=dtype
    )
    mask = torch.zeros(bsz, max_para_len, max_sen_len).float()

    for b_i in range(bsz):
        for sen_i, sen_l in enumerate(sen_lengths[b_i]):
            padded_seqs[b_i, sen_i, :sen_l] = sequences[b_i][sen_i]
            mask[b_i, sen_i, :sen_l] = 1
    return padded_seqs, mask  # , sen_lengths

In [ ]:
# Example

test_data_list = [[1, 2, 3], [1, 2], [3, 4, 7, 9]]
pad_sequences_1d(test_data_list, dtype=torch.long)

### Dataset

1. Prepare Dataset
2. Prepare Dataloader

In [ ]:
import os
import yaml

from easydict import EasyDict


GLOBAL_OPT = None


class BaseOptions(object):
    def __init__(self, config_path):
        self.config_path = config_path
        self.opt = {
            "use_amp": False,
            "use_compile": False,
            "use_focal_loss": False,
            "use_flash_attention": False,
            "model_variant": "baseline",
        }

    @property
    def option(self):
        if len(self.opt) == 0:
            raise RuntimeError("option is empty. Did you run parse()?")
        return self.opt

    def update(self, yaml_file):
        with open(yaml_file, "r") as f:
            yml = yaml.load(f, Loader=yaml.FullLoader)
            if yml:
                self.opt.update(yml)
        self.opt = EasyDict(self.opt)
        global GLOBAL_OPT
        GLOBAL_OPT = self.opt

    def parse(self):
        with open(self.config_path, "r") as f:
            yml = yaml.load(f, Loader=yaml.FullLoader)
            if yml:
                self.opt.update(yml)

        self.opt = EasyDict(self.opt)
        global GLOBAL_OPT
        GLOBAL_OPT = self.opt
        self.opt.ckpt_filepath = os.path.join(
            self.opt.results_dir, self.opt.ckpt_filename
        )
        self.opt.train_log_filepath = os.path.join(
            self.opt.results_dir, self.opt.train_log_filename
        )
        self.opt.eval_log_filepath = os.path.join(
            self.opt.results_dir, self.opt.eval_log_filename
        )



class Settings:
    """Application settings loaded from environment variables."""

    ROOT_DIR = Path("./").resolve()
    DEVICE = detect_device()

    """
    Pipeline settings:
    """


settings = Settings()




import json
import numpy as np
from sklearn.metrics import precision_recall_curve

"""
Copied from MMAction2
https://github.com/open-mmlab/mmaction2/blob/master/mmaction/core/evaluation/eval_detection.py
"""


def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(line.strip("\n")) for line in f.readlines()]


def compute_temporal_iou_batch_paired(pred_windows, gt_windows):
    """compute intersection-over-union along temporal axis for each pair of windows in pred_windows and gt_windows.
    Args:
        pred_windows: np.ndarray, (N, 2), [st (float), ed (float)] * N
        gt_windows: np.ndarray, (N, 2), [st (float), ed (float)] * N
    Returns:
        iou (float): np.ndarray, (N, )

    References:
        for np.divide with zeros, see https://stackoverflow.com/a/37977222
    """
    intersection = np.maximum(
        0,
        np.minimum(pred_windows[:, 1], gt_windows[:, 1])
        - np.maximum(pred_windows[:, 0], gt_windows[:, 0]),
    )
    union = np.maximum(pred_windows[:, 1], gt_windows[:, 1]) - np.minimum(
        pred_windows[:, 0], gt_windows[:, 0]
    )  # not the correct union though
    return np.divide(
        intersection, union, out=np.zeros_like(intersection), where=union != 0
    )


def compute_temporal_iou_batch_cross(spans1, spans2):
    """
    Args:
        spans1: (N, 2) np.ndarray, each row defines a span [st, ed]
        spans2: (M, 2) np.ndarray, ...

    Returns:
        iou: (N, M) np.ndarray
        union: (N, M) np.ndarray
    >>> spans1 = np.array([[0, 0.2, 0.9], [0.5, 1.0, 0.2]])
    >>> spans2 = np.array([[0, 0.3], [0., 1.0]])
    >>> compute_temporal_iou_batch_cross(spans1, spans2)
    (tensor([[0.6667, 0.2000],
         [0.0000, 0.5000]]),
     tensor([[0.3000, 1.0000],
             [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = np.maximum(spans1[:, None, 0], spans2[None, :, 0])  # (N, M)
    right = np.minimum(spans1[:, None, 1], spans2[None, :, 1])  # (N, M)

    inter = np.clip(right - left, 0, None)  # (N, M)
    union = areas1[:, None] + areas2[None, :] - inter  # (N, M)

    iou = inter / union
    return iou, union


def interpolated_precision_recall(precision, recall):
    """Interpolated AP - VOCdevkit from VOC 2011.

    Args:
        precision (np.ndarray): The precision of different thresholds.
        recall (np.ndarray): The recall of different thresholds.

    Returns：
        float: Average precision score.
    """
    mprecision = np.hstack([[0], precision, [0]])
    mrecall = np.hstack([[0], recall, [1]])
    for i in range(len(mprecision) - 1)[::-1]:
        mprecision[i] = max(mprecision[i], mprecision[i + 1])
    idx = np.where(mrecall[1::] != mrecall[0:-1])[0] + 1
    ap = np.sum((mrecall[idx] - mrecall[idx - 1]) * mprecision[idx])
    return ap


def compute_average_precision_detection(
    ground_truth, prediction, tiou_thresholds=np.linspace(0.5, 0.95, 10)
):
    """Compute average precision (detection task) between ground truth and
    predictions data frames. If multiple predictions occurs for the same
    predicted segment, only the one with highest score is matches as true
    positive. This code is greatly inspired by Pascal VOC devkit.

    Args:
        ground_truth (list[dict]): List containing the ground truth instances
            (dictionaries). Required keys are 'video-id', 't-start' and
            't-end'.
        prediction (list[dict]): List containing the prediction instances
            (dictionaries). Required keys are: 'video-id', 't-start', 't-end'
            and 'score'.
        tiou_thresholds (np.ndarray): A 1darray indicates the temporal
            intersection over union threshold, which is optional.
            Default: ``np.linspace(0.5, 0.95, 10)``.

    Returns:
        Float: ap, Average precision score.
    """
    num_thresholds = len(tiou_thresholds)
    num_gts = len(ground_truth)
    num_preds = len(prediction)
    ap = np.zeros(num_thresholds)
    if len(prediction) == 0:
        return ap

    num_positive = float(num_gts)
    lock_gt = np.ones((num_thresholds, num_gts)) * -1
    # Sort predictions by decreasing score order.
    prediction.sort(key=lambda x: -x["score"])
    # Initialize true positive and false positive vectors.
    tp = np.zeros((num_thresholds, num_preds))
    fp = np.zeros((num_thresholds, num_preds))

    # Adaptation to query faster
    ground_truth_by_videoid = {}
    for i, item in enumerate(ground_truth):
        item["index"] = i
        ground_truth_by_videoid.setdefault(item["video-id"], []).append(item)

    # Assigning true positive to truly grount truth instances.
    for idx, pred in enumerate(prediction):
        if pred["video-id"] in ground_truth_by_videoid:
            gts = ground_truth_by_videoid[pred["video-id"]]
        else:
            fp[:, idx] = 1
            continue

        _pred = np.array(
            [
                [pred["t-start"], pred["t-end"]],
            ]
        )
        _gt = np.array([[gt["t-start"], gt["t-end"]] for gt in gts])
        tiou_arr = compute_temporal_iou_batch_cross(_pred, _gt)[0]

        tiou_arr = tiou_arr.reshape(-1)
        # We would like to retrieve the predictions with highest tiou score.
        tiou_sorted_idx = tiou_arr.argsort()[::-1]
        for t_idx, tiou_threshold in enumerate(tiou_thresholds):
            for j_idx in tiou_sorted_idx:
                if tiou_arr[j_idx] < tiou_threshold:
                    fp[t_idx, idx] = 1
                    break
                if lock_gt[t_idx, gts[j_idx]["index"]] >= 0:
                    continue
                # Assign as true positive after the filters above.
                tp[t_idx, idx] = 1
                lock_gt[t_idx, gts[j_idx]["index"]] = idx
                break

            if fp[t_idx, idx] == 0 and tp[t_idx, idx] == 0:
                fp[t_idx, idx] = 1

    tp_cumsum = np.cumsum(tp, axis=1).astype(float)
    fp_cumsum = np.cumsum(fp, axis=1).astype(float)
    recall_cumsum = tp_cumsum / num_positive

    precision_cumsum = tp_cumsum / (tp_cumsum + fp_cumsum)

    for t_idx in range(len(tiou_thresholds)):
        ap[t_idx] = interpolated_precision_recall(
            precision_cumsum[t_idx, :], recall_cumsum[t_idx, :]
        )
    return ap


def get_ap(y_true, y_predict, interpolate=True, point_11=False):
    """
    Average precision in different formats: (non-) interpolated and/or 11-point approximated
    point_11=True and interpolate=True corresponds to the 11-point interpolated AP used in
    the PASCAL VOC challenge up to the 2008 edition and has been verfied against the vlfeat implementation
    The exact average precision (interpolate=False, point_11=False) corresponds to the one of vl_feat

    :param y_true: list/ numpy vector of true labels in {0,1} for each element
    :param y_predict: predicted score for each element
    :param interpolate: Use interpolation?
    :param point_11: Use 11-point approximation to average precision?
    :return: average precision

    ref: https://github.com/gyglim/video2gif_dataset/blob/master/v2g_evaluation/__init__.py

    """
    # Check inputs
    assert len(y_true) == len(y_predict), (
        "Prediction and ground truth need to be of the same length"
    )
    if len(set(y_true)) == 1:
        if y_true[0] == 0:
            return 0  # True labels are all zeros
            # raise ValueError('True labels cannot all be zero')
        else:
            return 1
    else:
        assert sorted(set(y_true)) == [0, 1], (
            "Ground truth can only contain elements {0,1}"
        )

    # Compute precision and recall
    precision, recall, _ = precision_recall_curve(y_true, y_predict)
    recall = recall.astype(np.float32)

    if interpolate:  # Compute the interpolated precision
        for i in range(1, len(precision)):
            precision[i] = max(precision[i - 1], precision[i])

    if point_11:  # Compute the 11-point approximated AP
        precision_11 = [
            precision[np.where(recall >= t)[0][-1]] for t in np.arange(0, 1.01, 0.1)
        ]
        return np.mean(precision_11)
    else:  # Compute the AP using precision at every additionally recalled sample
        indices = np.where(np.diff(recall))
        return np.mean(precision[indices])


# --- File: src.standalone_eval.eval ---
"""
Copyright $today.year LY Corporation

LY Corporation licenses this file to you under the Apache License,
version 2.0 (the "License"); you may not use this file except in compliance
with the License. You may obtain a copy of the License at:

  https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS, WITHOUT
WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the
License for the specific language governing permissions and limitations
under the License.

MIT License

Copyright (c) 2021 Jie Lei

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

import numpy as np
from collections import OrderedDict, defaultdict
import json
import time
import copy
import multiprocessing as mp
from standalone_eval.utils import (
    compute_average_precision_detection,
    compute_temporal_iou_batch_cross,
    compute_temporal_iou_batch_paired,
    load_jsonl,
    get_ap,
)


def compute_average_precision_detection_wrapper(
    input_triple, tiou_thresholds=np.linspace(0.5, 0.95, 10)
):
    qid, ground_truth, prediction = input_triple
    scores = compute_average_precision_detection(
        ground_truth, prediction, tiou_thresholds=tiou_thresholds
    )
    return qid, scores


def compute_mr_ap(
    submission,
    ground_truth,
    iou_thds=np.linspace(0.5, 0.95, 10),
    max_gt_windows=None,
    max_pred_windows=10,
    num_workers=8,
    chunksize=50,
):
    iou_thds = [float(f"{e:.2f}") for e in iou_thds]
    pred_qid2data = defaultdict(list)
    for d in submission:
        pred_windows = (
            d["pred_relevant_windows"][:max_pred_windows]
            if max_pred_windows is not None
            else d["pred_relevant_windows"]
        )
        qid = d["qid"]
        for w in pred_windows:
            pred_qid2data[qid].append(
                {
                    "video-id": d["qid"],  # in order to use the API
                    "t-start": w[0],
                    "t-end": w[1],
                    "score": w[2],
                }
            )

    gt_qid2data = defaultdict(list)
    for d in ground_truth:
        gt_windows = (
            d["relevant_windows"][:max_gt_windows]
            if max_gt_windows is not None
            else d["relevant_windows"]
        )
        qid = d["qid"]
        for w in gt_windows:
            gt_qid2data[qid].append(
                {"video-id": d["qid"], "t-start": w[0], "t-end": w[1]}
            )
    qid2ap_list = {}
    # start_time = time.time()
    data_triples = [
        [qid, gt_qid2data[qid], pred_qid2data[qid]] for qid in pred_qid2data
    ]
    from functools import partial

    compute_ap_from_triple = partial(
        compute_average_precision_detection_wrapper, tiou_thresholds=iou_thds
    )

    if num_workers > 1:
        with mp.Pool(num_workers) as pool:
            for qid, scores in pool.imap_unordered(
                compute_ap_from_triple, data_triples, chunksize=chunksize
            ):
                qid2ap_list[qid] = scores
    else:
        for data_triple in data_triples:
            qid, scores = compute_ap_from_triple(data_triple)
            qid2ap_list[qid] = scores

    # print(f"compute_average_precision_detection {time.time() - start_time:.2f} seconds.")
    ap_array = np.array(list(qid2ap_list.values()))  # (#queries, #thd)
    ap_thds = ap_array.mean(0)  # mAP at different IoU thresholds.
    iou_thd2ap = dict(zip([str(e) for e in iou_thds], ap_thds))
    iou_thd2ap["average"] = np.mean(ap_thds)
    # formatting
    iou_thd2ap = {k: float(f"{100 * v:.2f}") for k, v in iou_thd2ap.items()}
    return iou_thd2ap


def compute_mr_r1(submission, ground_truth, iou_thds=np.linspace(0.5, 0.95, 10)):
    """If a predicted segment has IoU >= iou_thd with one of the 1st GT segment, we define it positive"""
    iou_thds = [float(f"{e:.2f}") for e in iou_thds]
    pred_qid2window = {
        d["qid"]: d["pred_relevant_windows"][0][:2] for d in submission
    }  # :2 rm scores
    # gt_qid2window = {d["qid"]: d["relevant_windows"][0] for d in ground_truth}
    gt_qid2window = {}
    for d in ground_truth:
        cur_gt_windows = d["relevant_windows"]
        cur_qid = d["qid"]
        cur_max_iou_idx = 0
        if len(cur_gt_windows) > 0:  # select the GT window that has the highest IoU
            cur_ious = compute_temporal_iou_batch_cross(
                np.array([pred_qid2window[cur_qid]]), np.array(d["relevant_windows"])
            )[0]
            cur_max_iou_idx = np.argmax(cur_ious)
        gt_qid2window[cur_qid] = cur_gt_windows[cur_max_iou_idx]

    qids = list(pred_qid2window.keys())
    pred_windows = np.array([pred_qid2window[k] for k in qids]).astype(float)
    gt_windows = np.array([gt_qid2window[k] for k in qids]).astype(float)
    pred_gt_iou = compute_temporal_iou_batch_paired(pred_windows, gt_windows)
    iou_thd2recall_at_one = {}
    for thd in iou_thds:
        iou_thd2recall_at_one[str(thd)] = float(
            f"{np.mean(pred_gt_iou >= thd) * 100:.2f}"
        )
    return iou_thd2recall_at_one


def get_window_len(window):
    return window[1] - window[0]


def get_data_by_range(submission, ground_truth, len_range):
    """keep queries with ground truth window length in the specified length range.
    Args:
        submission:
        ground_truth:
        len_range: [min_l (int), max_l (int)]. the range is (min_l, max_l], i.e., min_l < l <= max_l
    """
    min_l, max_l = len_range
    if min_l == 0 and max_l == 150:  # min and max l in dataset
        return submission, ground_truth

    # only keep ground truth with windows in the specified length range
    # if multiple GT windows exists, we only keep the ones in the range
    ground_truth_in_range = []
    gt_qids_in_range = set()
    for d in ground_truth:
        rel_windows_in_range = [
            w for w in d["relevant_windows"] if min_l < get_window_len(w) <= max_l
        ]
        if len(rel_windows_in_range) > 0:
            d = copy.deepcopy(d)
            d["relevant_windows"] = rel_windows_in_range
            ground_truth_in_range.append(d)
            gt_qids_in_range.add(d["qid"])

    # keep only submissions for ground_truth_in_range
    submission_in_range = []
    for d in submission:
        if d["qid"] in gt_qids_in_range:
            submission_in_range.append(copy.deepcopy(d))

    return submission_in_range, ground_truth_in_range


def eval_moment_retrieval(submission, ground_truth, verbose=True):
    # length_ranges = [[0, 10], [10, 30], [30, 150], [0, 150], ]  #
    # range_names = ["short", "middle", "long", "full"]
    length_ranges = [[0, 1500]]  # TODO: cover all examples?
    range_names = ["full"]

    ret_metrics = {}
    for l_range, name in zip(length_ranges, range_names):
        if verbose:
            start_time = time.time()
        _submission, _ground_truth = get_data_by_range(
            submission, ground_truth, l_range
        )
        print(
            f"{name}: {l_range}, {len(_ground_truth)}/{len(ground_truth)}="
            f"{100 * len(_ground_truth) / len(ground_truth):.2f} examples."
        )
        iou_thd2average_precision = compute_mr_ap(
            _submission, _ground_truth, num_workers=8, chunksize=50
        )
        iou_thd2recall_at_one = compute_mr_r1(_submission, _ground_truth)
        ret_metrics[name] = {
            "MR-mAP": iou_thd2average_precision,
            "MR-R1": iou_thd2recall_at_one,
        }
        if verbose:
            print(
                f"[eval_moment_retrieval] [{name}] {time.time() - start_time:.2f} seconds"
            )
    return ret_metrics


def compute_hl_hit1(qid2preds, qid2gt_scores_binary):
    qid2max_scored_clip_idx = {
        k: np.argmax(v["pred_saliency_scores"]) for k, v in qid2preds.items()
    }
    hit_scores = np.zeros((len(qid2preds), 3))
    qids = list(qid2preds.keys())
    for idx, qid in enumerate(qids):
        pred_clip_idx = qid2max_scored_clip_idx[qid]
        gt_scores_binary = qid2gt_scores_binary[qid]  # (#clips, 3)
        if pred_clip_idx < len(gt_scores_binary):
            hit_scores[idx] = gt_scores_binary[pred_clip_idx]
    # aggregate scores from 3 separate annotations (3 workers) by taking the max.
    # then average scores from all queries.
    hit_at_one = float(f"{100 * np.mean(np.max(hit_scores, 1)):.2f}")
    return hit_at_one


def compute_hl_ap(qid2preds, qid2gt_scores_binary, num_workers=8, chunksize=50):
    qid2pred_scores = {k: v["pred_saliency_scores"] for k, v in qid2preds.items()}
    ap_scores = np.zeros((len(qid2preds), 3))  # (#preds, 3)
    qids = list(qid2preds.keys())
    input_tuples = []
    for idx, qid in enumerate(qids):
        for w_idx in range(3):  # annotation score idx
            y_true = qid2gt_scores_binary[qid][:, w_idx]
            y_predict = np.array(qid2pred_scores[qid])
            input_tuples.append((idx, w_idx, y_true, y_predict))

    if num_workers > 1:
        with mp.Pool(num_workers) as pool:
            for idx, w_idx, score in pool.imap_unordered(
                compute_ap_from_tuple, input_tuples, chunksize=chunksize
            ):
                ap_scores[idx, w_idx] = score
    else:
        for input_tuple in input_tuples:
            idx, w_idx, score = compute_ap_from_tuple(input_tuple)
            ap_scores[idx, w_idx] = score

    # it's the same if we first average across different annotations, then average across queries
    # since all queries have the same #annotations.
    mean_ap = float(f"{100 * np.mean(ap_scores):.2f}")
    return mean_ap


def compute_ap_from_tuple(input_tuple):
    idx, w_idx, y_true, y_predict = input_tuple
    if len(y_true) < len(y_predict):
        # print(f"len(y_true) < len(y_predict) {len(y_true), len(y_predict)}")
        y_predict = y_predict[: len(y_true)]
    elif len(y_true) > len(y_predict):
        # print(f"len(y_true) > len(y_predict) {len(y_true), len(y_predict)}")
        _y_predict = np.zeros(len(y_true))
        _y_predict[: len(y_predict)] = y_predict
        y_predict = _y_predict

    score = get_ap(y_true, y_predict)
    return idx, w_idx, score


def mk_gt_scores(gt_data, clip_length=2):
    """gt_data, dict,"""
    num_clips = int(gt_data["duration"] / clip_length)
    saliency_scores_full_video = np.zeros((num_clips, 3))
    relevant_clip_ids = np.array(gt_data["relevant_clip_ids"])  # (#relevant_clip_ids, )
    saliency_scores_relevant_clips = np.array(
        gt_data["saliency_scores"]
    )  # (#relevant_clip_ids, 3)
    saliency_scores_full_video[relevant_clip_ids] = saliency_scores_relevant_clips
    return saliency_scores_full_video  # (#clips_in_video, 3)  the scores are in range [0, 4]


def eval_highlight(submission, ground_truth, verbose=True):
    """
    Args:
        submission:
        ground_truth:
        verbose:
    """
    qid2preds = {d["qid"]: d for d in submission}
    qid2gt_scores_full_range = {
        d["qid"]: mk_gt_scores(d) for d in ground_truth
    }  # scores in range [0, 4]
    # gt_saliency_score_min: int, in [0, 1, 2, 3, 4]. The minimum score for a positive clip.
    gt_saliency_score_min_list = [2, 3, 4]
    saliency_score_names = ["Fair", "Good", "VeryGood"]
    highlight_det_metrics = {}
    for gt_saliency_score_min, score_name in zip(
        gt_saliency_score_min_list, saliency_score_names
    ):
        start_time = time.time()
        qid2gt_scores_binary = {
            k: (v >= gt_saliency_score_min).astype(float)
            for k, v in qid2gt_scores_full_range.items()
        }  # scores in [0, 1]
        hit_at_one = compute_hl_hit1(qid2preds, qid2gt_scores_binary)
        mean_ap = compute_hl_ap(qid2preds, qid2gt_scores_binary)
        highlight_det_metrics[f"HL-min-{score_name}"] = {
            "HL-mAP": mean_ap,
            "HL-Hit1": hit_at_one,
        }
        if verbose:
            print(
                f"Calculating highlight scores with min score {gt_saliency_score_min} ({score_name})"
            )
            print(f"Time cost {time.time() - start_time:.2f} seconds")
    return highlight_det_metrics


def eval_submission(submission, ground_truth, verbose=True, match_number=True):
    """
    Args:
        submission: list(dict), each dict is {
            qid: str,
            query: str,
            vid: str,
            pred_relevant_windows: list([st, ed]),
            pred_saliency_scores: list(float), len == #clips in video.
                i.e., each clip in the video will have a saliency score.
        }
        ground_truth: list(dict), each dict is     {
          "qid": 7803,
          "query": "Man in gray top walks from outside to inside.",
          "duration": 150,
          "vid": "RoripwjYFp8_360.0_510.0",
          "relevant_clip_ids": [13, 14, 15, 16, 17]
          "saliency_scores": [[4, 4, 2], [3, 4, 2], [2, 2, 3], [2, 2, 2], [0, 1, 3]]
               each sublist corresponds to one clip in relevant_clip_ids.
               The 3 elements in the sublist are scores from 3 different workers. The
               scores are in [0, 1, 2, 3, 4], meaning [Very Bad, ..., Good, Very Good]
        }
        verbose:
        match_number:

    Returns:

    """
    pred_qids = set([e["qid"] for e in submission])
    gt_qids = set([e["qid"] for e in ground_truth])
    if match_number:
        assert pred_qids == gt_qids, (
            "qids in ground_truth and submission must match. "
            "use `match_number=False` if you wish to disable this check"
        )
    else:  # only leave the items that exists in both submission and ground_truth
        shared_qids = pred_qids.intersection(gt_qids)
        submission = [e for e in submission if e["qid"] in shared_qids]
        ground_truth = [e for e in ground_truth if e["qid"] in shared_qids]

    eval_metrics = {}
    eval_metrics_brief = OrderedDict()
    if "pred_relevant_windows" in submission[0]:
        moment_ret_scores = eval_moment_retrieval(
            submission, ground_truth, verbose=verbose
        )
        eval_metrics.update(moment_ret_scores)
        moment_ret_scores_brief = {
            "MR-full-mAP": moment_ret_scores["full"]["MR-mAP"]["average"],
            "MR-full-mAP@0.5": moment_ret_scores["full"]["MR-mAP"]["0.5"],
            "MR-full-mAP@0.75": moment_ret_scores["full"]["MR-mAP"]["0.75"],
            "MR-full-R1@0.5": moment_ret_scores["full"]["MR-R1"]["0.5"],
            "MR-full-R1@0.7": moment_ret_scores["full"]["MR-R1"]["0.7"],
        }
        eval_metrics_brief.update(
            sorted(
                [(k, v) for k, v in moment_ret_scores_brief.items()], key=lambda x: x[0]
            )
        )

    if "pred_saliency_scores" in submission[0]:
        highlight_det_scores = eval_highlight(submission, ground_truth, verbose=verbose)
        eval_metrics.update(highlight_det_scores)
        highlight_det_scores_brief = dict(
            [
                (f"{k}-{sub_k.split('-')[1]}", v[sub_k])
                for k, v in highlight_det_scores.items()
                for sub_k in v
            ]
        )
        eval_metrics_brief.update(highlight_det_scores_brief)

    # sort by keys
    final_eval_metrics = OrderedDict()
    final_eval_metrics["brief"] = eval_metrics_brief
    final_eval_metrics.update(
        sorted([(k, v) for k, v in eval_metrics.items()], key=lambda x: x[0])
    )
    return final_eval_metrics


def eval_main():
    import argparse

    parser = argparse.ArgumentParser(
        description="Moments and Highlights Evaluation Script"
    )
    parser.add_argument(
        "--submission_path", type=str, help="path to generated prediction file"
    )
    parser.add_argument("--gt_path", type=str, help="path to GT file")
    parser.add_argument("--save_path", type=str, help="path to save the results")
    parser.add_argument("--not_verbose", action="store_true")
    args = parser.parse_args()

    verbose = not args.not_verbose
    submission = load_jsonl(args.submission_path)
    gt = load_jsonl(args.gt_path)
    results = eval_submission(submission, gt, verbose=verbose)
    if verbose:
        print(json.dumps(results, indent=4))

    with open(args.save_path, "w") as f:
        f.write(json.dumps(results, indent=4))


if __name__ == "__main__":
    eval_main()


# --- File: src.utils.basic_utils ---
import os
import json
import zipfile
import numpy as np
import pickle
import torch
from collections import OrderedDict
import pandas as pd
from pathlib import Path
from typing import Any, Dict, Union

import yaml


def save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt):
    checkpoint = {
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "lr_scheduler": lr_scheduler.state_dict(),
        "epoch": epoch_i,
        "opt": opt,
    }
    torch.save(checkpoint, opt.ckpt_filepath)


def rename_latest_to_best(latest_file_paths):
    best_file_paths = [e.replace("latest", "best") for e in latest_file_paths]
    for src, tgt in zip(latest_file_paths, best_file_paths):
        os.renames(src, tgt)


def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)


def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_json(filename):
    with open(filename, "r") as f:
        return json.load(f)


def save_json(data, filename, save_pretty=False, sort_keys=False):
    with open(filename, "w") as f:
        if save_pretty:
            f.write(json.dumps(data, indent=4, sort_keys=sort_keys))
        else:
            json.dump(data, f)


def load_jsonl(filename):
    with open(filename, "r") as f:
        return [json.loads(line.strip("\n")) for line in f.readlines()]


def save_jsonl(data, filename):
    """data is a list"""
    with open(filename, "w") as f:
        f.write("\n".join([json.dumps(e) for e in data]))


def save_lines(list_of_str, filepath):
    with open(filepath, "w") as f:
        f.write("\n".join(list_of_str))


def read_lines(filepath):
    with open(filepath, "r") as f:
        return [e.strip("\n") for e in f.readlines()]


def mkdirp(p):
    if not os.path.exists(p):
        os.makedirs(p)


def flat_list_of_lists(nested_list):
    """flatten a list of lists [[1,2], [3,4]] to [1,2,3,4]"""
    return [item for sublist in nested_list for item in sublist]


def convert_to_seconds(hms_time):
    """convert '00:01:12' to 72 seconds.
    :hms_time (str): time in comma separated string, e.g. '00:01:12'
    :return (int): time in seconds, e.g. 72
    """
    times = [float(t) for t in hms_time.split(":")]
    return times[0] * 3600 + times[1] * 60 + times[2]


def get_video_name_from_url(url):
    return url.split("/")[-1][:-4]


def merge_dicts(list_dicts):
    merged_dict = list_dicts[0].copy()
    for i in range(1, len(list_dicts)):
        merged_dict.update(list_dicts[i])
    return merged_dict


def l2_normalize_np_array(np_array, eps=1e-5):
    """np_array: np.ndarray, (*, D), where the last dim will be normalized"""
    return np_array / (np.linalg.norm(np_array, axis=-1, keepdims=True) + eps)


def make_zipfile(
    src_dir,
    save_path,
    enclosing_dir="",
    exclude_dirs=None,
    exclude_extensions=None,
    exclude_dirs_substring=None,
):
    """make a zip file of root_dir, save it to save_path.
    exclude_paths will be excluded if it is a subdir of root_dir.
    An enclosing_dir is added is specified.
    """
    abs_src = os.path.abspath(src_dir)
    with zipfile.ZipFile(save_path, "w") as zf:
        for dirname, subdirs, files in os.walk(src_dir):
            if exclude_dirs is not None:
                for e_p in exclude_dirs:
                    if e_p in subdirs:
                        subdirs.remove(e_p)
            if exclude_dirs_substring is not None:
                to_rm = []
                for d in subdirs:
                    if exclude_dirs_substring in d:
                        to_rm.append(d)
                for e in to_rm:
                    subdirs.remove(e)
            arcname = os.path.join(enclosing_dir, dirname[len(abs_src) + 1 :])
            zf.write(dirname, arcname)
            for filename in files:
                if exclude_extensions is not None:
                    if os.path.splitext(filename)[1] in exclude_extensions:
                        continue  # do not zip it
                absname = os.path.join(dirname, filename)
                arcname = os.path.join(enclosing_dir, absname[len(abs_src) + 1 :])
                zf.write(absname, arcname)


class AverageMeter(object):
    """Computes and stores the average and current/max/min value"""

    def __init__(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0
        self.max = -1e10
        self.min = 1e10

    def update(self, val, n=1):
        self.max = max(val, self.max)
        self.min = min(val, self.min)
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


def dissect_by_lengths(np_array, lengths, dim=0, assert_equal=True):
    """Dissect an array (N, D) into a list a sub-array,
    np_array.shape[0] == sum(lengths), Output is a list of nd arrays, singlton dimention is kept"""
    if assert_equal:
        assert len(np_array) == sum(lengths)
    length_indices = [
        0,
    ]
    for i in range(len(lengths)):
        length_indices.append(length_indices[i] + lengths[i])
    if dim == 0:
        array_list = [
            np_array[length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 1:
        array_list = [
            np_array[:, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    elif dim == 2:
        array_list = [
            np_array[:, :, length_indices[i] : length_indices[i + 1]]
            for i in range(len(lengths))
        ]
    else:
        raise NotImplementedError
    return array_list


def get_ratio_from_counter(counter_obj, threshold=200):
    keys = counter_obj.keys()
    values = counter_obj.values()
    filtered_values = [counter_obj[k] for k in keys if k > threshold]
    return float(sum(filtered_values)) / sum(values)


def get_counter_dist(counter_object, sort_type="none"):
    _sum = sum(counter_object.values())
    dist = {k: float(f"{100 * v / _sum:.2f}") for k, v in counter_object.items()}
    if sort_type == "value":
        dist = OrderedDict(sorted(dist.items(), reverse=True))
    return dist


def get_show_name(vid_name):
    """
    get tvshow name from vid_name
    :param vid_name: video clip name
    :return: tvshow name
    """
    show_list = ["friends", "met", "castle", "house", "grey"]
    vid_name_prefix = vid_name.split("_")[0]
    show_name = vid_name_prefix if vid_name_prefix in show_list else "bbt"
    return show_name


def get_abspaths_by_ext(dir_path, ext=(".jpg",)):
    """Get absolute paths to files in dir_path with extensions specified by ext.
    Note this function does work recursively.
    """
    if isinstance(ext, list):
        ext = tuple(ext)
    if isinstance(ext, str):
        ext = tuple(
            [
                ext,
            ]
        )
    filepaths = [
        os.path.join(root, name)
        for root, dirs, files in os.walk(dir_path)
        for name in files
        if name.endswith(tuple(ext))
    ]
    return filepaths


def get_basename_no_ext(path):
    """'/data/movienet/240p_keyframe_feats/tt7672188.npz' --> 'tt7672188'"""
    return os.path.splitext(os.path.split(path)[1])[0]


def dict_to_markdown(d, max_str_len=120):
    # convert list into its str representation
    d = {k: v.__repr__() if isinstance(v, list) else v for k, v in d.items()}
    # truncate string that is longer than max_str_len
    if max_str_len is not None:
        d = {k: v[-max_str_len:] if isinstance(v, str) else v for k, v in d.items()}
    return pd.DataFrame(d, index=[0]).transpose().to_markdown()


def get_dir_size(path="."):
    total_size = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    # entry.stat() is cached on some systems, making this very fast
                    total_size += entry.stat().size
                elif entry.is_dir():
                    # Recursively call the function for subdirectories
                    total_size += get_dir_size(entry.path)
    except PermissionError:
        # Handle folders you don't have access to
        return 0
    return total_size


def read_yaml(file_path: Union[str, Path]) -> Dict[str, Any]:
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"YAML file not found: {file_path}")

    with open(file_path, "r", encoding="utf-8") as f:
        try:
            data = yaml.safe_load(f)
            return data if data is not None else {}
        except yaml.YAMLError as e:
            raise yaml.YAMLError(f"Error parsing YAML file {file_path}: {e}")


def write_yaml(
    data: Dict[str, Any],
    file_path: Union[str, Path],
    default_flow_style: bool = False,
    sort_keys: bool = False,
) -> None:
    file_path = Path(file_path)
    file_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with open(file_path, "w", encoding="utf-8") as f:
            yaml.dump(
                data,
                f,
                default_flow_style=default_flow_style,
                sort_keys=sort_keys,
                allow_unicode=True,
            )
    except IOError as e:
        raise IOError(f"Error writing YAML file {file_path}: {e}")


# --- File: src.utils.log_utils ---
import time
import json
import wandb


def write_log(opt, epoch_i: int, loss_meters, metrics=None, mode="train"):
    # log
    if mode == "train":
        to_write = opt.train_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i + 1,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
        )
        filename = opt.train_log_filepath
    else:
        to_write = opt.eval_log_txt_formatter.format(
            time_str=time.strftime("%Y_%m_%d_%H_%M_%S"),
            epoch=epoch_i,
            loss_str=" ".join(
                ["{} {:.4f}".format(k, v.avg) for k, v in loss_meters.items()]
            ),
            eval_metrics_str=json.dumps(metrics),
        )
        filename = opt.eval_log_filepath

    import os
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, "a") as f:
        f.write(to_write)


class WandbLogger:
    def __init__(
        self, project_name: str, run_name: str, config: dict = None, entity: str = None
    ):
        """
        Initializes the W&B run.
        """
        self.run = wandb.init(
            project=project_name,
            name=run_name,
            config=config,
            entity=entity,
            reinit=True,
        )
        self.best_accuracy = 0.0

    def log_metrics(self, metrics, step, prefix="eval"):
        """
        Logs a dictionary of metrics.

        Args:
            metrics (dict): The dictionary from your eval function.
            step (int): Current global step or epoch.
            prefix (str): Dashboard grouping (e.g., 'train' or 'eval').
        """
        # Format keys: {'loss': 0.5} -> {'eval/loss': 0.5}
        log_dict = {f"{prefix}/{k}": v for k, v in metrics.items()}

        # Log to W&B
        self.run.log(log_dict, step=step)

        # Optional: Track Best Metric Logic
        if "accuracy" in metrics:
            if metrics["accuracy"] > self.best_accuracy:
                self.best_accuracy = metrics["accuracy"]
                self.run.summary["best_accuracy"] = self.best_accuracy
                print(f"New best accuracy: {self.best_accuracy:.4f}")

    def log_artifact(self, model_path, name="model-checkpoint"):
        """Save your model file to W&B"""
        artifact = wandb.Artifact(name, type="model")
        artifact.add_file(model_path)
        self.run.log_artifact(artifact)

    def finish(self):
        """Closes the W&B run"""
        self.run.finish()


# --- File: src.utils.model_utils ---
import torch
import copy


def count_parameters(model: torch.nn.Module, verbose: bool = True) -> dict:
    """Count number of parameters in PyTorch model,
    References: https://discuss.pytorch.org/t/how-do-i-check-the-number-of-parameters-of-a-model/4325/7.

    from utils.utils import count_parameters
    count_parameters(model)
    import sys
    sys.exit(1)
    """
    n_all = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_frozen = n_all - n_trainable
    if verbose:
        print(
            "Parameter Count: all {:,d}; trainable {:,d}; frozen {:,d}".format(
                n_all, n_trainable, n_frozen
            )
        )
    return {
        "n_all": n_all,
        "n_trainable": n_trainable,
        "n_frozen": n_frozen,
    }


def detect_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    elif torch.backends.mps.is_available():
        return torch.device("mps")
    else:
        return torch.device("cpu")


class ModelEMA(torch.nn.Module):
    def __init__(self, model, decay=0.999, device=None):
        super().__init__()
        # make a copy of the model for accumulating moving average of weights
        self.module = copy.deepcopy(model)
        self.module.eval()
        self.decay = decay
        self.device = device  # perform ema on different device from model if set
        if self.device is not None:
            self.module.to(device=device)

    def _update(self, model, update_fn):
        with torch.no_grad():
            for ema_v, model_v in zip(
                self.module.state_dict().values(), model.state_dict().values()
            ):
                if self.device is not None:
                    model_v = model_v.to(device=self.device)
                ema_v.copy_(update_fn(ema_v, model_v))

    def update(self, model):
        self._update(
            model, update_fn=lambda e, m: self.decay * e + (1.0 - self.decay) * m
        )

    def set(self, model):
        self._update(model, update_fn=lambda e, m: m)


# --- File: src.utils.span_utils ---
"""Module for span-related utility functions."""  # noqa: WPS402

import math
from typing import Any, List, Tuple

import torch
from torch import Tensor

EPS = 1e-6


def span_xx_to_cxw(xx_spans: Tensor) -> Tensor:
    """
    Convert spans from start-end format to center-width format.

    Args:
        xx_spans (Tensor): Tensor of shape (#windows, 2), where each row represents a span with the format (start, end).

    Returns:
        Tensor: Tensor where each row represents a span in the format (center, width).

    Examples:
        >>> spans = torch.Tensor([[0, 1], [0.2, 0.4]])
        >>> span_xx_to_cxw(spans)
        tensor([[0.5000, 1.0000],
                [0.3000, 0.2000]])

        >>> spans = torch.Tensor([[[0, 1], [0.2, 0.4]]])
        >>> span_xx_to_cxw(spans)
        tensor([[[0.5000, 1.0000],
                [0.3000, 0.2000]]])
    """
    center = xx_spans.sum(-1) * 0.5
    width = xx_spans[..., 1] - xx_spans[..., 0]
    return torch.stack([center, width], dim=-1)


def span_cxw_to_xx(cxw_spans: Tensor) -> Tensor:
    """
    Convert spans from center-width format to start-end format.

    Args:
        cxw_spans (Tensor): Tensor of shape (#wndws, 2), each row represents a span with the format (center, width).

    Returns:
        Tensor: A tensor of the same shape as `cxw_spans`, where each row represents a span in the format (start, end).

    Examples:
        >>> spans = torch.Tensor([[0.5000, 1.0000], [0.3000, 0.2000]])
        >>> span_cxw_to_xx(spans)
        tensor([[0.0000, 1.0000],
                [0.2000, 0.4000]])

        >>> spans = torch.Tensor([[[0.5000, 1.0000], [0.3000, 0.2000]]])
        >>> span_cxw_to_xx(spans)
        tensor([[[0.0000, 1.0000],
                [0.2000, 0.4000]]])
    """
    x_start = cxw_spans[..., 0] - 0.5 * cxw_spans[..., 1]
    x_end = cxw_spans[..., 0] + 0.5 * cxw_spans[..., 1]
    return torch.stack([x_start, x_end], dim=-1)


def temporal_iou(spans1: Tensor, spans2: Tensor) -> Tuple[Tensor, Tensor]:
    """
    Calculate the temporal Intersection over Union (IoU) and union of pairs of spans.

    Args:
        spans1 (Tensor): A (N, 2) torch.Tensor, where each row defines a span [start, end].
        spans2 (Tensor): A (M, 2) torch.Tensor, where each row defines a span [start, end].

    Returns:
        Tuple[Tensor, Tensor]: A tuple containing two (N, M) tensors:
            - The first tensor contains the IoU for each pair of spans.
            - The second tensor contains the union for each pair of spans.

    Examples:
        >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
        >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
        >>> temporal_iou(test_spans1, test_spans2)
        (tensor([[0.6667, 0.2000],
                [0.0000, 0.5000]]),
        tensor([[0.3000, 1.0000],
                [0.8000, 1.0000]]))
    """
    areas1 = spans1[:, 1] - spans1[:, 0]  # (N, )
    areas2 = spans2[:, 1] - spans2[:, 0]  # (M, )

    left = torch.max(spans1[:, None, 0], spans2[:, 0])  # noqa: WPS221
    right = torch.min(spans1[:, None, 1], spans2[:, 1])  # noqa: WPS221

    inter = (right - left).clamp(min=0)  # (N, M)
    union = areas1[:, None] + areas2 - inter  # (N, M)

    iou = inter / union
    return iou, union


def temporal_intersection_over_pred(gt_spans: Tensor, pred_spans: Tensor) -> Tensor:  # noqa: WPS118
    """
    Calculate the Intersection over Prediction (IoP) for pairs of ground truth and predicted spans.

    This function computes the IoP for each pair of ground truth and predicted spans.
    The IoP is defined as the intersection of the spans divided by the span of the predicted span.

    Args:
        gt_spans (Tensor): A (N, 2) tensor, where each row represents a ground truth span [start, end].
        pred_spans (Tensor): A (M, 2) tensor, where each row represents a predicted span [start, end].

    Returns:
        Tensor: A (N, M) tensor containing the IoP for each pair of ground truth and predicted spans.
    """
    left = torch.max(gt_spans[:, None, 0], pred_spans[:, 0])  # noqa: WPS221
    right = torch.min(gt_spans[:, None, 1], pred_spans[:, 1])  # noqa: WPS221
    inter = (right - left).clamp(min=0)  # (N, M)
    return inter / (pred_spans[:, 1] - pred_spans[:, 0])  # inter_over_pred


def generalized_temporal_iou(spans1: Tensor, spans2: Tensor) -> Tensor:
    """
    Calculate the Generalized Intersection over Union (GIoU) for pairs of spans.

    This function computes the GIoU for each pair of spans. The GIoU is an extension of the IoU metric that also takes
    into account the size of the smallest enclosing span that contains both spans in the pair.
    It provides a more accurate measure of overlap, especially when the spans do not intersect.

    Args:
        spans1 (Tensor): A (N, 2) Tensor, where each row defines a span in [start, end] format.
        spans2 (Tensor): A (M, 2) Tensor, where each row defines a span in [start, end] format.

    Returns:
        Tensor: A (N, M) tensor containing the GIoU for each pair of spans.

    Examples:
        >>> test_spans1 = torch.Tensor([[0, 0.2], [0.5, 1.0]])
        >>> test_spans2 = torch.Tensor([[0, 0.3], [0., 1.0]])
        >>> generalized_temporal_iou(test_spans1, test_spans2)
        tensor([[ 0.6667,  0.2000],
                [-0.2000,  0.5000]])

    References:
        - Generalized IoU: https://giou.stanford.edu/
        - DETR implementation of gIoU: https://github.com/facebookresearch/detr/blob/master/util/box_ops.py#L40
    """
    spans1 = spans1.float()
    spans2 = spans2.float()
    # Sort prediction spans to ensure start <= end, preventing assertion crashes on noisy early iterations
    spans1 = torch.stack([torch.min(spans1, dim=-1)[0], torch.max(spans1, dim=-1)[0]], dim=-1)
    spans2 = torch.stack([torch.min(spans2, dim=-1)[0], torch.max(spans2, dim=-1)[0]], dim=-1)
    assert (spans1[:, 1] >= spans1[:, 0]).all()
    assert (spans2[:, 1] >= spans2[:, 0]).all()
    iou, union = temporal_iou(spans1, spans2)

    left = torch.min(spans1[:, None, 0], spans2[:, 0])  # noqa: WPS221
    right = torch.max(spans1[:, None, 1], spans2[:, 1])  # noqa: WPS221
    enclosing_area = (right - left).clamp(min=0)  # (N, M)

    return iou - (enclosing_area - union) / enclosing_area


def temporal_intersection_criteria(pred_spans: Tensor, gt_spans: Tensor) -> Tuple[Tensor, Tensor]:
    """
    Calculate the temporal Intersection over GT area and intersection of pairs of spans.

    Args:
        pred_spans (Tensor): A (N, 2) torch.Tensor, where each row defines a span [start, end].
        gt_spans (Tensor): A (M, 2) torch.Tensor, where each row defines a span [start, end].

    Returns:
        Tuple[Tensor, Tensor]: A tuple containing two (N, M) tensors:
            - The first tensor contains the Intersection over GT
            - The second tensor contains the intersection for each pair of spans.

    Examples:
        >>> pred_spans = torch.Tensor([[0, 0.2], [0.5, 1.0]])
        >>> gt_spans = torch.Tensor([[0, 0.3], [0., 1.0]])
        >>> temporal_iou(pred_spans, gt_spans)
        (tensor([[0.6667, 0.2000],
                [0.0000, 0.5000]]),
        tensor([[0.2000, 0.2000],
                [0.0000, 0.5000]]))
    """
    gt_areas = gt_spans[:, 1] - gt_spans[:, 0]  # (M, )

    left = torch.max(pred_spans[:, None, 0], gt_spans[:, 0])  # noqa: WPS221
    right = torch.min(pred_spans[:, None, 1], gt_spans[:, 1])  # noqa: WPS221

    inter = (right - left).clamp(min=0)  # (N, M)
    return inter / (gt_areas + EPS), inter


class SpanList:
    """
    This class represents a set of spans.

    The spans are represented as a Nx2 Tensor.
    In order to uniquely determine the spans with respect to an video, we also store the corresponding video sizes.
    """

    def __init__(self, spans: Tensor, size: int, mode="xx"):
        """Initialize the SpanList.

        Args:
            spans (Tensor): A Nx2 Tensor representing the spans.
            size (int): The size of the video.
            mode (str): Format of the span. Defaults to "xx".

        Raises:
            ValueError: If the mode is not "xx" or "cxw".
        """
        if mode not in {"xx", "cxw"}:
            raise ValueError("mode should be 'xx' or 'cxw'")

        self.spans = spans
        self.size = size
        self.mode = mode
        self.extra_fields: dict = {}

    def add_field(self, field: str, field_data: Any):
        """Add a field to the SpanList.

        Args:
            field (str): The name of the field.
            field_data (Any): The data to be stored in the field.
        """
        self.extra_fields[field] = field_data

    def get_field(self, field: str) -> Any:
        """Get the data stored in a field.

        Args:
            field (str): The name of the field.

        Returns:
            Any: The data stored in the field.
        """
        return self.extra_fields[field]

    def fields(self) -> List[str]:
        """Get the names of the fields stored in the SpanList.

        Returns:
            List[str]: The names of the fields stored in the SpanList.
        """
        return list(self.extra_fields.keys())

    def copy_extra_fields(self, spans: "SpanList") -> None:
        """Copy the extra fields from another SpanList.

        Args:
            spans (SpanList): The SpanList to copy the extra fields from.
        """
        for key, value in spans.extra_fields.items():
            self.extra_fields[key] = value

    def _split_into_xx(self) -> Tuple[Tensor, Tensor]:
        """Split the spans into the format (xmin, xmax).

        Returns:
            Tuple[Tensor, Tensor]: The spans in the format (xmin, xmax).
        """
        if self.mode == "xx":
            xmin, xmax = self.spans.split(1, dim=-1)  # type: ignore
            return xmin, xmax  # noqa: WPS331
        xmin, width = self.spans.split(1, dim=-1)  # type: ignore
        return xmin, xmin + (width - 1).clamp(min=0)

    def convert(self, mode: str) -> "SpanList":
        """Convert the spans to a different format.

        Args:
            mode (str): The format to convert the spans to.

        Returns:
            "SpanList": The spans in the new format.

        Raises:
            ValueError: If the mode is not "xx" or "cxw".
        """
        if mode not in {"xx", "cxw"}:
            raise ValueError("mode should be 'xx' or 'cxw'")
        if mode == self.mode:
            return self

        xmin, xmax = self._split_into_xx()
        if mode == "xx":
            spans = torch.cat((xmin, xmax), dim=-1)
            spans_list = SpanList(spans, self.size, mode=mode)
        else:
            spans = torch.cat((xmin, xmax - xmin + 1), dim=-1)
            spans_list = SpanList(spans, self.size, mode=mode)
        spans_list.copy_extra_fields(self)
        return spans_list

    def __getitem__(self, item: int) -> "SpanList":
        """Get a subset of the spans.

        Args:
            item (int): The index of the span to get.

        Returns:
            SpanList: The subset of spans.
        """
        spans = SpanList(self.spans[item], self.size, self.mode)
        for key, value in self.extra_fields.items():
            spans.add_field(key, value[item])
        return spans

    def __len__(self) -> int:
        """Get the number of spans.

        Returns:
            int: The number of spans.
        """
        return self.spans.shape[0]

    def __repr__(self):
        """Get a string representation of the SpanList.

        Returns:
            str: A string representation of the SpanList.
        """
        string = self.__class__.__name__ + "("  # noqa: WPS336
        string += f"num_boxes={len(self)}, "  # noqa: WPS336,WPS237
        string += f"video_length={self.size}, "  # noqa: WPS336
        string += f"mode={self.mode})"  # noqa: WPS336
        return string


def cat_boxlist(spans: List["SpanList"]):
    """
    Concatenate a list of SpanList (having the same video size) into a single SpanList.

    Args:
        spans (List["SpanList"]): A list of SpanList to concatenate.

    Returns:
        List["SpanList"]: The concatenated SpanList.
    """
    assert isinstance(spans, (list, tuple))
    assert all(isinstance(span, SpanList) for span in spans)

    size = spans[0].size
    assert all(span.size == size for span in spans)

    mode = spans[0].mode
    assert all(span.mode == mode for span in spans)

    fields = set(spans[0].fields())
    assert all(set(span.fields()) == fields for span in spans)

    cat_boxes = SpanList(torch.cat([span.spans for span in spans], dim=0), size, mode)  # noqa: WPS221

    for field in fields:
        data = torch.cat([bbox.get_field(field) for bbox in spans], dim=0)
        cat_boxes.add_field(field, data)

    return cat_boxes


def encode_spans(gt_boxes: Tensor, anchors: Tensor) -> Tensor:
    """Encode spans into deltas between anchors and ground truth boxes.

    Args:
        gt_boxes (Tensor): Ground truth boxes in the format (start, end).
        anchors (Tensor): Anchors in the format (start, end).

    Returns:
        Tensor: Encoded deltas between anchors and ground truth boxes.
    """
    ex_widths = anchors[:, 1] - anchors[:, 0] + 1  # TODO: WHY DO WE ADD 1?
    ex_ctr_x = (anchors[:, 0] + anchors[:, 1]) / 2

    gt_widths = gt_boxes[:, 1] - gt_boxes[:, 0] + 1  # TODO: WHY DO WE ADD 1?
    gt_ctr_x = (gt_boxes[:, 0] + gt_boxes[:, 1]) / 2

    wx, ww = (10.0, 5.0)  # noqa: WPS111
    targets_dx = wx * (gt_ctr_x - ex_ctr_x) / ex_widths
    targets_dw = ww * torch.log(gt_widths / ex_widths)

    return torch.stack((targets_dx, targets_dw), dim=1)  # type: ignore


def decode_spans(preds: Tensor, anchors: Tensor) -> Tensor:
    """Decode deltas into spans.

    Args:
        preds (Tensor): Predictions in the format (dx, dw).
        anchors (Tensor): Anchors in the format (start, end).

    Returns:
        Tensor: Decoded spans.
    """
    anchors = anchors.to(preds.dtype)

    widths = anchors[:, 1] - anchors[:, 0] + 1  # TODO: WHY DO WE ADD 1?
    ctr_x = (anchors[:, 0] + anchors[:, 1]) / 2

    wx, ww = (10.0, 5.0)  # noqa: WPS111
    delta_w = preds[:, 1::2] / ww
    delta_x = preds[:, 0::2] / wx

    # Prevent sending too large values into torch.exp()
    delta_w = torch.clamp(delta_w, max=math.log(1000.0 / 16))  # noqa: WPS432

    pred_ctr_x = delta_x * widths[:, None] + ctr_x[:, None]
    pred_w = torch.exp(delta_w) * widths[:, None]

    pred_boxes = torch.zeros_like(preds)
    pred_boxes[:, 0::2] = pred_ctr_x - 0.5 * (pred_w - 1)
    pred_boxes[:, 1::2] = pred_ctr_x + 0.5 * (pred_w - 1)
    return pred_boxes


# --- File: src.models.qd_detr.postprocessing ---
"""
Copyright $today.year LY Corporation

LY Corporation licenses this file to you under the Apache License,
version 2.0 (the "License"); you may not use this file except in compliance
with the License. You may obtain a copy of the License at:

  https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS, WITHOUT
WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the
License for the specific language governing permissions and limitations
under the License.

Moment-DETR (https://github.com/jayleicn/moment_detr)
Copyright (c) 2021 Jie Lei

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

import torch
from tqdm import tqdm


class PostProcessorDETR:
    def __init__(
        self,
        clip_length=2,
        min_ts_val=0,
        max_ts_val=150,
        min_w_l=2,
        max_w_l=70,
        move_window_method="center",
        process_func_names=("clip_window_l", "clip_ts", "round_multiple"),
    ):
        """Initializes the PostProcessorDETR.

        Args:
            clip_length (int): Length of each clip.
            min_ts_val (float): Minimum timestamp value.
            max_ts_val (float): Maximum timestamp value.
            min_w_l (float): Minimum window length.
            max_w_l (float): Maximum window length.
            move_window_method (str): Method to move window.
            process_func_names (tuple): Names of processing functions.
        """
        self.clip_length = clip_length
        self.min_ts_val = min_ts_val
        self.max_ts_val = max_ts_val
        self.min_w_l = min_w_l
        self.max_w_l = max_w_l
        self.move_window_method = move_window_method
        self.process_func_names = process_func_names
        self.name2func = dict(
            clip_ts=self.clip_min_max_timestamps,
            round_multiple=self.round_to_multiple_clip_lengths,
        )

    def __call__(self, lines):
        """Processes the prediction lines.

        Args:
            lines (list): List of prediction lines.

        Returns:
            list: Processed lines.
        """
        processed_lines = []
        for line in tqdm(
            lines, desc=f"convert to multiples of clip_length={self.clip_length}"
        ):
            windows_and_scores = torch.tensor(line["pred_relevant_windows"])
            windows = windows_and_scores[:, :2]
            for func_name in self.process_func_names:
                windows = self.name2func[func_name](windows)
            line["pred_relevant_windows"] = torch.cat(
                [windows, windows_and_scores[:, 2:3]], dim=1
            ).tolist()
            line["pred_relevant_windows"] = [
                e[:2] + [float(f"{e[2]:.4f}")] for e in line["pred_relevant_windows"]
            ]
            processed_lines.append(line)
        return processed_lines

    def clip_min_max_timestamps(self, windows):
        """
        windows: (#windows, 2)  torch.Tensor
        ensure timestamps for all windows is within [min_val, max_val], clip is out of boundaries.
        """
        return torch.clamp(windows, min=self.min_ts_val, max=self.max_ts_val)

    def round_to_multiple_clip_lengths(self, windows):
        """
        windows: (#windows, 2)  torch.Tensor
        ensure the final window timestamps are multiples of `clip_length`
        """
        return torch.round(windows / self.clip_length) * self.clip_length


# --- File: src.utils.misc ---
"""
Copyright $today.year LY Corporation

LY Corporation licenses this file to you under the Apache License,
version 2.0 (the "License"); you may not use this file except in compliance
with the License. You may obtain a copy of the License at:

  https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS, WITHOUT
WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the
License for the specific language governing permissions and limitations
under the License.

Moment-DETR (https://github.com/jayleicn/moment_detr)
Copyright (c) 2021 Jie Lei

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

import torch


@torch.no_grad()
def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k
    output: (#items, #classes)
    target: int,
    """
    maxk = max(topk)
    num_items = output.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target)

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / num_items))
    return res


def inverse_sigmoid(x: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1 / x2)


# --- File: src.utils.tensor_utils ---
import numpy as np
import torch


def pad_sequences_1d(
    sequences, dtype=torch.long, device=torch.device("cpu"), fixed_length=None
):
    """Pad a single-nested list or a sequence of n-d array (torch.tensor or np.ndarray)
    into a (n+1)-d array, only allow the first dim has variable lengths.
    Args:
        sequences: list(n-d tensor or list)
        dtype: np.dtype or torch.dtype
        device:
        fixed_length: pad all seq in sequences to fixed length. All seq should have a length <= fixed_length.
            return will be of shape [len(sequences), fixed_length, ...]
    Returns:
        padded_seqs: ((n+1)-d tensor) padded with zeros
        mask: (2d tensor) of the same shape as the first two dims of padded_seqs,
              1 indicate valid, 0 otherwise
    Examples:
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=torch.long)
        >>> test_data_3d = [torch.randn(2,3,4), torch.randn(4,3,4), torch.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=torch.float)
        >>> test_data_list = [[1,2,3], [1,2], [3,4,7,9]]
        >>> pad_sequences_1d(test_data_list, dtype=np.float32)
        >>> test_data_3d = [np.random.randn(2,3,4), np.random.randn(4,3,4), np.random.randn(1,3,4)]
        >>> pad_sequences_1d(test_data_3d, dtype=np.float32)
    """
    if isinstance(sequences[0], list):
        if "torch" in str(dtype):
            sequences = [torch.tensor(s, dtype=dtype, device=device) for s in sequences]
        else:
            sequences = [np.asarray(s, dtype=dtype) for s in sequences]

    extra_dims = sequences[0].shape[
        1:
    ]  # the extra dims should be the same for all elements
    lengths = [len(seq) for seq in sequences]
    if fixed_length is not None:
        max_length = fixed_length
    else:
        max_length = max(lengths)
    if isinstance(sequences[0], torch.Tensor):
        assert "torch" in str(dtype), "dtype and input type does not match"
        padded_seqs = torch.zeros(
            (len(sequences), max_length) + extra_dims, dtype=dtype, device=device
        )
        mask = torch.zeros(
            (len(sequences), max_length), dtype=torch.float32, device=device
        )
    else:  # np
        assert "numpy" in str(dtype), "dtype and input type does not match"
        padded_seqs = np.zeros((len(sequences), max_length) + extra_dims, dtype=dtype)
        mask = np.zeros((len(sequences), max_length), dtype=np.float32)

    for idx, seq in enumerate(sequences):
        end = lengths[idx]
        padded_seqs[idx, :end] = seq
        mask[idx, :end] = 1
    return padded_seqs, mask  # , lengths


def pad_sequences_2d(sequences, dtype=torch.long):
    """Pad a double-nested list or a sequence of n-d torch tensor into a (n+1)-d tensor,
        only allow the first two dims has variable lengths
    Args:
        sequences: list(n-d tensor or list)
        dtype: torch.long for word indices / torch.float (float32) for other cases
    Returns:
    Examples:
        >>> test_data_list = [[[1, 3, 5], [3, 7, 4, 1]], [[98, 34, 11, 89, 90], [22], [34, 56]],]
        >>> pad_sequences_2d(test_data_list, dtype=torch.long)  # torch.Size([2, 3, 5])
        >>> test_data_3d = [torch.randn(2,2,4), torch.randn(4,3,4), torch.randn(1,5,4)]
        >>> pad_sequences_2d(test_data_3d, dtype=torch.float)  # torch.Size([2, 3, 5])
        >>> test_data_3d2 = [[torch.randn(2,4), ], [torch.randn(3,4), torch.randn(5,4)]]
        >>> pad_sequences_2d(test_data_3d2, dtype=torch.float)  # torch.Size([2, 3, 5])
    # TODO add support for numpy array
    """
    bsz = len(sequences)
    para_lengths = [len(seq) for seq in sequences]
    max_para_len = max(para_lengths)
    sen_lengths = [[len(word_seq) for word_seq in seq] for seq in sequences]
    max_sen_len = max([max(e) for e in sen_lengths])

    if isinstance(sequences[0], torch.Tensor):
        extra_dims = sequences[0].shape[2:]
    elif isinstance(sequences[0][0], torch.Tensor):
        extra_dims = sequences[0][0].shape[1:]
    else:
        sequences = [
            [torch.Tensor(word_seq, dtype=dtype) for word_seq in seq]
            for seq in sequences
        ]
        extra_dims = ()

    padded_seqs = torch.zeros(
        (bsz, max_para_len, max_sen_len) + extra_dims, dtype=dtype
    )
    mask = torch.zeros(bsz, max_para_len, max_sen_len).float()

    for b_i in range(bsz):
        for sen_i, sen_l in enumerate(sen_lengths[b_i]):
            padded_seqs[b_i, sen_i, :sen_l] = sequences[b_i][sen_i]
            mask[b_i, sen_i, :sen_l] = 1
    return padded_seqs, mask  # , sen_lengths


# --- File: src.utils.calc_utils ---
import torch


def inverse_sigmoid(x: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1 / x2)


# --- File: src.data.vocab ---
from typing import Dict, List, Optional

import torch
import torch.nn as nn


def _log_class_usage(klass):
    identifier = "torchtext"
    if klass and hasattr(klass, "__name__"):
        identifier += f".{klass.__name__}"
    torch._C._log_api_usage_once(identifier)


class Vocab(nn.Module):
    __jit_unused_properties__ = ["is_jitable"]
    r"""Creates a vocab object which maps tokens to indices.

    Args:
        vocab (torch.classes.torchtext.Vocab or torchtext._torchtext.Vocab): a cpp vocab object.
    """

    def __init__(self, vocab) -> None:
        super(Vocab, self).__init__()
        self.vocab = vocab
        _log_class_usage(__class__)

    @property
    def is_jitable(self):
        return isinstance(self.vocab, torch._C.ScriptObject)

    @torch.jit.export
    def forward(self, tokens: List[str]) -> List[int]:
        r"""Calls the `lookup_indices` method

        Args:
            tokens: a list of tokens used to lookup their corresponding `indices`.

        Returns:
            The indices associated with a list of `tokens`.
        """
        return self.vocab.lookup_indices(tokens)

    @torch.jit.export
    def __len__(self) -> int:
        r"""
        Returns:
            The length of the vocab.
        """
        return len(self.vocab)

    @torch.jit.export
    def __contains__(self, token: str) -> bool:
        r"""
        Args:
            token: The token for which to check the membership.

        Returns:
            Whether the token is member of vocab or not.
        """
        return self.vocab.__contains__(token)

    @torch.jit.export
    def __getitem__(self, token: str) -> int:
        r"""
        Args:
            token: The token used to lookup the corresponding index.

        Returns:
            The index corresponding to the associated token.
        """
        return self.vocab[token]

    @torch.jit.export
    def set_default_index(self, index: Optional[int]) -> None:
        r"""
        Args:
            index: Value of default index. This index will be returned when OOV token is queried.
        """
        self.vocab.set_default_index(index)

    @torch.jit.export
    def get_default_index(self) -> Optional[int]:
        r"""
        Returns:
            Value of default index if it is set.
        """
        return self.vocab.get_default_index()

    @torch.jit.export
    def insert_token(self, token: str, index: int) -> None:
        r"""
        Args:
            token: The token used to lookup the corresponding index.
            index: The index corresponding to the associated token.
        Raises:
            RuntimeError: If `index` is not in range [0, Vocab.size()] or if `token` already exists in the vocab.
        """
        self.vocab.insert_token(token, index)

    @torch.jit.export
    def append_token(self, token: str) -> None:
        r"""
        Args:
            token: The token used to lookup the corresponding index.

        Raises:
            RuntimeError: If `token` already exists in the vocab
        """
        self.vocab.append_token(token)

    @torch.jit.export
    def lookup_token(self, index: int) -> str:
        r"""
        Args:
            index: The index corresponding to the associated token.

        Returns:
            token: The token used to lookup the corresponding index.

        Raises:
            RuntimeError: If `index` not in range [0, itos.size()).
        """
        return self.vocab.lookup_token(index)

    @torch.jit.export
    def lookup_tokens(self, indices: List[int]) -> List[str]:
        r"""
        Args:
            indices: The `indices` used to lookup their corresponding`tokens`.

        Returns:
            The `tokens` associated with `indices`.

        Raises:
            RuntimeError: If an index within `indices` is not int range [0, itos.size()).
        """
        return self.vocab.lookup_tokens(indices)

    @torch.jit.export
    def lookup_indices(self, tokens: List[str]) -> List[int]:
        r"""
        Args:
            tokens: the tokens used to lookup their corresponding `indices`.

        Returns:
            The 'indices` associated with `tokens`.
        """
        return self.vocab.lookup_indices(tokens)

    @torch.jit.export
    def get_stoi(self) -> Dict[str, int]:
        r"""
        Returns:
            Dictionary mapping tokens to indices.
        """
        return self.vocab.get_stoi()

    @torch.jit.export
    def get_itos(self) -> List[str]:
        r"""
        Returns:
            List mapping indices to tokens.
        """
        return self.vocab.get_itos()

    def __prepare_scriptable__(self):
        r"""Return a JITable Vocab."""
        if not self.is_jitable:
            cpp_vocab = torch.classes.torchtext.Vocab(
                self.vocab.itos_, self.vocab.default_index_
            )
            return Vocab(cpp_vocab)
        return self


# --- File: src.models.augments.text ---
import random

import nltk
from nltk.corpus import wordnet
from transformers import pipeline

# Download necessary nltk data
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


def get_synonyms(word):
    """Get synonyms for a word using WordNet."""
    synonyms = set()
    for syn in wordnet.synsets(word):
        for lemma in syn.lemmas():
            synonyms.add(lemma.name())
    if word in synonyms:
        synonyms.remove(word)
    return list(synonyms)


def synonym_replacement(text, n=1):
    """Replace n words in the text with their synonyms."""
    words = text.split()
    new_words = words.copy()
    random_word_list = list(set([word for word in words if word.isalnum()]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word)
        if len(synonyms) >= 1:
            synonym = random.choice(synonyms)
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break
    return " ".join(new_words)


def random_deletion(text, p=0.1):
    """Randomly delete words with probability p."""
    words = text.split()
    if len(words) == 1:
        return text
    new_words = []
    for word in words:
        r = random.uniform(0, 1)
        if r > p:
            new_words.append(word)
    if len(new_words) == 0:
        rand_int = random.randint(0, len(words) - 1)
        return words[rand_int]
    return " ".join(new_words)


def random_swap(text, n=1):
    """Randomly swap two words in the text n times."""
    words = text.split()
    new_words = words.copy()
    for _ in range(n):
        if len(new_words) < 2:
            break
        idx1, idx2 = random.sample(range(len(new_words)), 2)
        new_words[idx1], new_words[idx2] = new_words[idx2], new_words[idx1]
    return " ".join(new_words)


def random_insertion(text, n=1):
    """Randomly insert n words into the text."""
    words = text.split()
    new_words = words.copy()
    for _ in range(n):
        add_word(new_words)
    return " ".join(new_words)


def add_word(new_words):
    """Helper function to add a random word."""
    synonyms = []
    counter = 0
    while len(synonyms) < 1:
        random_word = new_words[random.randint(0, len(new_words) - 1)]
        synonyms = get_synonyms(random_word)
        counter += 1
        if counter >= 10:
            return
    random_synonym = random.choice(synonyms)
    random_idx = random.randint(0, len(new_words) - 1)
    new_words.insert(random_idx, random_synonym)


def back_translation(text, src_lang="en", intermediate_lang="fr"):
    """Translate to intermediate language and back."""
    try:
        translator = pipeline(
            "translation", model=f"Helsinki-NLP/opus-mt-{src_lang}-{intermediate_lang}"
        )
        back_translator = pipeline(
            "translation", model=f"Helsinki-NLP/opus-mt-{intermediate_lang}-{src_lang}"
        )

        translated = translator(text)[0]["translation_text"]
        back_translated = back_translator(translated)[0]["translation_text"]
        return back_translated
    except Exception as e:
        print(f"Back translation failed: {e}")
        return text


def contextual_augmentation(text, model_name="distilgpt2"):
    """Use a language model to generate augmented text."""
    try:
        generator = pipeline("text-generation", model=model_name)
        # Simple approach: generate continuation and take part of it
        generated = generator(
            text, max_length=len(text.split()) + 10, num_return_sequences=1
        )[0]["generated_text"]
        # Extract a similar length text
        words = generated.split()
        original_len = len(text.split())
        return " ".join(words[:original_len])
    except Exception as e:
        print(f"Contextual augmentation failed: {e}")
        return text


def augment_text(text, techniques=None, **kwargs):
    """Apply multiple augmentation techniques."""
    if techniques is None:
        techniques = [
            "synonym_replacement",
            "random_deletion",
            "random_swap",
            "random_insertion",
        ]

    augmented = text
    for tech in techniques:
        if tech == "synonym_replacement":
            n = kwargs.get("synonym_n", 1)
            augmented = synonym_replacement(augmented, n)
        elif tech == "random_deletion":
            p = kwargs.get("deletion_p", 0.1)
            augmented = random_deletion(augmented, p)
        elif tech == "random_swap":
            n = kwargs.get("swap_n", 1)
            augmented = random_swap(augmented, n)
        elif tech == "random_insertion":
            n = kwargs.get("insertion_n", 1)
            augmented = random_insertion(augmented, n)
        elif tech == "back_translation":
            src = kwargs.get("src_lang", "en")
            inter = kwargs.get("intermediate_lang", "fr")
            augmented = back_translation(augmented, src, inter)
        elif tech == "contextual_augmentation":
            model = kwargs.get("model_name", "distilgpt2")
            augmented = contextual_augmentation(augmented, model)

    return augmented


# --- File: src.models.augments.audio ---
import os
import random
from typing import Callable, Dict, Optional, Union

import pandas as pd
import torch
import torchaudio
from torch.utils.data import Dataset


class AudioAugmentor:
    """Safe audio augmentations for audio-text retrieval."""

    def __init__(self, sr: int = 44100):
        self.sr = sr
        self.augmentations = [
            self.add_noise,
            self.random_gain,
            self.time_shift,
            self.polarity_inversion,
            self.speed_perturb,
            self.spec_augment_on_waveform,
        ]

    def __call__(self, waveform: torch.Tensor) -> torch.Tensor:
        # Apply 1-3 random augmentations
        n_augs = random.randint(1, 3)
        chosen = random.sample(self.augmentations, min(n_augs, len(self.augmentations)))
        for aug in chosen:
            waveform = aug(waveform)
        return waveform

    def add_noise(
        self, waveform: torch.Tensor, snr_db_range: tuple[float, float] = (15, 40)
    ) -> torch.Tensor:
        snr_db = random.uniform(*snr_db_range)
        noise = torch.randn_like(waveform)
        signal_power = waveform.norm(p=2)
        noise_power = noise.norm(p=2)
        if noise_power == 0:
            return waveform
        scale = signal_power / (10 ** (snr_db / 20) * noise_power)
        return waveform + scale * noise

    def random_gain(
        self, waveform: torch.Tensor, min_db: float = -6, max_db: float = 6
    ) -> torch.Tensor:
        gain_db = random.uniform(min_db, max_db)
        return waveform * (10 ** (gain_db / 20))

    def time_shift(
        self, waveform: torch.Tensor, max_shift: float = 0.1
    ) -> torch.Tensor:
        shift = int(waveform.shape[-1] * random.uniform(-max_shift, max_shift))
        return torch.roll(waveform, shifts=shift, dims=-1)

    def polarity_inversion(self, waveform: torch.Tensor) -> torch.Tensor:
        return -waveform if random.random() > 0.5 else waveform

    def speed_perturb(
        self, waveform: torch.Tensor, factor_range: tuple[float, float] = (0.95, 1.05)
    ) -> torch.Tensor:
        factor = random.uniform(*factor_range)
        resampled = torchaudio.functional.resample(
            waveform, orig_freq=self.sr, new_freq=int(self.sr * factor)
        )
        return resampled

    def spec_augment_on_waveform(self, waveform: torch.Tensor) -> torch.Tensor:
        """Apply small random zero-out segments (simplified SpecAugment on waveform)."""
        clip_fraction = random.uniform(0.0, 0.05)
        clip_len = int(waveform.shape[-1] * clip_fraction)
        if clip_len > 0:
            start = random.randint(0, waveform.shape[-1] - clip_len)
            waveform = waveform.clone()
            waveform[..., start : start + clip_len] = 0
        return waveform


class TextAugmentor:
    """Safe text augmentations for audio-text retrieval (meaning-preserving)."""

    def __init__(self) -> None:
        self.templates = [
            "{caption}",
            "The sound of {caption_lower}",
            "Audio recording of {caption_lower}",
            "A clip where {caption_lower}",
            "You can hear {caption_lower}",
            "An audio where {caption_lower}",
        ]

    def __call__(self, caption: str) -> str:
        # Apply one random text augmentation
        aug = random.choice(
            [
                self.template_wrap,
                self.light_synonym_replace,
                self.identity,
            ]
        )
        return aug(caption)

    def identity(self, caption: str) -> str:
        return caption

    def template_wrap(self, caption: str) -> str:
        template = random.choice(self.templates)
        return template.format(caption=caption, caption_lower=caption.lower())

    def light_synonym_replace(self, caption: str, n: int = 1) -> str:
        """Replace 1 word with a simple synonym (no NLTK dependency)."""
        # Lightweight synonyms for common audio description words
        synonyms = {
            "loud": ["noisy", "blaring"],
            "quiet": ["soft", "gentle"],
            "fast": ["quick", "rapid"],
            "slow": ["unhurried", "gradual"],
            "big": ["large", "huge"],
            "small": ["tiny", "little"],
            "many": ["several", "numerous"],
            "walks": ["strolls", "steps"],
            "runs": ["jogs", "sprints"],
            "talks": ["speaks", "chats"],
            "sings": ["vocalizes", "chants"],
            "plays": ["performs"],
            "hits": ["strikes", "taps"],
            "falls": ["drops", "descends"],
            "moves": ["shifts", "travels"],
            "flows": ["streams", "runs"],
            "blows": ["gusts", "whooshes"],
        }
        words = caption.split()
        replaced = False
        indices = list(range(len(words)))
        random.shuffle(indices)
        for idx in indices:
            word_lower = words[idx].lower().strip(".,!?")
            if word_lower in synonyms and not replaced:
                replacement = random.choice(synonyms[word_lower])
                # Preserve original casing roughly
                if words[idx][0].isupper():
                    replacement = replacement.capitalize()
                words[idx] = replacement
                replaced = True
                break
        return " ".join(words)


class ClothoDataset(Dataset):
    """
    Clotho dataset for Language-Based Audio Retrieval.

    Compatible with HuggingFace Trainer — returns a dict from __getitem__.

    Args:
        audio_dir:       Path to the directory containing .wav files
        captions_csv:    Path to the captions CSV file
        audio_processor: A callable that processes raw waveform into model inputs
                         (e.g., ClapProcessor, Wav2Vec2Processor, or a mel-spec transform)
        text_tokenizer:  A callable tokenizer (e.g., AutoTokenizer)
        sr:              Target sample rate
        max_audio_len:   Max audio length in seconds (clips/pads to this)
        train:           If True, apply augmentations
        text_max_length: Max token length for text
    """

    def __init__(
        self,
        audio_dir: str,
        captions_csv: str,
        audio_processor: Optional[Callable] = None,
        text_tokenizer: Optional[Callable] = None,
        sr: int = 44100,
        max_audio_len: float = 30.0,
        train: bool = True,
        text_max_length: int = 77,
    ):
        self.audio_dir = audio_dir
        self.sr = sr
        self.max_audio_len = max_audio_len
        self.max_samples = int(sr * max_audio_len)
        self.train = train
        self.audio_processor = audio_processor
        self.text_tokenizer = text_tokenizer
        self.text_max_length = text_max_length

        # Load captions and melt into (file_name, caption) pairs
        captions_df = pd.read_csv(captions_csv)
        caption_cols = [c for c in captions_df.columns if c.startswith("caption")]

        self.data = captions_df.melt(
            id_vars=["file_name"],
            value_vars=caption_cols,
            var_name="caption_number",
            value_name="caption",
        ).reset_index(drop=True)

        # Drop any rows with missing captions
        self.data = self.data.dropna(subset=["caption"]).reset_index(drop=True)

        # Augmentors (only used in train mode)
        self.audio_augmentor = AudioAugmentor(sr=sr) if train else None
        self.text_augmentor = TextAugmentor() if train else None

    def __len__(self) -> int:
        return len(self.data)

    def _load_audio(self, file_name: str) -> torch.Tensor:
        """Load and preprocess audio to a fixed length."""
        path = os.path.join(self.audio_dir, file_name)
        waveform, orig_sr = torchaudio.load(path)

        # Resample if needed
        if orig_sr != self.sr:
            waveform = torchaudio.functional.resample(waveform, orig_sr, self.sr)

        # Convert to mono
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)

        # Pad or truncate to max_samples
        if waveform.shape[-1] > self.max_samples:
            # Random crop during training, center crop during eval
            if self.train:
                start = random.randint(0, waveform.shape[-1] - self.max_samples)
            else:
                start = (waveform.shape[-1] - self.max_samples) // 2
            waveform = waveform[..., start : start + self.max_samples]
        elif waveform.shape[-1] < self.max_samples:
            pad_len = self.max_samples - waveform.shape[-1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))

        return waveform  # shape: (1, max_samples)

    def __getitem__(self, idx: int) -> Dict[str, Union[torch.Tensor, str]]:
        row = self.data.iloc[idx]
        file_name = row["file_name"]
        caption = row["caption"]

        # --- Load audio ---
        waveform = self._load_audio(file_name)

        # --- Audio augmentation (train only) ---
        if self.train and self.audio_augmentor:
            waveform = self.audio_augmentor(waveform)

        # --- Text augmentation (train only) ---
        if self.train and self.text_augmentor:
            caption = self.text_augmentor(caption)

        # --- Process audio through processor (e.g., CLAP, mel-spec) ---
        if self.audio_processor is not None:
            audio_inputs = self.audio_processor(
                audios=waveform.squeeze(0).numpy(),
                sampling_rate=self.sr,
                return_tensors="pt",
                padding="max_length",
                max_length=self.max_samples,
            )
            # Flatten batch dim added by processor
            audio_inputs = {k: v.squeeze(0) for k, v in audio_inputs.items()}
        else:
            # Return raw waveform if no processor
            audio_inputs = {"input_values": waveform.squeeze(0)}

        # --- Tokenize text ---
        if self.text_tokenizer is not None:
            text_inputs = self.text_tokenizer(
                caption,
                padding="max_length",
                max_length=self.text_max_length,
                truncation=True,
                return_tensors="pt",
            )
            # Flatten batch dim
            text_inputs = {k: v.squeeze(0) for k, v in text_inputs.items()}
        else:
            text_inputs = {"caption": caption}

        # --- Combine into a single dict (HF Trainer compatible) ---
        output = {}
        output.update(audio_inputs)
        output["input_ids"] = text_inputs.get("input_ids", caption)
        output["attention_mask"] = text_inputs.get(
            "attention_mask",
            torch.ones_like(text_inputs.get("input_ids", torch.tensor([]))),
        )

        return output


# --- File: src.models.components.positional_encoding.base ---
import math

import torch
from torch import nn


class TrainablePositionalEncoding(nn.Module):
    """Construct the embeddings from word, position and token_type embeddings."""

    def __init__(self, max_position_embeddings, hidden_size, dropout=0.1):
        super(TrainablePositionalEncoding, self).__init__()
        self.position_embeddings = nn.Embedding(max_position_embeddings, hidden_size)
        self.LayerNorm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_feat):
        """
        Args:
            input_feat: (N, L, D)
        """
        bsz, seq_length = input_feat.shape[:2]
        position_ids = torch.arange(
            seq_length, dtype=torch.long, device=input_feat.device
        )
        position_ids = position_ids.unsqueeze(0).repeat(bsz, 1)  # (N, L)

        position_embeddings = self.position_embeddings(position_ids)

        embeddings = self.LayerNorm(input_feat + position_embeddings)
        embeddings = self.dropout(embeddings)
        return embeddings


class PositionEmbeddingSine(nn.Module):
    """
    This is a more standard version of the position embedding, very similar to the one
    used by the Attention is all you need paper, generalized to work on images. (To 1D sequences)
    """

    def __init__(
        self, num_pos_feats=64, temperature=10000, normalize=False, scale=None
    ):
        super().__init__()
        self.num_pos_feats = num_pos_feats
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, x, mask):
        """
        Args:
            x: torch.tensor, (batch_size, L, d)
            mask: torch.tensor, (batch_size, L), with 1 as valid

        Returns:

        """
        assert mask is not None
        x_embed = mask.cumsum(1, dtype=torch.float32)  # (bsz, L)
        if self.normalize:
            eps = 1e-6
            x_embed = x_embed / (x_embed[:, -1:] + eps) * self.scale

        dim_t = torch.arange(self.num_pos_feats, dtype=torch.float32, device=x.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.num_pos_feats)

        pos_x = x_embed[:, :, None] / dim_t  # (bsz, L, num_pos_feats)
        pos_x = torch.stack(
            (pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3
        ).flatten(2)  # (bsz, L, num_pos_feats*2)
        return pos_x  # .permute(0, 2, 1)  # (bsz, num_pos_feats*2, L)


class PositionEmbeddingLearned(nn.Module):
    """
    Absolute pos embedding, learned.
    """

    def __init__(self, num_pos_feats=256):
        super().__init__()
        self.row_embed = nn.Embedding(50, num_pos_feats)
        self.col_embed = nn.Embedding(50, num_pos_feats)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.uniform_(self.row_embed.weight)
        nn.init.uniform_(self.col_embed.weight)

    def forward(self, x, mask):
        h, w = x.shape[-2:]
        i = torch.arange(w, device=x.device)
        j = torch.arange(h, device=x.device)
        x_emb = self.col_embed(i)
        y_emb = self.row_embed(j)
        pos = (
            torch.cat(
                [
                    x_emb.unsqueeze(0).repeat(h, 1, 1),
                    y_emb.unsqueeze(1).repeat(1, w, 1),
                ],
                dim=-1,
            )
            .permute(2, 0, 1)
            .unsqueeze(0)
            .repeat(x.shape[0], 1, 1, 1)
        )
        return pos


def build_position_encoding(args):
    N_steps = args.hidden_dim
    if args.position_embedding in ("v2", "sine"):
        # TODO find a better way of exposing other arguments
        position_embedding = PositionEmbeddingSine(N_steps, normalize=True)
    # elif args.position_embedding in ('v3', 'learned'):
    #     position_embedding = PositionEmbeddingLearned(N_steps)
    else:
        raise ValueError(f"not supported {args.position_embedding}")

    txt_pos_embed = TrainablePositionalEncoding(
        max_position_embeddings=args.max_q_l,
        hidden_size=args.hidden_dim,
        dropout=args.input_dropout,
    )
    return position_embedding, txt_pos_embed


# --- File: src.models.components.attention.multi_head ---
from typing import Optional

import torch
from torch import Tensor

import warnings
from typing import Tuple


from torch.nn.modules.linear import Linear
from torch.nn.init import constant_

from torch.nn.modules.module import Module


try:
    from torch.overrides import has_torch_function, handle_torch_function
except ImportError:
    from torch._overrides import has_torch_function, handle_torch_function

from torch.nn.functional import linear, pad, softmax, dropout


class MultiheadAttention(Module):
    r"""Allows the model to jointly attend to information
    from different representation subspaces.
    See reference: Attention Is All You Need
    .. math::
        \text{MultiHead}(Q, K, V) = \text{Concat}(head_1,\dots,head_h)W^O
        \text{where} head_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)
    Args:
        embed_dim: total dimension of the model.
        num_heads: parallel attention heads.
        dropout: a Dropout layer on attn_output_weights. Default: 0.0.
        bias: add bias as module parameter. Default: True.
        add_bias_kv: add bias to the key and value sequences at dim=0.
        add_zero_attn: add a new batch of zeros to the key and
                       value sequences at dim=1.
        kdim: total number of features in key. Default: None.
        vdim: total number of features in value. Default: None.
        Note: if kdim and vdim are None, they will be set to embed_dim such that
        query, key, and value have the same number of features.
    Examples::
        >>> multihead_attn = nn.MultiheadAttention(embed_dim, num_heads)
        >>> attn_output, attn_output_weights = multihead_attn(query, key, value)
    """

    bias_k: Optional[torch.Tensor]
    bias_v: Optional[torch.Tensor]

    def __init__(
        self,
        embed_dim,
        num_heads,
        dropout=0.0,
        bias=True,
        add_bias_kv=False,
        add_zero_attn=False,
        kdim=None,
        vdim=None,
    ):
        super(MultiheadAttention, self).__init__()
        self.embed_dim = embed_dim
        self.kdim = kdim if kdim is not None else embed_dim
        self.vdim = vdim if vdim is not None else embed_dim
        self._qkv_same_embed_dim = self.kdim == embed_dim and self.vdim == embed_dim

        self.num_heads = num_heads
        self.dropout = dropout
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == self.embed_dim, (
            "embed_dim must be divisible by num_heads"
        )

        vdim = vdim if vdim is not None else embed_dim
        self.out_proj = Linear(vdim, vdim)

        self.in_proj_bias = None
        self.in_proj_weight = None
        self.bias_k = self.bias_v = None
        self.q_proj_weight = None
        self.k_proj_weight = None
        self.v_proj_weight = None

        self.add_zero_attn = add_zero_attn

        self._reset_parameters()

    def _reset_parameters(self):
        constant_(self.out_proj.bias, 0.0)

    def __setstate__(self, state):
        # Support loading old MultiheadAttention checkpoints generated by v1.1.0
        if "_qkv_same_embed_dim" not in state:
            state["_qkv_same_embed_dim"] = True

        super(MultiheadAttention, self).__setstate__(state)

    def forward(
        self,
        query,
        key,
        value,
        key_padding_mask=None,
        need_weights=True,
        attn_mask=None,
    ):
        # type: (Tensor, Tensor, Tensor, Optional[Tensor], bool, Optional[Tensor]) -> Tuple[Tensor, Optional[Tensor]]
        r"""
        Args:
            query, key, value: map a query and a set of key-value pairs to an output.
                See "Attention Is All You Need" for more details.
            key_padding_mask: if provided, specified padding elements in the key will
                be ignored by the attention. When given a binary mask and a value is True,
                the corresponding value on the attention layer will be ignored. When given
                a byte mask and a value is non-zero, the corresponding value on the attention
                layer will be ignored
            need_weights: output attn_output_weights.
            attn_mask: 2D or 3D mask that prevents attention to certain positions. A 2D mask will be broadcasted for all
                the batches while a 3D mask allows to specify a different mask for the entries of each batch.
        Shape:
            - Inputs:
            - query: :math:`(L, N, E)` where L is the target sequence length, N is the batch size, E is
              the embedding dimension.
            - key: :math:`(S, N, E)`, where S is the source sequence length, N is the batch size, E is
              the embedding dimension.
            - value: :math:`(S, N, E)` where S is the source sequence length, N is the batch size, E is
              the embedding dimension.
            - key_padding_mask: :math:`(N, S)` where N is the batch size, S is the source sequence length.
              If a ByteTensor is provided, the non-zero positions will be ignored while the position
              with the zero positions will be unchanged. If a BoolTensor is provided, the positions with the
              value of ``True`` will be ignored while the position with the value of ``False`` will be unchanged.
            - attn_mask: 2D mask :math:`(L, S)` where L is the target sequence length, S is the source sequence length.
              3D mask :math:`(N*\text{num_heads}, L, S)` where N is the batch size, L is the target sequence length,
              S is the source sequence length. attn_mask ensure that position i is allowed to attend the unmasked
              positions. If a ByteTensor is provided, the non-zero positions are not allowed to attend
              while the zero positions will be unchanged. If a BoolTensor is provided, positions with ``True``
              is not allowed to attend while ``False`` values will be unchanged. If a FloatTensor
              is provided, it will be added to the attention weight.
            - Outputs:
            - attn_output: :math:`(L, N, E)` where L is the target sequence length, N is the batch size,
              E is the embedding dimension.
            - attn_output_weights: :math:`(N, L, S)` where N is the batch size,
              L is the target sequence length, S is the source sequence length.
        """
        if not self._qkv_same_embed_dim:
            return multi_head_attention_forward(
                query,
                key,
                value,
                self.embed_dim,
                self.num_heads,
                self.in_proj_weight,
                self.in_proj_bias,
                self.bias_k,
                self.bias_v,
                self.add_zero_attn,
                self.dropout,
                self.out_proj.weight,
                self.out_proj.bias,
                training=self.training,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                use_separate_proj_weight=True,
                q_proj_weight=self.q_proj_weight,
                k_proj_weight=self.k_proj_weight,
                v_proj_weight=self.v_proj_weight,
                out_dim=self.vdim,
            )
        else:
            return multi_head_attention_forward(
                query,
                key,
                value,
                self.embed_dim,
                self.num_heads,
                self.in_proj_weight,
                self.in_proj_bias,
                self.bias_k,
                self.bias_v,
                self.add_zero_attn,
                self.dropout,
                self.out_proj.weight,
                self.out_proj.bias,
                training=self.training,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                out_dim=self.vdim,
            )


def multi_head_attention_forward(
    query: Tensor,
    key: Tensor,
    value: Tensor,
    embed_dim_to_check: int,
    num_heads: int,
    in_proj_weight: Tensor,
    in_proj_bias: Tensor,
    bias_k: Optional[Tensor],
    bias_v: Optional[Tensor],
    add_zero_attn: bool,
    dropout_p: float,
    out_proj_weight: Tensor,
    out_proj_bias: Tensor,
    training: bool = True,
    key_padding_mask: Optional[Tensor] = None,
    need_weights: bool = True,
    attn_mask: Optional[Tensor] = None,
    use_separate_proj_weight: bool = False,
    q_proj_weight: Optional[Tensor] = None,
    k_proj_weight: Optional[Tensor] = None,
    v_proj_weight: Optional[Tensor] = None,
    static_k: Optional[Tensor] = None,
    static_v: Optional[Tensor] = None,
    out_dim: Optional[Tensor] = None,
) -> Tuple[Tensor, Optional[Tensor]]:
    r"""
    Args:
        query, key, value: map a query and a set of key-value pairs to an output.
            See "Attention Is All You Need" for more details.
        embed_dim_to_check: total dimension of the model.
        num_heads: parallel attention heads.
        in_proj_weight, in_proj_bias: input projection weight and bias.
        bias_k, bias_v: bias of the key and value sequences to be added at dim=0.
        add_zero_attn: add a new batch of zeros to the key and
                       value sequences at dim=1.
        dropout_p: probability of an element to be zeroed.
        out_proj_weight, out_proj_bias: the output projection weight and bias.
        training: apply dropout if is ``True``.
        key_padding_mask: if provided, specified padding elements in the key will
            be ignored by the attention. This is an binary mask. When the value is True,
            the corresponding value on the attention layer will be filled with -inf.
        need_weights: output attn_output_weights.
        attn_mask: 2D or 3D mask that prevents attention to certain positions. A 2D mask will be broadcasted for all
            the batches while a 3D mask allows to specify a different mask for the entries of each batch.
        use_separate_proj_weight: the function accept the proj. weights for query, key,
            and value in different forms. If false, in_proj_weight will be used, which is
            a combination of q_proj_weight, k_proj_weight, v_proj_weight.
        q_proj_weight, k_proj_weight, v_proj_weight, in_proj_bias: input projection weight and bias.
        static_k, static_v: static key and value used for attention operators.
    Shape:
        Inputs:
        - query: :math:`(L, N, E)` where L is the target sequence length, N is the batch size, E is
          the embedding dimension.
        - key: :math:`(S, N, E)`, where S is the source sequence length, N is the batch size, E is
          the embedding dimension.
        - value: :math:`(S, N, E)` where S is the source sequence length, N is the batch size, E is
          the embedding dimension.
        - key_padding_mask: :math:`(N, S)` where N is the batch size, S is the source sequence length.
          If a ByteTensor is provided, the non-zero positions will be ignored while the zero positions
          will be unchanged. If a BoolTensor is provided, the positions with the
          value of ``True`` will be ignored while the position with the value of ``False`` will be unchanged.
        - attn_mask: 2D mask :math:`(L, S)` where L is the target sequence length, S is the source sequence length.
          3D mask :math:`(N*num_heads, L, S)` where N is the batch size, L is the target sequence length,
          S is the source sequence length. attn_mask ensures that position i is allowed to attend the unmasked
          positions. If a ByteTensor is provided, the non-zero positions are not allowed to attend
          while the zero positions will be unchanged. If a BoolTensor is provided, positions with ``True``
          are not allowed to attend while ``False`` values will be unchanged. If a FloatTensor
          is provided, it will be added to the attention weight.
        - static_k: :math:`(N*num_heads, S, E/num_heads)`, where S is the source sequence length,
          N is the batch size, E is the embedding dimension. E/num_heads is the head dimension.
        - static_v: :math:`(N*num_heads, S, E/num_heads)`, where S is the source sequence length,
          N is the batch size, E is the embedding dimension. E/num_heads is the head dimension.
        Outputs:
        - attn_output: :math:`(L, N, E)` where L is the target sequence length, N is the batch size,
          E is the embedding dimension.
        - attn_output_weights: :math:`(N, L, S)` where N is the batch size,
          L is the target sequence length, S is the source sequence length.
    """
    if not torch.jit.is_scripting():
        tens_ops = (
            query,
            key,
            value,
            in_proj_weight,
            in_proj_bias,
            bias_k,
            bias_v,
            out_proj_weight,
            out_proj_bias,
        )
        if any([type(t) is not Tensor for t in tens_ops]) and has_torch_function(
            tens_ops
        ):
            return handle_torch_function(
                multi_head_attention_forward,
                tens_ops,
                query,
                key,
                value,
                embed_dim_to_check,
                num_heads,
                in_proj_weight,
                in_proj_bias,
                bias_k,
                bias_v,
                add_zero_attn,
                dropout_p,
                out_proj_weight,
                out_proj_bias,
                training=training,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                use_separate_proj_weight=use_separate_proj_weight,
                q_proj_weight=q_proj_weight,
                k_proj_weight=k_proj_weight,
                v_proj_weight=v_proj_weight,
                static_k=static_k,
                static_v=static_v,
            )
    tgt_len, bsz, embed_dim = query.size()
    assert embed_dim == embed_dim_to_check
    # allow MHA to have different sizes for the feature dimension
    assert key.size(0) == value.size(0) and key.size(1) == value.size(1)

    head_dim = embed_dim // num_heads
    v_head_dim = out_dim // num_heads
    assert head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
    scaling = float(head_dim) ** -0.5

    q = query * scaling
    k = key
    v = value

    if attn_mask is not None:
        assert (
            attn_mask.dtype == torch.float32
            or attn_mask.dtype == torch.float64
            or attn_mask.dtype == torch.float16
            or attn_mask.dtype == torch.uint8
            or attn_mask.dtype == torch.bool
        ), (
            "Only float, byte, and bool types are supported for attn_mask, not {}".format(
                attn_mask.dtype
            )
        )
        if attn_mask.dtype == torch.uint8:
            warnings.warn(
                "Byte tensor for attn_mask in nn.MultiheadAttention is deprecated. Use bool tensor instead."
            )
            attn_mask = attn_mask.to(torch.bool)

        if attn_mask.dim() == 2:
            attn_mask = attn_mask.unsqueeze(0)
            if list(attn_mask.size()) != [1, query.size(0), key.size(0)]:
                raise RuntimeError("The size of the 2D attn_mask is not correct.")
        elif attn_mask.dim() == 3:
            if list(attn_mask.size()) != [bsz * num_heads, query.size(0), key.size(0)]:
                raise RuntimeError("The size of the 3D attn_mask is not correct.")
        else:
            raise RuntimeError(
                "attn_mask's dimension {} is not supported".format(attn_mask.dim())
            )
        # attn_mask's dim is 3 now.

    # convert ByteTensor key_padding_mask to bool
    if key_padding_mask is not None and key_padding_mask.dtype == torch.uint8:
        warnings.warn(
            "Byte tensor for key_padding_mask in nn.MultiheadAttention is deprecated. Use bool tensor instead."
        )
        key_padding_mask = key_padding_mask.to(torch.bool)

    if bias_k is not None and bias_v is not None:
        if static_k is None and static_v is None:
            k = torch.cat([k, bias_k.repeat(1, bsz, 1)])
            v = torch.cat([v, bias_v.repeat(1, bsz, 1)])
            if attn_mask is not None:
                attn_mask = pad(attn_mask, (0, 1))
            if key_padding_mask is not None:
                key_padding_mask = pad(key_padding_mask, (0, 1))
        else:
            assert static_k is None, "bias cannot be added to static key."
            assert static_v is None, "bias cannot be added to static value."
    else:
        assert bias_k is None
        assert bias_v is None

    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)
    if k is not None:
        k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    if v is not None:
        v = v.contiguous().view(-1, bsz * num_heads, v_head_dim).transpose(0, 1)

    if static_k is not None:
        assert static_k.size(0) == bsz * num_heads
        assert static_k.size(2) == head_dim
        k = static_k

    if static_v is not None:
        assert static_v.size(0) == bsz * num_heads
        assert static_v.size(2) == v_head_dim
        v = static_v

    src_len = k.size(1)

    if key_padding_mask is not None:
        assert key_padding_mask.size(0) == bsz
        assert key_padding_mask.size(1) == src_len

    if add_zero_attn:
        src_len += 1
        k = torch.cat(
            [
                k,
                torch.zeros(
                    (k.size(0), 1) + k.size()[2:], dtype=k.dtype, device=k.device
                ),
            ],
            dim=1,
        )
        v = torch.cat(
            [
                v,
                torch.zeros(
                    (v.size(0), 1) + v.size()[2:], dtype=v.dtype, device=v.device
                ),
            ],
            dim=1,
        )
        if attn_mask is not None:
            attn_mask = pad(attn_mask, (0, 1))
        if key_padding_mask is not None:
            key_padding_mask = pad(key_padding_mask, (0, 1))

    attn_output_weights = torch.bmm(q, k.transpose(1, 2))
    assert list(attn_output_weights.size()) == [bsz * num_heads, tgt_len, src_len]

    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_output_weights.masked_fill_(attn_mask, float("-inf"))
        else:
            attn_output_weights += attn_mask

    if key_padding_mask is not None:
        attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, src_len)
        attn_output_weights = attn_output_weights.masked_fill(
            key_padding_mask.unsqueeze(1).unsqueeze(2),
            float("-inf"),
        )
        attn_output_weights = attn_output_weights.view(
            bsz * num_heads, tgt_len, src_len
        )

    # attn_output_weights = softmax(
    #     attn_output_weights, dim=-1)
    attn_output_weights = softmax(
        attn_output_weights - attn_output_weights.max(dim=-1, keepdim=True)[0], dim=-1
    )
    attn_output_weights = dropout(attn_output_weights, p=dropout_p, training=training)

    attn_output = torch.bmm(attn_output_weights, v)
    assert list(attn_output.size()) == [bsz * num_heads, tgt_len, v_head_dim]
    attn_output = attn_output.transpose(0, 1).contiguous().view(tgt_len, bsz, out_dim)
    attn_output = linear(attn_output, out_proj_weight, out_proj_bias)

    if need_weights:
        # average attention weights over heads
        attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, src_len)
        return attn_output, attn_output_weights.sum(dim=1) / num_heads
    else:
        return attn_output, None


# --- File: src.models.feature_extractor.query_extract ---
import argparse
from pathlib import Path

import numpy as np
import torch

# pyrefly: ignore [missing-import]
from msclap import CLAP
from tqdm import tqdm


def dump_audio(data_dir, save_dir, extractor, model_name):
    list_data = sorted(data_dir.glob("*.wav"))
    print(list_data)
    if len(list_data) == 0:
        print(f"No audio files found in {data_dir}")
        return

    # Setup the directory to save the audio features"
    dir_save_feats = save_dir / model_name
    dir_save_feats.mkdir(exist_ok=True, parents=True)

    # Loop through the audio files and extract the audio featseddings
    print("dump audio data...")
    for path_wav in tqdm(list_data):
        path_feats = dir_save_feats / f"{path_wav.stem}.npz"
        if path_feats.exists():
            continue
        feat, proj_feat = extractor.extract_audio_feats(str(path_wav))

        np.savez(path_feats, features=feat)


class ClapExtractor:
    def __init__(self, win_sec, hop_sec):
        # if gpu is available, use it
        self.use_cuda = torch.cuda.is_available()
        self.wrapper = CLAP(use_cuda=self.use_cuda, version="2023")
        if self.use_cuda:
            self.wrapper.clap.caption_encoder = self.wrapper.clap.caption_encoder.cuda()
            print("Inference on GPU")
        else:
            print("Inference on CPU")

        self.sl_win = SlidingWindos(win_sec, hop_sec)

    @torch.no_grad()
    def extract_audio_feats(self, path_wav):
        audio, sr = self.wrapper.read_audio(path_wav, resample=True)
        frames = self.sl_win(audio[0], sr)
        frames = frames.cuda() if self.use_cuda else frames

        feats = self.wrapper.clap.audio_encoder.base(frames)["embedding"]
        proj_feats = self.wrapper.clap.audio_encoder.projection(feats)

        feats = feats.cpu().numpy()
        proj_feats = proj_feats.cpu().numpy()

        return feats, proj_feats


class SlidingWindos:
    def __init__(self, win_sec, hop_sec):
        self.win_sec = win_sec
        self.hop_sec = hop_sec

    def __call__(self, audio, sr):
        """
        Perform sliding window processing on a 1D tensor with center-based cutting.

        Parameters:
        audio (torch.tensor): 1D tensor.
        win_sec (float): Length of each window.
        hop_sec (float): Number of elements to move the window at each step.
        sr (int): Sampling rate.

        Returns:
        torch.tensor: 2D tensor with shape (num_windows, win_length).
        """
        if audio.ndim != 1:
            raise ValueError("Input audio must be 1D tensor.")

        win_length = int(self.win_sec * sr)
        hop_length = int(self.hop_sec * sr)

        windows = audio.unfold(0, win_length, hop_length)

        return torch.tensor(windows)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("data_dir", type=Path, help="audio data directory")
    parser.add_argument("save_dir", type=Path, help="directory to save the features")
    parser.add_argument("--win_sec", type=float, default=1.0, help="window length")
    parser.add_argument("--hop_sec", type=float, default=1.0, help="hop length")
    parser.add_argument("--model_name", default="clap", help="model name")
    args = parser.parse_args()

    if args.model_name == "clap":
        extractor = ClapExtractor(args.win_sec, args.hop_sec)
    else:
        raise ValueError(f"Invalid model name: {args.model_name}")

    dump_audio(
        args.data_dir,
        args.save_dir,
        extractor,
        args.model_name,
    )


# --- File: src.models.feature_extractor.audio_extract ---
import argparse
from pathlib import Path

import numpy as np
import torch
from msclap import CLAP
from tqdm import tqdm


def dump_audio(data_dir, save_dir, extractor, model_name):
    list_data = sorted(data_dir.glob("*.wav"))
    print(list_data)
    if len(list_data) == 0:
        print(f"No audio files found in {data_dir}")
        return

    # Setup the directory to save the audio features"
    dir_save_feats = save_dir / model_name
    dir_save_feats.mkdir(exist_ok=True, parents=True)

    # Loop through the audio files and extract the audio featseddings
    print("dump audio data...")
    for path_wav in tqdm(list_data):
        path_feats = dir_save_feats / f"{path_wav.stem}.npz"
        if path_feats.exists():
            continue
        feat, proj_feat = extractor.extract_audio_feats(str(path_wav))

        np.savez(path_feats, features=feat)


class ClapExtractor:
    def __init__(self, win_sec, hop_sec):
        # if gpu is available, use it
        self.use_cuda = torch.cuda.is_available()
        self.wrapper = CLAP(use_cuda=self.use_cuda, version="2023")
        if self.use_cuda:
            self.wrapper.clap.caption_encoder = self.wrapper.clap.caption_encoder.cuda()
            print("Inference on GPU")
        else:
            print("Inference on CPU")

        self.sl_win = SlidingWindows(win_sec, hop_sec)

    @torch.no_grad()
    def extract_audio_feats(self, path_wav):
        audio, sr = self.wrapper.read_audio(path_wav, resample=True)
        duration = len(audio[0]) / sr
        print(f"Audio duration: {duration:.2f} seconds")
        frames = self.sl_win(audio[0], sr)
        print(f"Number of windows: {frames.shape[0]}")
        frames = frames.cuda() if self.use_cuda else frames

        feats = self.wrapper.clap.audio_encoder.base(frames)["embedding"]
        proj_feats = self.wrapper.clap.audio_encoder.projection(feats)

        feats = feats.cpu().numpy()
        proj_feats = proj_feats.cpu().numpy()

        return feats, proj_feats


class SlidingWindows:
    def __init__(self, win_sec, hop_sec):
        self.win_sec = win_sec
        self.hop_sec = hop_sec

    def __call__(self, audio, sr):
        """
        Perform sliding window processing on a 1D tensor with center-based cutting.

        Parameters:
        audio (torch.tensor): 1D tensor.
        win_sec (float): Length of each window.
        hop_sec (float): Number of elements to move the window at each step.
        sr (int): Sampling rate.

        Returns:
        torch.tensor: 2D tensor with shape (num_windows, win_length).
        """
        if audio.ndim != 1:
            raise ValueError("Input audio must be 1D tensor.")

        win_length = int(self.win_sec * sr)
        hop_length = int(self.hop_sec * sr)

        windows = audio.unfold(0, win_length, hop_length)

        return torch.tensor(windows)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("data_dir", type=Path, help="audio data directory")
    parser.add_argument("save_dir", type=Path, help="directory to save the features")
    parser.add_argument("--win_sec", type=float, default=1.0, help="window length")
    parser.add_argument("--hop_sec", type=float, default=1.0, help="hop length")
    parser.add_argument("--model_name", default="clap", help="model name")
    args = parser.parse_args()

    if args.model_name == "clap":
        extractor = ClapExtractor(args.win_sec, args.hop_sec)
    else:
        raise ValueError(f"Invalid model name: {args.model_name}")

    dump_audio(
        args.data_dir,
        args.save_dir,
        extractor,
        args.model_name,
    )


# --- File: src.models.components.mamba.custom ---
import torch
import torch.nn as nn
# pyrefly: ignore [missing-import]
from mamba_ssm import Mamba
from .cross_attention import MambaCrossAttention


class SparseContextAttention(nn.Module):
    def __init__(self, d_model, num_heads=8, top_k  =32):
        super().__init__()
        self.top_k = top_k
        self.similarity = nn.Linear(d_model, d_model)
        self.cross_attn = MambaCrossAttention(d_model, num_heads)

    def forward(self, chunk, context):
        # Find top-k most similar context tokens
        sim = torch.matmul(chunk, self.similarity(context).transpose(-2, -1))
        _, top_indices = torch.topk(sim, self.top_k, dim=-1)

        # Gather sparse context
        sparse_context = torch.gather(
            context, 1, top_indices.expand(-1, -1, context.shape[-1])
        )

        return self.cross_attn(chunk, sparse_context)


class HierarchicalMamba(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.mamba_local = Mamba(d_model)
        self.mamba_global = Mamba(d_model)
        self.cross_attn_local = MambaCrossAttention(d_model)
        self.cross_attn_global = MambaCrossAttention(d_model)

    def forward(self, x, chunk_size=2048):
        # Process chunks locally
        chunks = x.reshape(-1, chunk_size, x.shape[-1])
        local_out = self.mamba_local(chunks)
        local_out = local_out.reshape_as(x)

        # Downsample for global context
        global_context = x[:, ::8]  # Every 8th token
        global_out = self.mamba_global(global_context)

        # Cross-attend locally to global
        local_attended = self.cross_attn_local(
            local_out, global_out.unsqueeze(1).expand_as(local_out)
        )

        return local_attended + local_out


class TemporalBiasMamba(nn.Module):
    def __init__(self, d_model, num_heads=8, max_distance=5):
        super().__init__()
        self.cross_attn = MambaCrossAttention(d_model, num_heads)
        self.max_distance = max_distance
        self.temporal_bias = nn.Embedding(2 * max_distance + 1, num_heads)

    def forward(self, chunk, context, chunk_idx):
        # Compute attention with temporal distance penalty
        attn = self.cross_attn(chunk, context)

        # Bias toward nearby chunks
        dist = abs(chunk_idx - torch.arange(context.shape[0]))
        bias = self.temporal_bias(
            torch.clamp(dist, -self.max_distance, self.max_distance)
        )

        return attn * (1 + bias.unsqueeze(-1))


class DualStreamMamba(nn.Module):
    """
    Use two streams:

    Token stream: Mamba processes token-by-token within chunk
    Chunk stream: Cross-attn to summary of other chunks
    """

    def __init__(self, d_model):
        super().__init__()
        self.mamba_tokens = Mamba(d_model)
        self.chunk_summarizer = nn.Linear(d_model, d_model)
        self.cross_attn = MambaCrossAttention(d_model)

    def forward(self, chunks):  # [num_chunks, chunk_size, d_model]
        # Token-level processing
        token_outs = torch.stack([self.mamba_tokens(c) for c in chunks])

        # Chunk-level summaries
        chunk_summaries = self.chunk_summarizer(chunks.mean(dim=1))

        # Cross-attend tokens to chunk context
        results = []
        for i, tok_out in enumerate(token_outs):
            context = torch.cat([chunk_summaries[:i], chunk_summaries[i + 1 :]])
            cross = self.cross_attn(tok_out, context)
            results.append(tok_out + cross)

        return torch.stack(results)


# --- File: src.models.components.mamba.gate_attention ---
import torch
import torch.nn as nn
# pyrefly: ignore [missing-import]
from mamba_ssm import Mamba


class AttentionGatedMamba(nn.Module):
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.mamba = Mamba(d_model, d_state=d_state)
        self.context_gate = nn.Sequential(nn.Linear(d_model * 2, d_model), nn.Sigmoid())

    def forward(self, x, context):
        mamba_out = self.mamba(x)
        gate = self.context_gate(torch.cat([mamba_out, context], dim=-1))
        return mamba_out * gate


# --- File: src.models.components.mamba.dual_stream ---
import torch
import torch.nn as nn
# pyrefly: ignore [missing-import]
from mamba_ssm import Mamba

from .cross_attention import MambaCrossAttention


class DualStreamMamba(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.mamba_tokens = Mamba(d_model)
        self.chunk_summarizer = nn.Linear(d_model, d_model)
        self.cross_attn = MambaCrossAttention(d_model)

    def forward(self, chunks):  # [num_chunks, chunk_size, d_model]
        # Token-level processing
        token_outs = torch.stack([self.mamba_tokens(c) for c in chunks])

        # Chunk-level summaries
        chunk_summaries = self.chunk_summarizer(chunks.mean(dim=1))

        # Cross-attend tokens to chunk context
        results = []
        for i, tok_out in enumerate(token_outs):
            context = torch.cat([chunk_summaries[:i], chunk_summaries[i + 1 :]])
            cross = self.cross_attn(tok_out, context)
            results.append(tok_out + cross)

        return torch.stack(results)


# --- File: src.models.components.mamba.cross_attention ---
import torch
import torch.nn as nn
# pyrefly: ignore [missing-import]
from mamba_ssm import Mamba


class MambaCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads=8):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Query from primary sequence
        self.to_q = nn.Linear(d_model, d_model)
        # Key, Value from context sequence
        self.to_k = nn.Linear(d_model, d_model)
        self.to_v = nn.Linear(d_model, d_model)

        self.scale = self.head_dim**-0.5

    def forward(self, x, context):
        """
        x: [batch, seq_len, d_model] - primary sequence (query source)
        context: [batch, context_len, d_model] - context sequence (K, V source)
        """
        Q = self.to_q(x).reshape(-1, self.num_heads, x.shape[1], self.head_dim)
        K = self.to_k(context).reshape(
            -1, self.num_heads, context.shape[1], self.head_dim
        )
        V = self.to_v(context).reshape(
            -1, self.num_heads, context.shape[1], self.head_dim
        )

        # Attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        attn = torch.softmax(scores, dim=-1)
        out = torch.matmul(attn, V)

        return out.reshape(-1, x.shape[1], self.d_model)


class MambaWithCrossAttention(nn.Module):
    def __init__(self, d_model, d_state=16, num_heads=8):
        super().__init__()
        self.mamba = Mamba(d_model, d_state=d_state)
        self.cross_attn = MambaCrossAttention(d_model, num_heads)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, context=None):
        # Standard Mamba
        mamba_out = self.mamba(x)

        if context is not None:
            # Cross-attend to context
            cross_out = self.cross_attn(mamba_out, context)
            return self.norm(mamba_out + cross_out)  # Residual

        return mamba_out


# --- File: src.models.components.mamba.compressor ---
import torch.nn as nn
# pyrefly: ignore [missing-import]
from mamba_ssm import Mamba
from .cross_attention import MambaCrossAttention


class ContextCompressor(nn.Module):
    def __init__(self, d_model, compress_ratio=4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool1d(compress_ratio)
        self.compress = nn.Linear(d_model, d_model)

    def forward(self, chunk):
        # [batch, seq, d_model] -> [batch, compress_ratio, d_model]
        compressed = self.pool(chunk.transpose(1, 2)).transpose(1, 2)
        return self.compress(compressed)


class ChunkedMambaProcessor(nn.Module):
    def __init__(self, d_model, chunk_size=2048, overlap=512):
        super().__init__()
        self.mamba = Mamba(d_model, d_state=16)
        self.cross_attn = MambaCrossAttention(d_model)
        self.chunk_size = chunk_size
        self.overlap = overlap

    def forward(self, x, context_window=2):
        """
        x: [batch, total_seq, d_model]
        context_window: how many neighboring chunks to attend to
        """
        outputs = []
        stride = self.chunk_size - self.overlap

        for i in range(0, x.shape[1], stride):
            chunk = x[:, i : i + self.chunk_size]

            # Gather context from neighboring chunks
            start_ctx = max(0, i - context_window * self.chunk_size)
            end_ctx = min(x.shape[1], i + (context_window + 1) * self.chunk_size)
            context = x[:, start_ctx:end_ctx]

            # Process
            mamba_out = self.mamba(chunk)
            cross_out = self.cross_attn(mamba_out, context)
            outputs.append(cross_out)

        # Merge overlapping regions (average or learnable blend)
        return self._merge_chunks(outputs, stride)


# --- File: src.models.components.positional_encoding.rotary_pos_enc ---
# rope.py
import torch
import torch.nn as nn


class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000):
        super().__init__()
        self.dim = dim
        self.base = base
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len: int, device):
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_emb(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    """x: (..., seq_len, head_dim)"""
    x_ = x.float().reshape(*x.shape[:-1], -1, 2)
    cos = cos.reshape(1, 1, -1, 2) if cos.dim() == 2 else cos
    sin = sin.reshape(1, 1, -1, 2) if sin.dim() == 2 else sin

    x_out = torch.stack(
        [
            x_[..., 0] * cos[..., 0] - x_[..., 1] * sin[..., 0],
            x_[..., 1] * cos[..., 0] + x_[..., 0] * sin[..., 0],
        ],
        dim=-1,
    ).flatten(-2)
    return x_out.type_as(x)


# --- File: src.models.components.positional_encoding.alibi_pos_enc ---
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def get_alibi_slopes(n_heads: int):
    """Generate slopes for each head (geometric progression)"""
    start = 2 ** (-8.0 / n_heads)
    return torch.tensor(
        [start * (start**i) for i in range(n_heads)], dtype=torch.float32
    )


def create_alibi_bias(n_heads: int, seq_len: int, device: torch.device):
    """Create the ALiBi bias matrix: [n_heads, seq_len, seq_len]"""
    slopes = get_alibi_slopes(n_heads).to(device).view(n_heads, 1, 1)

    # Create distance matrix (negative for recency bias)
    pos = torch.arange(seq_len, device=device)
    distances = pos.unsqueeze(0) - pos.unsqueeze(1)  # [seq_len, seq_len]

    # Bias = -slope * distance  (only penalize past, but usually full matrix)
    alibi = distances.unsqueeze(0) * slopes  # [n_heads, seq_len, seq_len]
    return alibi  # You add this to attention scores (before softmax)


# Example integration in attention
class AttentionWithALiBi(nn.Module):
    def __init__(self, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.alibi_slopes = get_alibi_slopes(n_heads)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask=None):
        # q, k, v: [batch, heads, seq_len, head_dim]
        batch, heads, seq_len, _ = q.shape

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Add ALiBi bias
        alibi = create_alibi_bias(heads, seq_len, q.device)
        scores = scores + alibi

        if mask is not None:
            scores = scores + mask

        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        return output


# --- File: src.models.components.sg_detr_blocks.blocks.feed_forward ---
"""Feedforward neural network for feature mapping."""

from torch import Tensor, nn

SLOPE: float = 0.25


class FeedForwardNetwork(nn.Module):
    """A feedforward neural network for feature mapping.."""

    def __init__(self, input_dim: int, expansion_ratio: int = 4, dropout: float = 0.1):
        """
        Initialize the FeedForwardNetwork.

        Args:
            input_dim (int): The dimension of the input feature.
            expansion_ratio (int): The expansion ratio for the hidden layer dimension. Defaults to 4.
            dropout (float): The dropout probability. Defaults to 0.1.
        """
        super().__init__()
        self._dim = input_dim
        self._ratio = expansion_ratio
        self._dropout = dropout
        self._expanded_dim = int(input_dim * expansion_ratio)

        self.mapping = nn.Sequential(
            nn.Linear(input_dim, self._expanded_dim),
            nn.PReLU(init=SLOPE),
            nn.Dropout(p=dropout),
            nn.Linear(self._expanded_dim, input_dim),
        )

        self._init_parameters()

    def _init_parameters(self) -> None:  # noqa: WPS231
        """Initialize the parameters of the neural network layers."""
        for module in self.mapping[:-1].modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(
                    module.weight, gain=nn.init.calculate_gain("leaky_relu", SLOPE)
                )  # type: ignore
                if hasattr(module, "bias") and module.bias is not None:  # noqa: WPS421
                    nn.init.constant_(module.bias, 0)

        # There is no activation after last linear layer.
        for module in self.mapping[-1:].modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if hasattr(module, "bias") and module.bias is not None:  # noqa: WPS421
                    nn.init.constant_(module.bias, 0)

    def forward(self, embedding: Tensor) -> Tensor:
        """
        Forward pass of the network.

        Args:
            embedding (Tensor): The input feature tensor.

        Returns:
            Tensor: The output tensor after passing through the network.
        """
        return self.mapping(embedding)


class LinearLayer(nn.Module):
    """Linear layer configurable with layer normalization, dropout, and non-linearity."""

    def __init__(
        self,
        input_dim: int,
        output_dim: int,
        dropout: float = 0.1,
        normilize: bool = True,
        activate: bool = True,
    ) -> None:
        """Initialize the LinearLayer.

        Args:
            input_dim (int): The dimension of the input feature.
            output_dim (int): The dimension of the output feature.
            dropout (float): The dropout probability. Defaults to 0.1.
            normilize (bool): Whether to apply layer normalization. Defaults to True.
            activate (bool): Whether to apply non-linearity. Defaults to True.
        """
        super().__init__()
        norm = nn.LayerNorm(input_dim) if normilize else nn.Identity()
        activation = nn.ReLU() if activate else nn.Identity()
        self.mapper = nn.Sequential(
            *[  # noqa: WPS517
                norm,
                nn.Dropout(dropout),
                nn.Linear(input_dim, output_dim),
                activation,
            ],
        )

    def forward(self, emb: Tensor) -> Tensor:
        """
        Forward pass of the network.

        Args:
            emb (Tensor): The input feature tensor.

        Returns:
            Tensor: The output tensor after passing through the network.
        """
        return self.mapper(emb)  # (N, L, D)


class MLP(nn.Module):
    """Multi-layer perceptron."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        output_dim: int,
        num_layers: int,
        dropout: float = 0.1,
        normilize: bool = True,
        activate: bool = True,
    ) -> None:
        """
        Initialize the MLP.

        Args:
            input_dim (int): The dimension of the input feature.
            hidden_dim (int): The dimension of the hidden layer.
            output_dim (int): The dimension of the output feature.
            num_layers (int): The number of layers in the MLP.
            dropout (float): The dropout probability.
            normilize (bool): Whether to apply layer normalization.
            activate (bool): Whether to apply non-linearity.
        """
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers

        hidden_dims = [hidden_dim for _ in range(num_layers - 1)]
        input_dims = [input_dim] + hidden_dims
        output_dims = hidden_dims + [output_dim]

        list_of_layers = []
        for idx, (in_dim, out_dim) in enumerate(zip(input_dims, output_dims)):
            activate = activate if idx < num_layers - 1 else False
            layer = LinearLayer(in_dim, out_dim, dropout, normilize, activate)
            list_of_layers.append(layer)
        self.linear_mapper = nn.Sequential(*list_of_layers)
        self._init_parameters()

    def _init_parameters(self) -> None:  # noqa: WPS231
        """Initialize the parameters of the neural network layers."""
        for module in self.linear_mapper[:-1].modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(
                    module.weight, gain=nn.init.calculate_gain("relu")
                )  # type: ignore
                if hasattr(module, "bias") and module.bias is not None:  # noqa: WPS421
                    nn.init.constant_(module.bias, 0)

        for module in self.linear_mapper[-1:].modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if hasattr(module, "bias") and module.bias is not None:  # noqa: WPS421
                    nn.init.constant_(module.bias, 0)

    def forward(self, emb: Tensor) -> Tensor:
        """Forward pass of the network.

        Args:
            emb (Tensor): The input feature tensor.

        Returns:
            Tensor: The output tensor after passing through the network.
        """
        return self.linear_mapper(emb)


class SlimMLP(nn.Module):
    """Very simple multi-layer perceptron."""

    def __init__(
        self, input_dim: int, hidden_dim: int, output_dim: int, num_layers: int
    ) -> None:
        """
        Initialize the MLP.

        Args:
            input_dim (int): The dimension of the input feature.
            hidden_dim (int): The dimension of the hidden layer.
            output_dim (int): The dimension of the output feature.
            num_layers (int): The number of layers in the MLP.
        """
        super().__init__()
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.num_layers = num_layers

        hidden_dims = [hidden_dim for _ in range(num_layers - 1)]
        input_dims = [input_dim] + hidden_dims
        output_dims = hidden_dims + [output_dim]

        list_of_layers = []
        for idx, (in_dim, out_dim) in enumerate(zip(input_dims, output_dims)):
            layer = nn.Linear(in_dim, out_dim)
            non_linearity = (
                nn.ReLU(inplace=True) if idx < num_layers - 1 else nn.Identity()
            )  # noqa: WPS221
            list_of_layers.append(layer)
            list_of_layers.append(non_linearity)  # type: ignore
        self.linear_mapper = nn.Sequential(*list_of_layers)
        self._init_parameters()

    def _init_parameters(self) -> None:  # noqa: WPS231
        """Initialize the parameters of the neural network layers."""
        for module in self.linear_mapper[:-1].modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(
                    module.weight, gain=nn.init.calculate_gain("relu")
                )  # type: ignore
                if hasattr(module, "bias") and module.bias is not None:  # noqa: WPS421
                    nn.init.constant_(module.bias, 0)

        for module in self.linear_mapper[-1:].modules():
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if hasattr(module, "bias") and module.bias is not None:  # noqa: WPS421
                    nn.init.constant_(module.bias, 0)

    def forward(self, emb: Tensor) -> Tensor:
        """Forward pass of the network.

        Args:
            emb (Tensor): The input feature tensor.

        Returns:
            Tensor: The output tensor after passing through the network.
        """
        return self.linear_mapper(emb)


# --- File: src.models.components.sg_detr_blocks.utils.model_utils ---
"""Model utilities."""

from typing import List

import torch
from torch import Tensor, nn

MIN_ANCHOR_LENGTH: float = 0.0125
MAX_ANCHOR_LENGTH: float = 0.9875


def inverse_sigmoid(point: torch.Tensor, eps: float = 1e-3) -> torch.Tensor:
    """
    Inverse sigmoid function.

    Args:
        point (torch.Tensor): input tensor.
        eps (float): small value to avoid numerical instability.

    Returns:
        torch.Tensor: inverse sigmoid of the input tensor.
    """
    point = point.clamp(min=0, max=1)
    point1 = point.clamp(min=eps)
    point2 = (1 - point).clamp(min=eps)
    return torch.log(point1 / point2)


def init_weights(module: nn.Module) -> None:
    """
    Initialize weights for a module.

    Args:
        module (nn.Module): module to initialize weights for.
    """
    if isinstance(module, (nn.Linear, nn.Embedding)):
        module.weight.data.normal_(mean=0, std=0.02)  # noqa: WPS432
    elif isinstance(module, nn.LayerNorm):
        module.bias.data.zero_()
        module.weight.data.fill_(1.0)

    if isinstance(module, nn.Linear) and module.bias is not None:
        module.bias.data.zero_()


def get_reference_points(
    spatial_shapes: List[List[int]],
    valid_ratios: Tensor,
    device: torch.device,
) -> Tensor:  # noqa: WPS602
    """Get reference points for deformable attention.

    Args:
        spatial_shapes (List[List[int]]): List of spatial shapes (height, width) for each level.
        valid_ratios (Tensor): Portion of real tokens.
        device (torch.device): Device on which to perform the computation.

    Returns:
        Tensor: Reference points for deformable attention.
    """
    reference_points_list = []
    for lvl, (height, width) in enumerate(spatial_shapes):
        ref_y, ref_x = torch.meshgrid(
            torch.linspace(
                0.5, height - 0.5, height, dtype=torch.float32, device=device
            ),
            torch.linspace(0.5, width - 0.5, width, dtype=torch.float32, device=device),
        )
        x_scaler = valid_ratios[:, None, lvl, 1] * width
        y_scaler = valid_ratios[:, None, lvl, 1] * height
        ref_x = ref_x.reshape(-1)[None] / x_scaler
        ref_y = ref_y.reshape(-1)[None] / y_scaler
        ref = torch.stack((ref_x, ref_y), -1)
        reference_points_list.append(ref)
    reference_points = torch.cat(reference_points_list, 1)
    return reference_points[:, :, None] * valid_ratios[:, None]


def get_valid_ratio(mask: Tensor) -> Tensor:
    """
    Calculate the valid ratio of a mask tensor along height and width.

    Args:
        mask (Tensor): A tensor of shape (batch_size, height, width) representing the mask.

    Returns:
        Tensor: A tensor of shape (batch_size, 2) where the first column contains the
                valid width ratios and the second column contains the valid height ratios.
    """
    _, height, width = mask.shape
    valid_h = torch.sum(mask[:, :, 0], 1)
    valid_w = torch.sum(mask[:, 0, :], 1)
    valid_ratio_h = valid_h.float() / height
    valid_ratio_w = valid_w.float() / width
    return torch.stack([valid_ratio_w, valid_ratio_h], -1)


def gen_encoder_output_proposals(
    fpn_features: List[Tensor],
    memory_padding_masks: List[Tensor],
    default_widths: List[float],
):
    """
    Generate encoder output proposals.

    Args:
        fpn_features (List[Tensor]): Fpn features (List[batch_size, width, d_model]).
        memory_padding_masks(List[Tensor]): The padding masks tensor with shape (batch_size, width).
        default_widths (List[float]): The default width of the span for proposals.

    Returns:
        Tuple[Tensor, Tensor, Tensor]: A tuple containing:
            - output_memory (Tensor): The output memory tensor with shape (batch_size, sum(hw), d_model).
            - output_proposals (Tensor): The output proposals tensor with shape (batch_size, sum(hw), 4).
            - mask List[int]: mask of each scale.
    """
    spatial_shapes = [seq.size(1) for seq in fpn_features]
    proposals = []
    for lvl, (memory, lvl_mask, lvl_width) in enumerate(
        zip(fpn_features, memory_padding_masks, spatial_shapes)
    ):
        memory = memory.transpose(0, 1)
        _, batch_size, _ = memory.shape
        scale = torch.sum(lvl_mask, 1).unsqueeze(-1)

        # centers
        grid = torch.linspace(
            0, lvl_width - 1, lvl_width, dtype=torch.float32, device=memory.device
        )  # noqa: WPS221
        grid = (grid.unsqueeze(0).expand(batch_size, -1) + 0.5) / scale
        grid = grid.unsqueeze(-1)

        # widths
        width = torch.ones_like(grid) * default_widths[lvl]

        # concat them
        proposal = torch.cat((grid, width), -1)
        proposals.append(proposal)

    # prepare mask
    memory_padding_mask = torch.cat(memory_padding_masks, dim=1).unsqueeze(-1)
    output_proposals = torch.cat(proposals, 1)
    output_proposals_valid = (output_proposals > MIN_ANCHOR_LENGTH) & (
        output_proposals < MAX_ANCHOR_LENGTH
    )
    output_proposals_valid = output_proposals_valid.all(-1, keepdim=True)
    mask = torch.logical_and(memory_padding_mask, output_proposals_valid)

    # apply mask
    output_proposals = inverse_sigmoid(output_proposals)
    output_proposals = output_proposals.masked_fill(~mask, float("inf"))
    output_memory = torch.cat(fpn_features, 1)
    output_memory = output_memory.masked_fill(~mask, 0)
    return output_memory, output_proposals, mask


# --- File: src.models.components.sg_detr_blocks.blocks.position_encoding ---
"""Various positional encodings for the transformer."""

import math
from typing import Optional

import torch
from torch import Tensor, nn


class TrainablePositionalEncoding(nn.Module):
    """
    A module to add learnable positional encodings to the input features.

    Creates positional embeddings and applies layer norm and dropout to the sum of input features and these embeddings.

    Attributes:
        position_embeddings (nn.Embedding): Embedding layer for positional encodings.
        norm (nn.LayerNorm): Layer normalization.
        dropout (nn.Dropout): Dropout layer.
    """

    def __init__(self, dim, max_len: int, dropout: float = 0.1) -> None:
        """
        Initialize PositionalEncoding.

        Args:
            dim (int): The dimension of the input tensor.
            max_len (int): The maximum sequence length for positional encodings. Defaults to 5000.
            dropout (float): Dropout rate. Default: 0.1.
        """
        super().__init__()
        self.position_embeddings = nn.Embedding(max_len, dim)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input_feat: torch.Tensor) -> torch.Tensor:
        """
        Forward pass for adding positional encodings to input features.

        Args:
            input_feat (torch.Tensor): The input feature tensor of shape (N, L, D) where
                                    N is the batch size, L is the sequence length, and
                                    D is the feature dimension.

        Returns:
            torch.Tensor: The output tensor with positional encodings added to the input,
                        normalized and passed through dropout. Shape is (N, L, D).
        """
        bsz, seq_length = input_feat.shape[:2]
        position_ids = torch.arange(
            seq_length, dtype=torch.long, device=input_feat.device
        )
        position_ids = position_ids.unsqueeze(0).repeat(bsz, 1)  # (N, L)
        position_embeddings = self.position_embeddings(position_ids)
        embeddings = self.norm(input_feat + position_embeddings)
        return self.dropout(embeddings)


class PositionEmbeddingSine(nn.Module):
    """MRDETR implementation of PE."""

    def __init__(
        self,
        dim: int = 256,
        temperature: int = 10000,
        normalize: bool = True,
        scale: Optional[float] = None,
    ) -> None:
        """Init of the PositionEmbeddingSine.

        Args:
            dim (int): features dim.
            temperature (int): Defaults to 10000.
            normalize (bool): Defaults to True.
            scale (Optional[float]): Defaults to None.

        Raises:
            ValueError: normalize should be True if scale is passed
        """
        super().__init__()
        self.dim = dim
        self.temperature = temperature
        self.normalize = normalize
        if scale is not None and normalize is False:
            raise ValueError("normalize should be True if scale is passed")
        if scale is None:
            scale = 2 * math.pi
        self.scale = scale

    def forward(self, mask: Tensor) -> Tensor:
        """
        Forward pass of the PositionEmbeddingSine.

        Args:
            mask (Tensor): embedding mask (batch_size, L), with 1 as valid

        Returns:
            Tensor: PE embedding.
        """
        assert mask is not None
        x_embed = mask.cumsum(1, dtype=torch.float32)  # (bsz, L)
        if self.normalize:
            eps = 1e-6
            x_embed = x_embed / (x_embed[:, -1:] + eps)
            x_embed = x_embed * self.scale

        dim_t = torch.arange(self.dim, dtype=torch.float32, device=mask.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.dim)

        pos_x = x_embed[:, :, None] / dim_t
        sin_pos = pos_x[:, :, 0::2].sin()
        cos_pos = pos_x[:, :, 1::2].cos()
        return torch.stack((sin_pos, cos_pos), dim=3).flatten(2)


class PositionalEncoding(nn.Module):
    """
    Positional encoding for sequence data as introduced in [1].

    This module adds positional encodings to the input tensor to provide
    positional information to the model.

    References:
        1. Vaswani et al. (https://arxiv.org/abs/1706.03762)
    """

    fixed_pe_constant: float = 10000.0

    def __init__(self, dim: int, max_len: int = 5000, dropout: float = 0.1):
        """
        Initialize the PositionalEncoding module.

        Args:
            dim (int): The dimension of the input tensor.
            max_len (int): The maximum sequence length for positional encodings. Defaults to 5000.
            dropout (float): The dropout probability. Defaults to 0.1.
        """
        super().__init__()
        self._create_fixed_pe(max_len, dim)
        self.dropout = nn.Dropout(p=dropout)

    def _create_fixed_pe(self, max_len: int, dim: int):
        """
        Create fixed positional encodings based on the given maximum sequence length and dimension.

        Args:
            max_len (int): The maximum sequence length.
            dim (int): The dimension of the input tensor.
        """
        position = torch.arange(max_len).unsqueeze(1)

        div_size = -math.log(self.fixed_pe_constant) / dim
        div_term = torch.exp(torch.arange(0, dim, 2) * div_size)

        pos_enc = torch.zeros(1, max_len, dim)
        pos_enc[0, :, 0::2] = torch.sin(position * div_term)
        pos_enc[0, :, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pos_enc", pos_enc)

    def forward(self, embedding: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the PositionalEncoding module.

        Args:
            embedding (Tensor): The input tensor with shape (batch_size, sequence_length, dim).

        Returns:
            Tensor: The input tensor with positional encodings added.
        """
        embedding = embedding + self.pos_enc[:, : embedding.size(1)]  # type: ignore
        return self.dropout(embedding)


def gen_sineembed_for_position(
    pos_tensor: Tensor, d_model: int, temperature: int = 10000
) -> Tensor:
    """Generate sine embeddings for position.

    Args:
        pos_tensor (Tensor): position tensor (anchor points)
        d_model (int): dimension of the model
        temperature (int): temperature of the pos emb.

    Returns:
        Tensor: sine embeddings for position
    """
    scale = 2 * math.pi
    dim_t = torch.arange(d_model // 2, dtype=torch.float32, device=pos_tensor.device)
    dim_t = temperature ** (2 * (dim_t // 2) / (d_model // 2))

    # prepare PE embedding for center value
    center_embed = pos_tensor[:, :, 0] * scale
    pos_x = center_embed[:, :, None] / dim_t
    pos_x = torch.stack(  # noqa: WPS317
        (
            pos_x[:, :, 0::2].sin(),
            pos_x[:, :, 1::2].cos(),
        ),
        dim=3,
    ).flatten(2)

    # prepare PE embedding for width value
    span_embed = pos_tensor[:, :, 1] * scale
    pos_w = span_embed[:, :, None] / dim_t
    pos_w = torch.stack(  # noqa: WPS317
        (
            pos_w[:, :, 0::2].sin(),
            pos_w[:, :, 1::2].cos(),
        ),
        dim=3,
    ).flatten(2)

    return torch.cat((pos_x, pos_w), dim=2)


# --- File: src.models.components.sg_detr_blocks.utils.stacker ---
"""Module to create stack of modules."""

import copy

from torch import nn


def get_clones(module: nn.Module, number_layers: int) -> nn.ModuleList:
    """Create stack of modules.

    Args:
        module (nn.Module): The module to stack.
        number_layers (int): Number of layer to stack.

    Returns:
        nn.ModuleList: The stacked modules.
    """
    return nn.ModuleList([copy.deepcopy(module) for _ in range(number_layers)])


# --- File: src.models.components.sg_detr_blocks.blocks.conv_blocks ---
"""Convolutional blocks."""

from typing import List

from torch import Tensor, nn


class TransposedLayerNorm(nn.Module):
    """LayerNorm(emb.T).T."""

    def __init__(self, dim: int) -> None:
        """Initialize LayerNorm.

        Args:
            dim (int): embedding dim.
        """
        super().__init__()
        self.norm = nn.LayerNorm(dim)

    def forward(self, embedding: Tensor) -> Tensor:
        """Forward pass of the custom LayerNorm.

        Args:
            embedding (Tensor): input tensor

        Returns:
            Tensor: Normed tensor.
        """
        embedding = embedding.transpose(1, 2)
        embedding = self.norm(embedding)
        return embedding.transpose(1, 2)


class ConvBlock1D(nn.Module):
    """1D convolution block with optional upsampling."""

    def __init__(
        self,
        in_channels: int,
        hidden_dim: int,
        out_channels: int,
        num_layers: int,
        kernel_size: int = 3,
        stride: int = 1,
        use_norm: bool = True,
        last_activate: bool = False,
        upscale: bool = False,
    ) -> None:
        """Intialize a 1D convolution block.

        Args:
            in_channels (int): Number of input channels.
            hidden_dim (int): Hidden dimension.
            out_channels (int): Number of output channels.
            num_layers (int): Number of layers.
            kernel_size (int): Kernel size of the convolution. Defaults to 3.
            stride (int): Stride of the convolution. Defaults to 1.
            use_norm (bool): Whether to use normalization or not.
            last_activate (bool): Whether to use activation on the last layer or not.
            upscale (bool): Whether to use upscale before convolution or not.
        """
        super().__init__()
        hiddens = [hidden_dim for _ in range(num_layers - 1)]
        self.num_layers = len(hiddens) + 1

        activations: List[nn.Module] = []
        for idx in range(self.num_layers):
            if idx < len(hiddens) or last_activate:
                activations.append(nn.ReLU(inplace=True))
            else:
                activations.append(nn.Identity())

        layers: List[nn.Module] = []
        input_channels = [in_channels] + hiddens
        output_channels = hiddens + [out_channels]
        for in_channel, out_channel, activate in zip(
            input_channels, output_channels, activations
        ):
            conv = nn.Conv1d(
                in_channel,
                out_channel,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                bias=not use_norm,
            )

            layer = [
                nn.Upsample(scale_factor=2, mode="linear")
                if upscale
                else nn.Identity(),
                conv,
                TransposedLayerNorm(out_channel) if use_norm else nn.Identity(),
                activate,
            ]
            layers.extend(layer)

        self.layers = nn.Sequential(*layers)
        self._init_weights()

    def _init_weights(self) -> None:
        """Initialize weights of the convolution block."""
        for module in self.modules():
            if isinstance(module, nn.Conv1d):
                nn.init.kaiming_uniform_(module.weight, a=1)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)

    def forward(self, embeds: Tensor) -> Tensor:
        """Forward Pass.

        Args:
            embeds (Tensor): Input Embeddings.

        Returns:
            Tensor: Output Embeddings.
        """
        embeds = embeds.permute(0, 2, 1)
        embeds = self.layers(embeds)
        return embeds.permute(0, 2, 1)


# --- File: src.models.components.sg_detr_blocks.blocks.attention ---
"""
DAB-DETR MultiheadAttention that support query, key, and value to have different dimensions.

Note: Query, key, and value projections are removed.
"""

from typing import Optional, Tuple

import torch
from torch import Tensor
from torch.nn.functional import dropout, linear, pad, softmax
from torch.nn.modules.linear import Linear
from torch.nn.modules.module import Module


class DABMultiheadAttention(Module):
    """MHA with no projection and dummy tokens support."""

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        dropout_prob: float = 0.1,
        add_zero_attn: bool = False,
        kdim: Optional[int] = None,
        vdim: Optional[int] = None,
        num_dummies: Optional[int] = None,
    ):
        """Initialize DABMultiheadAttention.

        Args:
            embed_dim (int): The embedding dimension.
            num_heads (int): The number of attention heads.
            dropout_prob (float): The dropout probability.
            add_zero_attn (bool): Whether to add a column of zeros to the attention weights. Defaults to False.
            kdim (Optional[int]): The dimension of the key. Defaults to None.
            vdim (Optional[int]): The dimension of the value. Defaults to None.
            num_dummies (Optional[int]): The number of dummy tokens. Defaults to None.
        """
        super().__init__()
        self.embed_dim = embed_dim
        self.kdim = kdim if kdim is not None else embed_dim
        self.vdim = vdim if vdim is not None else embed_dim
        self._qkv_same_embed_dim = self.kdim == embed_dim and self.vdim == embed_dim

        self.num_heads = num_heads
        self.num_dummies = num_dummies
        self.dropout_prob = dropout_prob
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == self.embed_dim, (
            "embed_dim must be divisible by num_heads"
        )

        self.out_proj = Linear(self.vdim, self.vdim)

        self.add_zero_attn = add_zero_attn

    def forward(
        self,
        query: Tensor,
        key: Tensor,
        value: Tensor,
        key_padding_mask: Optional[Tensor] = None,
        need_weights: bool = True,
        attn_mask: Optional[Tensor] = None,
        dummy=True,
        saliency_scores: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Optional[Tensor]]:
        """Forward pass for DABMultiheadAttention.

        Args:
            query (Tensor): Query tensor of shape (L, N, E) where L is the target sequence length, N is the batch size,
            key (Tensor): Key tensor of shape (S, N, E) where S is the source sequence length, N is the batch size,
            value (Tensor): Value tensor of shape (S, N, E) where S is the source sequence length, N is the batch size,
            key_padding_mask (Optional[Tensor]): Key padding mask of shape (N, S) where N is the batch size,
            need_weights (bool): Whether to return attention weights. Defaults to True.
            attn_mask (Optional[Tensor]): Attention mask of shape (L, S) where L is the target sequence length
            dummy (bool): Whether to use dummy tokens. Defaults to True.

        Returns:
            Tuple[Tensor, Optional[Tensor]]: The output tensor and the attention weights
        """
        return multi_head_attention_forward(
            query,
            key,
            value,
            self.embed_dim,
            self.num_heads,
            self.add_zero_attn,
            self.dropout_prob,
            self.out_proj.weight,
            self.out_proj.bias,
            num_dummies=self.num_dummies,
            out_dim=self.vdim,
            training=self.training,
            key_padding_mask=key_padding_mask,
            need_weights=need_weights,
            attn_mask=attn_mask,
            dummy=dummy,
            saliency_scores=saliency_scores,
        )


class FlashMultiheadAttention(Module):
    """MHA wrapper that uses PyTorch's F.scaled_dot_product_attention (Flash Attention) when enabled,
    falling back to DABMultiheadAttention for compatibility or when custom logic (like dummy tokens
    or saliency weights) is required.
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        dropout_prob: float = 0.1,
        add_zero_attn: bool = False,
        kdim: Optional[int] = None,
        vdim: Optional[int] = None,
        num_dummies: Optional[int] = None,
        use_flash_attention: bool = False,
    ):
        """Initialize FlashMultiheadAttention."""
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout_prob = dropout_prob
        self.add_zero_attn = add_zero_attn
        self.kdim = kdim if kdim is not None else embed_dim
        self.vdim = vdim if vdim is not None else embed_dim
        self.num_dummies = num_dummies
        self.use_flash_attention = use_flash_attention

        self.dab_attn = DABMultiheadAttention(
            embed_dim=embed_dim,
            num_heads=num_heads,
            dropout_prob=dropout_prob,
            add_zero_attn=add_zero_attn,
            kdim=kdim,
            vdim=vdim,
            num_dummies=num_dummies,
        )

    def forward(
        self,
        query: Tensor,
        key: Tensor,
        value: Tensor,
        key_padding_mask: Optional[Tensor] = None,
        need_weights: bool = True,
        attn_mask: Optional[Tensor] = None,
        dummy: bool = True,
        saliency_scores: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Optional[Tensor]]:
        """Forward pass for FlashMultiheadAttention."""
        # Check global config if local setting is False
        use_flash_attn = self.use_flash_attention
        if not use_flash_attn:
            pass  # GLOBAL_OPT is already globally defined
            if GLOBAL_OPT is not None:
                use_flash_attn = GLOBAL_OPT.get("use_flash_attention", False)

        # Fallback conditions:
        # 1. use_flash_attn is False
        # 2. Need attention weights (scaled_dot_product_attention doesn't return them directly)
        # 3. Custom saliency scores are provided
        # 4. Dummy tokens are used (i.e. dummy=True and num_dummies is not None)
        # 5. add_zero_attn is True
        if (
            not use_flash_attn
            or need_weights
            or saliency_scores is not None
            or (dummy and self.num_dummies is not None)
            or self.add_zero_attn
        ):
            return self.dab_attn(
                query=query,
                key=key,
                value=value,
                key_padding_mask=key_padding_mask,
                need_weights=need_weights,
                attn_mask=attn_mask,
                dummy=dummy,
                saliency_scores=saliency_scores,
            )

        tgt_len, bsz, embed_dim = query.size()
        head_dim = embed_dim // self.num_heads
        v_head_dim = self.vdim // self.num_heads

        # Shape preparation: (L, N, E) -> (N, H, L, head_dim)
        q = query.view(tgt_len, bsz, self.num_heads, head_dim).permute(1, 2, 0, 3)
        k = key.view(-1, bsz, self.num_heads, head_dim).permute(1, 2, 0, 3)
        v = value.view(-1, bsz, self.num_heads, v_head_dim).permute(1, 2, 0, 3)

        sdpa_attn_mask = None

        # 1. Process attention mask
        if attn_mask is not None:
            if attn_mask.dim() == 2:
                sdpa_attn_mask = attn_mask.unsqueeze(0).unsqueeze(0)
            elif attn_mask.dim() == 3:
                sdpa_attn_mask = attn_mask.view(bsz, self.num_heads, tgt_len, -1)
            
            if sdpa_attn_mask.dtype == torch.bool:
                # Invert because SDPA expects True for elements to keep (opposite of masked_fill)
                sdpa_attn_mask = ~sdpa_attn_mask

        # 2. Process key padding mask
        if key_padding_mask is not None:
            k_mask = (~key_padding_mask.to(torch.bool)).unsqueeze(1).unsqueeze(2)
            if sdpa_attn_mask is not None:
                sdpa_attn_mask = sdpa_attn_mask & k_mask
            else:
                sdpa_attn_mask = k_mask

        dropout_p = self.dropout_prob if self.training else 0.0

        output = torch.nn.functional.scaled_dot_product_attention(
            q, k, v,
            attn_mask=sdpa_attn_mask,
            dropout_p=dropout_p,
            is_causal=False,
        )

        output = output.permute(2, 0, 1, 3).contiguous().view(tgt_len, bsz, self.vdim)
        output = self.dab_attn.out_proj(output)

        return output, None


def prepare_attention_mask(
    attn_mask: Tensor, query: Tensor, key: Tensor, bsz: int, num_heads: int
) -> Tensor:
    """Prepare attention mask for multi-head attention.

    Args:
        attn_mask (Tensor): Attention mask to be prepared
        query (Tensor): Query tensor of shape (L, N, E) where L is the target sequence length, N is the batch size
        key (Tensor): Key tensor of shape (S, N, E) where S is the source sequence length, N is the batch size
        bsz (int): The batch size
        num_heads (int): The number of attention heads

    Raises:
        RuntimeError: If the size of the 2D attn_mask is not correct.
        RuntimeError: If the size of the 3D attn_mask is not correct.
        RuntimeError: If attn_mask's dimension is not supported.

    Returns:
        Tensor: The prepared attention mask
    """
    if attn_mask.dtype == torch.uint8:
        attn_mask = attn_mask.to(torch.bool)

    if attn_mask.dim() == 2:
        attn_mask = attn_mask.unsqueeze(0)
        if list(attn_mask.size()) != [1, query.size(0), key.size(0)]:
            raise RuntimeError("The size of the 2D attn_mask is not correct.")
    elif attn_mask.dim() == 3:
        if list(attn_mask.size()) != [bsz * num_heads, query.size(0), key.size(0)]:
            raise RuntimeError("The size of the 3D attn_mask is not correct.")
    else:
        raise RuntimeError(f"attn_mask's dimension {attn_mask.dim()} is not supported")

    return attn_mask


def multi_head_attention_forward(
    query: Tensor,
    key: Tensor,
    value: Tensor,
    embed_dim_to_check: int,
    num_heads: int,
    add_zero_attn: bool,
    dropout_p: float,
    out_proj_weight: Tensor,
    out_proj_bias: Tensor,
    out_dim: int,
    num_dummies: Optional[int] = None,
    training: bool = True,
    key_padding_mask: Optional[Tensor] = None,
    need_weights: bool = True,
    attn_mask: Optional[Tensor] = None,
    dummy=True,
    saliency_scores: Optional[Tensor] = None,
) -> Tuple[Tensor, Optional[Tensor]]:
    """Forward pass for DABMultiheadAttention.

    Args:
        query (Tensor): Query tensor of shape (L, N, E) where L is the target sequence length, N is the batch size
        key (Tensor): Key tensor of shape (S, N, E) where S is the source sequence length, N is the batch size
        value (Tensor): Value tensor of shape (S, N, E) where S is the source sequence length, N is the batch size
        embed_dim_to_check (int): The embedding dimension to check for compatibility
        num_heads (int): The number of attention heads
        num_dummies (Optional[int]): The number of dummy tokens
        add_zero_attn (bool): Whether to add a column of zeros to the attention weights
        dropout_p (float): The dropout probability
        out_proj_weight (Tensor): The weight tensor for the output projection
        out_proj_bias (Tensor): The bias tensor for the output projection
        out_dim (int): The output dimension.
        training (bool): Whether to use dropout. Defaults to True.
        key_padding_mask (Optional[Tensor]): Key padding mask of shape (N, S) where N is the batch size
        need_weights (bool): Whether to return attention weights. Defaults to True.
        attn_mask (Optional[Tensor]): Attention mask of shape (L, S) where L is the target sequence length
        dummy (bool): Whether to use dummy tokens. Defaults to True.

    Returns:
        Tuple[Tensor, Optional[Tensor]]: The output tensor and the attention weights
    """
    tgt_len, bsz, embed_dim = query.size()
    head_dim = embed_dim // num_heads
    v_head_dim = out_dim // num_heads

    assert embed_dim == embed_dim_to_check
    assert key.size(0) == value.size(0)
    assert key.size(1) == value.size(1)
    assert head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"

    scaling = float(head_dim) ** -0.5
    q = query * scaling
    k = key
    v = value

    if attn_mask is not None:
        attn_mask = prepare_attention_mask(attn_mask, query, key, bsz, num_heads)

    # convert ByteTensor key_padding_mask to bool
    if key_padding_mask is not None and key_padding_mask.dtype == torch.uint8:
        key_padding_mask = key_padding_mask.to(torch.bool)

    q = q.contiguous().view(tgt_len, bsz * num_heads, head_dim).transpose(0, 1)
    if k is not None:
        k = k.contiguous().view(-1, bsz * num_heads, head_dim).transpose(0, 1)
    if v is not None:
        v = v.contiguous().view(-1, bsz * num_heads, v_head_dim).transpose(0, 1)

    src_len = k.size(1)

    if key_padding_mask is not None:
        assert key_padding_mask.size(0) == bsz
        assert key_padding_mask.size(1) == src_len

    if add_zero_attn:
        src_len += 1
        k = torch.cat(
            [
                k,
                torch.zeros(
                    (k.size(0), 1) + k.size()[2:], dtype=k.dtype, device=k.device
                ),
            ],
            dim=1,
        )
        v = torch.cat(
            [
                v,
                torch.zeros(
                    (v.size(0), 1) + v.size()[2:], dtype=v.dtype, device=v.device
                ),
            ],
            dim=1,
        )
        if attn_mask is not None:
            attn_mask = pad(attn_mask, (0, 1))  # pylint: disable=not-callable
        if key_padding_mask is not None:
            key_padding_mask = pad(key_padding_mask, (0, 1))  # pylint: disable=not-callable

    attn_output_weights = torch.bmm(q, k.transpose(1, 2))
    assert list(attn_output_weights.size()) == [bsz * num_heads, tgt_len, src_len]

    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_output_weights.masked_fill_(attn_mask, float("-inf"))
        else:
            attn_output_weights = attn_output_weights + attn_mask

    if key_padding_mask is not None:
        attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, src_len)
        attn_output_weights = attn_output_weights.masked_fill(
            key_padding_mask.unsqueeze(1).unsqueeze(2),
            float("-inf"),
        )
        attn_output_weights = attn_output_weights.view(
            bsz * num_heads, tgt_len, src_len
        )

    if attn_output_weights.size(-1) != 0:
        attn_output_weights_max = torch.max(attn_output_weights, dim=-1, keepdim=True)[
            0
        ]
        attn_output_weights = softmax(
            attn_output_weights - attn_output_weights_max, dim=-1
        )
    else:
        attn_output_weights = softmax(attn_output_weights, dim=-1)

    if saliency_scores is not None:
        scores = torch.repeat_interleave(saliency_scores, num_heads, dim=0)[:, :, None]
        attn_output_weights = attn_output_weights * scores

    attn_output_weights = dropout(attn_output_weights, p=dropout_p, training=training)

    if dummy:
        attn_output = torch.bmm(
            attn_output_weights[:, :, num_dummies:], v[:, num_dummies:, :]
        )
    else:
        attn_output = torch.bmm(attn_output_weights, v)

    assert list(attn_output.size()) == [bsz * num_heads, tgt_len, v_head_dim]
    attn_output = attn_output.transpose(0, 1).contiguous().view(tgt_len, bsz, out_dim)
    attn_output = linear(attn_output, out_proj_weight, out_proj_bias)  # pylint: disable=not-callable

    if need_weights:
        attn_output_weights = attn_output_weights.view(bsz, num_heads, tgt_len, src_len)
        return attn_output, attn_output_weights.sum(dim=1) / num_heads
    return attn_output, None


# --- File: src.models.components.sg_detr_blocks.utils.params ---
"""Select params."""

from typing import List, Optional

from torch import nn


def get_params_by_name(
    model: nn.Module,
    include_prefixes: Optional[List[str]] = None,
    exclude_prefixes: Optional[List[str]] = None,
) -> List[nn.Parameter]:
    """
    Retrieve model parameters based on name prefixes.

    Args:
        model (nn.Module): The model from which parameters are retrieved.
        include_prefixes (Optional[List[str]]): List of prefixes to include.
        exclude_prefixes (Optional[List[str]]): List of prefixes to exclude.

    Returns:
        List[nn.Parameter]: A list of parameters that match the include and exclude criteria.
    """
    params = []
    for name, param in model.named_parameters():
        if include_prefixes and not any(
            name.startswith(prefix) for prefix in include_prefixes
        ):
            continue
        if exclude_prefixes and any(
            name.startswith(prefix) for prefix in exclude_prefixes
        ):
            continue
        params.append(param)
    return params


# --- File: src.__main__ ---
import argparse


def main():
    parser = argparse.ArgumentParser(description="DCASE 2026 Challenge")
    parser.add_argument(
        "pipeline",
        type=str,
        choices=["train", "evaluate", "create_submission"],
        help="The pipeline to run.",
    )
    parser.add_argument(
        "--config", "-c", type=str, required=True, help="Path to the YAML configuration file."
    )
    parser.add_argument(
        "--resume",
        "-r",
        type=str,
        help="Specify model path for fine-tuning (train pipeline). If None, train the model from scratch.",
    )
    parser.add_argument(
        "--model_path",
        "-m",
        type=str,
        help="Model checkpoint path (required for evaluate and create_submission pipelines).",
    )
    parser.add_argument(
        "--split",
        "-s",
        type=str,
        default="val",
        choices=["val", "test"],
        help="Split name for evaluate: val or test",
    )

    args = parser.parse_args()

    option_manager = BaseOptions(args.config)
    option_manager.parse()
    opt = option_manager.option

    if args.pipeline == "train":
        train_main = main  # Assumes train.py main is currently defined as main
        train_main(opt, args.resume)
        
    elif args.pipeline == "evaluate":
        if not args.model_path:
            parser.error("--model_path is required for evaluate pipeline")
        pass
        opt.model_path = args.model_path
        opt.eval_split_name = args.split
        start_inference(opt)
        
    elif args.pipeline == "create_submission":
        if not args.model_path:
            parser.error("--model_path is required for create_submission pipeline")
        pass
        opt.model_path = args.model_path
        opt.eval_split_name = "private"
        start_inference(opt)


if __name__ == "__main__":
    main()


# --- File: src.models.components._encoders ---
import torch
from transformers import AutoModel, AutoProcessor, AutoTokenizer




class CLAPEncoder:
    model_id = "laion/clap-htsat-unfused"

    def __init__(self):
        self.processor = AutoProcessor.from_pretrained(
            self.model_id, dtype=torch.float16, device_map=settings.DEVICE
        )
        self.model = AutoModel.from_pretrained(self.model_id)

    def encode_audio(self, audio):
        inputs = self.processor(
            audio, return_tensors="pt", padding=True, truncation=True
        )
        outputs = self.model(**inputs)
        return outputs.last_hidden_state

    def encode_text(self, text):
        inputs = self.processor(
            text, return_tensors="pt", padding=True, truncation=True
        )
        outputs = self.model(**inputs)
        return outputs.last_hidden_state


class BERTEncoder:
    model_id = ""

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModel.from_pretrained(self.model_id)

    def encode_text(self, text):
        inputs = self.tokenizer(
            text, return_tensors="pt", padding=True, truncation=True
        )
        outputs = self.model(**inputs)
        return outputs.last_hidden_state


class ModernBERTEncoder:
    model_id = ""

    def __init__(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.model = AutoModel.from_pretrained(self.model_id)

    def encode_text(self, text):
        inputs = self.tokenizer(
            text, return_tensors="pt", padding=True, truncation=True
        )
        outputs = self.model(**inputs)
        return outputs.last_hidden_state


# --- File: src.models.qd_detr.matcher ---
import torch
from scipy.optimize import linear_sum_assignment
from torch import nn

    generalized_temporal_iou,
    span_cxw_to_xx,
)


class HungarianMatcher(nn.Module):
    """This class computes an assignment between the targets and the predictions of the network

    For efficiency reasons, the targets don't include the no_object. Because of this, in general,
    there are more predictions than targets. In this case, we do a 1-to-1 matching of the best predictions,
    while the others are un-matched (and thus treated as non-objects).
    """

    def __init__(
        self,
        cost_class: float = 1,
        cost_span: float = 1,
        cost_giou: float = 1,
        span_loss_type: str = "l1",
        max_a_l: int = 75,
    ):
        """Creates the matcher

        Params:
            cost_span: This is the relative weight of the L1 error of the span coordinates in the matching cost
            cost_giou: This is the relative weight of the giou loss of the spans in the matching cost
        """
        super().__init__()
        self.cost_class = cost_class
        self.cost_span = cost_span
        self.cost_giou = cost_giou
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        self.foreground_label = 0
        assert cost_class != 0 or cost_span != 0 or cost_giou != 0, (
            "all costs cant be 0"
        )

    @torch.no_grad()
    def forward(self, outputs, targets):
        """Performs the matching

        Params:
            outputs: This is a dict that contains at least these entries:
                 "pred_spans": Tensor of dim [batch_size, num_queries, 2] with the predicted span coordinates,
                    in normalized (cx, w) format
                 ""pred_logits": Tensor of dim [batch_size, num_queries, num_classes] with the classification logits

            targets: This is a list of targets (len(targets) = batch_size), where each target is a dict containing:
                 "spans": Tensor of dim [num_target_spans, 2] containing the target span coordinates. The spans are
                    in normalized (cx, w) format

        Returns:
            A list of size batch_size, containing tuples of (index_i, index_j) where:
                - index_i is the indices of the selected predictions (in order)
                - index_j is the indices of the corresponding selected targets (in order)
            For each batch element, it holds:
                len(index_i) = len(index_j) = min(num_queries, num_target_spans)
        """
        bs, num_queries = outputs["pred_spans"].shape[:2]
        targets = targets["span_labels"]

        # Also concat the target labels and spans
        out_prob = (
            outputs["pred_logits"].flatten(0, 1).softmax(-1)
        )  # [batch_size * num_queries, num_classes]
        tgt_spans = torch.cat(
            [v["spans"] for v in targets]
        )  # [num_target_spans in batch, 2]
        tgt_ids = torch.full(
            [len(tgt_spans)], self.foreground_label,
            dtype=torch.int64,
            device=out_prob.device
        )  # [total #spans in the batch]

        # Compute the classification cost. Contrary to the loss, we don't use the NLL,
        # but approximate it in 1 - prob[target class].
        # The 1 is a constant that doesn't change the matching, it can be omitted.
        cost_class = -out_prob[
            :, tgt_ids
        ]  # [batch_size * num_queries, total #spans in the batch]

        if self.span_loss_type == "l1":
            # We flatten to compute the cost matrices in a batch
            out_spans = outputs["pred_spans"].flatten(
                0, 1
            )  # [batch_size * num_queries, 2]

            # Compute the L1 cost between spans
            cost_span = torch.cdist(
                out_spans, tgt_spans, p=1
            )  # [batch_size * num_queries, total #spans in the batch]

            # Compute the giou cost between spans
            # [batch_size * num_queries, total #spans in the batch]
            cost_giou = -generalized_temporal_iou(
                span_cxw_to_xx(out_spans), span_cxw_to_xx(tgt_spans)
            )
        else:
            pred_spans = outputs["pred_spans"]  # (bsz, #queries, max_v_l * 2)
            pred_spans = pred_spans.view(bs * num_queries, 2, self.max_v_l).softmax(
                -1
            )  # (bsz * #queries, 2, max_v_l)
            cost_span = (
                -pred_spans[:, 0][:, tgt_spans[:, 0]]
                - pred_spans[:, 1][:, tgt_spans[:, 1]]
            )  # (bsz * #queries, #spans)
            # pred_spans = pred_spans.repeat(1, n_spans, 1, 1).flatten(0, 1)  # (bsz * #queries * #spans, max_v_l, 2)
            # tgt_spans = tgt_spans.view(1, n_spans, 2).repeat(bs * num_queries, 1, 1).flatten(0, 1)  # (bsz * #queries * #spans, 2)
            # cost_span = pred_spans[tgt_spans]
            # cost_span = cost_span.view(bs * num_queries, n_spans)

            # giou
            cost_giou = 0

        # Final cost matrix
        C = (
            self.cost_span * cost_span
            + self.cost_giou * cost_giou
            + self.cost_class * cost_class
        )
        C = C.view(bs, num_queries, -1).cpu()

        sizes = [len(v["spans"]) for v in targets]
        indices = [
            linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))
        ]
        return [
            (
                torch.as_tensor(i, dtype=torch.int64),
                torch.as_tensor(j, dtype=torch.int64),
            )
            for i, j in indices
        ]


class HungarianEventMatcher(nn.Module):
    """This class computes an assignment between the targets and the predictions of the network

    For efficiency reasons, the targets don't include the no_object. Because of this, in general,
    there are more predictions than targets. In this case, we do a 1-to-1 matching of the best predictions,
    while the others are un-matched (and thus treated as non-objects).
    """

    def __init__(
        self,
        cost_span: float = 1,
        cost_giou: float = 1,
        span_loss_type: str = "l1",
        max_v_l: int = 75,
    ):
        """Creates the matcher

        Params:
            cost_span: This is the relative weight of the L1 error of the span coordinates in the matching cost
            cost_giou: This is the relative weight of the giou loss of the spans in the matching cost
        """
        super().__init__()
        self.cost_span = cost_span
        self.cost_giou = cost_giou
        self.span_loss_type = span_loss_type
        self.max_v_l = max_v_l
        self.foreground_label = 0
        assert cost_span != 0 or cost_giou != 0, "all costs cant be 0"

    @torch.no_grad()
    def forward(self, outputs, targets):
        """Performs the matching

        Params:
            outputs: This is a dict that contains at least these entries:
                 "pred_spans": Tensor of dim [batch_size, num_queries, 2] with the predicted span coordinates,
                    in normalized (cx, w) format
                 ""pred_logits": Tensor of dim [batch_size, num_queries, num_classes] with the classification logits

            targets: This is a list of targets (len(targets) = batch_size), where each target is a dict containing:
                 "spans": Tensor of dim [num_target_spans, 2] containing the target span coordinates. The spans are
                    in normalized (cx, w) format

        Returns:
            A list of size batch_size, containing tuples of (index_i, index_j) where:
                - index_i is the indices of the selected predictions (in order)
                - index_j is the indices of the corresponding selected targets (in order)
            For each batch element, it holds:
                len(index_i) = len(index_j) = min(num_queries, num_target_spans)
        """
        bs, num_queries = outputs.shape[:2]

        # Also concat the target labels and spans
        tgt_spans = torch.cat([v for v in targets])  # [num_target_spans in batch, 2]

        # We flatten to compute the cost matrices in a batch
        out_spans = outputs.flatten(0, 1)  # [batch_size * num_queries, 2]

        # Compute the L1 cost between spans
        cost_span = torch.cdist(
            out_spans, tgt_spans, p=1
        )  # [batch_size * num_queries, total #spans in the batch]

        # Compute the giou cost between spans
        # [batch_size * num_queries, total #spans in the batch]
        cost_giou = -generalized_temporal_iou(
            span_cxw_to_xx(out_spans), span_cxw_to_xx(tgt_spans)
        )

        # Final cost matrix
        C = self.cost_span * cost_span + self.cost_giou * cost_giou
        C = C.view(bs, num_queries, -1).cpu()

        sizes = [len(v) for v in targets]
        indices = [
            linear_sum_assignment(c[i]) for i, c in enumerate(C.split(sizes, -1))
        ]
        return [
            (
                torch.as_tensor(i, dtype=torch.int64),
                torch.as_tensor(j, dtype=torch.int64),
            )
            for i, j in indices
        ]


def build_matcher(args):
    return HungarianMatcher(
        cost_span=args.set_cost_span,
        cost_giou=args.set_cost_giou,
        cost_class=args.set_cost_class,
        span_loss_type=args.span_loss_type,
        max_a_l=args.max_a_l,
    )


def build_event_matcher(args):
    return HungarianEventMatcher(
        cost_span=args.set_cost_span,
        cost_giou=args.set_cost_giou,
        span_loss_type=args.span_loss_type,
        max_a_l=args.max_a_l,
    )


# --- File: src.models.components.sg_detr_blocks.blocks.anchors ---
"""Module for generating anchors."""

from typing import Iterable, List, Optional, Tuple

import numpy as np
import torch
from torch import Tensor, nn




def get_width_cntrs(anchor: np.ndarray) -> Tuple[float, float]:
    """Calculate the width and x center of a given anchor (window).

    Args:
        anchor (np.ndarray): A 1-dimensional array representing the anchor (window).

    Returns:
        Tuple[float, float]: A tuple containing the width and the x center of the anchor.
    """
    width = anchor[1] - anchor[0] + 1
    x_ctr = anchor[0] + 0.5 * (width - 1)
    return width, x_ctr


def make_anchors(width: np.ndarray, x_ctr: float) -> np.ndarray:
    """Generate a set of anchors based on the given widths around a center point.

    Args:
        width (np.ndarray): A vector of widths.
        x_ctr (float): The center x-coordinate around which to generate the anchors.

    Returns:
        np.ndarray: A 2D array representing a set of anchors (windows).
    """
    width = width[:, np.newaxis]
    left_border = x_ctr - 0.5 * (width - 1)
    right_border = x_ctr + 0.5 * (width - 1)
    return np.hstack((left_border, right_border))


def _scale_enum(anchor: np.ndarray, scales: np.ndarray) -> np.ndarray:
    """Generate a set of anchors by scaling the base anchor.

    Args:
        anchor (np.ndarray): A 1-dimensional array representing the base anchor.
        scales (np.ndarray): A 1-dimensional array of scaling factors.

    Returns:
        np.ndarray: A 2D array representing the enumerated anchors.
    """
    width, x_center = get_width_cntrs(anchor)
    scaled_width = width * scales
    return make_anchors(scaled_width, x_center)


def generate_anchors(stride: float, sizes: Tuple[int, ...]) -> Tensor:
    """Generate a matrix of anchor boxes in (x1, x2) format.

    Anchors are centered on stride / 2 and have (approximate) sqrt areas of the specified sizes.

    Args:
        stride (float): The stride of the feature map.
        sizes (Tuple[int, ...]): A tuple of anchor sizes.

    Returns:
        Tensor: A tensor representing the generated anchors.
    """
    scales = np.array(sizes, dtype=np.float64) / stride
    anchor = np.array([1, stride], dtype=np.float64) - 0.5
    anchors = _scale_enum(anchor, scales)
    return torch.from_numpy(anchors)


# pylint: disable=abstract-method
class BufferList(nn.Module):
    """A custom module similar to `nn.ParameterList`, but for managing buffers.

    This class stores a list of buffers, which are tensors that do not require gradients.

    Attributes:
        _buffers (OrderedDict): An ordered dictionary containing the registered buffers.
    """

    def __init__(self, buffers: Optional[Iterable[Tensor]] = None) -> None:
        """Initialize the BufferList with an optional iterable of buffers.

        Args:
            buffers (Optional[Iterable[Tensor]]): An optional iterable of buffers to initialize the list with.
        """
        super().__init__()
        if buffers is not None:
            self.extend(buffers)

    def extend(self, buffers: Iterable[Tensor]) -> "BufferList":
        """Add multiple buffers to the BufferList.

        Args:
            buffers (Iterable[Tensor]): An iterable of buffers to be added to the list.

        Returns:
            BufferList: The BufferList instance (self) to allow method chaining.
        """
        offset = len(self)
        for idx, buffer in enumerate(buffers):
            self.register_buffer(str(offset + idx), buffer)
        return self

    def __len__(self) -> int:
        """Return the number of buffers in the list.

        Returns:
            int: The count of buffers.
        """
        return len(self._buffers)

    def __iter__(self):
        """Return an iterator over the buffers.

        Returns:
            Iterator: An iterator over the buffer values.
        """
        return iter(self._buffers.values())


class AnchorGenerator(nn.Module):
    """For a set of image sizes and feature maps, computes a set of anchors."""

    def __init__(
        self,
        anchor_sizes: Tuple[int, ...] = (4, 16, 32, 64),
        anchor_strides: Tuple[float, ...] = (0.5, 1, 2, 4),
    ) -> None:
        """Initialize the AnchorGenerator module.

        Args:
            anchor_sizes (Tuple[int, ...]): A tuple of anchor sizes.
            anchor_strides (Tuple[float, ...]): A tuple of anchor strides.
        """
        super().__init__()
        assert len(anchor_strides) == len(anchor_sizes), "Only support FPN now"
        cell_anchors = [
            generate_anchors(stride, (size,))
            for stride, size in zip(anchor_strides, anchor_sizes)
        ]
        self.strides = anchor_strides
        self.cell_anchors = BufferList(cell_anchors)

    def num_anchors_per_location(self) -> List[int]:
        """Return the number of anchors per location.

        Returns:
            List[int]: A list of integers representing the number of anchors per location.
        """
        return [len(cell_anchors) for cell_anchors in self.cell_anchors]

    def grid_anchors(self, grid_sizes: List[int]) -> List[Tensor]:
        """Generate anchors for a set of grid sizes.

        Args:
            grid_sizes (List[int]): A list of grid sizes.

        Returns:
            List[Tensor]: A list of tensors representing the generated anchors.
        """
        anchors = []
        for size, stride, base_anchors in zip(
            grid_sizes, self.strides, self.cell_anchors
        ):
            device = base_anchors.device
            shift_x = torch.arange(
                0, size * stride, step=stride, dtype=torch.float32, device=device
            )  # noqa: WPS221
            shifts = torch.stack((shift_x, shift_x), dim=1)
            shifts = shifts.view(-1, 1, 2)
            base_anchors = base_anchors.view(1, -1, 2)
            anchors.append((shifts + base_anchors).reshape(-1, 2))
        return anchors

    def forward(
        self, feature_maps: List[Tensor], real_video_len: Tensor
    ) -> List[List[SpanList]]:
        """Generate anchors for a set of feature maps.

        Args:
            feature_maps (List[Tensor]): A list of feature maps.
            real_video_len (Tensor): A tensor representing the real video length.

        Returns:
            List[List[SpanList]]: A list of lists of SpanList objects representing the generated anchors.
        """
        grid_sizes = [feature_map.shape[1] for feature_map in feature_maps]
        anchors_over_all_feature_maps = self.grid_anchors(grid_sizes)
        anchors = []
        for size in real_video_len:
            anchors_in_image = []
            for anchors_per_feature_map in anchors_over_all_feature_maps:
                spanlist = SpanList(anchors_per_feature_map, size, mode="xx")
                anchors_in_image.append(spanlist)
            anchors.append(anchors_in_image)
        return anchors


# --- File: src.models.components.sg_detr_blocks.utils.schemas ---
# pylint: disable=too-few-public-methods
"""Data schemas."""

from typing import Any, Dict, List, Optional

from pydantic import BaseModel, ConfigDict, Field
from torch import Tensor




class AuxDetectorOutput(BaseModel):
    """
    Detector output schema.

    Attributes:
        cls_logits (Optional[List[Tensor]]): logits predicted by AUX head.
        bbox_regression (Optional[List[Tensor]]): spans predicted by AUX head.
        bbox_ctrness (Optional[List[Tensor]]): centerness predicted by AUX head.
        anchors (Optional[List[List[SpanList]]]): anchors generated by AUX head.
        matched_gts (Optional[List[Tensor]]): matched gts.
        anchors_spans (Optional[List[SpanList]]): positive anchors.
        selected_features (Optional[List[Tensor]]): matched encoder features.
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)
    cls_logits: Optional[List[Tensor]] = Field(
        None, description="Logits predicted by AUX head."
    )
    bbox_regression: Optional[List[Tensor]] = Field(
        None, description="Spans predicted by AUX head."
    )
    bbox_ctrness: Optional[List[Tensor]] = Field(
        None, description="Centerness predicted by AUX head."
    )
    anchors: Optional[List[List[SpanList]]] = Field(
        None, description="Anchors generated by AUX head."
    )  # noqa: WPS234
    matched_gts: Optional[List[Tensor]] = Field(None, description="Matched gts.")
    anchors_spans: Optional[List[Tensor]] = Field(None, description="Positive anchors")
    selected_features: Optional[List[Tensor]] = Field(
        None, description="Matched encoder features."
    )


class QueryProposalsOutput(BaseModel):
    """
    Schema for the output of get_query_proposals method.

    Attributes:
        query_embs (Tensor): Selected content queries (batch_size, num_queries, d_model).
        refpoint_embed_detach (Tensor): Detached reference points embeddings with shape (batch_size, num_queries, 2).
        refpoint_embed_undetach (Tensor): Undetached reference points embeddings with shape (bs, num_queries, 2).
        class_logit_enc (Tensor): Encoder top classes with shape (batch_size, num_queries, 1).
        iou_logit_enc (Tensor): Encoder top ious with shape (batch_size, num_queries, 1).
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)

    query_embs: Tensor = Field(
        ..., description="Selected content queries (batch_size, num_queries, d_model)."
    )
    refpoint_embed_detach: Tensor = Field(
        ...,
        description="Detached reference points embeddings with shape (batch_size, num_queries, 2).",
    )
    refpoint_embed_enc: Tensor = Field(
        ...,
        description="Undetached reference points embeddings with shape (batch_size, num_queries, 2).",
    )
    class_logit_enc: Tensor = Field(
        ..., description="Encoder top classes with shape (batch_size, num_queries, 1)."
    )
    iou_logit_enc: Tensor = Field(
        ..., description="Encoder top ious with shape (batch_size, num_queries, 1)."
    )


class DetEncoderOutput(BaseModel):
    """
    Detection Encoder output schema.

    Attributes:
        memory (Tensor): Output of the regular encoder.
        vid_pos (Tensor): Positional embeddings.
        vid_mask (Tensor): Mask for the source sequence.
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)
    memory: Tensor = Field(..., description="Output of the encoder.")
    vid_pos: Tensor = Field(..., description="Positional embeddings.")
    vid_mask: Tensor = Field(..., description="Padding mask.")


class DetectorOutput(BaseModel):
    """
    Detector output schema.

    Attributes:
        outputs_class (Tensor): predicted labels. Shape: [num_queries, batch_size, 2]
        outputs_coord (Tensor): reference points. Shape: [num_queries, batch_size, 2]
        offsets (Tensor): predicted offsets for anchors (need only for losses). Shape: [num_queries, batch_size, 2]
        quality_scores (Optional[Tensor]): predicted IOU score for spans. [num_queries, batch_size, 2]
        co_info (Optional[Dict[str, Any]]): support output for the colab postprocessing.
        dn_info (Optional[Dict[str, Any]]): support output for the denoise postprocessing.
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)
    outputs_class: Tensor = Field(
        ..., description="predicted labels. Shape: [num_queries, batch_size, 2]"
    )
    outputs_coord: Tensor = Field(
        ..., description="reference points. Shape: [num_queries, batch_size, 2]"
    )
    offsets: Tensor = Field(
        ...,
        description="predicted offsets for anchors (need only for losses). Shape: [num_queries, batch_size, 2]",
    )
    quality_scores: Optional[Tensor] = Field(
        ..., description="predicted IOU score for spans. [num_queries, bs, 2]"
    )
    co_info: Optional[Dict[str, Any]] = Field(
        ..., description="support output for the colab postprocessing."
    )
    dn_info: Optional[Dict[str, Any]] = Field(
        ..., description="support output for the denoise postprocessing."
    )


class MomentEncoderOutput(BaseModel):
    """
    Moment Encoder output schema.

    Attributes:
        relevant_clips_mask (Optional[Tensor]): Mask for relevant clips
        irrel_clips_mask (Optional[Tensor]): Mask for irrelevant clips
        moment_token (Optional[Tensor]): Moment token inhanced with relevant clips representation
        moment_memory (Optional[Tensor]): Relevant clips representation (output from SA encoder)
        non_moment_token (Optional[Tensor]): Non-moment token inhanced with irrelevant clips representation
        non_moment_memory (Optional[Tensor]): Irrelevant clips representation (output from SA encoder)
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)
    relevant_clips_mask: Optional[Tensor] = Field(
        None, description="Mask for relevant clips"
    )
    irrelevant_clips_mask: Optional[Tensor] = Field(
        None, description="Mask for irrelevant clips"
    )
    moment_token: Optional[Tensor] = Field(
        None, description="Moment token inhanced with relevant clips representation"
    )
    moment_memory: Optional[Tensor] = Field(
        None, description="Relevant clips representation (output from SA encoder)"
    )
    non_moment_token: Optional[Tensor] = Field(
        None,
        description="Non-moment token inhanced with irrelevant clips representation",
    )
    non_moment_memory: Optional[Tensor] = Field(
        None,
        description="Irrelevant clips representation (output from SA encoder)",
    )


class SentenceEncoderOutput(BaseModel):
    """
    Sentence Encoder output schema.

    Attributes:
        model_config (dict): Model configuration
        sent_txt_token (Optional[Tensor]): sentence tokens inhanced with query representation
        sent_dummy_token (Optional[Tensor]): sentence tokens inhanced with dummy representation
        sent_words_memory (Optional[Tensor]): memory of words inhanced with query representation
        sent_dummy_memory (Optional[Tensor]): memory of words inhanced with dummy representation
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)
    sent_txt_token: Optional[Tensor] = Field(
        None, description="Sentence tokens inhanced with query representation"
    )
    sent_dummy_token: Optional[Tensor] = Field(
        None, description="Sentence tokens inhanced with dummy representation"
    )
    sent_words_memory: Optional[Tensor] = Field(
        None, description="Memory of words inhanced with query representation"
    )
    sent_dummy_memory: Optional[Tensor] = Field(
        None, description="Memory of words inhanced with dummy representation"
    )


class InferenceOutputWrapper(BaseModel):
    """Inference output schema.

    Attributes:
        pred_logits: predicted logits.
        pred_spans: predicted spans.
        saliency_scores: predicted saliency scores.
    """

    model_config = ConfigDict(arbitrary_types_allowed=True)
    pred_logits: Tensor = Field(..., description="Predicted logits")
    pred_spans: Tensor = Field(..., description="Predicted spans")
    saliency_scores: Tensor = Field(..., description="Predicted saliency scores")


# --- File: src.data.dataset ---
import logging
import random
from os.path import join
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch



from torch.utils.data import Dataset

logger = logging.getLogger(__name__)


    class StartEndDataset(Dataset):
        """One line in data loaded from data_path."
        {
        "qid": 7803,
        "query": "Man in gray top walks from outside to inside.",
        "duration": 150,
        "vid": "RoripwjYFp8_360.0_510.0",
        "relevant_clip_ids": [13, 14, 15, 16, 17],
        "relevant_windows": [[26, 36]]
        }
        """

        def __init__(
            self,
            data_path: str,
            a_feat_dir: str,
            q_feat_dir: str,
            q_feat_type: str = "last_hidden_state",
            a_feat_type: str = "pann",
            max_q_l: int = 32,
            max_a_l: int = 75,
            ctx_mode: str = "video",
            clip_len: int = 2,
            max_windows: int = 5,
            span_loss_type: str = "l1",
            load_labels: bool = True,
        ) -> None:
            """Initializes the StartEndDataset.

            Args:
                data_path (str): Path to the data file.
                a_feat_dir (str): Directory for audio features.
                q_feat_dir (str): Directory for query features.
                q_feat_type (str): Type of query features.
                a_feat_type (str): Type of audio features.
                max_q_l (int): Maximum query length.
                max_a_l (int): Maximum audio length.
                ctx_mode (str): Context mode.
                clip_len (int): Clip length.
                max_windows (int): Maximum number of windows.
                span_loss_type (str): Type of span loss.
                load_labels (bool): Whether to load labels.
            """
            self.data_path = data_path
            self.a_feat_dir = a_feat_dir
            self.q_feat_dir = q_feat_dir
            self.q_feat_type = q_feat_type
            self.a_feat_type = a_feat_type

            if max_a_l == -1:
                max_a_l = 100000000

            if max_q_l == -1:
                max_q_l = 100

            self.max_q_l = max_q_l
            self.max_a_l = max_a_l

            self.ctx_mode = ctx_mode
            self.use_tef = "tef" in ctx_mode
            self.use_audio = "audio" in ctx_mode
            self.clip_len = clip_len
            self.max_windows = max_windows  # maximum number of windows to use as labels
            self.span_loss_type = span_loss_type
            self.load_labels = load_labels
            self.data = self.load_data()

        def load_data(self) -> List[Dict[str, Any]]:
            datalist = load_jsonl(self.data_path)
            return datalist

        def __len__(self) -> int:
            """Returns the length of the dataset.

            Returns:
                int: Length of the dataset.
            """
            return len(self.data)

        def __getitem__(self, index: int) -> Dict[str, Any]:
            """Gets the item at the given index.

            Args:
                index (int): Index.

            Returns:
                Dict[str, Any]: Item data.
            """
            meta = self.data[index]

            model_inputs = dict()
            model_inputs["query_feat"] = self._get_query_feat_by_qid(
                meta["qid"]
            )  # (Dq, ) or (Lq, Dq)
            model_inputs["audio_feat"] = self._get_audio_feat_by_vid(meta["vid"])
            ctx_l = len(model_inputs["audio_feat"])

            # if self.use_tef:
            #     tef_st = torch.arange(0, ctx_l, 1.0) / ctx_l
            #     tef_ed = tef_st + 1.0 / ctx_l
            #     tef = torch.stack([tef_st, tef_ed], dim=1)  # (Lv, 2)
            #     model_inputs["audio_feat"] = torch.cat(
            #         [model_inputs["audio_feat"], tef], dim=1
            #     )

            if self.use_tef:
                duration = meta["duration"]  # Total video duration in seconds
                clip_indices = torch.arange(0, ctx_l, 1.0)  # [0, 1, 2, ..., ctx_l-1]
                tef_st = (clip_indices * self.clip_len) / duration  # Normalized start times
                tef_ed = (
                    (clip_indices + 1) * self.clip_len
                ) / duration  # Normalized end times
                tef_ed = torch.clamp(tef_ed, max=1.0)  # Ensure it doesn't exceed 1.0
                tef = torch.stack([tef_st, tef_ed], dim=1)  # (ctx_l, 2)
                model_inputs["audio_feat"] = torch.cat(
                    [model_inputs["audio_feat"], tef], dim=1
                )
            if self.load_labels:
                model_inputs["span_labels"] = self.get_span_labels(
                    meta["relevant_windows"], ctx_l, meta["duration"]
                )
                (
                    model_inputs["saliency_pos_labels"],
                    model_inputs["saliency_neg_labels"],
                    model_inputs["saliency_all_labels"],
                ) = self.get_saliency_labels_sub_as_query(
                    meta["relevant_windows"][0], ctx_l
                )

            return dict(meta=meta, model_inputs=model_inputs)

        def get_saliency_labels_sub_as_query(
            self, gt_window: List[float], ctx_l: int, max_n: int = 2
        ) -> Tuple[List[int], List[int], np.ndarray]:
            """
            Generate saliency labels for contrastive learning based on ground truth temporal window.

            This method samples positive and negative clip indices for training the saliency prediction
            head. Positive clips are sampled from within the ground truth window, while negative clips
            are sampled from outside it. Also creates a binary score array indicating relevant regions.

            Args:
                gt_window (List[float]): Ground truth time window [start_time, end_time] in seconds.
                ctx_l (int): Total number of clips/segments in the audio/video context.
                max_n (int, optional): Maximum number of positive and negative clips to sample. Defaults to 2.

            Returns:
                Tuple[List[int], List[int], np.ndarray]:
                    - pos_clip_indices: List of positive clip indices (from within gt_window).
                    - neg_clip_indices: List of negative clip indices (from outside gt_window).
                    - score_array: Binary numpy array of shape (ctx_l,) with 1s for relevant clips.

            Example:
                >>> dataset = StartEndDataset(...)
                >>> gt_window = [26.0, 36.0]  # 10-second window
                >>> ctx_l = 75  # 75 clips total
                >>> pos_indices, neg_indices, scores = dataset.get_saliency_labels_sub_as_query(gt_window, ctx_l, max_n=2)
                >>> print(f"Positive clips: {pos_indices}")  # e.g., [13, 17]
                >>> print(f"Negative clips: {neg_indices}")  # e.g., [5, 45]
                >>> print(f"Score array shape: {scores.shape}")  # (75,)
                >>> print(f"Relevant clips: {np.where(scores == 1)[0]}")  # e.g., [13 14 15 16 17]
            """
            gt_st = int(gt_window[0] / self.clip_len)
            gt_ed = max(0, min(int(gt_window[1] / self.clip_len), ctx_l) - 1)

            if gt_st > gt_ed:
                gt_st = gt_ed

            if gt_st != gt_ed:
                pos_clip_indices = random.sample(range(gt_st, gt_ed + 1), k=max_n)
            else:
                pos_clip_indices = [gt_st, gt_st]

            neg_pool = list(range(0, gt_st)) + list(
                range(gt_ed + 1, ctx_l)
            )  # to fix bugs / works..?
            try:
                neg_clip_indices = random.sample(neg_pool, k=max_n)
            except ValueError:
                neg_clip_indices = pos_clip_indices

            score_array = np.zeros(ctx_l)
            score_array[gt_st : gt_ed + 1] = 1

            return pos_clip_indices, neg_clip_indices, score_array

        def get_saliency_labels(
            self,
            rel_clip_ids: List[int],
            scores: List[List[float]],
            ctx_l: int,
            max_n: int = 1,
            add_easy_negative: bool = True,
        ) -> Tuple[List[int], List[int]]:
            """Sum the scores from the three annotations, then take the two clips with the
            maximum scores as positive, and two with the minimum scores as negative.
            Args:
                rel_clip_ids: list(int), list of relevant clip ids
                scores: list([anno1_score, anno2_score, anno3_score]),
                ctx_l: int
                max_n: int, #clips to use as positive and negative, for easy and hard negative, respectively.
                add_easy_negative: bool, if True, sample eay negative outside the relevant_clip_ids.
            """
            # indices inside rel_clip_ids
            scores = np.array(scores)  # (#rel_clips, 3)
            agg_scores = np.sum(scores, 1)  # (#rel_clips, )
            sort_indices = np.argsort(agg_scores)  # increasing

            # indices in the whole video
            # the min(_, ctx_l-1) here is incorrect, but should not cause
            # much troubles since this should be rarely used.
            hard_pos_clip_indices = [
                min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[-max_n:]
            ]
            hard_neg_clip_indices = [
                min(rel_clip_ids[idx], ctx_l - 1) for idx in sort_indices[:max_n]
            ]
            easy_pos_clip_indices = []
            easy_neg_clip_indices = []
            if add_easy_negative:
                easy_neg_pool = list(set(range(ctx_l)) - set(rel_clip_ids))
                if len(easy_neg_pool) >= max_n:
                    easy_pos_clip_indices = random.sample(rel_clip_ids, k=max_n)
                    easy_neg_clip_indices = random.sample(easy_neg_pool, k=max_n)
                else:  # copy the hard ones
                    easy_pos_clip_indices = hard_pos_clip_indices
                    easy_neg_clip_indices = hard_neg_clip_indices

            pos_clip_indices = hard_pos_clip_indices + easy_pos_clip_indices
            neg_clip_indices = hard_neg_clip_indices + easy_neg_clip_indices
            return pos_clip_indices, neg_clip_indices

        def get_span_labels(
            self, windows: List[List[float]], ctx_l: int, duration: float
        ) -> torch.Tensor:
            """
            windows: list([st, ed]) in seconds. E.g. [[26, 36]], corresponding st_ed clip_indices [[13, 17]] (inclusive)
                Note a maximum of `self.max_windows` windows are used.
            returns Tensor of shape (#windows, 2), each row is [center, width] normalized by video length
            """
            if len(windows) > self.max_windows:
                random.shuffle(windows)
                windows = windows[: self.max_windows]
            if self.span_loss_type == "l1":
                windows = torch.Tensor(windows) / duration  # normalized windows in xx
                windows = span_xx_to_cxw(windows)  # normalized windows in cxw
            elif self.span_loss_type == "ce":
                windows = torch.Tensor(
                    [
                        [
                            int(w[0] / self.clip_len),
                            min(int(w[1] / self.clip_len), ctx_l) - 1,
                        ]
                        for w in windows
                    ]
                ).long()  # inclusive
            else:
                raise NotImplementedError
            return windows

        def _get_query_feat_by_qid(self, qid: int) -> np.ndarray:
            """Gets query features by qid.

            Args:
                qid (int): Query id.

            Returns:
                np.ndarray: Query features.
            """
            q_feat_path = join(self.q_feat_dir, f"qid{qid}.npz")
            q_feat = np.load(q_feat_path)["last_hidden_state"]
            return q_feat

        def _get_audio_feat_by_vid(self, vid: str) -> torch.Tensor:
            """Gets audio features by vid.

            Args:
                vid (str): Video id.

            Returns:
                torch.Tensor: Audio features.
            """
            _feat_path = join(self.a_feat_dir, f"{vid}.npz")
            _feat = np.load(_feat_path)["features"][: self.max_a_l].astype(np.float32)
            _feat = l2_normalize_np_array(_feat)
            return torch.from_numpy(_feat)


    def start_end_collate(
        batch: List[Dict[str, Any]],
    ) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:
        """Collates the batch for start end dataset.

        Args:
            batch (List[Dict[str, Any]]): Batch.

        Returns:
            Tuple[List[Dict[str, Any]], Dict[str, Any]]: Batch meta, batched data.
        """
        batch_meta = [e["meta"] for e in batch]

        model_inputs_keys = batch[0]["model_inputs"].keys()
        batched_data = dict()
        for k in model_inputs_keys:
            if k == "span_labels":
                batched_data[k] = [
                    dict(spans=e["model_inputs"]["span_labels"]) for e in batch
                ]
                continue
            if k in ["saliency_pos_labels", "saliency_neg_labels"]:
                batched_data[k] = torch.LongTensor([e["model_inputs"][k] for e in batch])
                continue
            if k == "saliency_all_labels":
                pad_data, mask_data = pad_sequences_1d(
                    [e["model_inputs"][k] for e in batch],
                    dtype=np.float32,
                    fixed_length=None,
                )
                batched_data[k] = torch.tensor(pad_data, dtype=torch.float32)
                continue

            if batch[0]["model_inputs"][k].dtype == torch.float32:
                batched_data[k] = pad_sequences_1d(
                    [e["model_inputs"][k] for e in batch],
                    dtype=torch.float32,
                    fixed_length=None,
                )
            else:
                batched_data[k] = pad_sequences_1d(
                    [torch.from_numpy(e["model_inputs"][k]) for e in batch],
                    dtype=torch.float32,
                    fixed_length=None,
                )
        return batch_meta, batched_data


    def prepare_batch_inputs(
        batched_model_inputs: Dict[str, Any],
        device: torch.device,
        non_blocking: bool = False,
    ) -> Tuple[Dict[str, torch.Tensor], Optional[Dict[str, Any]]]:
        """Prepares batch inputs.

        Args:
            batched_model_inputs (Dict[str, Any]): Batched model inputs.
            device (torch.device): Device.
            non_blocking (bool): Non blocking.

        Returns:
            Tuple[Dict[str, torch.Tensor], Optional[Dict[str, Any]]]: Model inputs, targets.
        """
        model_inputs = dict(
            src_txt=batched_model_inputs["query_feat"][0].to(
                device, non_blocking=non_blocking
            ),
            src_txt_mask=batched_model_inputs["query_feat"][1].to(
                device, non_blocking=non_blocking
            ),
        )

        if "audio_feat" in batched_model_inputs:
            model_inputs["src_aud"] = batched_model_inputs["audio_feat"][0].to(
                device, non_blocking=non_blocking
            )
            model_inputs["src_aud_mask"] = batched_model_inputs["audio_feat"][1].to(
                device, non_blocking=non_blocking
            )

        targets = {}
        if "span_labels" in batched_model_inputs:
            targets["span_labels"] = [
                dict(spans=e["spans"].to(device, non_blocking=non_blocking))
                for e in batched_model_inputs["span_labels"]
            ]
        if "saliency_pos_labels" in batched_model_inputs:
            for name in ["saliency_pos_labels", "saliency_neg_labels"]:
                targets[name] = batched_model_inputs[name].to(
                    device, non_blocking=non_blocking
                )

        if "saliency_all_labels" in batched_model_inputs:
            targets["saliency_all_labels"] = batched_model_inputs["saliency_all_labels"].to(
                device, non_blocking=non_blocking
            )

        targets = None if len(targets) == 0 else targets
        return model_inputs, targets


# --- File: src.models.qd_detr.transformer ---
import copy
from typing import Optional

import torch
import torch.nn.functional as F
from torch import nn, Tensor
import math



class MLP(nn.Module):
    """Very simple multi-layer perceptron (also called FFN)"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )

    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x


def inverse_sigmoid(x, eps=1e-3):
    """Applies inverse sigmoid transformation to the input tensor.

    Args:
        x (torch.Tensor): Input tensor.
        eps (float): Small value for numerical stability.

    Returns:
        torch.Tensor: Inverse sigmoid of x.
    """
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1 / x2)


def gen_sineembed_for_position(pos_tensor):
    """Generates sine embeddings for position tensors.

    Args:
        pos_tensor (torch.Tensor): Position tensor.

    Returns:
        torch.Tensor: Sine embeddings.
    """
    # n_query, bs, _ = pos_tensor.size()
    # sineembed_tensor = torch.zeros(n_query, bs, 256)
    scale = 2 * math.pi
    dim_t = torch.arange(128, dtype=torch.float32, device=pos_tensor.device)
    dim_t = 10000 ** (2 * (dim_t // 2) / 128)
    center_embed = pos_tensor[:, :, 0] * scale
    pos_x = center_embed[:, :, None] / dim_t
    pos_x = torch.stack(
        (pos_x[:, :, 0::2].sin(), pos_x[:, :, 1::2].cos()), dim=3
    ).flatten(2)

    span_embed = pos_tensor[:, :, 1] * scale
    pos_w = span_embed[:, :, None] / dim_t
    pos_w = torch.stack(
        (pos_w[:, :, 0::2].sin(), pos_w[:, :, 1::2].cos()), dim=3
    ).flatten(2)

    pos = torch.cat((pos_x, pos_w), dim=2)
    return pos


class Transformer(nn.Module):
    def __init__(
        self,
        d_model=512,
        nhead=8,
        num_queries=2,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
        return_intermediate_dec=False,
        query_dim=2,
        keep_query_pos=False,
        query_scale_type="cond_elewise",
        num_patterns=0,
        modulate_t_attn=True,
        bbox_embed_diff_each_layer=False,
    ):
        """Initializes the Transformer model.

        Args:
            d_model (int): Model dimension.
            nhead (int): Number of attention heads.
            num_queries (int): Number of queries.
            num_encoder_layers (int): Number of encoder layers.
            num_decoder_layers (int): Number of decoder layers.
            dim_feedforward (int): Feedforward dimension.
            dropout (float): Dropout rate.
            activation (str): Activation function.
            normalize_before (bool): Whether to normalize before.
            return_intermediate_dec (bool): Whether to return intermediate decoder outputs.
            query_dim (int): Query dimension.
            keep_query_pos (bool): Whether to keep query position.
            query_scale_type (str): Query scale type.
            num_patterns (int): Number of patterns.
            modulate_t_attn (bool): Whether to modulate temporal attention.
            bbox_embed_diff_each_layer (bool): Whether bbox embed differs each layer.
        """
        super().__init__()

        t2v_encoder_layer = T2V_TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.t2v_encoder = TransformerEncoder(
            t2v_encoder_layer, num_encoder_layers, encoder_norm
        )

        # TransformerEncoderLayerThin
        encoder_layer = TransformerEncoderLayer(
            d_model, nhead, dim_feedforward, dropout, activation, normalize_before
        )
        encoder_norm = nn.LayerNorm(d_model) if normalize_before else None
        self.encoder = TransformerEncoder(
            encoder_layer, num_encoder_layers, encoder_norm
        )

        # TransformerDecoderLayerThin
        decoder_layer = TransformerDecoderLayer(
            d_model,
            nhead,
            dim_feedforward,
            dropout,
            activation,
            normalize_before,
            keep_query_pos=keep_query_pos,
        )
        decoder_norm = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(
            decoder_layer,
            num_decoder_layers,
            decoder_norm,
            return_intermediate=return_intermediate_dec,
            d_model=d_model,
            query_dim=query_dim,
            keep_query_pos=keep_query_pos,
            query_scale_type=query_scale_type,
            modulate_t_attn=modulate_t_attn,
            bbox_embed_diff_each_layer=bbox_embed_diff_each_layer,
        )

        self._reset_parameters()

        self.d_model = d_model
        self.nhead = nhead
        self.dec_layers = num_decoder_layers
        self.num_queries = num_queries
        self.num_patterns = num_patterns

    def _reset_parameters(self):
        """Resets the parameters of the model."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, mask, query_embed, pos_embed, audio_length):
        """
        Args:
            src: (batch_size, L, d)
            mask: (batch_size, L)
            query_embed: (#queries, d)
            pos_embed: (batch_size, L, d) the same as src

        Returns:

        """
        # flatten NxCxHxW to HWxNxC
        bs, seq_len, d = src.shape
        src = src.permute(1, 0, 2)  # (L, batch_size, d)
        pos_embed = pos_embed.permute(1, 0, 2)  # (L, batch_size, d)
        refpoint_embed = query_embed.unsqueeze(1).repeat(
            1, bs, 1
        )  # (#queries, batch_size, d)

        src = self.t2v_encoder(
            src, src_key_padding_mask=mask, pos=pos_embed, audio_length=audio_length
        )  # (L, batch_size, d)
        # print('after encoder : ',src.shape)
        src = src[: audio_length + 1]
        mask = mask[:, : audio_length + 1]
        pos_embed = pos_embed[: audio_length + 1]

        memory = self.encoder(
            src, src_key_padding_mask=mask, pos=pos_embed
        )  # (L, batch_size, d)
        memory_global, memory_local = memory[0], memory[1:]
        mask_local = mask[:, 1:]
        pos_embed_local = pos_embed[1:]

        tgt = torch.zeros(refpoint_embed.shape[0], bs, d, device=src.device)
        hs, references = self.decoder(
            tgt,
            memory_local,
            memory_key_padding_mask=mask_local,
            pos=pos_embed_local,
            refpoints_unsigmoid=refpoint_embed,
        )  # (#layers, #queries, batch_size, d)
        # hs = hs.transpose(1, 2)  # (#layers, batch_size, #qeries, d)
        # memory = memory.permute(1, 2, 0)  # (batch_size, d, L)
        memory_local = memory_local.transpose(0, 1)  # (batch_size, L, d)
        return hs, references, memory_local, memory_global


class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None, return_intermediate=False):
        """Initializes the TransformerEncoder.

        Args:
            encoder_layer: Encoder layer module.
            num_layers (int): Number of layers.
            norm: Normalization layer.
            return_intermediate (bool): Whether to return intermediate outputs.
        """
        super().__init__()
        self.layers = _get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate

    def forward(
        self,
        src,
        mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        **kwargs,
    ):
        """Forward pass through the TransformerEncoder.

        Args:
            src (torch.Tensor): Source tensor.
            mask (Optional[Tensor]): Mask tensor.
            src_key_padding_mask (Optional[Tensor]): Key padding mask.
            pos (Optional[Tensor]): Position tensor.
            **kwargs: Additional arguments.

        Returns:
            torch.Tensor: Output tensor.
        """
        output = src

        intermediate = []

        for layer in self.layers:
            output = layer(
                output,
                src_mask=mask,
                src_key_padding_mask=src_key_padding_mask,
                pos=pos,
                **kwargs,
            )
            if self.return_intermediate:
                intermediate.append(output)

        if self.norm is not None:
            output = self.norm(output)

        if self.return_intermediate:
            return torch.stack(intermediate)

        return output


class TransformerDecoder(nn.Module):
    def __init__(
        self,
        decoder_layer,
        num_layers,
        norm=None,
        return_intermediate=False,
        d_model=256,
        query_dim=2,
        keep_query_pos=False,
        query_scale_type="cond_elewise",
        modulate_t_attn=False,
        bbox_embed_diff_each_layer=False,
    ):
        super().__init__()
        self.layers = _get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = norm
        self.return_intermediate = return_intermediate
        assert return_intermediate
        self.query_dim = query_dim

        assert query_scale_type in ["cond_elewise", "cond_scalar", "fix_elewise"]
        self.query_scale_type = query_scale_type
        if query_scale_type == "cond_elewise":
            self.query_scale = MLP(d_model, d_model, d_model, 2)
        elif query_scale_type == "cond_scalar":
            self.query_scale = MLP(d_model, d_model, 1, 2)
        elif query_scale_type == "fix_elewise":
            self.query_scale = nn.Embedding(num_layers, d_model)
        else:
            raise NotImplementedError(
                "Unknown query_scale_type: {}".format(query_scale_type)
            )

        self.ref_point_head = MLP(d_model, d_model, d_model, 2)

        # self.bbox_embed = None
        # for DAB-deter
        if bbox_embed_diff_each_layer:
            self.bbox_embed = nn.ModuleList(
                [MLP(d_model, d_model, 2, 3) for i in range(num_layers)]
            )
        else:
            self.bbox_embed = MLP(d_model, d_model, 2, 3)
        # init bbox_embed
        if bbox_embed_diff_each_layer:
            for bbox_embed in self.bbox_embed:
                nn.init.constant_(bbox_embed.layers[-1].weight.data, 0)
                nn.init.constant_(bbox_embed.layers[-1].bias.data, 0)
        else:
            nn.init.constant_(self.bbox_embed.layers[-1].weight.data, 0)
            nn.init.constant_(self.bbox_embed.layers[-1].bias.data, 0)
        self.d_model = d_model
        self.modulate_t_attn = modulate_t_attn
        self.bbox_embed_diff_each_layer = bbox_embed_diff_each_layer

        if modulate_t_attn:
            self.ref_anchor_head = MLP(d_model, d_model, 1, 2)

        if not keep_query_pos:
            for layer_id in range(num_layers - 1):
                self.layers[layer_id + 1].ca_qpos_proj = None

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        refpoints_unsigmoid: Optional[Tensor] = None,  # num_queries, bs, 2
    ):
        output = tgt

        intermediate = []
        reference_points = refpoints_unsigmoid.sigmoid()
        ref_points = [reference_points]

        for layer_id, layer in enumerate(self.layers):
            obj_center = reference_points[..., : self.query_dim]
            # get sine embedding for the query vector
            query_sine_embed = gen_sineembed_for_position(obj_center)
            # print('line230', query_sine_embed.shape)
            query_pos = self.ref_point_head(query_sine_embed)
            # print('line232',query_sine_embed.shape)
            # For the first decoder layer, we do not apply transformation over p_s
            if self.query_scale_type != "fix_elewise":
                if layer_id == 0:
                    pos_transformation = 1
                else:
                    pos_transformation = self.query_scale(output)
            else:
                pos_transformation = self.query_scale.weight[layer_id]

            # apply transformation
            # print(query_sine_embed.shape) # 10 32 512
            query_sine_embed = query_sine_embed * pos_transformation

            # modulated HW attentions
            if self.modulate_t_attn:
                reft_cond = self.ref_anchor_head(output).sigmoid()  # nq, bs, 1
                # print(reft_cond.shape, reft_cond[..., 0].shape) # 10 32 1, 10 32
                # print(obj_center.shape, obj_center[..., 1].shape) # 10 32 2, 10 32
                # print(query_sine_embed.shape) # 10 32 256

                query_sine_embed *= (reft_cond[..., 0] / obj_center[..., 1]).unsqueeze(
                    -1
                )

            output = layer(
                output,
                memory,
                tgt_mask=tgt_mask,
                memory_mask=memory_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=memory_key_padding_mask,
                pos=pos,
                query_pos=query_pos,
                query_sine_embed=query_sine_embed,
                is_first=(layer_id == 0),
            )

            # iter update
            if self.bbox_embed is not None:
                if self.bbox_embed_diff_each_layer:
                    tmp = self.bbox_embed[layer_id](output)
                else:
                    tmp = self.bbox_embed(output)
                tmp[..., : self.query_dim] += inverse_sigmoid(reference_points)
                new_reference_points = tmp[..., : self.query_dim].sigmoid()
                if layer_id != self.num_layers - 1:
                    ref_points.append(new_reference_points)
                reference_points = new_reference_points.detach()

            if self.return_intermediate:
                intermediate.append(self.norm(output))

        if self.norm is not None:
            output = self.norm(output)
            if self.return_intermediate:
                intermediate.pop()
                intermediate.append(output)

        if self.return_intermediate:
            if self.bbox_embed is not None:
                return [
                    torch.stack(intermediate).transpose(1, 2),
                    torch.stack(ref_points).transpose(1, 2),
                ]
            else:
                return [
                    torch.stack(intermediate).transpose(1, 2),
                    reference_points.unsqueeze(0).transpose(1, 2),
                ]

        return output.unsqueeze(0)


class TransformerEncoderLayerThin(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        """Initializes the TransformerEncoderLayerThin.

        Args:
            d_model (int): Model dimension.
            nhead (int): Number of heads.
            dim_feedforward (int): Feedforward dimension.
            dropout (float): Dropout rate.
            activation (str): Activation function.
            normalize_before (bool): Whether to normalize before.
        """
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        # self.linear1 = nn.Linear(d_model, dim_feedforward)
        # self.dropout = nn.Dropout(dropout)
        # self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.linear = nn.Linear(d_model, d_model)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        # self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        """Adds positional embedding to the tensor.

        Args:
            tensor (torch.Tensor): Input tensor.
            pos (Optional[Tensor]): Positional embedding.

        Returns:
            torch.Tensor: Tensor with positional embedding.
        """
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(
            q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src2 = self.linear(src2)
        src = src + self.dropout(src2)
        src = self.norm(src)
        # src = src + self.dropout1(src2)
        # src = self.norm1(src)
        # src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        # src = src + self.dropout2(src2)
        # src = self.norm2(src)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        """not used"""
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(
            q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)


class T2V_TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before
        self.nhead = nhead

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        audio_length=None,
    ):

        assert audio_length is not None

        # print('before src shape :', src.shape)
        pos_src = self.with_pos_embed(src, pos)
        global_token, q, k, v = (
            src[0].unsqueeze(0),
            pos_src[1 : audio_length + 1],
            pos_src[audio_length + 1 :],
            src[audio_length + 1 :],
        )

        # print(src_key_padding_mask.shape) # torch.Size([32, 102])
        # print(src_key_padding_mask[:, 1:76].permute(1,0).shape) # torch.Size([75, 32])
        # print(src_key_padding_mask[:, 76:].shape) # torch.Size([32, 26])

        qmask, kmask = (
            src_key_padding_mask[:, 1 : audio_length + 1].unsqueeze(2),
            src_key_padding_mask[:, audio_length + 1 :].unsqueeze(1),
        )
        attn_mask = (
            torch.matmul(qmask.float(), kmask.float()).bool().repeat(self.nhead, 1, 1)
        )
        # print(attn_mask.shape)
        # print(attn_mask[0][0])
        # print(q.shape) 75 32 256
        # print(k.shape) 26 32 256

        src2 = self.self_attn(
            q,
            k,
            value=v,
            attn_mask=attn_mask,
            key_padding_mask=src_key_padding_mask[:, audio_length + 1 :],
        )[0]
        src2 = src[1 : audio_length + 1] + self.dropout1(src2)
        src3 = self.norm1(src2)
        src3 = self.linear2(self.dropout(self.activation(self.linear1(src3))))
        src2 = src2 + self.dropout2(src3)
        src2 = self.norm2(src2)
        src2 = torch.cat([global_token, src2], dim=0)
        src = torch.cat([src2, src[audio_length + 1 :]])
        # print('after src shape :',src.shape)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        print("before src shape :", src.shape)
        src2 = self.norm1(src)
        pos_src = self.with_pos_embed(src2, pos)
        global_token, q, k, v = (
            src[0].unsqueeze(0),
            pos_src[1:76],
            pos_src[76:],
            src2[76:],
        )
        # print(q.shape) # 100 32 256

        src2 = self.self_attn(
            q,
            k,
            value=v,
            attn_mask=src_key_padding_mask[:, 1:76].permute(1, 0),
            key_padding_mask=src_key_padding_mask[:, 76:],
        )[0]
        src2 = src[1:76] + self.dropout1(src2)
        src3 = self.norm1(src2)
        src3 = self.linear2(self.dropout(self.activation(self.linear1(src3))))
        src2 = src2 + self.dropout2(src3)
        src2 = self.norm2(src2)
        src2 = torch.cat([global_token, src2], dim=0)
        src = torch.cat([src2, src[76:]])
        print("after src shape :", src.shape)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        **kwargs,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        # For tvsum, add kwargs
        return self.forward_post(src, src_mask, src_key_padding_mask, pos, **kwargs)


class TransformerEncoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(src, pos)
        src2 = self.self_attn(
            q, k, value=src, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

    def forward_pre(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        src2 = self.norm1(src)
        q = k = self.with_pos_embed(src2, pos)
        src2 = self.self_attn(
            q, k, value=src2, attn_mask=src_mask, key_padding_mask=src_key_padding_mask
        )[0]
        src = src + self.dropout1(src2)
        src2 = self.norm2(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src2))))
        src = src + self.dropout2(src2)
        return src

    def forward(
        self,
        src,
        src_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(src, src_mask, src_key_padding_mask, pos)
        return self.forward_post(src, src_mask, src_key_padding_mask, pos)


class TransformerDecoderLayer(nn.Module):
    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
        keep_query_pos=False,
        rm_self_attn_decoder=False,
    ):
        super().__init__()
        # Decoder Self-Attention
        if not rm_self_attn_decoder:
            self.sa_qcontent_proj = nn.Linear(d_model, d_model)
            self.sa_qpos_proj = nn.Linear(d_model, d_model)
            self.sa_kcontent_proj = nn.Linear(d_model, d_model)
            self.sa_kpos_proj = nn.Linear(d_model, d_model)
            self.sa_v_proj = nn.Linear(d_model, d_model)
            self.self_attn = MultiheadAttention(
                d_model, nhead, dropout=dropout, vdim=d_model
            )

            self.norm1 = nn.LayerNorm(d_model)
            self.dropout1 = nn.Dropout(dropout)

        # Decoder Cross-Attention
        self.ca_qcontent_proj = nn.Linear(d_model, d_model)
        self.ca_qpos_proj = nn.Linear(d_model, d_model)
        self.ca_kcontent_proj = nn.Linear(d_model, d_model)
        self.ca_kpos_proj = nn.Linear(d_model, d_model)
        self.ca_v_proj = nn.Linear(d_model, d_model)
        self.ca_qpos_sine_proj = nn.Linear(d_model, d_model)
        self.cross_attn = MultiheadAttention(
            d_model * 2, nhead, dropout=dropout, vdim=d_model
        )

        self.nhead = nhead
        self.rm_self_attn_decoder = rm_self_attn_decoder

        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

        self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before
        self.keep_query_pos = keep_query_pos

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
        query_sine_embed=None,
        is_first=False,
    ):

        # ========== Begin of Self-Attention =============
        if not self.rm_self_attn_decoder:
            # Apply projections here
            # shape: num_queries x batch_size x 256
            q_content = self.sa_qcontent_proj(
                tgt
            )  # target is the input of the first decoder layer. zero by default.
            q_pos = self.sa_qpos_proj(query_pos)
            k_content = self.sa_kcontent_proj(tgt)
            k_pos = self.sa_kpos_proj(query_pos)
            v = self.sa_v_proj(tgt)

            num_queries, bs, n_model = q_content.shape
            hw, _, _ = k_content.shape

            q = q_content + q_pos
            k = k_content + k_pos

            tgt2 = self.self_attn(
                q, k, value=v, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
            )[0]
            # ========== End of Self-Attention =============

            tgt = tgt + self.dropout1(tgt2)
            tgt = self.norm1(tgt)

        # ========== Begin of Cross-Attention =============
        # Apply projections here
        # shape: num_queries x batch_size x 256
        q_content = self.ca_qcontent_proj(tgt)
        k_content = self.ca_kcontent_proj(memory)
        v = self.ca_v_proj(memory)

        num_queries, bs, n_model = q_content.shape
        hw, _, _ = k_content.shape

        k_pos = self.ca_kpos_proj(pos)

        # For the first decoder layer, we concatenate the positional embedding predicted from
        # the object query (the positional embedding) into the original query (key) in DETR.
        if is_first or self.keep_query_pos:
            q_pos = self.ca_qpos_proj(query_pos)
            q = q_content + q_pos
            k = k_content + k_pos
        else:
            q = q_content
            k = k_content

        q = q.view(num_queries, bs, self.nhead, n_model // self.nhead)
        query_sine_embed = self.ca_qpos_sine_proj(query_sine_embed)
        query_sine_embed = query_sine_embed.view(
            num_queries, bs, self.nhead, n_model // self.nhead
        )
        q = torch.cat([q, query_sine_embed], dim=3).view(num_queries, bs, n_model * 2)
        k = k.view(hw, bs, self.nhead, n_model // self.nhead)
        k_pos = k_pos.view(hw, bs, self.nhead, n_model // self.nhead)
        k = torch.cat([k, k_pos], dim=3).view(hw, bs, n_model * 2)

        tgt2 = self.cross_attn(
            query=q,
            key=k,
            value=v,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        # ========== End of Cross-Attention =============

        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt


class TransformerDecoderLayerThin(nn.Module):
    """removed intermediate layer"""

    def __init__(
        self,
        d_model,
        nhead,
        dim_feedforward=2048,
        dropout=0.1,
        activation="relu",
        normalize_before=False,
    ):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self.multihead_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        # Implementation of Feedforward model
        self.linear1 = nn.Linear(d_model, d_model)
        # self.linear1 = nn.Linear(d_model, dim_feedforward)
        # self.dropout = nn.Dropout(dropout)
        # self.linear2 = nn.Linear(dim_feedforward, d_model)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        # self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        # self.dropout3 = nn.Dropout(dropout)

        # self.activation = _get_activation_fn(activation)
        self.normalize_before = normalize_before

    def with_pos_embed(self, tensor, pos: Optional[Tensor]):
        return tensor if pos is None else tensor + pos

    def forward_post(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        q = k = self.with_pos_embed(tgt, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt2 = self.linear1(tgt2)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        # tgt = tgt + self.dropout2(tgt2)
        # tgt = self.norm2(tgt)
        # tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        # tgt = tgt + self.dropout3(tgt2)
        # tgt = self.norm3(tgt)
        return tgt

    def forward_pre(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        tgt2 = self.norm1(tgt)
        q = k = self.with_pos_embed(tgt2, query_pos)
        tgt2 = self.self_attn(
            q, k, value=tgt2, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask
        )[0]
        tgt = tgt + self.dropout1(tgt2)
        tgt2 = self.norm2(tgt)
        tgt2 = self.multihead_attn(
            query=self.with_pos_embed(tgt2, query_pos),
            key=self.with_pos_embed(memory, pos),
            value=memory,
            attn_mask=memory_mask,
            key_padding_mask=memory_key_padding_mask,
        )[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt2 = self.norm3(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt2))))
        tgt = tgt + self.dropout3(tgt2)
        return tgt

    def forward(
        self,
        tgt,
        memory,
        tgt_mask: Optional[Tensor] = None,
        memory_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        memory_key_padding_mask: Optional[Tensor] = None,
        pos: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ):
        if self.normalize_before:
            return self.forward_pre(
                tgt,
                memory,
                tgt_mask,
                memory_mask,
                tgt_key_padding_mask,
                memory_key_padding_mask,
                pos,
                query_pos,
            )
        return self.forward_post(
            tgt,
            memory,
            tgt_mask,
            memory_mask,
            tgt_key_padding_mask,
            memory_key_padding_mask,
            pos,
            query_pos,
        )


def _get_clones(module, N):
    """Creates N clones of the module.

    Args:
        module: Module to clone.
        N (int): Number of clones.

    Returns:
        nn.ModuleList: List of cloned modules.
    """
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


def build_transformer(args):
    """Builds the transformer model.

    Args:
        args: Configuration arguments.

    Returns:
        Transformer: The transformer model.
    """
    return Transformer(
        d_model=args.hidden_dim,
        dropout=args.dropout,
        nhead=args.nheads,
        dim_feedforward=args.dim_feedforward,
        num_encoder_layers=args.enc_layers,
        num_decoder_layers=args.dec_layers,
        normalize_before=False,
        return_intermediate_dec=True,
        activation="prelu",
    )


def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    if activation == "prelu":
        return nn.PReLU()
    if activation == "selu":
        return F.selu
    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")


# --- File: src.models.components.attention.cross_modal_fusion ---
# cross_modal_fusion.py
import torch
import torch.nn as nn



class CrossModalCoAttention(nn.Module):
    """
    Co-Attention between Audio and Text.
    Allows both modalities to attend to each other.
    """

    def __init__(self, d_model=512, nhead=8, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead

        # Audio -> Text attention
        self.audio_to_text = MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        # Text -> Audio attention
        self.text_to_audio = MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

        # Feed-forward
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm_ffn = nn.LayerNorm(d_model)

    def forward(self, audio_feat, text_feat, audio_mask=None, text_mask=None):
        """
        audio_feat: (B, L_a, D)
        text_feat:  (B, L_t, D)
        """

        # Audio attends to Text
        audio2text, _ = self.audio_to_text(
            query=audio_feat,
            key=text_feat,
            value=text_feat,
            key_padding_mask=~text_mask if text_mask is not None else None,
        )
        audio_feat = self.norm1(audio_feat + self.dropout(audio2text))

        # Text attends to Audio
        text2audio, _ = self.text_to_audio(
            query=text_feat,
            key=audio_feat,
            value=audio_feat,
            key_padding_mask=~audio_mask if audio_mask is not None else None,
        )
        text_feat = self.norm2(text_feat + self.dropout(text2audio))

        # FFN
        audio_feat = audio_feat + self.dropout(self.ffn(audio_feat))
        audio_feat = self.norm_ffn(audio_feat)

        text_feat = text_feat + self.dropout(self.ffn(text_feat))
        text_feat = self.norm_ffn(text_feat)

        return audio_feat, text_feat


class CrossModalFusionBlock(nn.Module):
    """Stack multiple co-attention layers + final fusion"""

    def __init__(self, d_model=512, nhead=8, num_layers=2, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [CrossModalCoAttention(d_model, nhead, dropout) for _ in range(num_layers)]
        )

        self.final_fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

    def forward(self, audio_feat, text_feat, audio_mask=None, text_mask=None):
        for layer in self.layers:
            audio_feat, text_feat = layer(audio_feat, text_feat, audio_mask, text_mask)

        # Final fusion: concatenate + project
        fused = torch.cat([audio_feat, text_feat], dim=1)  # (B, L_a + L_t, D)

        return fused


# --- File: src.models.components.sg_detr_blocks.blocks.anchor ---
"""Anchors init."""

from typing import List, Tuple

import numpy as np
import torch
from torch import Tensor, nn




class RandomAnchor(nn.Module):
    """Random anchor Embedding."""

    def __init__(self, num_queries: int, uniform: bool = True) -> None:
        """
        Initialize the AnchorEmbedding module.

        Args:
            num_queries (int): Total number of anchors
            uniform (bool): if True, uniform distribution will be used
        """
        super().__init__()
        self.num_queries = num_queries
        self.center = nn.Embedding(num_queries, 1)
        self.width = nn.Embedding(num_queries, 1)

        # Initialize the weights
        if uniform:
            self.center.weight.data = inverse_sigmoid(
                torch.Tensor(np.random.uniform(0, 1, size=num_queries))
            )[:, None]
            self.width.weight.data = inverse_sigmoid(
                torch.Tensor(np.random.uniform(0, 0.5, size=num_queries))
            )[:, None]

    def get_reference_points(self) -> Tensor:
        """
        Get reference points as tensor [n_points, 2].

        Returns:
            Tensor: reference points as tensor [n_points, 2]
        """
        return torch.cat([self.center.weight, self.width.weight], dim=-1)

    def forward(self, idx: int) -> Tensor:
        """
        Forward pass for the AnchorEmbedding module.

        Args:
            idx (Tensor): Index tensor for the anchors.

        Returns:
            Tensor: Concatenated embeddings of centers and widths.
        """
        centers = self.center_embedding(idx)  # type: ignore
        widths = self.width_embedding(idx)  # type: ignore
        return torch.cat([centers, widths], dim=-1)


def distribute_points_on_triangle(
    n_references: int, min_width: float
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Distributes points uniformly along the perimeter of a triangle with vertices
    at (0, 0), (0.5, 1), (1, 0), adjusting for a minimum width on the bottom side.

    Args:
        n_references (int): Number of points to distribute along the perimeter.
        min_width (float): Minimum width adjustment for the bottom side.

    Returns:
        Tuple[np.ndarray, np.ndarray]: Two numpy arrays containing the x and y coordinates
                                        of the distributed points.
    """
    # Coordinates of the triangle vertices
    points = np.array([[0, 0], [0.5, 1], [1, 0]])

    # Lengths of the triangle sides
    side_lengths = np.linalg.norm(np.diff(points[[0, 1, 2, 0], :], axis=0), axis=1)
    perimeter = np.sum(side_lengths)

    # Determining the number of points on each side based on its length
    num_points_per_side = (side_lengths / perimeter * n_references).astype(int)
    num_points_per_side[-1] = (
        n_references - num_points_per_side[:-1].sum()
    )  # Adjusting the last segment

    def interpolate_points(p1: float, p2: float, num_points: int) -> np.ndarray:
        return np.linspace(p1, p2, num_points, endpoint=False)[1:]

    x_points: List[float] = []
    y_points: List[float] = []

    for point_idx in range(3):
        point_1 = points[point_idx]
        point_2 = points[(point_idx + 1) % 3]
        num_points = num_points_per_side[point_idx]

        # Adjusting the starting points for the left and right sides
        if point_idx == 0:
            point_1 = [min_width / 2, min_width]
        if point_idx == 1:
            point_2 = [1 - min_width / 2, min_width]

        interpolated_points = interpolate_points(point_1, point_2, num_points + 1)

        if point_idx == 2:  # Bottom side
            interpolated_points[:, 1] += min_width  # Increasing y for the bottom side

        x_points.extend(interpolated_points[:, 0])
        y_points.extend(interpolated_points[:, 1])

    return np.array(x_points), np.array(y_points)


class TriangleAnchor(nn.Module):
    """
    The embedding class for negative anchors.

    Separates the centers and widths of the anchors, which allows better adjustment during training.
    """

    def __init__(
        self,
        num_queries: int,
        min_width: float = 0.1,
    ) -> None:
        """
        Initialize the NegAnchorEmbedding module.
        """
        super().__init__()
        centers, widths = distribute_points_on_triangle(num_queries, min_width)
        self.num_queries = num_queries
        self.center = nn.Embedding(num_queries, 1)
        self.width = nn.Embedding(num_queries, 1)
        self.center.weight.data = inverse_sigmoid(torch.Tensor(centers))[:, None]
        self.width.weight.data = inverse_sigmoid(torch.Tensor(widths))[:, None]

    def get_reference_points(self) -> Tensor:
        """
        Get reference points as tensor [n_points, 2].

        Returns:
            Tensor: reference points as tensor [n_points, 2]
        """
        return torch.cat([self.center.weight, self.width.weight], dim=-1)

    def forward(self, idx: Tensor) -> Tensor:
        """
        Forward pass for the AnchorEmbedding module.

        Args:
            idx (Tensor): Index tensor for the anchors.

        Returns:
            Tensor: Concatenated embeddings of centers and widths.
        """
        centers = self.center(idx)  # Embedding for centers
        widths = self.width(idx)  # Embedding for widths
        return torch.cat([centers, widths], dim=-1)


class LayerAnchor(nn.Module):
    """
    The embedding class for anchors.

    Separates the centers and widths of the anchors, which allows to better adjust them during training.
    In addition, there is a special initialization for a video width token
    """

    def __init__(
        self,
        num_queries: int,
        ratios: Tuple[float, ...] = (0.5, 0.35),
        special_anchor: bool = True,
    ) -> None:
        """
        Initialize the AnchorEmbedding module.

        Args:
            num_queries (int): Total number of anchors
            ratios (Tuple[float, ...]): Defines anchor levels, each level consists of ratio * num_queries of anchors.
            special_anchor (bool): If True, a special anchor with parameters (0.5, 1) will be added.
        """
        super().__init__()
        self.num_queries = num_queries
        # Create two separate embeddings: one for the centers, the other for the widths
        layers_num_queries = (
            num_queries - 1 if special_anchor else num_queries
        )  # If there is a special anchor,
        # then remove one of the anchor from the distribution
        self.center = nn.Embedding(layers_num_queries, 1)
        self.width = nn.Embedding(layers_num_queries, 1)
        # Determine the number of anchors for each level
        ratios_n = [int(ratio * layers_num_queries) for ratio in ratios]
        # Determine the number of anchors for the last level
        ratios_n.append(layers_num_queries - sum(ratios_n))
        centers: List[float] = []
        widths: List[float] = []
        for ratio_n in ratios_n:
            # distribute the anchors evenly, their width is the same
            centers.extend(np.linspace(0, 1, ratio_n + 2)[1:-1])
            widths.extend([1 / ratio_n] * ratio_n)
        # Add a special anchor
        if special_anchor:
            centers.append(0.5)
            widths.append(1)
        # Initialize the weights
        self.center.weight.data = inverse_sigmoid(torch.Tensor(centers))[:, None]
        self.width.weight.data = inverse_sigmoid(torch.Tensor(widths))[:, None]

    def get_reference_points(self) -> Tensor:
        """
        Get reference points as tensor [n_points, 2].

        Returns:
            Tensor: reference points as tensor [n_points, 2]
        """
        return torch.cat([self.center.weight, self.width.weight], dim=-1)

    def forward(self, idx: int) -> Tensor:
        """
        Forward pass for the AnchorEmbedding module.

        Args:
            idx (Tensor): Index tensor for the anchors.

        Returns:
            Tensor: Concatenated embeddings of centers and widths.
        """
        centers = self.center_embedding(idx)  # type: ignore
        widths = self.width_embedding(idx)  # type: ignore
        return torch.cat([centers, widths], dim=-1)


# --- File: src.models.components.sg_detr_blocks.utils.aux_anchors ---
"""Aux anchors utils."""

from typing import Any, Dict, List, Tuple, Union

import torch
from torch import Tensor, nn
from torch.nn import Embedding





# pylint: disable=R0913,R0914
def prepare_anchors_codetr(  # noqa: WPS210, WPS234
    linear_mapper: Union[nn.Linear, nn.Sequential],
    matched_gts: List[torch.Tensor],
    anchors_per_seq: List[torch.Tensor],
    encoder_features_per_seq: List[torch.Tensor],
) -> Tuple[Tensor, Tensor, Dict[str, Any]]:  # noqa: WPS221
    """
    Add anchors from aux head as support reference points for detr during training.

    Args:
        linear_mapper (nn.Linear): linear mapper.
        matched_gts (List[torch.Tensor]): gt spans mathced to selected anchors.
        anchors_per_seq (List[torch.Tensor]): selected anchors from aux head.
        encoder_features_per_seq (List[torch.Tensor]): selected encoder features.

    Returns:
        Tuple[Tensor, Tensor, Dict[str, Any]]:
            - input_query_label: label query embedding for detr decoder
            - input_query_spans: reference points for detr decoder
            - mask_dict: aux information for support reference points. None if no support spans
    """
    device = encoder_features_per_seq[0].device
    pad_size = len(anchors_per_seq[0])

    # prepare gt spans and labels
    known = [torch.nonzero(torch.ones(len(span))) for span in matched_gts]  # noqa: WPS221
    positive_inds = torch.cat(
        [ones + idx * pad_size for idx, ones in enumerate(known)]
    ).to(device)  # noqa: WPS221

    gt_labels = torch.zeros(len(anchors_per_seq) * pad_size, device=device)
    gt_labels[positive_inds[..., 0]] = 1
    gt_spans = torch.cat(matched_gts)  # flatten the batch bboxes

    # get content query
    encoder_features = torch.stack(encoder_features_per_seq)
    input_query_label = linear_mapper(encoder_features)

    # get pos queries
    anchors_spans = torch.stack(anchors_per_seq)
    input_query_span = inverse_sigmoid(anchors_spans).to(torch.float32)

    input_query_label = input_query_label.transpose(0, 1)
    input_query_span = input_query_span.transpose(0, 1)

    mask_dict = {
        "known_lbs_bboxes": (gt_labels, gt_spans),
        "pad_size": pad_size,
    }
    return input_query_label, input_query_span, mask_dict


# pylint: disable=R0913,R0914
def prepare_anchors_codetr_legacy(  # noqa: WPS210, WPS234
    linear_mapper: nn.Linear,
    matched_gts: List[torch.Tensor],
    anchors_per_seq: List[torch.Tensor],
    encoder_features_per_seq: List[torch.Tensor],
    batch_size: int = 512,
) -> Tuple[Tensor, Tensor, Dict[str, Any]]:  # noqa: WPS221
    """
    Add anchors from aux head as support reference points for detr during training.

    Args:
        linear_mapper (nn.Linear): linear mapper.
        matched_gts (List[torch.Tensor]): gt spans mathced to selected anchors.
        anchors_per_seq (List[torch.Tensor]): selected anchors from aux head.
        encoder_features_per_seq (List[torch.Tensor]): selected encoder features.
        batch_size (int): batch size

    Returns:
        Tuple[Tensor, Tensor, Dict[str, Any]]:
            - input_query_label: label query embedding for detr decoder
            - input_query_spans: reference points for detr decoder
            - mask_dict: aux information for support reference points. None if no support spans
    """
    device = encoder_features_per_seq[0].device

    known = [
        torch.ones(len(span)) for span in matched_gts
    ]  # replace bboxes with ones in batch
    know_idx = [
        torch.nonzero(idxs) for idxs in known
    ]  # enumerate bboxes of each object in the batch
    known_num = [sum(idxs) for idxs in known]  # count bboxes in the batch

    # prepare indices
    tmp = torch.cat(known)  # flatten the indices
    known_indice = torch.nonzero(tmp).view(-1)  # enumerate all bboxes
    known_indice = known_indice.to(device)

    # prepare spans and labels
    gt_spans = torch.cat(matched_gts)  # flatten the batch bboxes
    gt_labels = torch.cat(known).long().to(device)
    anchors_spans = torch.cat(anchors_per_seq)

    # get the batch index for each span
    batch_idx = torch.cat(
        [torch.full_like(ones, idx) for idx, ones in enumerate(known)]
    )  # noqa: WPS221
    batch_idx = batch_idx.to(device)

    # padding shapes
    pad_size = int(max(known_num))  # max anchors per seq

    # add padding to labels
    encoder_features = torch.cat(encoder_features_per_seq)
    input_label_embed = linear_mapper(encoder_features)
    padding_label = torch.zeros(pad_size, input_label_embed.size(1)).to(device)
    input_query_label = padding_label.repeat(batch_size, 1, 1)

    # add padding to spans
    input_bbox_embed = inverse_sigmoid(anchors_spans).to(torch.float32)
    padding_bbox = torch.zeros(pad_size, 2, device=device)
    input_query_bbox = padding_bbox.repeat(batch_size, 1, 1)

    # map in order
    map_known_indice = torch.cat(
        [torch.tensor(range(int(num))) for num in known_num]
    ).long()  # noqa: WPS221
    input_query_label[(batch_idx.long(), map_known_indice)] = input_label_embed
    input_query_bbox[(batch_idx.long(), map_known_indice)] = input_bbox_embed

    input_query_label = input_query_label.transpose(0, 1)
    input_query_bbox = input_query_bbox.transpose(0, 1)

    mask_dict = {
        "known_indice": torch.as_tensor(known_indice).long(),
        "batch_idx": torch.as_tensor(batch_idx).long(),
        "map_known_indice": torch.as_tensor(map_known_indice).long(),
        "known_lbs_bboxes": (gt_labels, gt_spans),
        "know_idx": know_idx,
        "pad_size": pad_size,
    }

    return input_query_label, input_query_bbox, mask_dict


def recalculate_num_groups(num_groups: int, known_num: List[int]) -> int:
    """Recalculate num groups based on max number of spans.

    Args:
        num_groups (int): initial num of groups
        known_num (List[int]): number of gt spans in each sample

    Returns:
        int: recalulated num of groups
    """
    if int(max(known_num)) == 0:
        num_groups = 1
    else:
        num_groups = num_groups // (int(max(known_num) * 2))
    if num_groups < 1:
        num_groups = 1
    return num_groups


# pylint: disable=R0915
def prepare_anchors_dn(  # noqa: WPS210
    label_enc: Embedding,
    targets: Dict[str, Any],
    num_groups: int = 5,
    span_noise_scale: float = 0.4,
    negative_offset: float = 1.0,
    batch_size: int = 512,
) -> Tuple[Tensor, Tensor, Dict[str, Any]]:
    """
    Add noise to gt spans and use them as support regerence points during training.

    Args:
        label_enc (Embedding): target embeddings.
        targets (Dict[str, Any]): target dict contains "span_labels"
        num_groups (int): number of denoise groups
        span_noise_scale (float): noise scale for span coords
        negative_offset (float): offset for negative samples
        batch_size (int): batch size

    Returns:
        Tuple[Tensor, Tensor, Dict[str, Any]]:
            - input_query_label: label query embedding for detr decoder
            - input_query_spans: reference points for detr decoder
            - mask_dict: aux information for support reference points. None if no support spans
    """
    device = label_enc.weight.device
    tg_spans = targets["span_labels"]
    known = [
        torch.ones(len(span["spans"])) for span in tg_spans
    ]  # replace spans with ones in batch
    known_num = [sum(idxs) for idxs in known]  # count spans in the batch

    num_groups = recalculate_num_groups(num_groups, known_num)

    # prepare for the dn part
    tmp = torch.cat(known)  # flatten the indices
    known_indice = torch.nonzero(tmp).view(-1)  # enumerate all spans

    # get the batch index for each span
    spans = torch.cat([span["spans"] for span in tg_spans])  # flatten the batch spans
    labels = torch.cat(known).long()
    known_indice = known_indice.to(device)
    labels = labels.to(device)
    batch_idx = torch.cat(
        [torch.full_like(ones, idx) for idx, ones in enumerate(known)]
    )  # noqa: WPS221
    batch_idx = batch_idx.to(device)

    # add noise
    double_groups = 2 * num_groups
    known_indice = known_indice.repeat(double_groups, 1).view(-1)
    known_labels = labels.repeat(double_groups, 1).view(-1)
    known_bid = batch_idx.repeat(double_groups, 1).view(-1)
    known_spans = spans.repeat(double_groups, 1)
    known_labels_expaned = known_labels.clone()
    known_spans_expand = known_spans.clone()

    # padding shapes
    single_pad = int(max(known_num))
    pad_size = int(single_pad * double_groups)

    # positive/negative indices
    positive_idx = torch.tensor(range(len(spans)), dtype=torch.long, device=device)
    positive_idx = positive_idx[None].repeat(num_groups, 1)
    pos_shift = (
        torch.tensor(range(num_groups), dtype=torch.long, device=device)
        * len(spans)
        * 2
    )  # noqa: WPS221
    positive_idx += pos_shift.unsqueeze(1)
    positive_idx = positive_idx.flatten()
    negative_idx = positive_idx + len(spans)

    # apply noise on the box
    known_spans_expand_xx = span_cxw_to_xx(known_spans_expand)
    diff = torch.zeros_like(known_spans_expand, device=device)
    diff[:, :1] = known_spans_expand[:, 1:] / 2  # center diff
    diff[:, 1:] = known_spans_expand[:, 1:] / 2  # width diff

    rand_sign = torch.randint_like(
        known_spans_expand, low=0, high=2, dtype=torch.float32
    )  # exclusive
    rand_sign = rand_sign * 2.0 - 1.0  # noqa: WPS432
    rand_part = torch.rand_like(known_spans_expand)
    rand_part[negative_idx] += negative_offset
    modulation = rand_part * rand_sign
    modulated_diff = torch.mul(modulation, diff) * span_noise_scale

    # make diff scale dependent
    known_spans_expand_xx = known_spans_expand_xx + modulated_diff
    known_spans_expand_xx = known_spans_expand_xx.clamp(min=0.0, max=1.0)
    known_spans_expand = span_xx_to_cxw(known_spans_expand_xx)

    # prepare labels
    input_label_embed = label_enc(known_labels_expaned - 1)
    padding_label = torch.zeros(pad_size, batch_size, input_label_embed.size(1)).to(
        device
    )
    input_query_label = padding_label.transpose(0, 1)

    # prepare spans
    input_span_embed = inverse_sigmoid(known_spans_expand)
    input_query_spans = torch.zeros(batch_size, pad_size, 2, device=device)

    # map in order
    map_known_indice = torch.cat([torch.tensor(range(int(num))) for num in known_num])  # noqa: WPS221
    map_known_indice = torch.cat(
        [map_known_indice + single_pad * idx for idx in range(double_groups)]
    )  # noqa: WPS221
    map_known_indice = map_known_indice.long()
    input_query_label[(known_bid.long(), map_known_indice)] = input_label_embed
    input_query_spans[(known_bid.long(), map_known_indice)] = input_span_embed

    mask_dict = {
        "pad_size": pad_size,
        "num_groups": num_groups,
    }

    input_query_label = input_query_label.transpose(0, 1)
    input_query_spans = input_query_spans.transpose(0, 1)

    return input_query_label, input_query_spans, mask_dict


def aux_post_process(
    outputs_class: torch.Tensor,
    outputs_coord: torch.Tensor,
    quality_scores: torch.Tensor,
    offsets: torch.Tensor,
    co_dict: Dict[str, Any],
    dn_dict: Dict[str, Any],
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Separate denoise part from the output and put it in the mask_dict.

    Args:
        outputs_class (torch.Tensor): known and predicted classes.
        outputs_coord (torch.Tensor): known and predicted spans.
        quality_scores (torch.Tensor): known and predicted iou scores.
        offsets (torch.Tensor): calculated offsets.
        co_dict (Dict[str, Any]): collab paddings.
        dn_dict (Dict[str, Any]): dn paddings.

    Returns:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]: Predicted classes and spans, etc.
    """
    co_pad_size = co_dict["pad_size"] if co_dict is not None else 0
    dn_pad_size = dn_dict["pad_size"] if dn_dict is not None else 0
    tgt_pad_size = dn_pad_size + co_pad_size

    tgt_outputs_class = outputs_class[:, :, tgt_pad_size:, :]
    tgt_outputs_coord = outputs_coord[:, :, tgt_pad_size:, :]
    if quality_scores is not None:
        quality_scores = quality_scores[:, :, tgt_pad_size:, :]
    if offsets is not None:
        offsets = offsets[:, :, tgt_pad_size:, :]

    if dn_pad_size > 0:
        output_dn_class = outputs_class[:, :, :dn_pad_size, :]
        output_dn_coord = outputs_coord[:, :, :dn_pad_size, :]
        dn_dict["output_known_lbs_bboxes"] = (output_dn_class, output_dn_coord)

    if co_pad_size > 0:
        output_co_class = outputs_class[:, :, dn_pad_size:tgt_pad_size, :]
        output_co_coord = outputs_coord[:, :, dn_pad_size:tgt_pad_size, :]
        co_dict["output_known_lbs_bboxes"] = (output_co_class, output_co_coord)
    return tgt_outputs_class, tgt_outputs_coord, quality_scores, offsets


# --- File: src.models.components.sg_detr_blocks.blocks.multiscale ---
"""Prepare multiscale features for the transformer."""

from typing import List

import torch
from torch import Tensor, nn



INIT_CONST: float = 0.01


class DWSCond1d(nn.Module):
    """DWS conv."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        padding: int = 1,
        stride: int = 1,
        bias: bool = False,
    ):
        """Initialize DWS block.

        Args:
            in_channels (int): input channels.
            out_channels (int): output channels.
            kernel_size (int): kernel size. Defaults to 3.
            padding (int): padding size. Defaults to 1.
            stride (int): stride. Defaults to 1.
            bias (bool): whether to use bias or not. Defaults to False.
        """
        super().__init__()
        self.depthwise = nn.Conv1d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias,
        )
        self.pointwise = nn.Conv1d(out_channels, out_channels, kernel_size=1, bias=bias)

    def forward(self, emb: Tensor) -> Tensor:
        """Forward pass of the DWS block.

        Args:
            emb (Tensor): input tensor

        Returns:
            Tensor: output tensor
        """
        out = self.depthwise(emb)
        return self.pointwise(out)


class FPNSequence(nn.Module):
    """Feature Pyramid Network sequence."""

    def __init__(self, feature_dim: int) -> None:
        """Initialize a Feature Pyramid Network sequence.

        Args:
            feature_dim (int): Feature dimension.
        """
        super().__init__()
        self.upscale = nn.Sequential(
            nn.Upsample(scale_factor=2),
            DWSCond1d(feature_dim, feature_dim, kernel_size=3, padding=1, bias=False),  # noqa: WPS204
            TransposedLayerNorm(feature_dim),  # noqa: WPS204
            nn.GELU(),
            DWSCond1d(feature_dim, feature_dim, kernel_size=3, padding=1, bias=False),
            TransposedLayerNorm(feature_dim),
        )

        self.level0 = nn.Sequential(
            DWSCond1d(feature_dim, feature_dim, kernel_size=3, padding=1, bias=False),
            TransposedLayerNorm(feature_dim),
            nn.GELU(),
            DWSCond1d(feature_dim, feature_dim, kernel_size=3, padding=1, bias=False),
            TransposedLayerNorm(feature_dim),
        )

        self.level1 = nn.Sequential(
            DWSCond1d(
                feature_dim, feature_dim, kernel_size=3, stride=2, padding=1, bias=False
            ),
            TransposedLayerNorm(feature_dim),
            nn.GELU(),
            DWSCond1d(feature_dim, feature_dim, kernel_size=3, padding=1, bias=False),
            TransposedLayerNorm(feature_dim),
        )

        self.level2 = nn.Sequential(
            nn.GELU(),
            DWSCond1d(
                feature_dim, feature_dim, kernel_size=3, stride=2, padding=1, bias=False
            ),
            TransposedLayerNorm(feature_dim),
            nn.GELU(),
            DWSCond1d(feature_dim, feature_dim, kernel_size=3, padding=1, bias=False),
            TransposedLayerNorm(feature_dim),
        )
        self._init_params()

    def _init_params(self) -> None:
        """Init stem blocks."""
        for module in self.modules():
            if isinstance(module, nn.Conv1d):
                torch.nn.init.normal_(module.weight, std=INIT_CONST)
                if module.bias is not None:
                    torch.nn.init.constant_(module.bias, 0)  # type: ignore

    def forward(self, emb: torch.Tensor) -> List[torch.Tensor]:
        """Forward pass through the FPN sequence.

        Args:
            emb (torch.Tensor): Input tensor.

        Returns:
            List[torch.Tensor]: List of scaled features.
        """
        emb = emb.permute(0, 2, 1)
        levelu = self.upscale(emb).permute(0, 2, 1)  # 160
        level0 = self.level0(emb).permute(0, 2, 1)  # 80
        level1 = self.level1(emb)  # 40
        level2 = self.level2(level1).permute(0, 2, 1)  # 20
        return [levelu, level0, level1.permute(0, 2, 1), level2]


# --- File: src.models.components.sg_detr_blocks.blocks.pooling ---
"""Pooling modules."""

import torch
from torch import Tensor, nn





EPS: float = 1e-6
INIT_CONST: float = 0.02
GAMMA_CONST: float = 1e-4  # 1  #  A constant for the residual connection for pooling, since the query vector is
# initially random, it is better to add aggregation completely.


class AttentionPool2d(nn.Module):
    """Attention for Learned Aggregation."""

    def __init__(self, dim: int, bias: bool = True):
        """Initialize AttentionPool2d.

        Args:
            dim (int): Dimensionality of the input embeddings.
            bias (bool): If True, adds a learnable bias to the linear transformations. Default is True.
        """
        super().__init__()
        self.q_proj = nn.Linear(dim, dim, bias=bias)
        self.vk_proj = nn.Linear(dim, dim * 2, bias=bias)
        self.proj = nn.Linear(dim, dim)

    def forward(self, emb: Tensor, cls_q: Tensor) -> Tensor:
        """
        Forward pass of the AttentionPool2d module.

        Args:
            emb (Tensor): Input embeddings of shape (batch_size, seq_len, dim).
            cls_q (Tensor): Query tensor of shape (batch_size, dim).

        Returns:
            Tensor: Projected and pooled output tensor of shape (batch_size, dim).
        """
        emb = emb.transpose(1, 2)  # swap seq_len and dim
        batch_size, seq_len, dim = emb.shape

        # Compute query vector
        query = self.q_proj(cls_q.expand(batch_size, -1, -1))
        # Compute key and value vectors
        key_value = self.vk_proj(emb).reshape(batch_size, seq_len, 2, dim)
        key, value = key_value.permute(2, 0, 1, 3).chunk(2, 0)

        # Compute attention scores
        attn = torch.matmul(query, key.transpose(-2, -1))
        attn = torch.softmax(attn, dim=-1)
        # Compute weighted sum of values
        emb = torch.matmul(attn, value).transpose(1, 2).reshape(batch_size, dim)

        # Project and return pooled output
        return self.proj(emb)


class LearnedAggregationLayer(nn.Module):
    """Learned Aggregation from https://arxiv.org/abs/2112.13692."""

    def __init__(
        self,
        dim: int,
        expansion_ratio: int = 4,
        nhead: int = 8,
        dropout: float = 0.1,
        use_projections: bool = False,
        use_gamma: bool = True,
    ):
        """
        Initialize LearnedAggregationLayer.

        Args:
            dim (int): Dimensionality of the input embeddings.
            expansion_ratio (int): Expansion ratio for the feedforward network. Default is 4.
            nhead (int): Number of attention heads. Default is 8.
            dropout (float): Dropout rate. Default is 0.1.
            use_projections (bool): Whether to use projections. Default is False.
        """
        super().__init__()
        self.use_gamma = use_gamma
        if use_gamma:
            self.gamma_1 = nn.Parameter(GAMMA_CONST * torch.ones(dim))  # noqa: WPS114
            self.gamma_2 = nn.Parameter(GAMMA_CONST * torch.ones(dim))  # noqa: WPS114
        else:
            self.gamma_1, self.gamma_2 = 1.0, 1.0  # type: ignore

        self.attn = DABMultiheadAttention(dim, nhead)
        self.attn_norm = nn.LayerNorm(dim)
        self.ffn_norm = nn.LayerNorm(dim)
        self.ffn = FeedForwardNetwork(dim, expansion_ratio, dropout)
        self.use_projections = use_projections
        if use_projections:
            self.k_proj = nn.Linear(dim, dim)
            self.v_proj = nn.Linear(dim, dim)

        self.apply(self._init_weights)

    def forward(self, emb: Tensor, query: Tensor, key_padding_mask: Tensor) -> Tensor:
        """
        Forward pass of the LearnedAggregationLayer module.

        Args:
            emb (Tensor): Input embeddings of shape (batch_size, seq_len, dim).
            query (Tensor): Content embeddings of shape (1, batch_size, dim).
            key_padding_mask (Tensor): Mask tensor of shape (batch_size, seq_len).

        Returns:
            Tensor: Output tensor of shape (1, batch_size, dim).
        """
        emb = emb.transpose(0, 1)
        emb = self.attn_norm(emb)
        if self.use_projections:
            k_emb = self.k_proj(emb)
            v_emb = self.v_proj(emb)
        else:
            k_emb, v_emb = emb, emb

        key_padding_mask = key_padding_mask.to(torch.bool)
        attn_out, _ = self.attn(
            query, k_emb, v_emb, key_padding_mask=~key_padding_mask, dummy=False
        )
        output = query + self.gamma_1 * attn_out
        ffn_output = self.ffn(self.ffn_norm(output))
        output = output + self.gamma_2 * ffn_output
        return output

    @torch.no_grad()
    def _init_weights(self, module):
        """
        Initialize weights of the module.

        Args:
            module (nn.Module): A submodule of LearnedAggregationLayer.
        """
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=INIT_CONST)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)


class LearnedAggregation(nn.Module):
    """Learned Aggregation from https://arxiv.org/abs/2112.13692."""

    def __init__(
        self, dim, aggregation_layer: LearnedAggregationLayer, num_layers: int
    ):
        """Initialize a LearnedAggregation module.

        Args:
            dim (int): Dimensionality of the input embeddings.
            aggregation_layer (LearnedAggregationLayer): An instance of the transformer aggregation layer.
            num_layers (int): The number of layers in the learned aggregation module.
        """
        super().__init__()
        self.layers = get_clones(aggregation_layer, num_layers)
        self.num_layers = num_layers
        self.cls_q = nn.Parameter(torch.zeros(dim))
        self.norm = nn.LayerNorm(dim)
        nn.init.trunc_normal_(self.cls_q, std=INIT_CONST)

    def forward(self, emb: Tensor, key_padding_mask: Tensor) -> Tensor:
        """
        Forward pass of the LearnedAggregationLay module.

        Args:
            emb (Tensor): Input embeddings of shape (batch_size, seq_len, dim).
            key_padding_mask (Tensor): Mask tensor of shape (batch_size, seq_len).

        Returns:
            Tensor: Output tensor of shape (batch_size, 1, dim).
        """
        # (1, batch_size, 256)
        content = self.cls_q[None, None].repeat(1, emb.size(0), 1)
        output = content
        layer_n = 0
        for layer in self.layers:
            output = layer(
                emb=emb,
                query=output,
                key_padding_mask=key_padding_mask,
            )
            if layer_n < self.num_layers - 1:
                output = self.norm(output)
            layer_n += 1
        return output.transpose(0, 1)


class GRUFeatureExtractor(nn.Module):
    """GRU-based Feature Extractor."""

    def __init__(
        self,
        input_dim: int,
        hidden_dim: int,
        num_layers: int = 1,
        bidirectional: bool = False,
    ) -> None:
        """Initialize GRUFeatureExtractor.

        Args:
            input_dim (int): Dimensionality of the input.
            hidden_dim (int): Dimensionality of the hidden state.
            num_layers (int): Number of recurrent layers. Default is 1.
            bidirectional (bool): If True, use bidirectional GRU. Default is False.
        """
        super().__init__()
        self.gru = nn.GRU(  # type: ignore
            input_dim,
            hidden_dim,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True,
        )

    def forward(self, emb: Tensor) -> Tensor:
        """
        Forward pass of the GRUFeatureExtractor.

        Args:
            emb (Tensor): Input tensor of shape (batch_size, seq_length, input_dim).

        Returns:
            Tensor: Extracted features (hidden state)
        """
        # emb shape: (batch_size, seq_length, input_dim)
        _, hidden = self.gru(emb)

        # If bidirectional, concatenate the forward and backward hidden states
        if self.gru.bidirectional:
            # Combine the hidden states of both directions (forward and backward)
            hidden = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)  # noqa:WPS221
        else:
            # If not bidirectional, use the hidden state of the last layer
            hidden = hidden[-1, :, :]

        # Output the hidden state(s) as the global feature(s)
        # For a single layer, hidden shape: (batch_size, hidden_dim)
        # For multiple layers, hidden shape: (batch_size, num_layers * hidden_dim)
        return hidden


class GlobalMaxPooling(nn.Module):
    """Max Pooling Layer."""

    def forward(self, extracted_intervals: Tensor, masks: Tensor) -> Tensor:
        """
        Perform forward pass of the GlobalMaxPooling layer.

        Args:
            extracted_intervals (Tensor): The input tensor with shape [batch_size, num_intervals, model_dim].
            masks (Tensor): The mask tensor with shape [batch_size, num_intervals].

        Returns:
            Tensor: The tensor after applying global max pooling with shape [batch_size, 1, model_dim].
        """
        # Apply mask to the extracted_intervals
        masked_intervals = extracted_intervals * masks.unsqueeze(-1).float()
        # Set masked elements to negative infinity
        masked_intervals[masks == 0] = float("-inf")
        # Calculate the max pooling
        max_intervals, _ = masked_intervals.max(dim=1)  # [batch_size, model_dim]
        # Add an extra dimension to match the desired output shape
        return max_intervals.unsqueeze(1)  # [batch_size, 1, model_dim]


class GlobalMeanPooling(nn.Module):
    """Mean Pooling Layer."""

    def forward(self, extracted_intervals: Tensor, masks: Tensor) -> Tensor:
        """
        Perform forward pass of the GlobalMeanPooling layer.

        Args:
            extracted_intervals (Tensor): The input tensor with shape [batch_size, num_intervals, model_dim].
            masks (Tensor): The mask tensor with shape [batch_size, num_intervals].

        Returns:
            Tensor: The tensor after applying global mean pooling with shape [batch_size, 1, model_dim].
        """
        # Apply mask to the extracted_intervals
        masked_intervals = extracted_intervals * masks.unsqueeze(-1).float()

        # Calculate the sum and count for mean pooling
        sum_intervals = masked_intervals.sum(dim=1)  # [batch_size, model_dim]
        count_intervals = masks.sum(dim=1).unsqueeze(-1).float()  # [batch_size, 1]

        # Avoid division by zero
        mean_intervals = sum_intervals / (
            count_intervals + EPS
        )  # [batch_size, model_dim]

        # Add an extra dimension to match the desired output shape
        return mean_intervals.unsqueeze(1)  # [batch_size, 1, model_dim]


# --- File: src.models.components.sg_detr_blocks.blocks.layers ---
"""Module for the transformer layers."""

from typing import Optional

import torch
from torch import Tensor, nn





class Scale(nn.Module):
    """Scale distinguisher."""

    def __init__(self, init_value: float = 1.0) -> None:
        """Initialize Scale.

        Args:
            init_value (float): Init value. Defaults to 1.0.
        """
        super().__init__()
        self.scale = nn.Parameter(torch.FloatTensor([init_value]))

    def forward(self, emb: Tensor) -> Tensor:
        """Forward pass of the module.

        Args:
            emb (Tensor): Features from the certain scale.

        Returns:
            Tensor: Adjusted scale features.
        """
        return emb * self.scale


class DropPath(nn.Module):
    """Drop paths per sample (when applied in main path of residual blocks)."""

    def __init__(self, drop_prob: float = 0.1) -> None:
        """Initialize the DropPath module.

        Args:
            drop_prob (float): The dropout probability. Defaults to 0.1.
        """
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, path_value: Tensor) -> Tensor:
        """Forward pass of the DropPath module.

        Args:
            path_value (Tensor): The input tensor.

        Returns:
            Tensor: The output tensor after applying drop path.
        """
        if not self.training:
            return path_value

        path_value = path_value.permute(1, 0, 2)
        keep_prob = 1 - self.drop_prob
        shape = (path_value.shape[0],) + (1,) * (path_value.ndim - 1)
        mask = keep_prob + torch.rand(
            shape, dtype=path_value.dtype, device=path_value.device
        )
        mask.floor_()
        path_value = path_value.div(keep_prob) * mask
        return path_value.permute(1, 0, 2)


class TransformerEncoderLayer(nn.Module):
    """Transformer encoder layer."""

    def __init__(
        self,
        d_model: int,
        nhead: int = 8,
        expansion_ratio: int = 4,
        dropout: float = 0.1,
        droppath: float = 0.1,
    ):
        """Initialize the TransformerEncoderLayer.

        Args:
            d_model (int): The dimension of the input feature.
            nhead (int): The number of heads in the multihead attention.
            expansion_ratio (int): The expansion ratio for the hidden layer dimension of FFN. Defaults to 4.
            dropout (float): Dropout rate. Defaults to 0.1.
            droppath (float): Droppath rate. Defaults to 0.1.
        """
        super().__init__()
        self.d_model = d_model
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout)
        self._init_parameters()

        self.ffn = FeedForwardNetwork(d_model, expansion_ratio, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = DropPath(droppath)
        self.dropout2 = DropPath(droppath)

    def _init_parameters(self) -> None:
        for param in self.parameters():
            if param.dim() > 1:
                nn.init.xavier_uniform_(param)

    def forward(
        self,
        embedding: Tensor,
        position_embedding: Tensor,
        embedding_mask: Optional[Tensor] = None,
        embedding_key_padding_mask: Optional[Tensor] = None,
    ):
        """Forward pass of the TransformerEncoderLayer.

        Args:
            embedding (Tensor): The input tensor.
            position_embedding (Tensor): The positional embedding for the input tensor.
            embedding_mask (Optional[Tensor]): The mask for the input tensor. Defaults to None.
            embedding_key_padding_mask (Optional[Tensor]):  The key padding mask for the input tensor. Defaults to None.

        Returns:
            _type_: _description_
        """
        query = embedding + position_embedding
        key = query

        # Attention part
        att_out, _ = self.self_attn(
            query=query,
            key=key,
            value=embedding,
            attn_mask=embedding_mask,
            key_padding_mask=embedding_key_padding_mask,
        )
        embedding = embedding + self.dropout1(att_out)
        embedding = self.norm1(embedding)

        # FFN part
        ffn_out = self.ffn(embedding)
        embedding = embedding + self.dropout2(ffn_out)
        return self.norm2(embedding)


class T2ATransformerEncoderLayer(nn.Module):
    """Text to audio transformer encoder layer."""

    def __init__(
        self,
        d_model: int,
        nhead: int = 8,
        expansion_ratio: int = 4,
        dropout: float = 0.1,
        droppath: float = 0.1,
        num_dummies: int = 35,
        use_cross_attn_wo_dummy: bool = False,
        weight_attn_with_saliency: bool = False,
    ) -> None:
        """Initialize the T2V_TransformerEncoderLayer.

        Args:
            d_model (int): The dimension of the input feature.
            nhead (int): The number of heads in the multihead attention.
            expansion_ratio (int): The expansion ratio for the hidden layer dimension of FFN. Defaults to 4.
            dropout (float): Dropout rate. Defaults to 0.1.
            droppath (float): Droppath rate. Defaults to 0.1.
            num_dummies (int): The number of dummy tokens. Defaults to 35.
        """
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.use_cross_attn_wo_dummy = use_cross_attn_wo_dummy
        self.weight_attn_with_saliency = weight_attn_with_saliency

        if use_cross_attn_wo_dummy:
            self.self_attn = DABMultiheadAttention(d_model, nhead, dropout_prob=dropout)
        else:
            self.self_attn = DABMultiheadAttention(
                d_model, nhead, dropout_prob=dropout, num_dummies=num_dummies
            )

        self._init_parameters()

        self.ffn = FeedForwardNetwork(d_model, expansion_ratio, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = DropPath(droppath)
        self.dropout2 = DropPath(droppath)

    def _init_parameters(self) -> None:
        for param in self.parameters():
            if param.dim() > 1:
                nn.init.xavier_uniform_(param)

    def forward(
        self,
        emb: Tensor,
        pos_embed: Tensor,
        audio_length: int,
        emb_key_padding_mask: Optional[Tensor] = None,
        dummy: bool = True,
        saliency_scores: Optional[Tensor] = None,
    ):
        """Forward pass of the T2ATransformerEncoderLayer.

        Args:
            emb (Tensor): The input tensor.
            pos_embed (Tensor): The positional embedding for the input tensor.
            audio_length (int): The length of the audio (#clips).
            emb_key_padding_mask (Optional[Tensor]): The key padding mask for the input tensor. Defaults to None.
            dummy (bool): Whether to use dummy tokens. Defaults to True.

        Returns:
            Tuple[Tensor, Tensor]: The output tensor and the attention weights.
        """
        pos_emb = emb + pos_embed
        query = pos_emb[:audio_length]
        key = pos_emb[audio_length:]
        value = emb[audio_length:]

        attn_mask = None
        if emb_key_padding_mask is not None:
            query_mask = emb_key_padding_mask[:, :audio_length].unsqueeze(2)
            key_mask = emb_key_padding_mask[:, audio_length:].unsqueeze(1)
            attn_mask = torch.matmul(query_mask.float(), key_mask.float())
            attn_mask = torch.repeat_interleave(attn_mask.bool(), self.nhead, dim=0)
            emb_key_padding_mask = emb_key_padding_mask[:, audio_length:]

        if self.use_cross_attn_wo_dummy:
            kwargs = (
                {"saliency_scores": saliency_scores}
                if self.weight_attn_with_saliency
                else {}
            )
            attn_out, attn_weights = self.self_attn(
                query,
                key,
                value=value,
                attn_mask=attn_mask,
                key_padding_mask=emb_key_padding_mask,
                dummy=False,
                **kwargs,
            )
            attn_weights = None
        else:
            attn_out, attn_weights = self.self_attn(
                query,
                key,
                value,
                attn_mask=attn_mask,
                key_padding_mask=emb_key_padding_mask,
                dummy=dummy,
            )

        emb_aug = emb[:audio_length] + self.dropout1(attn_out)
        ffn_out = self.norm1(self.ffn(emb_aug))
        emb_aug = emb_aug + self.dropout2(ffn_out)
        emb_aug = self.norm2(emb_aug)
        src = torch.cat([emb_aug, emb[audio_length:]])
        return src, attn_weights


class DecoderSelfAttention(nn.Module):
    """Self-attention layer for the decoder."""

    def __init__(
        self,
        d_model: int,
        nhead: int,
        dropout: float = 0.1,
        droppath: float = 0.1,
    ):
        """Initialize the DecoderSelfAttention.

        Args:
            d_model (int): The dimension of the input feature.
            nhead (int): The number of heads in the multihead attention.
            dropout (float): Dropout rate. Defaults to 0.1.
            droppath (float): Droppath rate. Defaults to 0.1.
        """
        super().__init__()
        self.query_content_proj = nn.Linear(d_model, d_model)
        self.query_pos_proj = nn.Linear(d_model, d_model)
        self.key_content_proj = nn.Linear(d_model, d_model)
        self.key_pos_proj = nn.Linear(d_model, d_model)
        self.value_proj = nn.Linear(d_model, d_model)

        self.self_attn = DABMultiheadAttention(
            d_model, nhead, dropout_prob=dropout, vdim=d_model
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = DropPath(droppath)

    def forward(
        self,
        embedding: Tensor,
        embedding_mask: Optional[Tensor] = None,
        embedding_key_padding_mask: Optional[Tensor] = None,
        query_pos: Optional[Tensor] = None,
    ) -> Tensor:
        """Forward pass of the DecoderSelfAttention.

        Args:
            embedding (Tensor): The input tensor.
            embedding_mask (Optional[Tensor]): The mask for the input tensor. Defaults to None.
            embedding_key_padding_mask (Optional[Tensor]): The key padding mask for the input tensor. Defaults to None.
            query_pos (Optional[Tensor]): The positional embedding for the query.

        Returns:
            Tensor: The output tensor after applying self-attention.
        """
        # Apply projections here
        # shape: num_queries x batch_size x d_model
        query_content = self.query_content_proj(
            embedding
        )  # embedding is the input of the first decoder layer.
        query_pos = self.query_pos_proj(query_pos)
        key_content = self.key_content_proj(embedding)
        key_pos = self.key_pos_proj(query_pos)
        value = self.value_proj(embedding)

        query = query_content + query_pos
        key = key_content + key_pos

        att_out, _ = self.self_attn(
            query=query,
            key=key,
            value=value,
            attn_mask=embedding_mask,
            key_padding_mask=embedding_key_padding_mask,
            dummy=False,
        )

        embedding = embedding + self.dropout1(att_out)
        return self.norm1(embedding)


# pylint: disable=too-many-arguments,too-many-locals
class TransformerDecoderLayer(nn.Module):
    """Transformer decoder layer."""

    def __init__(
        self,
        d_model: int,
        cont_pos_tradeoff: int,
        nhead: int = 8,
        expansion_ratio: int = 4,
        dropout: float = 0.1,
        droppath: float = 0.1,
        rm_self_attn_decoder: bool = False,
    ) -> None:
        """
        Initialize the TransformerDecoderLayer.

        Args:
            d_model (int): The dimension of the input feature.
            cont_pos_tradeoff (int): Offset for the dim content/position dims.
            nhead (int): The number of heads in the multihead attention.
            expansion_ratio (int): The expansion ratio for the hidden layer dimension of FFN. Defaults to 4.
            dropout (float): Dropout rate. Defaults to 0.1.
            droppath (float): Droppath rate. Defaults to 0.1.
            rm_self_attn_decoder (bool): Whether to remove the self-attention layer in the decoder. Defaults to False.
        """
        super().__init__()
        self.nhead = nhead
        self.cont_pos_tradeoff = cont_pos_tradeoff
        self.rm_self_attn_decoder = rm_self_attn_decoder
        assert cont_pos_tradeoff % nhead == 0, (
            "cont_pos_tradeoff should be divisible by nhead"
        )

        # Decoder Self-Attention
        if not rm_self_attn_decoder:
            self.self_attention = DecoderSelfAttention(
                d_model, nhead, dropout=dropout, droppath=droppath
            )

        # Decoder Cross-Attention
        # query related mappers
        self.ca_qcontent_proj = nn.Linear(d_model, d_model + cont_pos_tradeoff)
        self.ca_qpos_sine_proj = nn.Linear(d_model, d_model - cont_pos_tradeoff)

        # key related mappers
        self.ca_kcontent_proj = nn.Linear(d_model, d_model + cont_pos_tradeoff)
        self.ca_kpos_proj = nn.Linear(d_model, d_model - cont_pos_tradeoff)

        # value mapper
        self.ca_value_proj = nn.Linear(d_model, d_model)

        # cross attention module
        self.cross_attn = DABMultiheadAttention(
            d_model * 2, nhead, dropout_prob=dropout, vdim=d_model
        )

        # init params
        self._init_parameters()

        # Feedforward model
        self.ffn = FeedForwardNetwork(d_model, expansion_ratio, dropout)

        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout2 = DropPath(droppath)
        self.dropout3 = DropPath(droppath)

    def _init_parameters(self) -> None:
        for param in self.parameters():
            if param.dim() > 1:
                nn.init.xavier_uniform_(param)

    def forward(  # noqa: WPS211
        self,
        tgt: Tensor,
        src: Tensor,
        query_sine_embed: Tensor,
        query_pos: Tensor,
        src_pos: Tensor,
        tgt_mask: Optional[Tensor] = None,
        src_mask: Optional[Tensor] = None,
        tgt_key_padding_mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
    ):
        """Forward pass of the TransformerDecoderLayer.

        Args:
            tgt (Tensor): Decoder content embedding from previous layer. Shape: [#Queries, batch_size, dim]
            src (Tensor): audio content embedding from the encoder. Shape: [Lv, batch_size, dim]
            query_sine_embed (Tensor): The positional embedding for the query (tgt). Shape: [#Queries, batch_size, dim]
            query_pos (Tensor): The positional embedding for the query (Used in SelfAttnetion block).
            src_pos (Tensor): The positional embedding for the audio.
            tgt_mask (Optional[Tensor]): The mask for the target tensor.
            src_mask (Optional[Tensor]): The mask for the audio tensor.
            tgt_key_padding_mask (Optional[Tensor]): The key padding mask for the target tensor.
            src_key_padding_mask (Optional[Tensor]): The key padding mask for the audio tensor.

        Returns:
            Tensor: The output tensor after applying the decoder layer.
        """
        # ========== Begin of Self-Attention =============
        if not self.rm_self_attn_decoder:
            tgt = self.self_attention(tgt, tgt_mask, tgt_key_padding_mask, query_pos)

        # ========== Begin of Cross-Attention =============
        # query related. shape: num_queries x batch_size x 256
        query = self.ca_qcontent_proj(tgt)
        query_sine_embed = self.ca_qpos_sine_proj(query_sine_embed)

        # audio related features
        key = self.ca_kcontent_proj(src)
        key_pos = self.ca_kpos_proj(src_pos)
        value = self.ca_value_proj(src)

        num_queries, batch_size, content_dim = query.shape
        _, _, pos_dim = query_sine_embed.shape
        length, _, _ = key.shape

        # concat query content and positional info
        query = query.view(
            num_queries, batch_size, self.nhead, content_dim // self.nhead
        )
        query_sine_embed = query_sine_embed.view(
            num_queries, batch_size, self.nhead, pos_dim // self.nhead
        )
        query = torch.cat([query, query_sine_embed], dim=3)
        query = query.view(num_queries, batch_size, content_dim + pos_dim)

        # concat audio content and PE embedding
        key = key.view(length, batch_size, self.nhead, content_dim // self.nhead)
        key_pos = key_pos.view(length, batch_size, self.nhead, pos_dim // self.nhead)
        key = torch.cat([key, key_pos], dim=3)
        key = key.view(length, batch_size, content_dim + pos_dim)

        tgt2, _ = self.cross_attn(
            query=query,
            key=key,
            value=value,
            attn_mask=src_mask,
            key_padding_mask=src_key_padding_mask,
            dummy=False,
        )
        # ========== End of Cross-Attention =============

        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        tgt2 = self.ffn(tgt)
        tgt = tgt + self.dropout3(tgt2)
        return self.norm3(tgt)


# --- File: src.models.components.sg_detr_blocks.blocks.decoder ---
"""Transformer decoder based on DAB DETR implementation."""

from typing import List, Optional, Tuple

import torch
from torch import Tensor, nn




    gen_sineembed_for_position,
)




# pylint: disable=too-many-arguments, too-many-locals
class TransformerDecoder(nn.Module):  # noqa: WPS230
    """Transformer decoder consisting of *args.decoder_layers* layers."""

    def __init__(  # noqa: C901, WPS211
        self,
        decoder_layer: TransformerDecoderLayer,
        num_layers: int,
        return_intermediate: bool = False,
        d_model: int = 256,
        query_dim: int = 2,
        temperature: int = 10000,
        predict_quality_score: bool = True,
        init_spans_with_zeros: bool = True,
    ) -> None:
        """
        Initialize a Transformer decoder.

        Args:
            decoder_layer (TransformerDecoderLayer): an instance of the TransformerDecoderLayer() class
            num_layers (int): number of decoder layers
            return_intermediate (bool): whether to return intermediate results
            d_model (int): dimension of the model
            query_dim (int): dimension of the query vector
            temperature (int): temperature of the pos emb.
            predict_quality_score (bool): predict iou of the predicted interval
            init_spans_with_zeros (bool): whether to init last mlp layer with zeros or not
        """
        super().__init__()
        self.layers = get_clones(decoder_layer, num_layers)
        self.num_layers = num_layers
        self.return_intermediate = return_intermediate
        self.query_dim = query_dim
        self.temperature = temperature
        self.predict_quality_score = predict_quality_score
        self.init_spans_with_zeros = init_spans_with_zeros
        self.d_model = d_model

        # mlp for query scale
        self.query_scale = SlimMLP(
            input_dim=d_model, hidden_dim=d_model, output_dim=d_model, num_layers=2
        )

        if predict_quality_score:
            # input_dim = decoder output + anchor embedding
            self.quality_score_embed = SlimMLP(
                input_dim=d_model * 2, hidden_dim=d_model, output_dim=1, num_layers=3
            )

        # mlp for query PE embeddings
        self.ref_point_head = SlimMLP(
            input_dim=d_model, hidden_dim=d_model, output_dim=d_model, num_layers=2
        )

        # modulation mlp
        self.ref_anchor_head = SlimMLP(
            input_dim=d_model, hidden_dim=d_model, output_dim=1, num_layers=2
        )

        # anchor regression
        self.span_embed = SlimMLP(
            input_dim=d_model, hidden_dim=d_model, output_dim=2, num_layers=3
        )
        self._init_parameters()

        # output norm
        self.norm = nn.LayerNorm(d_model)

    def _init_parameters(self) -> None:
        """Init parameters."""
        # init last reg layer with zeros
        for module in self.span_embed.linear_mapper.modules():
            if (
                isinstance(module, nn.Linear)
                and module.out_features == 2
                and self.init_spans_with_zeros
            ):
                nn.init.constant_(module.weight.data, 0)  # noqa: WPS219
                nn.init.constant_(module.bias.data, 0)  # noqa: WPS219

    def apply_cond_spatial_query(
        self, query_sine_embed: Tensor, output: Tensor
    ) -> Tensor:
        """Rescale the postional embs leverage conditional spatial query.

        Based on DAB-Implementation.

        Args:
            query_sine_embed (Tensor): PE embedding generated from query embs. Shape: [#Queries, batch_size, dim]
            output (Tensor): Content vector used as query. Shape: [#Queries, batch_size, dim]

        Returns:
            Tensor: Rescaled PE Embs.
        """
        pos_transformation = self.query_scale(output)
        # apply transformation
        return query_sine_embed * pos_transformation

    def update_reference_points(
        self, output: Tensor, reference_points: Tensor
    ) -> Tensor:
        """Update reference points based on DAB implementation.

        Args:
            output (Tensor): content vector from CA block
            reference_points (Tensor): anchor points

        Returns:
            Tensor: updated reference points
        """
        box_offsets = self.span_embed(output)
        new_boxes = box_offsets[..., : self.query_dim] + inverse_sigmoid(
            reference_points
        )  # noqa: WPS221
        return new_boxes.sigmoid()

    def forward(  # noqa: WPS210, C901
        self,
        src: Tensor,
        src_key_padding_mask: Tensor,
        src_pos: Tensor,
        content: Tensor,
        refpoints_unsigmoid: Tensor,
        content_key_padding_mask: Optional[Tensor] = None,
        content_mask: Optional[Tensor] = None,
        src_mask: Optional[Tensor] = None,
    ) -> Tuple[Tensor, Tensor, Optional[Tensor]]:
        """Forward pass of the Transformer decoder.

        Args:
            src (Tensor): source video features. Shape: [L_video, batch_size, dim]
            src_key_padding_mask (Tensor): mask for source video features. Shape: [batch_size, L_video]
            src_pos (Tensor): position embeddings for source video features. Shape: [L_video, batch_size, dim]
            content (Tensor): Zero inited embedding represented decoder input. Shape: [#Queries, batch_size, dim]
            refpoints_unsigmoid (Tensor): reference points or anchor points. Shape: [#Queries, batch_size, 2]
            content_key_padding_mask (Optional[Tensor]): mask for content
            content_mask (Optional[Tensor]): mask for content
            src_mask (Optional[Tensor]): mask for source video features

        Returns:
            Tuple[Tensor, Tensor, Optional[Tensor]]: Decoder outputs from each levele and spans, labels
            and predicted iou score(optional).
        """
        output = content

        intermediate = []
        reference_points = refpoints_unsigmoid.sigmoid()
        # when predicting the span, the previous anchor is used.
        # Otherwise it will be that the offsets will be predicted twice(one time for anchor, one time for span), For
        # this reason, anchors from 0 to the number of decoder layers -1 are returned
        ref_points = [reference_points]
        quality_scores: List[Tensor] = []

        for layer_id, layer in enumerate(self.layers):
            # get sine embedding for the query vector
            query_sine_embed = gen_sineembed_for_position(
                reference_points, self.d_model, temperature=self.temperature
            )

            # construct PE emb for self attention layer
            query_pos = self.ref_point_head(query_sine_embed)

            query_sine_embed = self.apply_cond_spatial_query(query_sine_embed, output)

            # modulated HW attentions
            reft_cond = self.ref_anchor_head(output).sigmoid().squeeze(2)  # nq, bs, 1
            obj_width = reference_points[..., 1]
            modulation_value = (reft_cond / obj_width).unsqueeze(-1)  # noqa: WPS221
            query_sine_embed = query_sine_embed * modulation_value  # noqa: WPS350

            output = layer(
                tgt=output,
                src=src,
                query_sine_embed=query_sine_embed,
                query_pos=query_pos,
                src_pos=src_pos,
                tgt_mask=content_mask,
                src_mask=src_mask,
                tgt_key_padding_mask=content_key_padding_mask,
                src_key_padding_mask=src_key_padding_mask,
            )

            # update anchor
            new_reference_points = self.update_reference_points(
                output, reference_points
            )

            if self.predict_quality_score:
                #  iou score is predicted by prediction and therefore the gradient should not flow by prediction
                reference_points_embed = gen_sineembed_for_position(
                    new_reference_points,
                    self.d_model,
                    temperature=self.temperature,
                ).detach()
                # the decoder outputs and the predicted span are used for prediction iou score
                score_data = torch.concat([output, reference_points_embed], dim=-1)
                quality_score = self.quality_score_embed(score_data)

            # Note: anchors are always used with detach (except for the first trainable anchor), since they must be
            # constants when predicting offset
            reference_points = new_reference_points.detach()

            # do not return anchors from the last layer
            if layer_id != self.num_layers - 1:
                ref_points.append(new_reference_points)

            if self.return_intermediate:
                intermediate.append(self.norm(output))
                if self.predict_quality_score:
                    quality_scores.append(quality_score)

        output = self.norm(output)
        if self.return_intermediate:
            intermediate.pop()
            intermediate.append(output)
            stacked_decoder_outputs = torch.stack(intermediate).transpose(1, 2)
            stacked_reference_points = torch.stack(ref_points).transpose(1, 2)
            if self.predict_quality_score:
                stacked_quality_scores = torch.stack(quality_scores).transpose(1, 2)
            else:
                stacked_quality_scores = None
            return (
                stacked_decoder_outputs,
                stacked_reference_points,
                stacked_quality_scores,
            )
        if self.predict_quality_score:
            return (
                output.unsqueeze(0),
                new_reference_points.unsqueeze(0),
                quality_score.unsqueeze(0),
            )
        return output.unsqueeze(0), new_reference_points.unsqueeze(0), None


# --- File: src.models.components.sg_detr_blocks.blocks.encoder ---
"""Module for Transformer Encoders."""

import math
from typing import Any, Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from torch.nn import functional as func


    T2ATransformerEncoderLayer,
    TransformerEncoderLayer,
)

    GlobalMaxPooling,
    GlobalMeanPooling,
    GRUFeatureExtractor,
    LearnedAggregation,
    LearnedAggregationLayer,
)

    MomentEncoderOutput,
    SentenceEncoderOutput,
)



TEMP: float = 3.0
A_PARAM: float = 0.0  # noqa: WPS358
B_PARAM: float = 20.0


class TransformerEncoder(nn.Module):
    """
    Transformer Encoder class that stacks multiple encoder layers.

    Attributes:
        num_layers (int): Number of encoder layers.
        layers (nn.ModuleList): List of duplicated encoder layers.
        return_intermediate (bool): Whether to return intermediate outputs from each layer.
    """

    def __init__(
        self,
        encoder_layer: TransformerEncoderLayer,
        num_layers: int,
        return_intermediate: bool = False,
    ) -> None:
        """Initialize TransformerEncoder.

        Args:
            encoder_layer (TransformerEncoderLayer): An instance of the TransformerEncoderLayer to be duplicated.
            num_layers (int): Number of layers to be stacked.
            return_intermediate (bool): If set to True, the encoder will return all intermediate representations.
        """
        super().__init__()
        self.num_layers = num_layers
        self.layers = get_clones(encoder_layer, num_layers)
        self.return_intermediate = return_intermediate

    def forward(
        self,
        src: Tensor,
        src_pos: Tensor,
        mask: Optional[Tensor] = None,
        src_key_padding_mask: Optional[Tensor] = None,
    ) -> Union[Tensor, List[Tensor]]:
        """
        Pass the input through the encoder layers in turn.

        Args:
            src (Tensor): The sequence to the encoder (required).
            src_pos (Tensor): The position of the sequence (required).
            mask (Optional[Tensor]): The mask for the src sequence (optional).
            src_key_padding_mask (Optional[Tensor]): The mask for the src keys per batch (optional).

        Returns:
            Union[Tensor, List[Tensor]]: Output of the last layer or intermediate outputs from all layers.
        """
        output = src

        intermediate = []

        for layer in self.layers:
            output = layer(
                output,
                src_pos,
                embedding_mask=mask,
                embedding_key_padding_mask=src_key_padding_mask,
            )

            if self.return_intermediate:
                intermediate.append(output)

        if self.return_intermediate:
            return torch.stack(intermediate)

        return output


class TransformerCATEEncoder(nn.Module):
    """
    A Transformer CATE Encoder module.

    This module applies a series of transformer encoder layers to the input data.

    Attributes:
        layers (nn.ModuleList): A list of identical transformer encoder layers.
        num_layers (int): The number of encoder layers.
        norm (nn.LayerNorm): A layer normalization module.
        return_intermediate (bool): If set to True, intermediate outputs of each layer will be returned.
    """

    def __init__(
        self,
        encoder_layer: T2ATransformerEncoderLayer,
        num_layers: int,
        return_intermediate: bool = False,
    ):
        """Initialize a TransformerCATEEncoder.

        Args:
            encoder_layer (T2ATransformerEncoderLayer): An instance of the transformer encoder layer.
            num_layers (int): The number of layers in the encoder.
            return_intermediate (bool): Whether to return intermediate outputs. Defaults to False.
        """
        super().__init__()
        self.layers = get_clones(encoder_layer, num_layers)
        self.num_layers = num_layers
        self.norm = nn.LayerNorm(encoder_layer.d_model)
        self.return_intermediate = return_intermediate

    def forward(
        self,
        src: Tensor,
        pos_embed: Tensor,
        audio_length: int,
        src_key_padding_mask: Optional[Tensor] = None,
        dummy: bool = True,
        saliency_scores: Optional[Tensor] = None,
    ):
        """
        Pass the input (and mask) through each layer in turn.

        Args:
            src (Tensor): The sequence to the encoder (required). Shape: [Lv + Ld + Lt, Bs, dim]
            pos_embed (Tensor): The position of the sequence (required). Shape: [Lv + Ld + Lt, Bs, dim]
            audio_length (int): The length of the audio (#clips).
            src_key_padding_mask (Optional[Tensor]): The mask for the src keys per batch.  Shape: [Bs, Lv + Ld + Lt]
            dummy (bool): Whether to use dummy tokens.

        Returns:
            Tensor: The encoded output.
            Tensor: The attention weights.
        """
        output = src
        intermediate = []
        attn_weights = None
        for layer in self.layers:
            output, attn_weight = layer(
                output,
                pos_embed,
                audio_length,
                emb_key_padding_mask=src_key_padding_mask,
                dummy=dummy,
                saliency_scores=saliency_scores,
            )
            output = self.norm(output)

            attn_weights = (
                attn_weight if attn_weights is None else attn_weights + attn_weight
            )

            if self.return_intermediate:
                intermediate.append(output)

        if attn_weights is not None:
            attn_weights /= self.num_layers  # type: ignore

        if self.norm is not None:
            output = self.norm(output)

        if self.return_intermediate:
            return torch.stack(intermediate), attn_weights

        return output, attn_weights


class DummyEncoder(nn.Module):
    """Encoder for dummy tokens."""

    def __init__(
        self,
        d_model: int,
        num_dummies: int,
        num_dummy_layers: int,
        dropout: float,
        droppath: float,
    ) -> None:
        """Initialize DummyEncoder.

        Args:
            d_model (int): dimension of the model
            num_dummies (int): number of dummy tokens
            num_dummy_layers (int): number of dummy layers
            dropout (float): dropout rate
            droppath (float): droppath rate
        """
        super().__init__()
        self.d_model = d_model
        self.num_dummies = num_dummies

        # define dummy tokens
        self.dummy_rep_token = torch.nn.Parameter(torch.randn(num_dummies, d_model))
        self.dummy_rep_pos = torch.nn.Parameter(torch.randn(num_dummies, d_model))

        # define self attention to set query-dummy tokens relations
        input_txt_sa_proj = TransformerEncoderLayer(
            d_model, dropout=dropout, droppath=droppath
        )
        self.txtproj_encoder = TransformerEncoder(input_txt_sa_proj, num_dummy_layers)

    def prepare_dummy_tokens(self, src_txt: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        """Prepare aux entities.

        Args:
            src_txt (Tensor): text features, (batch_size, L_txt, D_txt)

        Returns:
            Tuple[Tensor, Tensor, Tensor]:
                - token: token repeated for batch_size times
                - mask: mask for token
                - position: position embedding for token repeated for batch_size times
        """
        # dummy or sentence tokens
        dummy_tokens = self.dummy_rep_token[None].repeat(src_txt.shape[0], 1, 1)

        # dummy or sentence mask
        dummy_mask = torch.tensor([True for _ in range(self.num_dummies)])
        dummy_mask = dummy_mask.to(src_txt.device)
        dummy_mask = dummy_mask.repeat(src_txt.shape[0], 1)

        # dummy positions
        dummy_position = self.dummy_rep_pos[None].repeat(src_txt.shape[0], 1, 1)
        return dummy_tokens, dummy_mask, dummy_position

    def forward(
        self, src_txt: Tensor, src_txt_mask: Tensor, pos_txt: Tensor
    ) -> Tuple[Tensor, Tensor, Tensor]:
        """Forward pass.

        Create dummy tokens with query excluded content and concat them with text features.

        Args:
            src_txt (Tensor): intital text features, (batch_size, L_txt, D_txt)
            src_txt_mask (Tensor): mask for text features, (batch_size, L_txt)
            pos_txt (Tensor): position embedding for text features, (batch_size, L_txt, D_txt)

        Returns:
            Tuple[Tensor, Tensor, Tensor]: text, mask and position tensors concated with dummy tokens
        """
        # get dummy tokens
        dummy_tokens, dummy_mask, dummy_pos = self.prepare_dummy_tokens(src_txt)

        dummy_src_txt = torch.cat([dummy_tokens, src_txt], dim=1)
        dummy_src_txt_mask = torch.cat([dummy_mask, src_txt_mask], dim=1)
        dummy_pos_txt = torch.cat([dummy_pos, pos_txt], dim=1)

        # force dummy tokens to contain query excluding content
        dummy_src_txt = dummy_src_txt.permute(1, 0, 2)  # (L, batch_size, d)
        dummy_pos_txt = dummy_pos_txt.permute(1, 0, 2)  # (L, batch_size, d)
        memory = self.txtproj_encoder(
            dummy_src_txt,
            dummy_pos_txt,
            src_key_padding_mask=~(
                dummy_src_txt_mask.bool()
            ),  # Should be True for padding tokens
        )  # (L, batch_size, d_model)

        # concat inhanced dummy tokens with text embeddings
        dummy_token = memory[: self.num_dummies].permute(1, 0, 2)  # (batch_size, L, d)
        dummy_src_txt = torch.cat([dummy_token, src_txt], dim=1)
        dummy_pos_txt = dummy_pos_txt.permute(1, 0, 2)  # (batch_size, L, d)

        return dummy_src_txt, dummy_src_txt_mask, dummy_pos_txt


class SentenceEncoder(nn.Module):
    """Encoder for sentence tokens."""

    def __init__(
        self, d_model: int, num_sentence_layers: int, dropout: float, droppath: float
    ) -> None:
        """Initialize SentenceEncoder.

        Args:
            d_model (int): dimension of the model
            num_sentence_layers (int): number of sentence layers
            dropout (float): dropout rate
            droppath (float): droppath rate
        """
        super().__init__()
        self.d_model = d_model

        self.sent_rep_token = torch.nn.Parameter(torch.randn(1, d_model))
        self.sent_rep_pos = torch.nn.Parameter(torch.randn(1, d_model))

        # define sentence encoder to incorporate sentence representation
        scls_encoder_layer = TransformerEncoderLayer(
            d_model, dropout=dropout, droppath=droppath
        )
        self.sent_cls_encoder = TransformerEncoder(
            scls_encoder_layer, num_sentence_layers
        )

    def prepare_sentence_tokens(self, src_txt: Tensor) -> Tuple[Tensor, Tensor, Tensor]:
        """Prepare sentence tokens.

        Args:
            src_txt (Tensor): text features, (batch_size, L_txt, D_txt).

        Returns:
            Tuple[Tensor, Tensor, Tensor]:
                - token: token repeated for batch_size times
                - mask: mask for token
                - position: position embedding for token repeated for batch_size times
        """
        # dummy or sentence tokens
        sent_rep_token = self.sent_rep_token[None].repeat(src_txt.shape[0], 1, 1)

        # dummy or sentence mask
        mask = torch.tensor([[True]])
        mask = mask.to(src_txt.device)
        mask = mask.repeat(src_txt.shape[0], 1)

        # dummy positions
        position = self.sent_rep_pos[None].repeat(src_txt.shape[0], 1, 1)
        return sent_rep_token, mask, position

    def run_encoder(
        self, src: Tensor, mask: Tensor, pos: Tensor
    ) -> Tuple[Tensor, Tensor]:
        """Run self attention based encoder.

        Args:
            src (Tensor): input features, (batch_size, L_txt, D_txt)
            mask (Tensor): mask for input features, (batch_size, L_txt)
            pos (Tensor): position embedding for input features, (batch_size, L_txt, D_txt)

        Returns:
            Tuple[Tensor, Tensor]: token and memory tensors
        """
        src = src.permute(1, 0, 2)  # (L, batch_size, d)
        pos = pos.permute(1, 0, 2)  # (L, batch_size, d)
        memory = self.sent_cls_encoder(
            src, pos, src_key_padding_mask=~mask
        )  # True for padding tokens
        return memory[0], memory[1:]

    # pylint: disable=too-many-locals,too-many-arguments
    def forward(
        self,
        src_txt: Tensor,
        src_txt_mask: Tensor,
        pos_txt: Tensor,
        dummy_token: Optional[Tensor],
        dummy_mask: Optional[Tensor],
        dummy_pos: Optional[Tensor],
    ) -> SentenceEncoderOutput:
        """Forward pass.

        Args:
            src_txt (Tensor): intital text features, (batch_size, L_txt, D_txt)
            src_txt_mask (Tensor): mask for text features, (batch_size, L_txt)
            pos_txt (Tensor): position embedding for text features, (batch_size, L_txt, D_txt)
            dummy_token (Optional[Tensor]): dummy tokens, (batch_size, L_txt, D_txt)
            dummy_pos (Optional[Tensor]): dummy positions, (batch_size, L_txt, D_txt)
            dummy_mask (Optional[Tensor]): dummy mask, (batch_size, L_txt)

        Returns:
            SentenceEncoderOutput:
                - sent_txt_token: sentence tokens inhanced with query representation
                - sent_dummy_token: sentence tokens inhanced with dummy representation
                - sent_words_memory: memory of words inhanced with query representation
                - sent_dummy_memory: memory of words inhanced with dummy representation
        """
        sent_token, sent_mask, sent_pos = self.prepare_sentence_tokens(src_txt)

        # concat sentence token and text embeddings
        sent_txt = torch.cat([sent_token, src_txt], dim=1)
        sent_txt_mask = torch.cat([sent_mask, src_txt_mask.bool()], dim=1)
        sent_txt_pos = torch.cat([sent_pos, pos_txt], dim=1)

        # inhace sentence tokens with query representation
        sent_txt_token, sent_words_memory = self.run_encoder(
            sent_txt, sent_txt_mask, sent_txt_pos
        )

        if dummy_token is not None:
            assert dummy_mask is not None
            assert dummy_pos is not None
            # concat sentence token and dummy embeddings
            sent_dummy = torch.cat([sent_token, dummy_token], dim=1)
            sent_dummy_mask = torch.cat([sent_mask, dummy_mask.bool()], dim=1)
            sent_dummy_pos = torch.cat([sent_pos, dummy_pos], dim=1)

            # inhace sentence tokens with dummy representation (query excluding content)
            sent_dummy_token, sent_dummy_memory = self.run_encoder(
                sent_dummy, sent_dummy_mask, sent_dummy_pos
            )
        else:
            sent_dummy_token, sent_dummy_memory = None, None

        return SentenceEncoderOutput(
            sent_txt_token=sent_txt_token,
            sent_dummy_token=sent_dummy_token,
            sent_words_memory=sent_words_memory,
            sent_dummy_memory=sent_dummy_memory,
        )


class MomentEncoder(nn.Module):
    """Encoder for moment tokens."""

    def __init__(
        self, d_model: int, num_mcls_layers: int, dropout: float, droppath: float
    ) -> None:
        """Initialize MomentEncoder.

        Args:
            d_model (int): dimension of the model
            num_mcls_layers (int): number of moment layers
            dropout (float): dropout rate
            droppath (float): droppath rate
        """
        super().__init__()
        # define moment tokens
        self.moment_rep_token = torch.nn.Parameter(torch.randn(1, d_model))
        self.moment_rep_pos = torch.nn.Parameter(torch.randn(1, d_model))

        # define moment encoder to incorporate moment representation
        mcls_encoder_layer = TransformerEncoderLayer(
            d_model, dropout=dropout, droppath=droppath
        )
        self.mcls_encoder = TransformerEncoder(mcls_encoder_layer, num_mcls_layers)

    def prepare_moment_tokens(
        self,
        src_aud: Tensor,
        pos_aud: Tensor,
        src_aud_mask: Tensor,
        targets: dict,
        is_positive: bool,
    ) -> Tuple[Tensor, Tensor, Tensor, Tensor]:
        """Prepare moment tokens.

        We will use them in SA block. The idea is to inhance the moment and non moment tokens. That is why we create
        masks based on clip relevance or irrelevance.

        Args:
            src_aud (Tensor): audio features, (batch_size, L_aud, D_aud)
            pos_aud (Tensor): positional embedding for audio
            src_aud_mask (Tensor): audio mask, (batch_size, L_aud)
            targets (dict): targets dict
            is_positive (bool): True if we want to inhance moment tokens, False for non moment tokens

        Returns:
            Tuple[Tensor, Tensor, Tensor]: clip_mask, moment+vid emb, moment+vid position, moment+vid mask
        """
        # add moment token to audio mask
        mom_mask = torch.tensor([[True]]).to(src_aud_mask.device)
        mom_mask = mom_mask.repeat(src_aud_mask.shape[0], 1)
        mom_aud_mask = torch.cat([mom_mask, src_aud_mask.bool()], dim=1)

        # add moment token to clips mask
        clips_mask = torch.clamp(targets["relevant_clips"], 0, 1).bool()
        clips_mask = clips_mask if is_positive else ~clips_mask
        clips_mask = clips_mask * src_aud_mask.bool()
        mom_clips_mask = torch.cat([mom_mask, clips_mask], dim=1)

        # ignore relevant or irrelevant clips based on 'is_positive' flag
        mom_aud_mask = mom_aud_mask * mom_clips_mask

        # Concat moment and vid embeddings
        moment_token = self.moment_rep_token[None].repeat(src_aud.shape[0], 1, 1)
        mom_aud_emb = torch.cat([moment_token, src_aud], dim=1)

        # concat moment and vid positions
        moment_token_pos = self.moment_rep_pos[None].repeat(pos_aud.shape[0], 1, 1)
        mom_vid_pos = torch.cat([moment_token_pos, pos_aud], dim=1)
        return clips_mask, mom_aud_emb, mom_vid_pos, mom_aud_mask

    # pylint: disable=too-many-locals
    def forward(
        self,
        src_aud: Tensor,
        src_aud_mask: Tensor,
        pos_aud: Tensor,
        targets: Dict[str, Any],
    ) -> MomentEncoderOutput:
        """Forward pass of the moment encoder.

        Args:
            src_aud (Tensor): audio features, (batch_size, L_aud, D_aud)
            src_aud_mask (Tensor): audio mask, (batch_size, L_aud)
            pos_aud (Tensor): positional embedding for audio
            targets (Dict[str, Any]): targets dict contains 'relevant_clips' key

        Returns:
            MomentEncoderOutput: output of the encoder schema with the following attributes:
                - rel_clips_mask (Tensor): Mask for relevant clips
                - irrel_clips_mask (Tensor): Mask for irrelevant clips
                - moment_token (Tensor): Moment token inhanced with relevant clips representation
                - moment_memory (Tensor): Relevant clips representation (output from SA encoder)
                - non_moment_token (Tensor): Non-moment token inhanced with irrelevant clips representation
                - non_moment_memory (Tensor): Irrelevant clips representation (output from SA encoder)
        """
        rel_clips_mask, mom_aud_emb, mom_vid_pos, mom_aud_mask = (
            self.prepare_moment_tokens(
                src_aud,
                pos_aud,
                src_aud_mask,
                targets,
                is_positive=True,
            )
        )
        irrel_clips_mask, non_mom_aud_emb, non_mom_vid_pos, non_mom_aud_mask = (
            self.prepare_moment_tokens(
                src_aud,
                pos_aud,
                src_aud_mask,
                targets,
                is_positive=False,
            )
        )

        # moment token
        mom_aud_emb = mom_aud_emb.permute(1, 0, 2)  # (L, batch_size, dim)
        mom_vid_pos = mom_vid_pos.permute(1, 0, 2)  # (L, batch_size, dim)
        mmemory = self.mcls_encoder(
            mom_aud_emb, mom_vid_pos, src_key_padding_mask=~mom_aud_mask
        )  # True for padding
        moment_token, moment_memory = mmemory[0], mmemory[1:]

        non_mom_aud_emb = non_mom_aud_emb.permute(1, 0, 2)  # (L, batch_size, dim)
        non_mom_vid_pos = non_mom_vid_pos.permute(1, 0, 2)  # (L, batch_size, dim)
        nmmemory = self.mcls_encoder(
            non_mom_aud_emb, non_mom_vid_pos, src_key_padding_mask=~non_mom_aud_mask
        )
        non_moment_token, non_moment_memory = nmmemory[0], nmmemory[1:]

        return MomentEncoderOutput(
            relevant_clips_mask=rel_clips_mask,
            irrelevant_clips_mask=irrel_clips_mask,
            moment_token=moment_token,
            moment_memory=moment_memory,
            non_moment_token=non_moment_token,
            non_moment_memory=non_moment_memory,
        )


class Text2AudioEncoder(nn.Module):
    """Encoder to incorporate text representation into audio."""

    def __init__(
        self,
        d_model: int,
        num_dummies: int,
        num_t2v_layers: int,
        dropout: float,
        droppath: float,
        remove_dummy: bool = True,
        use_cross_attn_wo_dummy: bool = False,
        weight_attn_with_saliency: bool = False,
    ) -> None:
        """Initialize Text2AudioEncoder.

        Args:
            d_model (int): dimension of the model
            num_dummies (int): number of dummy tokens
            num_t2v_layers (int): number of encoder layers
            dropout (float): dropout rate
            droppath (float): droppath rate
            remove_dummy (bool): wther to remove dummy or not.
        """
        super().__init__()
        self.d_model = d_model
        self.use_cross_attn_wo_dummy = use_cross_attn_wo_dummy
        self.num_dummies = 0 if use_cross_attn_wo_dummy else num_dummies

        # define text to vision encoder to incorporate text representation into audio
        t2v_encoder_layer = T2ATransformerEncoderLayer(
            d_model,
            num_dummies=self.num_dummies,
            dropout=dropout,
            droppath=droppath,
            use_cross_attn_wo_dummy=use_cross_attn_wo_dummy,
            weight_attn_with_saliency=weight_attn_with_saliency,
        )
        self.t2v_encoder = TransformerCATEEncoder(t2v_encoder_layer, num_t2v_layers)
        self.remove_dummy = remove_dummy

    def forward(
        self,
        src: Tensor,
        mask: Tensor,
        pos: Tensor,
        batch_audio_len: int,
        saliency_scores: Tensor,
    ) -> Tuple[Tensor, Tensor, Tensor, Optional[Tensor]]:
        """Forward pass of the Text2AudioEncoder.

        Args:
            src (Tensor): audio features, (batch_size, L_aud + L_dummy + L_txt, D_aud)
            mask (Tensor): audio mask, (batch_size, L_aud + L_dummy + L_txt)
            pos (Tensor): position embedding for audio, (batch_size, L_aud + L_dummy + L_txt, D_aud)
            batch_audio_len (int): batch audio length

        Returns:
            Tuple[Tensor, Tensor, Tensor, Optional[Tensor]]:
                - audio tensor concated with saliency token
                - mask tensor concated with saliency token
                - position tensors concated with saliency token
                - attention weights from audio to text transformer
        """
        src = src.permute(1, 0, 2)  # (Lv + Ld + Lt, batch_size, dim)
        pos = pos.permute(1, 0, 2)  # (Lv + Ld + Lt, batch_size, dim)

        t2v_src, attn_weights = self.t2v_encoder(
            src,
            pos,
            audio_length=batch_audio_len,
            src_key_padding_mask=~mask,  # Should be True for padding tokens
            dummy=self.remove_dummy,
            saliency_scores=saliency_scores,
        )  # (L, batch_size, dim)
        t2v_src = t2v_src.permute(1, 0, 2)
        pos = pos.permute(1, 0, 2)
        return t2v_src, mask, pos, attn_weights  # type: ignore


class LocalSaliencyHead(nn.Module):
    """
    A neural network module to compute local saliency scores for audio and text embeddings.

    Attributes:
        logit_mode (str): The mode for calculating logits. Must be one of {"linear", "exp", "exp_b"}.
        sentence_pooling (LearnedAggregation): An instance of LearnedAggregation for sentence pooling.
        temp (nn.Parameter): A temperature parameter used in "exp" and "exp_b" logit modes.
        a (nn.Parameter): A parameter used in "exp_b" and "linear" logit modes.
        b (nn.Parameter): A parameter used in "linear" logit mode.
    """

    allowed_logit_modes = {"linear", "exp", "exp_b"}

    def __init__(
        self,
        model_dim: int,
        use_projections: bool = True,
        logit_mode: str = "linear",
        use_gamma: bool = True,
        num_aggregation_layers: int = 1,
    ) -> None:
        """
        Initialize the LocalSaliencyHead module.

        Args:
            model_dim (int): Dimension of the model.
            use_projections (bool): Whether to use projections in LearnedAggregation. Defaults to True.
            logit_mode (str): Mode for calculating logits. Must be one of {"linear", "exp", "exp_b"}.
        """
        super().__init__()
        self._validate_logit_mode(logit_mode)
        self.logit_mode = logit_mode
        sentence_pooling_layer = LearnedAggregationLayer(
            dim=model_dim, use_projections=use_projections, use_gamma=use_gamma
        )
        self.sentence_pooling = LearnedAggregation(
            model_dim, sentence_pooling_layer, num_aggregation_layers
        )
        self._initialize_parameters()

    def _validate_logit_mode(self, logit_mode: str) -> None:
        """
        Validate the logit_mode.

        Args:
            logit_mode (str): The mode for calculating logits.

        Raises:
            ValueError: If logit_mode is not one of {"linear", "exp", "exp_b"}.
        """
        if logit_mode not in self.allowed_logit_modes:
            raise ValueError(f"logit_mode must be one of {self.allowed_logit_modes}")

    def _initialize_parameters(self) -> None:
        """Initialize the parameters based on the logit_mode."""
        self.temp: Optional[nn.Parameter] = None
        self.a_param: Optional[nn.Parameter] = None
        self.b_param: Optional[nn.Parameter] = None
        if self.logit_mode == "exp":
            self.temp = nn.Parameter(torch.tensor(TEMP))
        elif self.logit_mode == "exp_b":
            self.temp = nn.Parameter(torch.tensor(TEMP))
            self.a_param = nn.Parameter(torch.tensor(A_PARAM))
        else:
            self.a_param = nn.Parameter(torch.tensor(A_PARAM))
            self.b_param = nn.Parameter(torch.tensor(B_PARAM))

    def saliency_scores(self, aud_emb: Tensor, txt_emb: Tensor) -> Tensor:
        """
        Compute saliency scores for audio and text embeddings.

        Args:
            aud_emb (Tensor): The audio embeddings.
            txt_emb (Tensor): The text embeddings.

        Returns:
            Tensor: The computed saliency scores.
        """
        txt_emb = self._normalize_embedding(txt_emb)
        aud_emb = self._normalize_embedding(aud_emb)
        scores = torch.sum(aud_emb * txt_emb, dim=-1)
        return self._apply_logit_mode(scores)

    def _normalize_embedding(self, emb: Tensor) -> Tensor:
        """
        Normalize the embeddings.

        Args:
            emb (Tensor): The embeddings to normalize.

        Returns:
            Tensor: The normalized embeddings.
        """
        return nn.functional.normalize(emb, p=2, dim=-1)

    def _apply_logit_mode(self, scores: Tensor) -> Tensor:
        """
        Apply the logit mode to the scores.

        Args:
            scores (Tensor): The computed scores.

        Returns:
            Tensor: The scores after applying the logit mode.
        """
        if self.logit_mode == "exp":
            return scores * torch.exp(self.temp)  # type: ignore
        if self.logit_mode == "exp_b":
            return scores * torch.exp(self.temp) + self.a_param  # type: ignore
        return scores * self.b_param + self.a_param

    def forward(
        self, src_aud: Tensor, src_txt: Tensor, src_txt_mask: Tensor
    ) -> Tuple[Tensor, Tensor]:
        """
        Forward pass of the LocalSaliencyHead module.

        Args:
            src_aud (Tensor): Source audio embeddings.
            src_txt (Tensor): Source text embeddings.
            src_txt_mask (Tensor): Source text mask.

        Returns:
            Tuple[Tensor, Tensor]: The saliency scores and the pooled sentence embeddings.
        """
        src_sent = self.sentence_pooling(src_txt, key_padding_mask=src_txt_mask)
        saliency_scores = self.saliency_scores(src_aud, src_sent)
        return saliency_scores, src_sent


class SaliencyAmplifier(nn.Module):
    """
    A neural network module to amplify features based on saliency scores.

    Attributes:
        alpha (nn.Parameter): A learnable parameter to scale the new features.
        mode (str): The mode for calculating saliency. Must be one of {"sigmoid", "softmax"}.
        use_mha (bool): Whether to use multi-head attention (MHA).
        mha (TransformerEncoderLayer): An instance of TransformerEncoderLayer for multi-head attention.
    """

    allowed_modes = {"sigmoid", "softmax", "sin"}

    def __init__(
        self,
        d_model: int,
        mode: str = "sigmoid",
        use_mha: bool = True,
        use_norm: bool = False,
        temperature: int = 10000,
    ) -> None:
        """
        Initialize the SaliencyAmplifier module.

        Args:
            d_model (int): Dimension of the model.
            mode (str): Mode for calculating saliency. Must be one of {"sigmoid", "softmax"}. Defaults to "sigmoid".
            use_mha (bool): Whether to use multi-head attention (MHA). Defaults to True.
            use_norm (bool):  Whether to use norm or not. Defaults to False.
            temperature (int): temp for sin emb.

        Raises:
            ValueError: If mode is not one of {"sigmoid", "softmax"}.
        """
        super().__init__()
        if mode not in self.allowed_modes:
            raise ValueError(f"mode must be one of {self.allowed_modes}")

        self.mode = mode
        self.d_model = d_model
        self.use_mha = use_mha
        self.use_norm = use_norm
        self.temperature = temperature

        self.norm = nn.LayerNorm(d_model) if use_norm else None
        self.mha = TransformerEncoderLayer(d_model=d_model) if use_mha else None

    def gen_sin_emb(self, scores: Tensor) -> Tensor:
        """Generate sine embeddings for scores.

        Args:
            scores (Tensor): local scores. Shape: [seq_len, batch_size]

        Returns:
            Tensor: sine embeddings for local scores
        """
        scale = 2 * math.pi
        dim_t = torch.arange(self.d_model, dtype=torch.float32, device=scores.device)
        dim_t = self.temperature ** (2 * (dim_t // 2) / self.d_model)

        scaled_scores = scores * scale
        sin_emb = scaled_scores[:, :, None] / dim_t
        return torch.stack(  # noqa: WPS317
            (
                sin_emb[:, :, 0::2].sin(),
                sin_emb[:, :, 1::2].cos(),
            ),
            dim=3,
        ).flatten(2)

    # pylint: disable=not-callable
    def forward(
        self,
        features: Tensor,
        saliency_scores: Tensor,
        pos: Tensor,
        aud_mask: Tensor,
    ) -> Tensor:
        """
        Forward pass of the SaliencyAmplifier module.

        Args:
            features (Tensor): The input features.
            saliency_scores (Tensor): The saliency scores.
            pos (Tensor): Positional encodings.
            aud_mask (Tensor): The mask for the audio sequence. Defaults to None.

        Returns:
            Tensor: The output features after saliency amplification.
        """
        if self.mode == "sin":
            saliency = saliency_scores.transpose(0, 1).detach()
            saliency_sine_embed = self.gen_sin_emb(saliency.sigmoid())
            output = features + saliency_sine_embed
        else:
            if self.mode == "sigmoid":
                saliency = torch.sigmoid(
                    saliency_scores.transpose(0, 1)
                )  # [seq_length, batch_size]
            elif self.mode == "softmax":
                masked_saliency_scores = saliency_scores.masked_fill(
                    ~aud_mask, float("-inf")
                )
                saliency = torch.softmax(
                    masked_saliency_scores, dim=1
                )  # Check temperature
                saliency = saliency.transpose(0, 1)

            new_features = (
                features * saliency[:, :, None]
            )  # [seq_length, batch_size, model_dim]
            output = features + new_features

        if self.use_norm:
            assert self.norm is not None
            output = self.norm(output)

        if self.use_mha:
            assert self.mha is not None
            output = self.mha(output, pos)

        return output


class MR2HD(nn.Module):
    """
    A neural network module for moment retrieval to highlight detection amplification.

    Attributes:
        score_mode (str): The mode for calculating scores. Must be one of {"probs", "iou", "probs_iou"}.
        aggregation_mode (str): The mode for aggregating features. One of {"attention", "gru", "none", "mean", "max"}.
        learned_aggregation (Optional[nn.Module]): An instance of a learned aggregation module or None.
        linear (nn.Linear): A linear layer for generating logits.
        temperature (nn.Parameter): A temperature parameter for scaling the scores.
    """

    def __init__(
        self,
        model_dim: int,
        aggregation_mode: str = "attention",
        score_mode: str = "probs",
    ) -> None:
        """
        Initialize the MR2HD module.

        Args:
            model_dim (int): Dimension of the model.
            aggregation_mode (str): Mode for aggregating features. One of {"attention", "gru", "none", "mean", "max"}.
            score_mode (str): Mode for calculating scores. One of {"probs", "iou", "probs_iou"}.
        """
        super().__init__()
        # check score and aggregation modes
        assert aggregation_mode in {"attention", "gru", "none", "mean", "max"}
        assert score_mode in {"probs", "iou", "probs_iou"}
        self.score_mode = score_mode
        self.aggregation_mode = aggregation_mode
        self.temperature = nn.Parameter(torch.tensor(TEMP))

        # init aggregation class
        if aggregation_mode == "attention":
            self.learned_aggregation: Optional[nn.Module] = LearnedAggregation(
                model_dim, LearnedAggregationLayer(model_dim), 1
            )
        elif aggregation_mode == "gru":
            self.learned_aggregation = GRUFeatureExtractor(model_dim, model_dim)
        elif aggregation_mode == "none":
            self.learned_aggregation = None
        elif aggregation_mode == "mean":
            self.learned_aggregation = GlobalMeanPooling()
        else:
            self.learned_aggregation = GlobalMaxPooling()

    def extract_intervals(
        self, src_aud: Tensor, predicted_spans_idxes: Tensor
    ) -> Tuple[Tensor, Tensor]:
        """
        Extract specified intervals from a batch of audio sequences and return these intervals along with masks.

        Args:
            src_aud (Tensor): A tensor of shape (bs, seq_length, model_dim) representing the input audio sequences.
            predicted_spans_idxes (Tensor): represents [bs, num_spans, 2] tensor for the intervals to be extracted.

        Returns:
            Tuple[Tensor, Tensor]:
                - extracted_intervals (Tensor): A tensor of shape (batch_size, num_spans, seq_length, model_dim)
                    containing the extracted intervals. Each interval is padded to match the seq_length.
                - masks (Tensor): A tensor of shape (batch_size, num_spans, seq_length) containing the masks
                    corresponding to the extracted intervals, 1 indicates the interval and 0 indicates the padding.
        """
        batch_size, seq_length, model_dim = src_aud.shape
        _, num_spans, _ = predicted_spans_idxes.shape

        # Create a list to store the extracted intervals for each batch
        extracted_intervals = []
        masks = []

        for batch_idx in range(batch_size):
            batch_intervals = []
            batch_masks = []
            for span_idx in range(num_spans):
                start_idx, end_idx = predicted_spans_idxes[batch_idx, span_idx].tolist()

                # Correct intervals where start_idx equals end_idx
                if start_idx == end_idx and end_idx < seq_length:
                    end_idx += 1
                if start_idx == end_idx and end_idx == seq_length:
                    start_idx -= 1

                # Ensure indices are within valid range
                start_idx = max(0, start_idx)
                end_idx = min(seq_length, end_idx)

                # Extract the interval
                length = end_idx - start_idx
                mask = [1] * length + [0] * (seq_length - length)

                mask = torch.tensor(mask).bool().to(src_aud.device)  # type: ignore
                interval = src_aud[batch_idx, start_idx:end_idx]
                padding = torch.zeros(seq_length - length, model_dim).to(src_aud.device)
                interval = torch.concat([interval, padding])  # [seq_length, model_dim]
                batch_intervals.append(interval)
                batch_masks.append(mask)

            extracted_intervals.append(torch.stack(batch_intervals))
            masks.append(torch.stack(batch_masks))  # type: ignore
        return torch.stack(extracted_intervals), torch.stack(masks)

    def forward(
        self,
        audio_features: Tensor,
        outputs_class: Tensor,
        quality_scores: Tensor,
        outputs_coord: Tensor,
        audio_text_features: Tensor,
    ) -> Tensor:
        """
        Forward pass of the MR2HD class.

        Args:
            audio_features (Tensor): shape [bs, seq_length, model_dim], audio features before cross-attention with text.
            outputs_class (Tensor): shape [bs, num_queries, 2], class logits (0 idx is positive class).
            outputs_coord (Tensor): shape [bs, num_queries, 2], spans in normalized [0, 1] format (center and width).
            quality_scores (Tensor): shape [bs, num_queries], quality scores for each query.
            audio_text_features (Tensor): shape [seq_length, bs, model_dim], audio features after cross-attn with text.

        Returns:
            Tensor: updated saliency scores, shape [bs, seq_length]
        """
        if self.learned_aggregation is None:
            return audio_text_features.transpose(0, 1)

        batch_size, audio_length, _ = audio_features.shape
        scores = torch.sigmoid(outputs_class)[..., 0]
        outputs_coord_xx = span_cxw_to_xx(outputs_coord)  # [batch_size, num_queries, 2]
        predicted_spans_idxes = torch.round(outputs_coord_xx * audio_length).to(int)  # type: ignore
        # predicted_spans_idxes shape: [batch_size, num_queries, 2]

        extracted_intervals, masks = self.extract_intervals(
            audio_features, predicted_spans_idxes
        )
        list_of_seqs: List[Tensor] = []
        for batch_idx in range(batch_size):
            batch_intervals = extracted_intervals[batch_idx]
            batch_masks = masks[batch_idx]

            # agregate features inside interval
            if self.aggregation_mode in {"attention", "max", "mean"}:
                aggregation_span_vectors = self.learned_aggregation(
                    batch_intervals, batch_masks
                )[:, 0, :]
            else:  # only for test ancient evil
                span_vectors = [
                    self.learned_aggregation(interval[mask][None, :, :])[0]
                    for interval, mask in zip(batch_intervals, batch_masks)
                ]
                aggregation_span_vectors = torch.stack(span_vectors)

            # get interval scores
            if self.score_mode == "probs":
                item_scores = scores[batch_idx]
            elif self.score_mode == "iou":
                item_scores = quality_scores[batch_idx]
            else:
                item_scores = torch.sqrt(scores[batch_idx] * quality_scores[batch_idx])

            # agregate span features
            mult = torch.softmax(
                item_scores * torch.exp(self.temperature), dim=0
            )  # [seq_length]
            vercor_of_seq = torch.sum(
                aggregation_span_vectors * mult[:, None], dim=0
            )  # [model_dim]
            list_of_seqs.append(vercor_of_seq)

        interval_features = torch.stack(list_of_seqs).unsqueeze(
            1
        )  # [batch_size, 1, model_dim]
        cosine_similarities = func.cosine_similarity(
            audio_features, interval_features, dim=-1
        )  # pylint: disable=E1102

        audio_text_features = audio_text_features.transpose(0, 1)
        cosine_similarities = cosine_similarities[:, :, None]
        return audio_text_features + audio_text_features * cosine_similarities


class CrossAttentionWithProbs(nn.Module):
    def __init__(self, model_dim: int, d_k: int, d_v: int) -> None:
        super(CrossAttentionWithProbs, self).__init__()
        self.w_query = nn.Linear(model_dim, d_k, bias=False)
        self.w_key = nn.Linear(model_dim, d_k, bias=False)
        self.w_value = nn.Linear(model_dim, d_v, bias=False)
        self.temperature = nn.Parameter(torch.tensor(3.0))

    def forward(
        self,
        aggregation_span_vectors: Tensor,
        audio_text_features: Tensor,
        probs: Tensor,
    ) -> Tensor:
        # Conversion to Q, K, V
        query = self.w_query(
            audio_text_features
        )  # shape: [batch_size, seq_length, d_k]
        key = self.w_key(aggregation_span_vectors)  # shape: [batch_size, n_spans, d_k]
        value = self.w_value(
            aggregation_span_vectors
        )  # shape: [batch_size, n_spans, d_v]

        scores = torch.matmul(query, key.transpose(-2, -1))
        scores = scores / torch.sqrt(torch.tensor(query.size(-1), dtype=torch.float32))
        # scores shape: [batch_size, seq_length, n_spans]

        scores_weighted = scores * torch.softmax(
            probs / self.temperature, dim=1
        ).unsqueeze(1)
        # scores_weighted shape: [batch_size, seq_length, n_spans]

        attention_weights = F.softmax(
            scores_weighted, dim=-1
        )  # shape: [batch_size, seq_length, n_spans]

        output = torch.matmul(
            attention_weights, value
        )  # shape: [batch_size, seq_length, d_v]

        return output


class MR2HD_V2(nn.Module):
    """
    A neural network module for moment retrieval to highlight detection amplification.

    Attributes:
        score_mode (str): The mode for calculating scores. Must be one of {"probs", "iou", "probs_iou"}.
        aggregation_mode (str): The mode for aggregating features. One of {"attention", "gru", "none", "mean", "max"}.
        learned_aggregation (Optional[nn.Module]): An instance of a learned aggregation module or None.
        linear (nn.Linear): A linear layer for generating logits.
        temperature (nn.Parameter): A temperature parameter for scaling the scores.
    """

    def __init__(
        self,
        model_dim: int,
        aggregation_mode: str = "attention",
        score_mode: str = "probs",
    ) -> None:
        """
        Initialize the MR2HD module.

        Args:
            model_dim (int): Dimension of the model.
            aggregation_mode (str): Mode for aggregating features. One of {"attention", "gru", "none", "mean", "max"}.
            score_mode (str): Mode for calculating scores. One of {"probs", "iou", "probs_iou"}.
        """
        super().__init__()
        # check score and aggregation modes
        assert aggregation_mode in {"attention", "gru", "none", "mean", "max"}
        assert score_mode in {"probs", "iou", "probs_iou"}
        self.score_mode = score_mode
        self.aggregation_mode = aggregation_mode
        # self.temperature = nn.Parameter(torch.tensor(TEMP))

        # init aggregation class
        if aggregation_mode == "attention":
            self.learned_aggregation: Optional[nn.Module] = LearnedAggregation(
                model_dim, LearnedAggregationLayer(model_dim), 1
            )
        elif aggregation_mode == "gru":
            self.learned_aggregation = GRUFeatureExtractor(model_dim, model_dim)
        elif aggregation_mode == "none":
            self.learned_aggregation = None
        elif aggregation_mode == "mean":
            self.learned_aggregation = GlobalMeanPooling()
        else:
            self.learned_aggregation = GlobalMaxPooling()
        self.cross_attentio_with_scores = CrossAttentionWithProbs(
            model_dim, model_dim, model_dim
        )
        # init linear mapper
        self.linear = nn.Linear(model_dim, 1)
        nn.init.constant_(self.linear.weight.data, 0)  # noqa: WPS219
        nn.init.constant_(self.linear.bias.data, 0)  # noqa: WPS219

    def extract_intervals(
        self, src_aud: Tensor, predicted_spans_idxes: Tensor
    ) -> Tuple[Tensor, Tensor]:
        batch_size, seq_length, model_dim = src_aud.shape
        _, num_spans, _ = predicted_spans_idxes.shape

        # Create a list to store the extracted intervals for each batch
        extracted_intervals = []
        masks = []

        for batch_idx in range(batch_size):
            batch_intervals = []
            batch_masks = []
            for span_idx in range(num_spans):
                start_idx, end_idx = predicted_spans_idxes[batch_idx, span_idx].tolist()

                # Correct intervals where start_idx equals end_idx
                if start_idx == end_idx and end_idx < seq_length:
                    end_idx += 1
                if start_idx == end_idx and end_idx == seq_length:
                    start_idx -= 1

                # Ensure indices are within valid range
                start_idx = max(0, start_idx)
                end_idx = min(seq_length, end_idx)

                # Extract the interval
                length = end_idx - start_idx
                mask = [1] * length + [0] * (seq_length - length)

                mask = torch.tensor(mask).bool().to(src_aud.device)  # type: ignore
                interval = src_aud[batch_idx, start_idx:end_idx]
                padding = torch.zeros(seq_length - length, model_dim).to(src_aud.device)
                interval = torch.concat([interval, padding])  # [seq_length, model_dim]
                batch_intervals.append(interval)
                batch_masks.append(mask)

            extracted_intervals.append(torch.stack(batch_intervals))
            masks.append(torch.stack(batch_masks))  # type: ignore
        return torch.stack(extracted_intervals), torch.stack(masks)

    def forward(
        self,
        saliency_scores: Tensor,
        audio_features: Tensor,
        outputs_class: Tensor,
        quality_scores: Tensor,
        outputs_coord: Tensor,
        audio_text_features: Tensor,
    ) -> Tensor:
        """
        Forward pass of the MR2HD class.

        Args:
            saliency_scores (Tensor): shape [bs, seq_length], initial saliency scores.
            audio_features (Tensor): shape [bs, seq_length, model_dim], audio features before cross-attention with text.
            outputs_class (Tensor): shape [bs, num_queries, 2], class logits (0 idx is positive class).
            outputs_coord (Tensor): shape [bs, num_queries, 2], spans in normalized [0, 1] format (center and width).
            quality_scores (Tensor): shape [bs, num_queries], quality scores for each query.
            audio_text_features (Tensor): shape [seq_length, bs, model_dim], audio features after cross-attn with text.

        Returns:
            Tensor: updated saliency scores, shape [bs, seq_length]
        """
        if self.learned_aggregation is None:
            logits = self.linear(audio_text_features.transpose(0, 1))[
                :, :, 0
            ]  # [batch_size, seq_length]
            return saliency_scores + logits  # [batch_size, seq_length]

        batch_size, audio_length, _ = audio_features.shape
        scores = torch.sigmoid(outputs_class)[..., 0]
        outputs_coord_xx = span_cxw_to_xx(outputs_coord)  # [batch_size, num_queries, 2]
        predicted_spans_idxes = torch.round(outputs_coord_xx * audio_length).to(int)  # type: ignore
        # predicted_spans_idxes shape [batch_size, num_queries, 2]

        extracted_intervals, masks = self.extract_intervals(
            audio_features, predicted_spans_idxes
        )
        # list_of_seqs: List[Tensor] = []
        list_of_aggregation_span_vectors = []
        for batch_idx in range(batch_size):
            batch_intervals = extracted_intervals[batch_idx]
            batch_masks = masks[batch_idx]

            # agregate features inside interval
            if self.aggregation_mode in {"attention", "max", "mean"}:
                aggregation_span_vectors = self.learned_aggregation(
                    batch_intervals, batch_masks
                )[:, 0, :]
            else:  # only for test ancient evil
                span_vectors = [
                    self.learned_aggregation(interval[mask][None, :, :])[0]
                    for interval, mask in zip(batch_intervals, batch_masks)
                ]
                aggregation_span_vectors = torch.stack(span_vectors)

            list_of_aggregation_span_vectors.append(aggregation_span_vectors)
        aggregation_span_vectors = torch.stack(list_of_aggregation_span_vectors)

        audio_text_features = audio_text_features.transpose(0, 1)
        audio_text_features_updated = self.cross_attentio_with_scores(
            aggregation_span_vectors, audio_text_features, scores
        )
        logits = self.linear(audio_text_features_updated)[
            :, :, 0
        ]  # [batch_size, seq_length]
        return saliency_scores + logits  # [batch_size, seq_length]


# --- File: src.models.components.sg_detr_blocks.blocks.audio ---
"""Audio module."""

from typing import Any, Optional, Tuple, Union

import torch
from torch import Tensor, nn





EPS: float = 1e-6
INIT_CONST: float = 0.01


class FFNFuser(nn.Module):
    """FFNDuser Encoder."""

    def __init__(self, dim: int):
        """Initialize FFNFuser.

        Args:
            dim (int): model dim.
        """
        super().__init__()
        self.dim = dim
        self.projector = nn.Linear(dim + dim, dim)
        self.output_norm = nn.LayerNorm(dim)  # noqa: WPS204

    def forward(self, audio: Tensor, video: Tensor, **_: Any) -> Tensor:
        """Forward pass of the FFNFuser.

        Args:
            audio (Tensor): audio embeddings.
            video (Tensor): video embeddings.
            _ (Any): other kwargs.

        Returns:
            Tensor: merged features.
        """
        concated_features = torch.cat((video, audio), dim=2)
        merged = self.projector(concated_features)
        return self.output_norm(merged)


class CrossAttentionLayer(nn.Module):  # noqa: WPS230
    """Cross Attnetion."""

    def __init__(
        self,
        dim: int,
        expansion_ratio: int = 4,
        nhead: int = 8,
        dropout: float = 0.1,
        droppath: float = 0.1,
    ):
        """
        Initialize LearnedAggregation.

        Args:
            dim (int): Dimensionality of the input embeddings.
            expansion_ratio (int): Expansion ratio for the feedforward network. Default is 4.
            nhead (int): Number of attention heads. Default is 8.
            dropout (float): Dropout rate. Default is 0.1.
            droppath (float): Droppath rate. Defaults to 0.1.
        """
        super().__init__()
        self.dim = dim
        self.nhead = nhead
        self.attn = nn.MultiheadAttention(dim, nhead, dropout=dropout)
        self.norm1 = nn.LayerNorm(dim)  # noqa: WPS204
        self.norm2 = nn.LayerNorm(dim)
        self.dropout1 = DropPath(droppath)  # noqa: WPS204
        self.dropout2 = DropPath(droppath)
        self.ffn = FeedForwardNetwork(dim, expansion_ratio, dropout)
        self.apply(self._init_weights)

    @torch.no_grad()
    def _init_weights(self, module):
        """
        Initialize weights of the module.

        Args:
            module (nn.Module): A submodule of LearnedAggregation.
        """
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=INIT_CONST)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(
        self,
        query_emb: Tensor,
        key_emb: Tensor,
        pos_emb: Tensor,
        mask: Tensor,
    ) -> Tensor:
        """
        Forward pass of the CrossAttentionLayer module.

        Args:
            query_emb (Tensor): query feature seqs. (batch_size, seq_len, dim).
            key_emb (Tensor): key feature seqs. (batch_size, seq_len, dim).
            pos_emb (Tensor): positional embedding. (batch_size, seq_len, dim).
            mask (Tensor): mask of shape (batch_size, seq_len).

        Returns:
            Tensor: Output tensor of shape (seq_len, batch_size, dim).
        """
        # prepare mask
        mask = mask.to(torch.bool)

        # prepare queries
        query_emb_pos = query_emb + pos_emb
        query_emb_pos = query_emb_pos.transpose(0, 1)
        query_emb = query_emb.transpose(0, 1)

        # prepare keys and values
        key_emb_pos = key_emb + pos_emb
        key_emb_pos = key_emb_pos.transpose(0, 1)
        value_emb = key_emb.transpose(0, 1)

        attn_out, _ = self.attn(
            query_emb_pos,
            key_emb_pos,
            value_emb,
            key_padding_mask=~mask,
        )

        output = query_emb + self.dropout1(attn_out)
        output = self.norm1(output)
        ffn_output = self.ffn(output)
        output = output + self.dropout2(ffn_output)
        output = self.norm2(output)
        return output.transpose(0, 1)


class CrossAttentionEncoder(nn.Module):
    """Co-Attnetion Encoder."""

    def __init__(
        self, num_layers: int, dim: int, dropout: float, droppath: float
    ) -> None:
        """Initialize CrossAttention Encoder.

        Args:
            num_layers (int): num layers.
            dim (int): model dim.
            dropout (float): Dropout rate.
            droppath (float): Droppath rate.
        """
        super().__init__()
        self.num_layers = num_layers

        # video to audio cross attention
        vid_to_aud_cross = CrossAttentionLayer(
            dim=dim, dropout=dropout, droppath=droppath
        )
        self.vid_to_aud_cross_layers = get_clones(vid_to_aud_cross, num_layers)

    def forward(
        self,
        vid_emb: Tensor,
        aud_emb: Tensor,
        pos_emb: Tensor,
        mask: Tensor,
    ) -> Tensor:
        """
        Forward pass of the CrossAttention Encoder.

        Args:
            vid_emb (Tensor): video feature seqs. (batch_size, seq_len, dim).
            aud_emb (Tensor): audio feature seqs. (batch_size, seq_len, dim).
            pos_emb (Tensor): positional embedding. (batch_size, seq_len, dim).
            mask (Tensor): mask of shape (batch_size, seq_len).

        Returns:
            Tensor: Output tensor of shape (seq_len, batch_size, dim).
        """
        for vid_aud_layer in self.vid_to_aud_cross_layers:
            vid_emb = vid_aud_layer(
                query_emb=vid_emb,
                key_emb=aud_emb,
                pos_emb=pos_emb,
                mask=mask,
            )
        return vid_emb


class CoAttentionEncoder(nn.Module):
    """Co-Attnetion Encoder."""

    def __init__(
        self, num_layers: int, dim: int, dropout: float, droppath: float
    ) -> None:
        """Initialize CoAttention Encoder.

        Args:
            num_layers (int): num layers.
            dim (int): model dim.
            dropout (float): Dropout rate.
            droppath (float): Droppath rate.
        """
        super().__init__()
        self.num_layers = num_layers

        self.mapper = nn.Sequential(nn.Linear(dim + dim, dim), nn.LayerNorm(dim))  # noqa: WPS221

        # video to audio cross attention
        vid_to_aud_cross = CrossAttentionLayer(
            dim=dim, dropout=dropout, droppath=droppath
        )
        self.vid_to_aud_cross_layers = get_clones(vid_to_aud_cross, num_layers)

        # audio to video cross attention
        aud_to_vid_cross = CrossAttentionLayer(
            dim=dim, dropout=dropout, droppath=droppath
        )
        self.aud_to_vid_cross_layers = get_clones(aud_to_vid_cross, num_layers)

    def forward(
        self,
        vid_emb: Tensor,
        aud_emb: Tensor,
        pos_emb: Tensor,
        mask: Tensor,
    ) -> Tensor:
        """
        Forward pass of the CoAttention Encoder.

        Args:
            vid_emb (Tensor): video feature seqs. (batch_size, seq_len, dim).
            aud_emb (Tensor): audio feature seqs. (batch_size, seq_len, dim).
            pos_emb (Tensor): positional embedding. (batch_size, seq_len, dim).
            mask (Tensor): mask of shape (batch_size, seq_len).

        Returns:
            Tensor: Output tensor of shape (seq_len, batch_size, dim).
        """
        for aud_vid_layer, vid_aud_layer in zip(
            self.aud_to_vid_cross_layers, self.vid_to_aud_cross_layers
        ):
            updated_audio = aud_vid_layer(
                query_emb=aud_emb,
                key_emb=vid_emb,
                pos_emb=pos_emb,
                mask=mask,
            )
            updated_video = vid_aud_layer(
                query_emb=vid_emb,
                key_emb=aud_emb,
                pos_emb=pos_emb,
                mask=mask,
            )
            aud_emb = updated_audio
            vid_emb = updated_video
        return self.mapper(torch.cat((vid_emb, aud_emb), dim=2))


class BottleneckLayer(nn.Module):  # noqa: WPS230
    """Bottleneck Layer."""

    def __init__(
        self,
        dim: int,
        nhead: int = 8,
        expansion_ratio: int = 4,
        dropout: float = 0.1,
        droppath: float = 0.1,
    ):
        """Initialize the Bottleneck Layer.

        Args:
            dim (int): model dim.
            nhead (int): number of attn head. Defaults to 8.
            expansion_ratio (int): ffn expansion ratio. Defaults to 4.
            dropout (float): dropout prob. Defaults to 0.1.
            droppath (float): droppath prob. Defaults to 0.1.
        """
        super().__init__()
        self.dims = dim
        self.nhead = nhead
        self.expansion_ratio = expansion_ratio
        self.dropout = dropout
        self.droppath = droppath

        self.att1 = nn.MultiheadAttention(dim, nhead, dropout=dropout)
        self.att2 = nn.MultiheadAttention(dim, nhead, dropout=dropout)
        self.att3 = nn.MultiheadAttention(dim, nhead, dropout=dropout)
        self.att4 = nn.MultiheadAttention(dim, nhead, dropout=dropout)

        self.ffn1 = FeedForwardNetwork(dim, expansion_ratio, dropout)
        self.ffn2 = FeedForwardNetwork(dim, expansion_ratio, dropout)

        self.droppath1 = DropPath(droppath)  # noqa: WPS204
        self.droppath2 = DropPath(droppath)
        self.droppath3 = DropPath(droppath)
        self.droppath4 = DropPath(droppath)
        self.droppath5 = DropPath(droppath)
        self.droppath6 = DropPath(droppath)

        self.norm1 = nn.LayerNorm(dim)  # noqa: WPS204
        self.norm2 = nn.LayerNorm(dim)
        self.norm3 = nn.LayerNorm(dim)
        self.norm4 = nn.LayerNorm(dim)
        self.norm5 = nn.LayerNorm(dim)
        self.norm6 = nn.LayerNorm(dim)

        self.apply(self._init_weights)

    @torch.no_grad()
    def _init_weights(self, module):
        """
        Initialize weights of the module.

        Args:
            module (nn.Module): A submodule of LearnedAggregation.
        """
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=INIT_CONST)
            if module.bias is not None:
                nn.init.constant_(module.bias, 0)

    def forward(
        self,
        video: Tensor,
        audio: Tensor,
        bottle_token: Tensor,
        pos_emb: Tensor,
        mask: Tensor,
    ) -> Tuple[Tensor, Tensor, Tensor]:
        """Forward pass of the BottleneckLayer.

        Args:
            video (Tensor): video embeddings.
            audio (Tensor): audio embeddings.
            bottle_token (Tensor): bottlneck tokens.
            pos_emb (Tensor): positioanl embedding.
            mask (Tensor): key padding mask

        Returns:
            Tuple[Tensor, Tensor, Tensor]: Updated video, audio and token embeddings.
        """
        mask = mask.to(torch.bool)
        normed_video = self.norm1(video)
        normed_audio = self.norm2(audio)
        normed_bottle_token = self.norm3(bottle_token)

        # token to audio-video
        key_video = normed_video + pos_emb
        key_audio = normed_audio + pos_emb
        token_video_attn, _ = self.att1(
            normed_bottle_token, key_video, normed_video, key_padding_mask=~mask
        )
        token_audio_attn, _ = self.att2(
            normed_bottle_token, key_audio, normed_audio, key_padding_mask=~mask
        )
        bottle_token = bottle_token + self.droppath1(token_video_attn)
        bottle_token = bottle_token + self.droppath2(token_audio_attn)

        # audio-video to token attn
        normed_bottle_token = self.norm4(bottle_token)
        query_video = normed_video + pos_emb
        query_audio = normed_audio + pos_emb
        video_token_attn, _ = self.att3(
            query_video, normed_bottle_token, normed_bottle_token
        )
        audio_token_attn, _ = self.att4(
            query_audio, normed_bottle_token, normed_bottle_token
        )
        video = video + self.droppath3(video_token_attn)
        audio = audio + self.droppath4(audio_token_attn)

        # apply ffn
        normed_video = self.norm5(video)
        normed_audio = self.norm6(audio)
        video = video + self.droppath5(self.ffn1(normed_video))
        audio = audio + self.droppath6(self.ffn2(normed_audio))

        return audio, video, bottle_token


class BottleneckEncoder(nn.Module):
    """Bottleneck Encoder."""

    def __init__(
        self,
        num_layers: int,
        dim: int,
        num_tokens: int,
        dropout: float,
        droppath: float,
    ) -> None:
        """Initialize Bottleneck Encoder.

        Args:
            num_layers (int): num layers.
            dim (int): model dim.
            num_tokens (int): number of bottleneck tokens.
            dropout (float): Dropout rate.
            droppath (float): Droppath rate.
        """
        super().__init__()
        self.num_layers = num_layers
        self.token = nn.Embedding(num_tokens, dim)
        self.mapper = nn.Sequential(nn.Linear(dim + dim, dim), nn.LayerNorm(dim))  # noqa: WPS221
        bottle_layer = BottleneckLayer(dim=dim, dropout=dropout, droppath=droppath)
        self.layers = get_clones(bottle_layer, num_layers)

    def forward(
        self,
        vid_emb: Tensor,
        aud_emb: Tensor,
        pos_emb: Tensor,
        mask: Tensor,
    ) -> Tensor:
        """
        Forward pass of the CoAttention Encoder.

        Args:
            vid_emb (Tensor): video feature seqs. (batch_size, seq_len, dim).
            aud_emb (Tensor): audio feature seqs. (batch_size, seq_len, dim).
            pos_emb (Tensor): positional embedding. (batch_size, seq_len, dim).
            mask (Tensor): seq mask of shape (batch_size, seq_len).

        Returns:
            Tensor: Output tensor of shape (seq_len, batch_size, dim).
        """
        token = self.token.weight.expand(vid_emb.size(0), -1, -1)
        token = token.transpose(0, 1)
        vid_emb = vid_emb.transpose(0, 1)
        aud_emb = aud_emb.transpose(0, 1)
        pos_emb = pos_emb.transpose(0, 1)
        for layer in self.layers:
            vid_emb, aud_emb, token = layer(vid_emb, aud_emb, token, pos_emb, mask)
        projected_emb = self.mapper(torch.cat((vid_emb, aud_emb), dim=2))
        return projected_emb.transpose(0, 1)


class AudioMerger(nn.Module):
    """Class to merge video and audio features."""

    def __init__(  # noqa: WPS231
        self,
        num_layers: int,
        merge_type: Optional[str],
        num_tokens: int,
        model_dim: int,
        dropout: float,
        droppath: float,
    ) -> None:
        """Initialize AudioMerger class.

        Args:
            num_layers (int): num layers to use.
            merge_type (str): one of {concat, crossattn, coatt}
            num_tokens (int): number of bottleneck tokens to use.
            model_dim (int): model dim.
            dropout (float): Dropout rate.
            droppath (float): Droppath rate.

        Raises:
            NotImplementedError: merge_type should be one of {concat, crossattn, coatt}
        """
        super().__init__()
        self.dropout = dropout
        self.droppath = droppath
        self.merge_type = merge_type
        self.num_layers = num_layers
        if self.merge_type == "concat":  # noqa: WPS223
            self.audio_fuser: Optional[
                Union[
                    FFNFuser,
                    CrossAttentionEncoder,
                    CoAttentionEncoder,
                    BottleneckEncoder,
                ]
            ] = FFNFuser(model_dim)
        elif self.merge_type == "crossattn":
            self.audio_fuser = CrossAttentionEncoder(
                num_layers=num_layers,
                dim=model_dim,
                dropout=dropout,
                droppath=droppath,
            )
        elif self.merge_type == "coattn":
            self.audio_fuser = CoAttentionEncoder(
                num_layers=num_layers,
                dim=model_dim,
                dropout=dropout,
                droppath=droppath,
            )
        elif self.merge_type == "bottleneck":
            self.audio_fuser = BottleneckEncoder(
                num_layers=num_layers,
                dim=model_dim,
                num_tokens=num_tokens,
                dropout=dropout,
                droppath=droppath,
            )
        elif self.merge_type is None:
            self.audio_fuser = None
        else:
            raise NotImplementedError(
                "merge_type should be one of {concat, crossattn, coatt, bottleneck}"
            )

    def forward(
        self,
        audio: Tensor,
        video: Tensor,
        pos_emb: Tensor,
        mask: Tensor,
    ) -> Tensor:
        """Forward pass of the audio merger.

        Args:
            audio (Tensor): audio features
            video (Tensor): video features
            pos_emb (Tensor): pos embedding
            mask (Tensor): key padding mask

        Returns:
            Tensor: Merged features
        """
        if self.audio_fuser is not None:
            return self.audio_fuser(video, audio, pos_emb=pos_emb, mask=mask)
        return video


# --- File: src.models.components.sg_detr_blocks.blocks.atss ---
"""ATSS head implementation."""

import math
from typing import Any, Dict, List, Optional, Tuple

import torch
from torch import Tensor, nn








INIT_CONST: float = 0.01


class ATSSClassificationHead(nn.Module):
    """A classification head of the ATSS."""

    def __init__(
        self,
        in_channels: int,
        num_anchors: int,
        num_convs: int = 3,
        prior_probability: float = 0.3,
    ) -> None:
        """Initialize classification head.

        Args:
            in_channels (int): number of channels of the input feature.
            num_anchors (int): number of anchors.
            num_convs (int): number of conv layer. Default: 3.
            prior_probability (float): probability of prior. Default: 0.3.
        """
        super().__init__()
        self.num_anchors = num_anchors
        self.stem = ConvBlock1D(
            in_channels=in_channels,
            hidden_dim=in_channels,
            out_channels=in_channels,
            num_layers=num_convs,
            kernel_size=3,
            last_activate=True,
        )
        self._stem_init()

        self.cls_logits = ConvBlock1D(
            in_channels=in_channels,
            hidden_dim=in_channels,
            out_channels=num_anchors,
            num_layers=1,
            kernel_size=3,
            use_norm=False,
        )
        self.cls_init(prior_probability)

    def _stem_init(self) -> None:
        """Init stem blocks."""
        for module in self.stem.modules():
            if isinstance(module, nn.Conv1d):
                torch.nn.init.normal_(module.weight, std=INIT_CONST)
                if module.bias is not None:
                    torch.nn.init.constant_(module.bias, 0)  # type: ignore

    def cls_init(self, prior_probability: float) -> None:
        """Init classification layer.

        Args:
            prior_probability (float): Prior prob.
        """
        for module in self.cls_logits.modules():
            if isinstance(module, nn.Conv1d):
                torch.nn.init.normal_(module.weight, std=INIT_CONST)
                if module.bias is not None:
                    prior_prob_logit = -math.log(
                        (1 - prior_probability) / prior_probability
                    )
                    torch.nn.init.constant_(module.bias, prior_prob_logit)

    def forward(self, features: List[Tensor]) -> List[Tensor]:
        """Forward pass of the classification head.

        Args:
            features (List[Tensor]): FPN features.

        Returns:
            List[Tensor]: Predicted logits for each scale.
        """
        all_cls_logits = []
        for feature in features:
            feature = self.stem(feature)
            all_cls_logits.append(self.cls_logits(feature))
        return all_cls_logits


class ATSSRegressionHead(nn.Module):
    """A regression head to use in ATSS, which combines regression branch and center-ness branch."""

    def __init__(
        self,
        in_channels: int,
        num_anchors: int,
        num_convs: int = 3,
        fpn_strides: Tuple[float, ...] = (0.5, 1, 2, 4),
    ) -> None:
        """Initialize aux regression head.

        Args:
            in_channels (int): number of channels of the input feature
            num_anchors (int): number of anchors.
            num_convs (int): number of conv layer. Default: 2.
            fpn_strides (Tuple[float, ...]): fpn strides.
        """
        super().__init__()
        self.num_anchors = num_anchors
        self.stem = ConvBlock1D(
            in_channels=in_channels,
            hidden_dim=in_channels,
            out_channels=in_channels,
            num_layers=num_convs,
            kernel_size=3,
            last_activate=True,
        )

        self.bbox_reg = ConvBlock1D(
            in_channels=in_channels,
            hidden_dim=in_channels,
            out_channels=2 * num_anchors,
            num_layers=1,
            kernel_size=3,
            use_norm=False,
        )
        self.bbox_ctrness = ConvBlock1D(
            in_channels=in_channels,
            hidden_dim=in_channels,
            out_channels=num_anchors,
            num_layers=1,
            kernel_size=3,
            use_norm=False,
        )

        # init modules
        self._init_modules()

        # scale adjuster
        self.scales = nn.ModuleList(
            [Scale(init_value=1.0) for _ in range(len(fpn_strides))]
        )  # noqa: WPS221
        self.fpn_strides = fpn_strides

    def _init_modules(self) -> None:
        """Init weights."""
        for module in self.modules():
            if isinstance(module, nn.Conv1d):
                torch.nn.init.normal_(module.weight, std=INIT_CONST)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)

    def forward(self, features: List[Tensor]) -> Tuple[List[Tensor], List[Tensor]]:
        """Forwar pass of the regression head.

        Args:
            features (List[Tensor]): input FPN embeddings.

        Returns:
            Tuple[List[Tensor], List[Tensor]]: [Offsets, Centerness]
        """
        all_bbox_regression = []
        all_bbox_ctrness = []

        for idx, feature in enumerate(features):
            bbox_feature = self.stem(feature)

            # predict centerness
            bbox_ctrness: Tensor = self.bbox_ctrness(bbox_feature)
            all_bbox_ctrness.append(bbox_ctrness)

            # predict spans
            bbox_regression: Tensor = self.bbox_reg(bbox_feature)
            bbox_regression = self.scales[idx](bbox_regression)
            all_bbox_regression.append(bbox_regression)

        return all_bbox_regression, all_bbox_ctrness


class ATSSHead(nn.Module):
    """A regression and classification head of the ATSS."""

    def __init__(
        self,
        in_channels: int,
        num_convs: int = 3,
        top_k_positive_anchors: int = 9,
        prior_probability: float = 0.3,
        fpn_strides: Tuple[float, ...] = (0.5, 1, 2, 4),
        anchor_sizes: Tuple[int, ...] = (4, 16, 32, 64),
    ) -> None:
        """Initialize ATSS module.

        Args:
            in_channels (int): number of channels of the input feature
            num_convs (int): number of conv layer of head. Default: 3.
            top_k_positive_anchors (int): num anchors to select as positive ones. Default: 9.
            prior_probability (float): probability of prior. Default: 0.3.
            fpn_strides (Tuple[float, ...]): fpn strides.
            anchor_sizes (Tuple[int, ...]): size of anchors.
        """
        super().__init__()
        self.fpn_strides = fpn_strides
        self.num_anchors = 1
        self.top_k_positive_anchors = top_k_positive_anchors
        self.classification_head = ATSSClassificationHead(
            in_channels, self.num_anchors, num_convs, prior_probability
        )
        self.regression_head = ATSSRegressionHead(
            in_channels, self.num_anchors, num_convs, fpn_strides
        )
        self.anchor_generator = AnchorGenerator(anchor_sizes, fpn_strides)

    def prepare_positive_locations(  # noqa: WPS210
        self,
        encoder_features: List[Tensor],
        anchors: List[List[SpanList]],
        targets: Dict[str, Any],
        meta: List[Dict[str, Any]],
    ) -> Tuple[List[Tensor], List[Tensor], List[Tensor]]:
        """Prepare positive locations for anchors based on targets.

        Args:
            encoder_features (List[Tensor]): Encoder features for each feature map.
            anchors (List[List[SpanList]]): List of anchor spans for each seq and level.
            targets (Dict[str, Any]): Dictionary containing target information.
            meta (List[Dict[str, Any]]): List of metadata dictionaries for each sequence.

        Returns:
            Tuple[List[Tensor], List[Tensor], List[Tensor]]:
                - List of matched ground truth spans for each seq.
                - List of anchor spans for each seq.
                - List of selected features for each seq.
        """
        targets_xx = [
            (span_cxw_to_xx(target["spans"]) * sample_meta["duration"]).type(torch.int)
            / 2
            for target, sample_meta in zip(targets["span_labels"], meta)
        ]

        cls_labels, matched_gts, anchors_all_lvls = prepare_matched_gt(
            targets_xx, anchors, self.top_k_positive_anchors
        )

        max_labels_num: int = 0
        for labels in cls_labels:
            cur_labels_sum = sum(labels)
            max_labels_num = (
                max_labels_num if max_labels_num > cur_labels_sum else cur_labels_sum
            )

        gt_all_seqs = []
        anchors_all_seqs = []
        selected_features = []
        stacked_encoder_features = torch.cat(encoder_features, 1)
        device = stacked_encoder_features.device
        for idx, (
            cls_labels_per_seq,
            matched_gts_per_seq,
            anchors_per_seq,
            feature_map,
        ) in enumerate(  # noqa: WPS352
            zip(cls_labels, matched_gts, anchors_all_lvls, stacked_encoder_features),
        ):
            # get padding size
            negative_padding_size = max_labels_num - cls_labels_per_seq.sum()

            # select positive and negative encoder features
            selected_features_seq = feature_map[cls_labels_per_seq.bool()]
            neg_inds = torch.nonzero(~cls_labels_per_seq.bool())
            indices = torch.randperm(neg_inds.shape[0], device=device)[
                :negative_padding_size
            ]
            neg_inds = neg_inds[indices, 0]
            neg_selected_features_seq = feature_map[neg_inds]
            selected_features_seq = torch.cat(
                (selected_features_seq, neg_selected_features_seq)
            )

            # select positive gts and anchors
            matched_gts_per_seq = matched_gts_per_seq[cls_labels_per_seq.bool()]
            pos_anchors_per_seq = anchors_per_seq[cls_labels_per_seq.bool()]
            neg_anchors_per_seq = anchors_per_seq[neg_inds]
            anchors_per_seq = torch.cat((pos_anchors_per_seq, neg_anchors_per_seq))

            # convert positive anchors and gts
            matched_gts_per_seq = span_xx_to_cxw(matched_gts_per_seq)
            anchors_per_seq = span_xx_to_cxw(anchors_per_seq)
            matched_gts_per_seq = (matched_gts_per_seq / meta[idx]["duration"]) * 2
            anchors_per_seq = (anchors_per_seq / meta[idx]["duration"]) * 2

            gt_all_seqs.append(matched_gts_per_seq)
            anchors_all_seqs.append(anchors_per_seq)
            selected_features.append(selected_features_seq)

        return gt_all_seqs, anchors_all_seqs, selected_features

    def forward(
        self,
        fpn_features: List[Tensor],
        real_video_len: Tensor,
        targets: Optional[Dict[str, Any]] = None,
        meta: Optional[List[Dict[str, Any]]] = None,
    ) -> AuxDetectorOutput:
        """Forward pass of the model.

        Args:
            fpn_features (List[Tensor]): input FPN embeddings.
            real_video_len (Tensor): real video length.
            targets (Optional[Dict[str, Any]]): target information.
            meta (Optional[List[Dict[str, Any]]]): meta information.

        Returns:
            AuxDetectorOutput: Auxiliary head output schema.
        """
        cls_logits = self.classification_head(fpn_features)
        bbox_regression, bbox_ctrness = self.regression_head(fpn_features)
        anchors = self.anchor_generator(fpn_features, real_video_len)
        if targets is not None and meta is not None:
            matched_gts, anchors_spans, selected_features = (
                self.prepare_positive_locations(
                    fpn_features,
                    anchors,
                    targets,
                    meta,
                )
            )
        else:
            matched_gts, anchors_spans, selected_features = None, None, None
        return AuxDetectorOutput(
            cls_logits=cls_logits,
            bbox_regression=bbox_regression,
            bbox_ctrness=bbox_ctrness,
            anchors=anchors,
            matched_gts=matched_gts,
            anchors_spans=anchors_spans,
            selected_features=selected_features,
        )


# --- File: src.models.qd_detr.model ---
import torch
import torch.nn.functional as F
import numpy as np
from torch import nn







    LocalSaliencyHead,
    Text2AudioEncoder,
    SaliencyAmplifier,
)


def inverse_sigmoid(x, eps=1e-3):
    """Applies inverse sigmoid transformation to the input tensor.

    Args:
        x (torch.Tensor): Input tensor.
        eps (float): Small value for numerical stability.

    Returns:
        torch.Tensor: Inverse sigmoid of x.
    """
    x = x.clamp(min=0, max=1)
    x1 = x.clamp(min=eps)
    x2 = (1 - x).clamp(min=eps)
    return torch.log(x1 / x2)


class QDDETR(nn.Module):
    def __init__(
        self,
        transformer,
        position_embed,
        txt_position_embed,
        aud_dim,
        txt_dim,
        num_queries,
        input_dropout,
        max_a_l,
        aux_loss=True,
        span_loss_type="l1",
        use_txt_pos=False,
        n_input_proj=2,
    ):
        """Initializes the model.
        Parameters:
            transformer: torch module of the transformer architecture. See transformer.py
            position_embed: torch module of the position_embedding, See position_encoding.py
            txt_position_embed: position_embedding for text
            txt_dim: int, text query input dimension
            num_queries: number of object queries, ie detection slot. This is the maximal number of objects
                         QD-DETR can detect in a single audio.
            aux_loss: True if auxiliary decoding losses (loss at each decoder layer) are to be used.
            max_a_l: int, maximum #clips in audio
            span_loss_type: str, one of [l1, ce]
                l1: (center-x, width) regression.
                ce: (st_idx, ed_idx) classification.
            # foreground_thd: float, intersection over prediction >= foreground_thd: labeled as foreground
            # background_thd: float, intersection over prediction <= background_thd: labeled background
        """
        super().__init__()
        self.num_queries = num_queries
        self.transformer = transformer
        self.position_embed = position_embed
        self.txt_position_embed = txt_position_embed
        hidden_dim = transformer.d_model
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        span_pred_dim = 2
        self.span_embed = MLP(hidden_dim, hidden_dim, span_pred_dim, 3)
        self.class_embed = nn.Linear(hidden_dim, 2)  # 0: background, 1: foreground
        self.use_txt_pos = use_txt_pos
        self.n_input_proj = n_input_proj
        self.query_embed = nn.Embedding(num_queries, 2)
        relu_args = [True] * 3
        relu_args[n_input_proj - 1] = False

        self.input_txt_proj = nn.Sequential(
            *[
                LinearLayer(
                    txt_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[0],
                ),
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[1],
                ),
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[2],
                ),
            ][:n_input_proj]
        )
        self.input_aud_proj = nn.Sequential(
            *[
                LinearLayer(
                    aud_dim + 2,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[0],
                ),  # add pos_embedding
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[1],
                ),
                LinearLayer(
                    hidden_dim,
                    hidden_dim,
                    layer_norm=True,
                    dropout=input_dropout,
                    relu=relu_args[2],
                ),
            ][:n_input_proj]
        )
        self.aux_loss = aux_loss

        self.saliency_proj1 = nn.Linear(hidden_dim, hidden_dim)
        self.saliency_proj2 = nn.Linear(hidden_dim, hidden_dim)

        self.hidden_dim = hidden_dim
        self.global_rep_token = torch.nn.Parameter(torch.randn(hidden_dim))
        self.global_rep_pos = torch.nn.Parameter(torch.randn(hidden_dim))
        self.local_saliency_head = LocalSaliencyHead(
            model_dim=hidden_dim,
            use_projections=True,
            logit_mode="linear",
            use_gamma=True,
            num_aggregation_layers=1,
        )

        self.txt2aud_encoder = Text2AudioEncoder(
            d_model=hidden_dim,
            num_dummies=0,
            num_t2v_layers=2,
            dropout=input_dropout,
            droppath=0.1,
            use_cross_attn_wo_dummy=True,
            weight_attn_with_saliency=True,
        )

        self.saliency_amplifier = SaliencyAmplifier(
            d_model=hidden_dim,
            mode="sigmoid",
            use_mha=True,
        )

    def forward(self, src_txt, src_txt_mask, src_aud, src_aud_mask):
        """The forward expects two tensors:
           - src_txt: [batch_size, L_txt, D_txt]
           - src_txt_mask: [batch_size, L_txt], containing 0 on padded pixels,
                will convert to 1 as padding later for transformer
           - src_aud: [batch_size, L_aud, D_aud]
           - src_aud_mask: [batch_size, L_aud], containing 0 on padded pixels,
                will convert to 1 as padding later for transformer

        It returns a dict with the following elements:
           - "pred_spans": The normalized boxes coordinates for all queries, represented as
                           (center_x, width). These values are normalized in [0, 1],
                           relative to the size of each individual image (disregarding possible padding).
                           See PostProcess for information on how to retrieve the unnormalized bounding box.
           - "aux_outputs": Optional, only returned when auxilary losses are activated. It is a list of
                            dictionnaries containing the two above keys for each decoder layer.
        """
        src_aud = self.input_aud_proj(src_aud)
        src_txt = self.input_txt_proj(src_txt)

        audio_length = src_aud.shape[1]

        # Position embedding (you can still use your current one or upgrade to RoPE later)
        pos_aud = self.position_embed(src_aud, src_aud_mask)
        pos_txt = (
            self.txt_position_embed(src_txt)
            if self.use_txt_pos
            else torch.zeros_like(src_txt)
        )

        # Get saliency scores
        saliency_scores, src_sent = self.local_saliency_head(
            src_aud, src_txt, src_txt_mask
        )

        # Concat audio and text
        src = torch.cat([src_aud, src_txt], dim=1)
        mask = torch.cat([src_aud_mask, src_txt_mask], dim=1).bool()
        pos = torch.cat([pos_aud, pos_txt], dim=1)

        # Saliency-Guided Cross Attention
        src_updated, mask_updated, pos_updated, attn_weights = self.txt2aud_encoder(
            src=src,
            mask=mask,
            pos=pos,
            batch_audio_len=audio_length,
            saliency_scores=torch.sigmoid(saliency_scores),
        )

        src = src_updated
        mask = mask_updated
        pos = pos_updated

        # (#layers, bsz, #queries, d), (bsz, L_aud+L_txt, d)

        # for global token
        mask_ = torch.tensor([[True]]).to(mask.device).repeat(mask.shape[0], 1)
        mask = torch.cat([mask_, mask], dim=1)
        src_ = self.global_rep_token.reshape([1, 1, self.hidden_dim]).repeat(
            src.shape[0], 1, 1
        )
        src = torch.cat([src_, src], dim=1)
        pos_ = self.global_rep_pos.reshape([1, 1, self.hidden_dim]).repeat(
            pos.shape[0], 1, 1
        )
        pos = torch.cat([pos_, pos], dim=1)

        audio_length = src_aud.shape[1]

        hs, reference, memory, memory_global = self.transformer(
            src, ~mask, self.query_embed.weight, pos, audio_length
        )

        # Apply Saliency Amplifier to audio memory
        aud_mem = memory  # memory is already (bsz, L_aud, d)
        aud_mem_amplified = self.saliency_amplifier(
            features=aud_mem.transpose(0, 1),  # (L_aud, bsz, d)
            saliency_scores=saliency_scores,
            pos=pos_aud.transpose(0, 1),  # (L_aud, bsz, d)
            aud_mask=src_aud_mask,
        )
        aud_mem = aud_mem_amplified.transpose(0, 1)  # back to (bsz, L_aud, d)

        # Re-construct memory with amplified audio features
        # Note: hs is already computed, but if hs should be influenced by amplified memory,
        # we might need to apply saliency_amplifier *before* the decoder.
        # SG-DETR applies it after det_encoder (TransformerEncoder) and before query_selector / FPN / detr_head.
        # We apply it to aud_mem which is used for saliency_scores computation below.

        outputs_class = self.class_embed(
            hs
        )  # (#layers, batch_size, #queries, #classes)
        reference_before_sigmoid = inverse_sigmoid(reference)
        tmp = self.span_embed(hs)
        outputs_coord = tmp + reference_before_sigmoid
        if self.span_loss_type == "l1":
            outputs_coord = outputs_coord.sigmoid()
        out = {"pred_logits": outputs_class[-1], "pred_spans": outputs_coord[-1]}

        # aud_mem is already extracted above

        ### Neg Pairs ###
        src_txt_neg = torch.cat([src_txt[1:], src_txt[0:1]], dim=0)
        src_txt_mask_neg = torch.cat([src_txt_mask[1:], src_txt_mask[0:1]], dim=0)
        src_neg = torch.cat([src_aud, src_txt_neg], dim=1)
        mask_neg = torch.cat([src_aud_mask, src_txt_mask_neg], dim=1).bool()

        mask_neg = torch.cat([mask_, mask_neg], dim=1)
        src_neg = torch.cat([src_, src_neg], dim=1)
        pos_neg = pos.clone()  # since it does not use actual content

        _, _, memory_neg, memory_global_neg = self.transformer(
            src_neg, ~mask_neg, self.query_embed.weight, pos_neg, audio_length
        )
        aud_mem_neg = memory_neg[:, : src_aud.shape[1]]

        out["saliency_scores"] = torch.sum(
            self.saliency_proj1(aud_mem)
            * self.saliency_proj2(memory_global).unsqueeze(1),
            dim=-1,
        ) / np.sqrt(self.hidden_dim)
        out["saliency_scores_neg"] = torch.sum(
            self.saliency_proj1(aud_mem_neg)
            * self.saliency_proj2(memory_global_neg).unsqueeze(1),
            dim=-1,
        ) / np.sqrt(self.hidden_dim)
        out["audio_mask"] = src_aud_mask
        if self.aux_loss:
            out["aux_outputs"] = [
                {"pred_logits": a, "pred_spans": b}
                for a, b in zip(outputs_class[:-1], outputs_coord[:-1])
            ]
        return out


class SetCriterion(nn.Module):
    """This class computes the loss for DETR.
    The process happens in two steps:
        1) we compute hungarian assignment between ground truth boxes and the outputs of the model
        2) we supervise each pair of matched ground-truth / prediction (supervise class and box)
    """

    def __init__(
        self,
        matcher,
        weight_dict,
        eos_coef,
        losses,
        span_loss_type,
        max_a_l,
        saliency_margin=1,
        use_focal_loss=False,
    ):
        """Create the criterion.
        Parameters:
            matcher: module able to compute a matching between targets and proposals
            weight_dict: dict containing as key the names of the losses and as values their relative weight.
            eos_coef: relative classification weight applied to the no-object category
            losses: list of all the losses to be applied. See get_loss for list of available losses.
            span_loss_type: str, [l1, ce]
            max_v_l: int,
            saliency_margin: float
        """
        super().__init__()
        self.matcher = matcher
        self.weight_dict = weight_dict
        self.losses = losses
        self.span_loss_type = span_loss_type
        self.max_a_l = max_a_l
        self.saliency_margin = saliency_margin
        self.use_focal_loss = use_focal_loss

        # foreground and background classification
        self.foreground_label = 0
        self.background_label = 1
        self.eos_coef = eos_coef
        empty_weight = torch.ones(2)
        empty_weight[-1] = (
            self.eos_coef
        )  # lower weight for background (index 1, foreground index 0)
        self.register_buffer("empty_weight", empty_weight)

    def _focal_loss(self, logits, targets, alpha=0.25, gamma=2.0):
        """Computes focal loss for classification."""
        log_prob = F.log_softmax(logits, dim=-1)
        prob = torch.exp(log_prob)
        log_pt = log_prob.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        pt = prob.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
        
        # Clamp pt and log_pt to guarantee numerical stability of gradients under mixed-precision (autocast)
        pt = torch.clamp(pt, min=1e-5, max=1.0 - 1e-5)
        log_pt = torch.clamp(log_pt, min=-12.0)
        
        alpha_t = torch.where(targets == self.foreground_label, alpha, 1.0 - alpha)
        loss = -alpha_t * ((1.0 - pt) ** gamma) * log_pt
        return loss.mean()

    def loss_spans(self, outputs, targets, indices):
        """Compute the losses related to the bounding boxes, the L1 regression loss and the GIoU loss
        targets dicts must contain the key "spans" containing a tensor of dim [nb_tgt_spans, 2]
        The target spans are expected in format (center_x, w), normalized by the image size.
        """
        assert "pred_spans" in outputs
        targets = targets["span_labels"]
        idx = self._get_src_permutation_idx(indices)
        src_spans = outputs["pred_spans"][idx]  # (#spans, max_v_l * 2)
        tgt_spans = torch.cat(
            [t["spans"][i] for t, (_, i) in zip(targets, indices)], dim=0
        )  # (#spans, 2)
        if self.span_loss_type == "l1":
            loss_span = F.l1_loss(src_spans, tgt_spans, reduction="none")
            loss_giou = 1 - torch.diag(
                generalized_temporal_iou(
                    span_cxw_to_xx(src_spans), span_cxw_to_xx(tgt_spans)
                )
            )
        else:  # ce
            n_spans = src_spans.shape[0]
            src_spans = src_spans.view(n_spans, 2, self.max_v_l).transpose(1, 2)
            loss_span = F.cross_entropy(src_spans, tgt_spans, reduction="none")
            loss_giou = loss_span.new_zeros([1])

        losses = {}
        losses["loss_span"] = loss_span.mean()
        losses["loss_giou"] = loss_giou.mean()
        return losses

    def loss_labels(self, outputs, targets, indices, log=True):
        """Classification loss (NLL)
        targets dicts must contain the key "labels" containing a tensor of dim [nb_target_boxes]
        """
        # TODO add foreground and background classifier.  use all non-matched as background.
        assert "pred_logits" in outputs
        src_logits = outputs["pred_logits"]  # (batch_size, #queries, #classes=2)
        # idx is a tuple of two 1D tensors (batch_idx, src_idx), of the same length == #objects in batch
        idx = self._get_src_permutation_idx(indices)
        target_classes = torch.full(
            src_logits.shape[:2],
            self.background_label,
            dtype=torch.int64,
            device=src_logits.device,
        )  # (batch_size, #queries)
        target_classes[idx] = self.foreground_label

        loss_ce = F.cross_entropy(
            src_logits.transpose(1, 2),
            target_classes,
            self.empty_weight,
            reduction="none",
        )
        losses = {"loss_label": loss_ce.mean()}

        if self.use_focal_loss:
            losses["loss_focal"] = self._focal_loss(src_logits, target_classes)

        if log:
            # TODO this should probably be a separate loss, not hacked in this one here
            losses["class_error"] = (
                100 - accuracy(src_logits[idx], self.foreground_label)[0]
            )
        return losses

    def loss_saliency(self, outputs, targets, indices, log=True):
        """higher scores for positive clips"""
        if "saliency_pos_labels" not in targets:
            return {"loss_saliency": 0}

        aud_token_mask = outputs["audio_mask"]

        # Neg pair loss
        saliency_scores_neg = outputs["saliency_scores_neg"].clone()  # (N, L)
        # loss_neg_pair = torch.sigmoid(saliency_scores_neg).mean()

        loss_neg_pair = (
            (-torch.log(1.0 - torch.sigmoid(saliency_scores_neg)) * aud_token_mask)
            .sum(dim=1)
            .mean()
        )

        saliency_scores = outputs["saliency_scores"].clone()  # (N, L)
        saliency_contrast_label = targets["saliency_all_labels"]

        saliency_scores = torch.cat([saliency_scores, saliency_scores_neg], dim=1)
        saliency_contrast_label = torch.cat(
            [saliency_contrast_label, torch.zeros_like(saliency_contrast_label)], dim=1
        )

        aud_token_mask = aud_token_mask.repeat([1, 2])
        saliency_scores = (
            aud_token_mask * saliency_scores + (1.0 - aud_token_mask) * -1e3
        )

        tau = 0.5
        loss_rank_contrastive = 0.0

        # for rand_idx in range(1, 13, 3):
        #     # 1, 4, 7, 10 --> 5 stages
        for rand_idx in range(1, 12):
            drop_mask = ~(saliency_contrast_label > 100)  # no drop
            pos_mask = (
                saliency_contrast_label >= rand_idx
            )  # positive when equal or higher than rand_idx

            if torch.sum(pos_mask) == 0:  # no positive sample
                continue
            else:
                batch_drop_mask = (
                    torch.sum(pos_mask, dim=1) > 0
                )  # negative sample indicator

            # drop higher ranks
            cur_saliency_scores = saliency_scores * drop_mask / tau + ~drop_mask * -1e3

            # numerical stability
            logits = (
                cur_saliency_scores
                - torch.max(cur_saliency_scores, dim=1, keepdim=True)[0]
            )

            # softmax
            exp_logits = torch.exp(logits)
            log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-6)

            mean_log_prob_pos = (pos_mask * log_prob * aud_token_mask).sum(1) / (
                pos_mask.sum(1) + 1e-6
            )

            loss = -mean_log_prob_pos * batch_drop_mask

            loss_rank_contrastive = loss_rank_contrastive + loss.mean()

        loss_rank_contrastive = loss_rank_contrastive / 12

        saliency_scores = outputs["saliency_scores"]  # (N, L)
        pos_indices = targets["saliency_pos_labels"]  # (N, #pairs)
        neg_indices = targets["saliency_neg_labels"]  # (N, #pairs)
        num_pairs = pos_indices.shape[1]  # typically 2 or 4
        batch_indices = torch.arange(len(saliency_scores)).to(saliency_scores.device)
        pos_scores = torch.stack(
            [
                saliency_scores[batch_indices, pos_indices[:, col_idx]]
                for col_idx in range(num_pairs)
            ],
            dim=1,
        )
        neg_scores = torch.stack(
            [
                saliency_scores[batch_indices, neg_indices[:, col_idx]]
                for col_idx in range(num_pairs)
            ],
            dim=1,
        )
        loss_saliency = (
            torch.clamp(self.saliency_margin + neg_scores - pos_scores, min=0).sum()
            / (len(pos_scores) * num_pairs)
            * 2
        )  # * 2 to keep the loss the same scale

        loss_saliency = loss_saliency + loss_rank_contrastive + loss_neg_pair
        return {"loss_saliency": loss_saliency}

    def _get_src_permutation_idx(self, indices):
        """Permutes predictions following the given indices.

        Args:
            indices: List of tuples (src, tgt) indices.

        Returns:
            tuple: batch_idx, src_idx
        """
        # permute predictions following indices
        batch_idx = torch.cat(
            [torch.full_like(src, i) for i, (src, _) in enumerate(indices)]
        )
        src_idx = torch.cat([src for (src, _) in indices])
        return batch_idx, src_idx  # two 1D tensors of the same length

    def _get_tgt_permutation_idx(self, indices):
        """Permutes targets following the given indices.

        Args:
            indices: List of tuples (src, tgt) indices.

        Returns:
            tuple: batch_idx, tgt_idx
        """
        # permute targets following indices
        batch_idx = torch.cat(
            [torch.full_like(tgt, i) for i, (_, tgt) in enumerate(indices)]
        )
        tgt_idx = torch.cat([tgt for (_, tgt) in indices])
        return batch_idx, tgt_idx

    def get_loss(self, loss, outputs, targets, indices, **kwargs):
        """Retrieves the loss function for the given loss type.

        Args:
            loss (str): Type of loss.
            outputs: Model outputs.
            targets: Ground truth targets.
            indices: Matched indices.
            **kwargs: Additional arguments.

        Returns:
            dict: Loss values.
        """
        loss_map = {
            "spans": self.loss_spans,
            "labels": self.loss_labels,
            "saliency": self.loss_saliency,
        }
        assert loss in loss_map, f"do you really want to compute {loss} loss?"
        return loss_map[loss](outputs, targets, indices, **kwargs)

    def forward(self, outputs, targets):
        """This performs the loss computation.
        Parameters:
             outputs: dict of tensors, see the output specification of the model for the format
             targets: list of dicts, such that len(targets) == batch_size.
                      The expected keys in each dict depends on the losses applied, see each loss' doc
        """
        outputs_without_aux = {k: v for k, v in outputs.items() if k != "aux_outputs"}

        # Retrieve the matching between the outputs of the last layer and the targets
        # list(tuples), each tuple is (pred_span_indices, tgt_span_indices)

        indices = self.matcher(outputs_without_aux, targets)
        losses_target = self.losses

        # Compute all the requested losses
        losses = {}
        for loss in losses_target:
            losses.update(self.get_loss(loss, outputs, targets, indices))

        # In case of auxiliary losses, we repeat this process with the output of each intermediate layer.
        if "aux_outputs" in outputs:
            for i, aux_outputs in enumerate(outputs["aux_outputs"]):
                indices = self.matcher(aux_outputs, targets)
                losses_target = self.losses

                for loss in losses_target:
                    if "saliency" == loss:  # skip as it is only in the top layer
                        continue
                    kwargs = {}
                    l_dict = self.get_loss(
                        loss, aux_outputs, targets, indices, **kwargs
                    )
                    l_dict = {k + f"_{i}": v for k, v in l_dict.items()}
                    losses.update(l_dict)
        return losses


class MLP(nn.Module):
    """Very simple multi-layer perceptron (also called FFN)"""

    def __init__(self, input_dim, hidden_dim, output_dim, num_layers):
        super().__init__()
        self.num_layers = num_layers
        h = [hidden_dim] * (num_layers - 1)
        self.layers = nn.ModuleList(
            nn.Linear(n, k) for n, k in zip([input_dim] + h, h + [output_dim])
        )

    def forward(self, x):
        """Forward pass through the MLP.

        Args:
            x (torch.Tensor): Input tensor.

        Returns:
            torch.Tensor: Output tensor.
        """
        for i, layer in enumerate(self.layers):
            x = F.relu(layer(x)) if i < self.num_layers - 1 else layer(x)
        return x


class LinearLayer(nn.Module):
    """linear layer configurable with layer normalization, dropout, ReLU."""

    def __init__(self, in_hsz, out_hsz, layer_norm=True, dropout=0.1, relu=True):
        super(LinearLayer, self).__init__()
        self.relu = relu
        self.layer_norm = layer_norm
        if layer_norm:
            self.LayerNorm = nn.LayerNorm(in_hsz)
        layers = [nn.Dropout(dropout), nn.Linear(in_hsz, out_hsz)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        """(N, L, D)"""
        if self.layer_norm:
            x = self.LayerNorm(x)
        x = self.net(x)
        if self.relu:
            x = F.relu(x, inplace=True)
        return x  # (N, L, D)


def build_model(args):
    """Builds the QD-DETR model and criterion.

    Args:
        args: Configuration arguments.

    Returns:
        tuple: model, criterion
    """
    # the `num_classes` naming here is somewhat misleading.
    # it indeed corresponds to `max_obj_id + 1`, where max_obj_id
    # is the maximum id for a class in your dataset. For example,
    # COCO has a max_obj_id of 90, so we pass `num_classes` to be 91.
    # As another example, for a dataset that has a single class with id 1,
    # you should pass `num_classes` to be 2 (max_obj_id + 1).
    # For more details on this, check the following discussion
    # https://github.com/facebookresearch/qd_detr/issues/108#issuecomment-650269223
    device = torch.device(args.device)
    transformer = build_transformer(args)
    position_embedding, txt_position_embedding = build_position_encoding(args)

    model = QDDETR(
        transformer,
        position_embedding,
        txt_position_embedding,
        max_a_l=args.max_a_l,
        txt_dim=args.t_feat_dim,
        aud_dim=args.a_feat_dim,
        aux_loss=args.aux_loss,
        num_queries=args.num_queries,
        input_dropout=args.input_dropout,
        span_loss_type=args.span_loss_type,
        n_input_proj=args.n_input_proj,
    )

    matcher = build_matcher(args)
    use_focal_loss = args.get("use_focal_loss", False)
    weight_dict = {
        "loss_span": args.span_loss_coef,
        "loss_giou": args.giou_loss_coef,
        "loss_label": 0.0 if use_focal_loss else args.label_loss_coef,
        "loss_saliency": args.lw_saliency,
    }
    if use_focal_loss:
        weight_dict["loss_focal"] = args.label_loss_coef

    if args.aux_loss:
        aux_weight_dict = {}
        for i in range(args.dec_layers - 1):
            aux_weight_dict.update(
                {k + f"_{i}": v for k, v in weight_dict.items() if k != "loss_saliency"}
            )
        weight_dict.update(aux_weight_dict)

    losses = ["spans", "labels", "saliency"]
    criterion = SetCriterion(
        matcher=matcher,
        weight_dict=weight_dict,
        losses=losses,
        eos_coef=args.eos_coef,
        span_loss_type=args.span_loss_type,
        max_a_l=args.max_a_l,
        saliency_margin=args.saliency_margin,
        use_focal_loss=use_focal_loss,
    )
    criterion.to(device)
    return model, criterion


# --- File: src.models.components.sg_detr_blocks.blocks.detector ---
"""DETR Transformer class."""

import math
from typing import Any, Dict, List, Optional, Tuple

import torch
from torch import Tensor, nn
from torch.nn import functional as func





    TransformerDecoderLayer,
    TransformerEncoderLayer,
)

    prepare_anchors_codetr,
    prepare_anchors_dn,
)

    gen_encoder_output_proposals,
    inverse_sigmoid,
)

    DetectorOutput,
    DetEncoderOutput,
    QueryProposalsOutput,
)

EPS: float = 0.01
MIN_CONST: float = -1e7


class QuerySelector(nn.Module):  # noqa: WPS230
    """Class to select suqries for decoder."""

    def __init__(
        self,
        model_dim: int,
        num_queries: int,
        prior_prob: float = 0.35,
        default_widths: List[float] = [0.05, 0.2, 0.4, 0.85],
        init_spans_with_zeros: bool = True,
    ):
        """Initialize QuerySelector.

        Args:
            model_dim (int): Hidden dimension of the model.
            num_queries (int): number of queries to predict.
            prior_prob (float): prior foreground prob.
            default_widths (float): default width of encoder's anchors.
            init_spans_with_zeros (bool): whether to init last mlp layer with zeros or not
        """
        super().__init__()
        self.model_dim = model_dim
        self.num_queries = num_queries
        self.prior_prob = prior_prob
        self.default_widths = default_widths
        self.init_spans_with_zeros = init_spans_with_zeros

        self.enc_output = nn.Linear(model_dim, model_dim)
        self.enc_output_norm = nn.LayerNorm(model_dim)
        self.enc_out_span_embed = SlimMLP(
            input_dim=model_dim, hidden_dim=model_dim, output_dim=2, num_layers=3
        )
        self.enc_out_class_embed = nn.Linear(model_dim, 1)
        self.enc_out_iou_embed = nn.Linear(model_dim, 1)
        self._init_parameters(prior_prob)

    def _init_parameters(self, prior_prob: float) -> None:
        """Init parameters.

        Args:
            prior_prob (float): prior foreground prob.
        """
        # init cls layer with prior prob
        bias_value = math.log(prior_prob / (1 - prior_prob))
        torch.nn.init.normal_(self.enc_out_class_embed.weight, std=EPS)  # noqa: WPS432
        torch.nn.init.normal_(self.enc_out_iou_embed.weight, std=EPS)  # noqa: WPS432
        self.enc_out_class_embed.bias.data.fill_(bias_value)
        self.enc_out_iou_embed.bias.data.fill_(bias_value * 2)

        # init last reg layer with zeros
        for module in self.enc_out_span_embed.linear_mapper.modules():
            if (
                isinstance(module, nn.Linear)
                and module.out_features == 2
                and self.init_spans_with_zeros
            ):
                nn.init.constant_(module.weight.data, 0)  # noqa: WPS219
                nn.init.constant_(module.bias.data, 0)  # noqa: WPS219

        # init mapper
        nn.init.xavier_uniform_(self.enc_output.weight.data)
        nn.init.constant_(self.enc_output.bias.data, 0)

    def get_query_proposals(
        self,
        memory_aux: Tensor,
        output_proposals: Tensor,
        mask: Tensor,
    ) -> QueryProposalsOutput:
        """Prepare query proposals.

        Args:
            memory_aux (Tensor): Query memory tensor from the encoder. Shape: [bs, seq, dim]
            output_proposals (Tensor): Tensor containing the initial proposals. Shape: [bs, seq, 2]
            mask (Tensor): mask for irrelevant embs.

        Returns:
            QueryProposalsOutput: Query proposal schema.
        """
        # map memory features
        memory_aux = self.enc_output_norm(self.enc_output(memory_aux))

        # predict objectness and iou scores
        enc_outputs_class_unselected = self.enc_out_class_embed(memory_aux)
        enc_outputs_iou_unselected = self.enc_out_iou_embed(memory_aux)

        # combine them
        enc_outputs_combo_unselected = torch.sqrt(
            enc_outputs_class_unselected.sigmoid()
            * enc_outputs_iou_unselected.sigmoid(),
        )
        enc_outputs_combo_unselected = enc_outputs_combo_unselected.masked_fill(
            ~mask, float("-inf")
        )

        # predict reference points
        enc_outputs_offsets_unselected = self.enc_out_span_embed(memory_aux)
        enc_outputs_coord_unselected = output_proposals + enc_outputs_offsets_unselected

        # Find the most relevant indices
        topk_proposals = torch.topk(
            enc_outputs_combo_unselected[..., 0], self.num_queries, dim=1
        )[1]  # noqa: WPS221
        topk_proposals = topk_proposals.unsqueeze(-1)

        # gather features
        query_topk = topk_proposals.repeat(1, 1, self.model_dim)
        query_embs = torch.gather(memory_aux, 1, query_topk).detach()

        # gather spans
        spans_topk = topk_proposals.repeat(1, 1, 2)
        refpoint_embed_undetach = torch.gather(
            enc_outputs_coord_unselected, 1, spans_topk
        )  # unsigmoid
        refpoint_embed_detach = refpoint_embed_undetach.detach()

        # gather logits
        class_logit_enc = torch.gather(
            enc_outputs_class_unselected, 1, topk_proposals[..., [0]]
        )
        iou_logit_enc = torch.gather(
            enc_outputs_iou_unselected, 1, topk_proposals[..., [0]]
        )

        return QueryProposalsOutput(
            query_embs=query_embs,
            refpoint_embed_detach=refpoint_embed_detach,
            refpoint_embed_enc=refpoint_embed_undetach.sigmoid(),
            class_logit_enc=class_logit_enc,
            iou_logit_enc=iou_logit_enc,
        )

    @staticmethod
    def _prepare_mask(fpn_features: List[Tensor], mask: Tensor) -> List[Tensor]:  # noqa: WPS602
        spatial_shapes = [seq.size(1) for seq in fpn_features]
        original_seq_length = mask.shape[1]

        updated_masks = []
        for seq_length in spatial_shapes:
            if seq_length > original_seq_length:
                scale = seq_length // original_seq_length
                updated_mask = mask.repeat_interleave(scale, dim=1)
                updated_masks.append(updated_mask)
            elif seq_length < original_seq_length:
                scale = original_seq_length // seq_length
                updated_mask = func.max_pool1d(
                    mask.float(), kernel_size=scale, stride=scale
                )  # type: ignore
                updated_masks.append(updated_mask.bool())
            else:
                updated_masks.append(mask)
        return updated_masks

    def forward(
        self, multiscale: List[Tensor], vid_mask: Tensor
    ) -> QueryProposalsOutput:
        """Forward pass of the QuerySelector.

        Args:
            multiscale (List[Tensor]): multiscale features.
            vid_mask (Tensor): mask for the source sequence. Shape: [batch_size, Lv]

        Returns:
            QueryProposalsOutput: query proposals.
        """
        masks = self._prepare_mask(multiscale, vid_mask)

        # compute reference points
        memory_aux, output_proposals, mask = gen_encoder_output_proposals(
            fpn_features=multiscale,
            memory_padding_masks=masks,
            default_widths=self.default_widths,
        )

        # get proposal outputs
        return self.get_query_proposals(memory_aux, output_proposals, mask)


class DetectorEncoder(nn.Module):
    """Stack of the encoder layers."""

    def __init__(
        self,
        model_dim: int,
        num_encoder_layers: int = 3,
        dropout: float = 0.1,
        droppath: float = 0.1,
    ) -> None:
        """Initialize Detector Encoder.

        Args:
            model_dim (int): Hidden dimension of the model
            num_encoder_layers (int): Number of encoder layers.
            dropout (float): Dropout rate
            droppath (float): Droppath rate
        """
        super().__init__()
        self.model_dim = model_dim
        self.enc_layers = num_encoder_layers

        # Init Encoder
        general_encoder_layer = TransformerEncoderLayer(
            model_dim, dropout=dropout, droppath=droppath
        )
        self.encoder = TransformerEncoder(general_encoder_layer, num_encoder_layers)

    def forward(
        self,
        src: Tensor,
        mask: Tensor,
        pos: Tensor,
        video_length: Tensor,
    ) -> DetEncoderOutput:
        """Forward pass of the Transformer.

        Args:
            src (Tensor): source sequence. Shape: [Lv, batch_size, dim]
            mask (Tensor): mask for the source sequence. Shape: [batch_size, Lv]
            pos (Tensor): positional embedding. Shape: [Lv, batch_size, dim]
            video_length (Tensor): length of the video. Shape: [batch_size]

        Returns:
            DetEncoderOutput: output of the encoder
        """
        vid_src = src[:video_length]  # (L_video, batch_size, dim)
        vid_mask = mask[:, :video_length]  # (batch_size, L_video)
        vid_pos = pos[:video_length]  # (L_video, batch_size, dim)

        # encoder forward pass
        memory = self.encoder(vid_src, vid_pos, src_key_padding_mask=~vid_mask)

        return DetEncoderOutput(memory=memory, vid_pos=vid_pos, vid_mask=vid_mask)


class MomentDetector(nn.Module):  # noqa: WPS230
    """Transformer module from DETR."""

    def __init__(  # noqa: WPS211
        self,
        reference: Optional[nn.Module],
        model_dim: int = 512,
        cont_pos_tradeoff: int = 0,
        num_queries: int = 25,
        use_rpn: bool = True,
        use_encoder_features: bool = True,
        num_decoder_layers: int = 3,
        dropout: float = 0.1,
        droppath: float = 0.1,
        temperature: int = 10000,
        prior_prob: float = 0.35,
        unique_content_queries: bool = True,
        init_spans_with_zeros: bool = True,
        return_intermediate_dec: bool = True,
        predict_quality_score: bool = True,
        num_groups: int = 5,
        span_noise_scale: float = 0.4,
        negative_offset: float = 1.0,
        look_at_target: bool = False,
        aux_anchors_type: Tuple[str, ...] = (),
    ):
        """
        Initialize a Transformer module.

        Args:
            reference (nn.Module): anchors generator.
            model_dim (int): hidden dimension of the model
            cont_pos_tradeoff (int): Offset for the dim content/position dims.
            num_queries (int): number of queries
            use_rpn (bool): whether to use RPN as anchor generator or reference.
            use_encoder_features (bool): whether to use encoder features as content queries or not.
            num_decoder_layers (int): number of decoder layers
            dropout (float): dropout rate
            droppath (float): droppath rate
            temperature (int): temperature of the pos emb.
            prior_prob (float): prior foreground prob.
            unique_content_queries (bool): unique content embeddings.
            init_spans_with_zeros (bool): whether to init last mlp layer with zeros or not
            return_intermediate_dec (bool): whether to return intermediate results of the decoder
            predict_quality_score (bool): predict iou of the predicted interval
            num_groups (int): number of noised gt groups.
            span_noise_scale (float): noise scale for the bbox
            negative_offset (float): offset for negative samples
            look_at_target (bool): if true aux anchors can see main queries
            aux_anchors_type (Tuple[str]): aux anchors preparation method. Could be of {"collab", "denoise"}
        """
        super().__init__()
        self.model_dim = model_dim
        self.cont_pos_tradeoff = cont_pos_tradeoff
        self.dec_layers = num_decoder_layers
        self.num_groups = num_groups
        self.span_noise_scale = span_noise_scale
        self.use_rpn = use_rpn
        self.use_encoder_features = use_encoder_features
        self.look_at_target = look_at_target
        self.aux_anchors_type = aux_anchors_type
        self.unique_content_queries = unique_content_queries
        self.init_spans_with_zeros = init_spans_with_zeros
        self.predict_quality_score = predict_quality_score
        self.negative_offset = negative_offset

        # defince query tokens for decoder layers
        self.num_queries: int = num_queries
        self.refpoint_embed = None if use_rpn else reference

        # content query encoder
        self.init_content_queries(num_queries, model_dim, aux_anchors_type)

        decoder_layer = TransformerDecoderLayer(
            d_model=model_dim,
            cont_pos_tradeoff=cont_pos_tradeoff,
            dropout=dropout,
            droppath=droppath,
        )

        self.decoder = TransformerDecoder(
            decoder_layer,
            num_decoder_layers,
            return_intermediate=return_intermediate_dec,
            d_model=model_dim,
            temperature=temperature,
            predict_quality_score=predict_quality_score,
            init_spans_with_zeros=init_spans_with_zeros,
        )
        # define heads for classification and box regression
        self.span_embed = SlimMLP(
            input_dim=model_dim, hidden_dim=model_dim, output_dim=2, num_layers=3
        )
        self.class_embed = nn.Linear(model_dim, 1)
        self._init_parameters(prior_prob)

    def init_content_queries(
        self, num_queries: int, model_dim: int, aux_anchors_type: Tuple[str, ...]
    ) -> None:
        """Init contnent queries.

        Args:
            num_queries (int): number of queries to use.
            model_dim (int): model dim.
            aux_anchors_type (Tuple[str, ...]): list of auxiliary anchors.
        """
        if self.use_rpn and self.use_encoder_features:
            self.label_enc = None
        else:
            if self.unique_content_queries:
                self.label_enc = nn.Embedding(num_queries, model_dim)
            else:
                self.label_enc = nn.Embedding(1, model_dim)
            nn.init.normal_(self.label_enc.weight.data, std=EPS)
        if "collab" in aux_anchors_type:
            self.co_mapper = nn.Sequential(
                nn.Linear(model_dim, model_dim), nn.LayerNorm(model_dim)
            )
            nn.init.normal_(self.co_mapper[0].weight.data, std=EPS)
        if "denoise" in aux_anchors_type:
            self.dn_label_enc = nn.Embedding(1, model_dim)
            nn.init.normal_(self.dn_label_enc.weight.data, std=EPS)

    def _init_parameters(self, prior_prob: float) -> None:
        """Init parameters.

        Args:
            prior_prob (float): prior foreground prob.
        """
        # init cls layer with prior prob
        bias_value = math.log(prior_prob / (1 - prior_prob))
        nn.init.normal_(self.class_embed.weight, std=EPS)  # noqa: WPS432
        self.class_embed.bias.data.fill_(bias_value)

        # init last reg layer with zeros
        for module in self.span_embed.linear_mapper.modules():
            if (
                isinstance(module, nn.Linear)
                and module.out_features == 2
                and self.init_spans_with_zeros
            ):
                nn.init.constant_(module.weight.data, 0)  # noqa: WPS219
                nn.init.constant_(module.bias.data, 0)  # noqa: WPS219

    def prepare_regular_detr(
        self,
        proposals: Optional[QueryProposalsOutput],
        refpoint_emb: Tensor,
        batch_size: int = 512,
    ) -> Tuple[Tensor, Tensor]:
        """
        Prepare reference points for detr decoder.

        Args:
            proposals (QueryProposalsOutput): query selector output.
            refpoint_emb (Tensor): positional queries as anchor points
            batch_size (int): batch size

        Returns:
            Tuple[Tensor, Tensor]:
                - input_query_label: label query embedding for detr decoder
                - input_query_spans: reference points for detr decoder
        """
        # prepare content queries
        if self.use_rpn and self.use_encoder_features:
            assert proposals is not None
            input_query_label = proposals.query_embs.transpose(0, 1)
        else:
            if self.label_enc is None:
                raise ValueError("label_enc cannot be None when using anchors")
            input_query_label = self.label_enc.weight[:, None, :]
            if self.unique_content_queries:
                input_query_label = input_query_label.repeat(1, batch_size, 1)
            else:
                input_query_label = input_query_label.repeat(
                    self.num_queries, batch_size, 1
                )

        # prepare pos queries
        input_query_span = (
            refpoint_emb if self.use_rpn else refpoint_emb.repeat(batch_size, 1, 1)
        )
        input_query_span = input_query_span.transpose(0, 1)
        return input_query_label, input_query_span

    def get_collab_queries(  # noqa: WPS234
        self,
        matched_gts: Optional[Tensor],
        anchors_spans: Optional[Tensor],
        encoder_features: Optional[Tensor],
        batch_size: int,
        device: torch.device,
    ) -> Tuple[Optional[Tensor], Optional[Tensor], Optional[Dict[str, Any]]]:  # noqa: WPS221
        """Get coolab queries.

        Args:
            memory_local (Tensor): Output from the encoder. Shape: [Lv, batch_size, dim]
            matched_gts (Optional[Tensor]): gt spans mathced to selected anchors.
            anchors_spans (Optional[Tensor]): selected anchors.
            encoder_features (Optional[Tensor]): selected encoder features.

        Returns:
            Tuple[Optional[Tensor], Optional[Tensor], Optional[Dict[str, Any]]]: prepared collab info
        """
        if (
            "collab" in self.aux_anchors_type
            and self.training
            and matched_gts is not None
            and anchors_spans is not None
        ):
            co_query_label, co_query_span, co_info = prepare_anchors_codetr(
                linear_mapper=self.co_mapper,  # type: ignore
                matched_gts=matched_gts,  # type: ignore
                anchors_per_seq=anchors_spans,  # type: ignore
                encoder_features_per_seq=encoder_features,  # type: ignore
            )
        else:
            co_query_label = torch.empty((0, batch_size, self.model_dim), device=device)
            co_query_span = torch.empty((0, batch_size, 2), device=device)
            co_info = None
        return co_query_label, co_query_span, co_info

    def get_denoise_queries(  # noqa: WPS234
        self,
        targets: Optional[Dict[str, Any]],
        batch_size: int,
        device: torch.device,
    ) -> Tuple[Optional[Tensor], Optional[Tensor], Optional[Dict[str, Any]]]:  # noqa: WPS221
        """Get denoise queries.

        Args:
            targets (Optional[Dict[str, Any]]): gt spans mathced to selected anchors.
            batch_size (int): batch size
            device (torch.device): device

        Returns:
            Tuple[Optional[Tensor], Optional[Tensor], Optional[Dict[str, Any]]]: prepared denoise info
        """
        if (
            "denoise" in self.aux_anchors_type
            and self.training
            and self.num_groups > 0
            and targets is not None
        ):
            dn_query_label, dn_query_span, dn_info = prepare_anchors_dn(
                label_enc=self.dn_label_enc,
                targets=targets,
                num_groups=self.num_groups,
                span_noise_scale=self.span_noise_scale,
                negative_offset=self.negative_offset,
                batch_size=batch_size,
            )
        else:
            dn_query_label = torch.empty((0, batch_size, self.model_dim), device=device)
            dn_query_span = torch.empty((0, batch_size, 2), device=device)
            dn_info = None
        return dn_query_label, dn_query_span, dn_info

    def _get_attention_mask(
        self,
        co_info: Optional[Dict[str, Any]],
        dn_info: Optional[Dict[str, Any]],
    ) -> Optional[Tensor]:
        """Compute attention mask.

        Args:
            co_info (Optional[Dict[str, Any]]): collaborative anchors info
            dn_info (Optional[Dict[str, Any]]): denoise anchors info

        Returns:
            Optional[Tensor]: computed attention mask
        """
        if co_info is None and dn_info is None:
            return None

        tgt_size = self.num_queries
        colab_size = co_info["pad_size"] if co_info is not None else 0
        denoise_size = dn_info["pad_size"] if dn_info is not None else 0
        attn_mask = torch.zeros(
            tgt_size + colab_size + denoise_size,
            tgt_size + colab_size + denoise_size,
            device=self.class_embed.weight.device,
        ).bool()

        total_mask = colab_size + denoise_size
        total_mask = total_mask if self.look_at_target else total_mask + tgt_size

        if dn_info is not None:
            num_groups = dn_info["num_groups"]
            double_pad = denoise_size / num_groups
            # match query cannot see the reconstruct
            attn_mask[denoise_size:, :denoise_size] = True
            # reconstruct cannot see each other
            for idx in range(num_groups):
                double_idx = int(double_pad * idx)
                double_idx_p = int(double_pad * (idx + 1))
                attn_mask[double_idx:double_idx_p, double_idx_p:total_mask] = True
                attn_mask[double_idx:double_idx_p, :double_idx] = True

        if co_info is not None:
            attn_mask[denoise_size + colab_size :, : denoise_size + colab_size] = True
            attn_mask[
                denoise_size : denoise_size + colab_size,
                denoise_size + colab_size : total_mask,
            ] = True

        return attn_mask

    def _predict_spans(
        self, output: Tensor, reference: Tensor
    ) -> Tuple[Tensor, Tensor]:
        """
        Predict spans.

        Args:
            output (Tensor): content vector from CA block
            reference (Tensor): anchor points

        Returns:
            Tuple[Tensor, Tensor]: predicter spans, shape [batch_size, quary_num, 2] and offsets
        """
        offset = self.span_embed(output)
        reference_before_sigmoid = inverse_sigmoid(reference)
        outputs_coord = offset + reference_before_sigmoid
        outputs_coord = outputs_coord.sigmoid()
        # due to the fact that the usual offset that is calculated above adds up before sigmoids
        final_offset = outputs_coord - reference
        return outputs_coord, final_offset

    # pylint: disable=too-many-locals
    def forward(  # noqa: R0913 C901
        self,
        memory_local: Tensor,
        vid_mask: Tensor,
        vid_pos: Tensor,
        matched_gts: Optional[List[Tensor]],
        anchors_spans: Optional[List[Tensor]],
        encoder_features: Optional[List[Tensor]],
        proposals: Optional[QueryProposalsOutput],
        targets: Optional[Dict[str, Any]] = None,
    ) -> DetectorOutput:
        """Forward pass of the Transformer.

        Args:
            memory_local (Tensor): Output from the encoder. Shape: [Lv, batch_size, dim]
            vid_mask (Tensor): mask for the source sequence. Shape: [batch_size, Lv]
            vid_pos (Tensor): positional embedding. Shape: [Lv, batch_size, dim]
            matched_gts (Optional[List[Tensor]]): gt spans mathced to selected anchors.
            anchors_spans (Optional[List[Tensor]]): selected anchors.
            encoder_features (Optional[List[Tensor]]): selected encoder features.
            proposals (Optional[QueryProposalsOutput]): proposals info.
            targets (Optional[Dict[str, Any]]): target meta information. Defaults to None.

        Returns:
            DetectorOutput: detector output schema.
        """
        batch_size = memory_local.size(1)
        device = memory_local.device
        if self.use_rpn:
            assert proposals is not None
            ref_points = proposals.refpoint_embed_detach
        else:
            ref_points = self.refpoint_embed.get_reference_points()  # type: ignore

        input_query_label, input_query_span = self.prepare_regular_detr(
            proposals=proposals,
            refpoint_emb=ref_points,
            batch_size=batch_size,
        )
        co_query_label, co_query_span, co_info = self.get_collab_queries(
            matched_gts,  # type: ignore
            anchors_spans,  # type: ignore
            encoder_features,  # type: ignore
            batch_size,
            device,
        )
        dn_query_label, dn_query_span, dn_info = self.get_denoise_queries(
            targets, batch_size, device
        )
        attn_mask = self._get_attention_mask(co_info, dn_info)
        input_query_label = torch.cat(
            [dn_query_label, co_query_label, input_query_label], dim=0
        )  # type: ignore
        input_query_span = torch.cat(
            [dn_query_span, co_query_span, input_query_span], dim=0
        )  # type: ignore

        hs, reference_points, quality_score = self.decoder(  # noqa: WPS111
            src=memory_local,
            src_key_padding_mask=~vid_mask,
            src_pos=vid_pos,
            content=input_query_label,
            content_mask=attn_mask,
            refpoints_unsigmoid=input_query_span,
        )  # (#layers, #queries, batch_size, dim)

        # get positive class and coords
        outputs_class: Tensor = self.class_embed(
            hs
        )  # (#layers, batch_size, #queries, 1)
        outputs_coord, offset = self._predict_spans(hs, reference_points)

        return DetectorOutput(
            outputs_class=outputs_class,
            outputs_coord=outputs_coord,
            offsets=offset,
            quality_scores=quality_score,
            co_info=co_info,
            dn_info=dn_info,
        )


# --- File: src.pipelines.evaluate ---
import argparse
import pprint

from tqdm import tqdm
import os
from collections import defaultdict
from easydict import EasyDict

import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))






import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader








import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s.%(msecs)03d:%(levelname)s:%(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)


def eval_epoch_post_processing(submission, opt, gt_data, save_submission_filename):
    """Evaluates epoch with post processing.

    Args:
        submission: Submission.
        opt: Options.
        gt_data: Ground truth data.
        save_submission_filename: Filename.

    Returns:
        tuple: metrics, latest_file_paths
    """
    logger.info("Saving/Evaluating before nms results")
    submission_path = os.path.join(opt.results_dir, save_submission_filename)
    save_jsonl(submission, submission_path)

    if opt.eval_split_name in ["val", "test"]:
        metrics = eval_submission(submission, gt_data)
        save_metrics_path = submission_path.replace(".jsonl", "_metrics.json")
        save_json(metrics, save_metrics_path, save_pretty=True, sort_keys=False)
        latest_file_paths = [submission_path, save_metrics_path]
    else:
        metrics = None
        latest_file_paths = [
            submission_path,
        ]

    return metrics, latest_file_paths


@torch.no_grad()
def compute_mr_results(model, eval_loader, opt, criterion=None):
    """Computes MR results.

    Args:
        model: Model.
        eval_loader: Eval loader.
        opt: Options.
        criterion: Criterion.

    Returns:
        tuple: mr_res, loss_meters
    """
    batch_input_fn = prepare_batch_inputs
    loss_meters = defaultdict(AverageMeter)

    mr_res = []
    for batch in tqdm(eval_loader, desc="compute st ed scores"):
        query_meta = batch[0]
        model_inputs, targets = batch_input_fn(batch[1], opt.device)
        outputs = model(**model_inputs)

        # compose predictions
        pred_spans = outputs["pred_spans"].cpu()  # (bsz, #queries, 2)
        prob = F.softmax(
            outputs["pred_logits"], -1
        )  # (batch_size, #queries, #classes=2)
        scores = prob[
            ..., 0
        ].cpu()  # * (batch_size, #queries)  foreground label is 0, we directly take it

        for idx, (meta, spans, score) in enumerate(zip(query_meta, pred_spans, scores)):
            spans = span_cxw_to_xx(spans) * meta["duration"]
            cur_ranked_preds = torch.cat([spans, score[:, None]], dim=1).tolist()
            cur_ranked_preds = sorted(
                cur_ranked_preds, key=lambda x: x[2], reverse=True
            )
            cur_ranked_preds = [
                [float(f"{e:.4f}") for e in row] for row in cur_ranked_preds
            ]

            cur_query_pred = dict(
                qid=meta["qid"],
                query=meta["query"],
                vid=meta["vid"],
                pred_relevant_windows=cur_ranked_preds,
            )

            mr_res.append(cur_query_pred)

        if criterion:
            loss_dict = criterion(outputs, targets)
            weight_dict = criterion.weight_dict
            losses = sum(
                loss_dict[k] * weight_dict[k]
                for k in loss_dict.keys()
                if k in weight_dict
            )
            loss_dict["loss_overall"] = float(losses)
            for k, v in loss_dict.items():
                loss_meters[k].update(
                    float(v) * weight_dict[k] if k in weight_dict else float(v)
                )

    post_processor = PostProcessorDETR(
        clip_length=opt.clip_length,
        min_ts_val=0,
        max_ts_val=300,
        min_w_l=1,
        max_w_l=300,
        move_window_method="left",
        process_func_names=("clip_ts", "round_multiple"),
    )

    mr_res = post_processor(mr_res)
    return mr_res, loss_meters


def get_eval_res(model, eval_loader, opt, criterion):
    """compute and save query and video proposal embeddings"""
    eval_res, eval_loss_meters = compute_mr_results(model, eval_loader, opt, criterion)
    return eval_res, eval_loss_meters


def eval_epoch(model, eval_dataset, opt, save_submission_filename, criterion):
    """Evaluates epoch.

    Args:
        model: Model.
        eval_dataset: Eval dataset.
        opt: Options.
        save_submission_filename: Filename.
        criterion: Criterion.

    Returns:
        tuple: metrics, eval_loss_meters, latest_file_paths
    """
    logger.info("Generate submissions")
    model.eval()
    criterion.eval()

    eval_loader = DataLoader(
        eval_dataset,
        collate_fn=start_end_collate,
        batch_size=opt.eval_bsz,
        num_workers=opt.num_workers,
        shuffle=False,
    )

    submission, eval_loss_meters = get_eval_res(model, eval_loader, opt, criterion)
    metrics, latest_file_paths = eval_epoch_post_processing(
        submission, opt, eval_dataset.data, save_submission_filename
    )
    return metrics, eval_loss_meters, latest_file_paths


def setup_model(opt):
    """setup model/optimizer/scheduler and load checkpoints when needed"""
    logger.info("setup model/optimizer/scheduler")
    model, criterion = build_model_qd_detr(opt)

    if opt.device in ["cuda", "mps"]:
        logger.info(f"{opt.device} enabled.")
        model.to(opt.device)
        criterion.to(opt.device)

    param_dicts = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad]}
    ]
    optimizer = torch.optim.AdamW(param_dicts, lr=opt.lr, weight_decay=opt.wd)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, opt.lr_drop)

    return model, criterion, optimizer, lr_scheduler


def start_inference(opt):
    """Starts inference.

    Args:
        opt: Options.
    """
    logger.info("Setup config, data and model...")

    # dataset & data loader
    dataset_config = EasyDict(
        data_path=opt.val_path if opt.eval_split_name == "val" else opt.test_path,
        ctx_mode=opt.ctx_mode,
        a_feat_dir=opt.a_feat_dir,
        q_feat_dir=opt.t_feat_dir,
        q_feat_type="last_hidden_state",
        a_feat_type=opt.a_feat_type,
        max_q_l=opt.max_q_l,
        max_a_l=opt.max_a_l,
        clip_len=opt.clip_length,
        max_windows=opt.max_windows,
        span_loss_type=opt.span_loss_type,
        load_labels=True,
    )

    eval_dataset = StartEndDataset(**dataset_config)
    model, criterion, _, _ = setup_model(opt)
    checkpoint = torch.load(opt.model_path, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    logger.info("Model checkpoint: {}".format(opt.model_path))

    logger.info("Starting inference...")
    save_submission_filename = "submission.jsonl"

    with torch.no_grad():
        metrics, eval_loss_meters, latest_file_paths = eval_epoch(
            model, eval_dataset, opt, save_submission_filename, criterion
        )
    logger.info("metrics_no_nms {}".format(pprint.pformat(metrics["brief"], indent=4)))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", "-c", type=str, required=True, help="config path")
    parser.add_argument(
        "--model_path", "-m", type=str, required=True, help="model checkpoint path"
    )
    parser.add_argument(
        "--split",
        "-s",
        type=str,
        default="val",
        choices=["val", "test"],
        help="split name: val or test",
    )
    args = parser.parse_args()
    option_manager = BaseOptions(args.config)
    option_manager.parse()
    opt = option_manager.option

    opt.model_path = args.model_path
    opt.eval_split_name = args.split
    start_inference(opt)


# --- File: src.pipelines.create_submission ---
import argparse

from tqdm import tqdm
import os
from easydict import EasyDict

import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))





import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader








import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s.%(msecs)03d:%(levelname)s:%(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)


def eval_epoch_post_processing(submission, opt, gt_data, save_submission_filename):
    logger.info("Saving/Evaluating before nms results")
    submission_path = os.path.join(opt.results_dir, save_submission_filename)
    save_jsonl(submission, submission_path)

    if opt.eval_split_name in ["val", "test"]:
        metrics = eval_submission(submission, gt_data)
        save_metrics_path = submission_path.replace(".jsonl", "_metrics.json")
        save_json(metrics, save_metrics_path, save_pretty=True, sort_keys=False)
        latest_file_paths = [submission_path, save_metrics_path]
    else:
        metrics = None
        latest_file_paths = [
            submission_path,
        ]

    return metrics, latest_file_paths


@torch.no_grad()
def compute_mr_results(model, eval_loader, opt, criterion=None):
    batch_input_fn = prepare_batch_inputs
    mr_res = []
    for batch in tqdm(eval_loader, desc="compute st ed scores"):
        query_meta = batch[0]
        model_inputs, targets = batch_input_fn(batch[1], opt.device)
        outputs = model(**model_inputs)

        # compose predictions
        pred_spans = outputs["pred_spans"].cpu()  # (bsz, #queries, 2)
        prob = F.softmax(
            outputs["pred_logits"], -1
        )  # (batch_size, #queries, #classes=2)
        scores = prob[
            ..., 0
        ].cpu()  # * (batch_size, #queries)  foreground label is 0, we directly take it

        for idx, (meta, spans, score) in enumerate(zip(query_meta, pred_spans, scores)):
            spans = span_cxw_to_xx(spans) * meta["duration"]
            cur_ranked_preds = torch.cat([spans, score[:, None]], dim=1).tolist()
            cur_ranked_preds = sorted(
                cur_ranked_preds, key=lambda x: x[2], reverse=True
            )
            cur_ranked_preds = [
                [float(f"{e:.4f}") for e in row] for row in cur_ranked_preds
            ]
            cur_query_pred = dict(
                qid=meta["qid"],
                query=meta["query"],
                vid=meta["vid"],
                pred_relevant_windows=cur_ranked_preds,
            )
            mr_res.append(cur_query_pred)

    post_processor = PostProcessorDETR(
        clip_length=opt.clip_length,
        min_ts_val=0,
        max_ts_val=300,
        min_w_l=1,
        max_w_l=300,
        move_window_method="left",
        process_func_names=("clip_ts", "round_multiple"),
    )

    mr_res = post_processor(mr_res)

    # finally remove scores from each prediction
    results = []
    for mr in mr_res:
        mr["pred_relevant_windows"] = [
            [start, end] for start, end, _ in mr["pred_relevant_windows"]
        ]
        results.append(mr)
    return results


def get_eval_res(model, eval_loader, opt, criterion):
    """compute and save query and video proposal embeddings"""
    eval_res = compute_mr_results(model, eval_loader, opt, criterion)
    return eval_res


def eval_epoch(model, eval_dataset, opt, save_submission_filename, criterion):
    logger.info("Generate submissions")
    model.eval()
    criterion.eval()

    eval_loader = DataLoader(
        eval_dataset,
        collate_fn=start_end_collate,
        batch_size=opt.eval_bsz,
        num_workers=opt.num_workers,
        shuffle=False,
    )

    submission = get_eval_res(model, eval_loader, opt, criterion)
    logger.info("Saving/Evaluating before nms results")
    submission_path = os.path.join(opt.results_dir, save_submission_filename)
    save_jsonl(submission, submission_path)


def setup_model(opt):
    """setup model/optimizer/scheduler and load checkpoints when needed"""
    logger.info("setup model/optimizer/scheduler")
    model, criterion = build_model_qd_detr(opt)

    if opt.device == "cuda":
        logger.info("CUDA enabled.")
        model.to(opt.device)
        criterion.to(opt.device)

    param_dicts = [
        {"params": [p for n, p in model.named_parameters() if p.requires_grad]}
    ]
    optimizer = torch.optim.AdamW(param_dicts, lr=opt.lr, weight_decay=opt.wd)
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, opt.lr_drop)
    return model, criterion, optimizer, lr_scheduler


def start_inference(opt):
    logger.info("Setup config, data and model...")

    # dataset & data loader
    dataset_config = EasyDict(
        data_path=opt.submission_path,
        ctx_mode=opt.ctx_mode,
        a_feat_dir=opt.a_sub_feat_dir,
        q_feat_dir=opt.t_sub_feat_dir,
        q_feat_type="last_hidden_state",
        a_feat_type=opt.a_feat_type,
        max_q_l=opt.max_q_l,
        max_a_l=opt.max_a_l,
        clip_len=opt.clip_length,
        max_windows=opt.max_windows,
        span_loss_type=opt.span_loss_type,
        load_labels=False,
    )

    eval_dataset = StartEndDataset(**dataset_config)
    model, criterion, _, _ = setup_model(opt)
    checkpoint = torch.load(opt.model_path, weights_only=False)
    model.load_state_dict(checkpoint["model"])
    logger.info("Model checkpoint: {}".format(opt.model_path))

    logger.info("Starting inference...")
    save_submission_filename = "private_submission.jsonl"

    with torch.no_grad():
        eval_epoch(model, eval_dataset, opt, save_submission_filename, criterion)

    logger.info("Done")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", "-c", type=str, required=True, help="config path")
    parser.add_argument(
        "--model_path", "-m", type=str, required=True, help="model checkpoint path"
    )
    args = parser.parse_args()
    option_manager = BaseOptions(args.config)
    option_manager.parse()
    opt = option_manager.option
    opt.model_path = args.model_path
    opt.eval_split_name = "private"
    start_inference(opt)


# --- File: src.pipelines.train ---
import os
import pprint
import random
import argparse
import copy
import numpy as np
from tqdm import tqdm, trange
from collections import defaultdict

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from easydict import EasyDict

import sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname(__file__), "..")))






    AverageMeter,
    save_checkpoint,
    rename_latest_to_best,
)



import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    format="%(asctime)s.%(msecs)03d:%(levelname)s:%(name)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    level=logging.INFO,
)


def set_seed(seed, use_cuda=True):
    """Sets the random seed.

    Args:
        seed (int): Seed.
        use_cuda (bool): Use cuda.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if use_cuda:
        torch.cuda.manual_seed_all(seed)


def train_epoch(model, criterion, train_loader, optimizer, opt, epoch_i, scaler=None):
    """Trains for one epoch.

    Args:
        model: Model.
        criterion: Criterion.
        train_loader: Train loader.
        optimizer: Optimizer.
        opt: Options.
        epoch_i (int): Epoch index.
        scaler: GradScaler for AMP. Defaults to None.
    """
    logger.info(f"[Epoch {epoch_i + 1}]")
    model.train()
    criterion.train()

    # init meters
    loss_meters = defaultdict(AverageMeter)

    use_amp = opt.get("use_amp", False)
    device_type = "cuda" if "cuda" in str(opt.device) else ("mps" if "mps" in str(opt.device) else "cpu")
    
    # Configure autocast keyword args
    autocast_kwargs = {"device_type": device_type, "enabled": use_amp}
    if device_type in ["cpu", "mps"]:
        autocast_kwargs["dtype"] = torch.bfloat16

    num_training_examples = len(train_loader)
    for batch_idx, batch in tqdm(
        enumerate(train_loader), desc="Training Iteration", total=num_training_examples
    ):
        model_inputs, targets = prepare_batch_inputs(batch[1], opt.device)

        optimizer.zero_grad()

        with torch.amp.autocast(**autocast_kwargs):
            outputs = (
                model(**model_inputs, targets=targets)
                if opt.model_name == "cg_detr"
                else model(**model_inputs)
            )
            loss_dict = criterion(outputs, targets)
            losses = sum(
                loss_dict[k] * criterion.weight_dict[k]
                for k in loss_dict.keys()
                if k in criterion.weight_dict
            )

        if scaler is not None:
            scaler.scale(losses).backward()
            if opt.grad_clip > 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), opt.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            losses.backward()
            if opt.grad_clip > 0:
                nn.utils.clip_grad_norm_(model.parameters(), opt.grad_clip)
            optimizer.step()

        loss_dict["loss_overall"] = float(losses)
        for k, v in loss_dict.items():
            loss_meters[k].update(
                float(v) * criterion.weight_dict[k]
                if k in criterion.weight_dict
                else float(v)
            )

    write_log(opt, epoch_i, loss_meters)


def train(model, criterion, optimizer, lr_scheduler, train_dataset, val_dataset, opt):
    """Trains the model.

    Args:
        model: Model.
        criterion: Criterion.
        optimizer: Optimizer.
        lr_scheduler: LR scheduler.
        train_dataset: Train dataset.
        val_dataset: Val dataset.
        opt: Options.
    """
    opt.train_log_txt_formatter = "{time_str} [Epoch] {epoch:03d} [Loss] {loss_str}\n"
    opt.eval_log_txt_formatter = "{time_str} [Epoch] {epoch:03d} [Loss] {loss_str} [Metrics] {eval_metrics_str}\n"
    save_submission_filename = "latest_{}_val_preds.jsonl".format(opt.dset_name)

    train_loader = DataLoader(
        train_dataset,
        collate_fn=start_end_collate,
        batch_size=opt.bsz,
        num_workers=opt.num_workers,
        shuffle=True,
    )

    if opt.model_ema:
        logger.info("Using model EMA...")
        model_ema = ModelEMA(model, decay=opt.ema_decay)

    # Setup AMP GradScaler for CUDA if enabled
    scaler = torch.amp.GradScaler() if opt.get("use_amp", False) and "cuda" in str(opt.device) else None

    prev_best_score = 0
    for epoch_i in trange(opt.n_epoch, desc="Epoch"):
        train_epoch(model, criterion, train_loader, optimizer, opt, epoch_i, scaler=scaler)
        lr_scheduler.step()

        if opt.model_ema:
            model_ema.update(model)

        if (epoch_i + 1) % opt.eval_epoch_interval == 0:
            with torch.no_grad():
                if opt.model_ema:
                    metrics, eval_loss_meters, latest_file_paths = eval_epoch(
                        model_ema.module,
                        val_dataset,
                        opt,
                        save_submission_filename,
                        criterion,
                    )
                else:
                    metrics, eval_loss_meters, latest_file_paths = eval_epoch(
                        model, val_dataset, opt, save_submission_filename, criterion
                    )

            write_log(opt, epoch_i, eval_loss_meters, metrics=metrics, mode="val")
            logger.info("metrics {}".format(pprint.pformat(metrics["brief"], indent=4)))

            stop_score = metrics["brief"]["MR-full-R1@0.7"]

            if stop_score > prev_best_score:
                prev_best_score = stop_score
                save_checkpoint(model, optimizer, lr_scheduler, epoch_i, opt)
                logger.info("The checkpoint file has been updated.")
                rename_latest_to_best(latest_file_paths)


def main(opt, resume=None):
    """Main function.

    Args:
        opt: Options.
        resume: Resume path.
    """
    logger.info("Setup config, data and model...")
    set_seed(opt.seed)

    # dataset & data loader
    dataset_config = EasyDict(
        data_path=opt.train_path,
        ctx_mode=opt.ctx_mode,
        a_feat_dir=opt.a_feat_dir,
        q_feat_dir=opt.t_feat_dir,
        q_feat_type="last_hidden_state",
        a_feat_type=opt.a_feat_type,
        max_q_l=opt.max_q_l,
        max_a_l=opt.max_a_l,
        clip_len=opt.clip_length,
        max_windows=opt.max_windows,
        span_loss_type=opt.span_loss_type,
        load_labels=True,
    )

    train_dataset = StartEndDataset(**dataset_config)
    copied_eval_config = copy.deepcopy(dataset_config)
    copied_eval_config.data_path = opt.val_path
    eval_dataset = StartEndDataset(**copied_eval_config)

    # prepare model
    model, criterion, optimizer, lr_scheduler = setup_model(opt)

    logger.info(f"Model {model}")
    count_parameters(model, verbose=True)

    if resume is not None:
        checkpoint = torch.load(resume, weights_only=False)
        model.load_state_dict(checkpoint["model"])
        logger.info("Loaded model checkpoint: {}".format(resume))

    if opt.get("use_compile", False):
        logger.info("Compiling model with torch.compile...")
        try:
            model = torch.compile(model, mode="max-autotune")
            logger.info("Model compilation enabled successfully.")
        except Exception as e:
            logger.warning(f"Failed to compile model: {e}. Falling back to uncompiled model.")

    logger.info("Start Training...")

    # start training
    train(model, criterion, optimizer, lr_scheduler, train_dataset, eval_dataset, opt)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", "-c", type=str, required=True, help="config path")
    parser.add_argument(
        "--resume",
        "-r",
        type=str,
        help="specify model path for fine-tuning. If None, train the model from scratch.",
    )
    args = parser.parse_args()
    option_manager = BaseOptions(args.config)
    option_manager.parse()
    opt = option_manager.option
    main(opt, args.resume)


# --- END SRC BUNDLE ---
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))  # Ensure src/ is in PYTHONPATH if needed



In [ ]:
def get_dir_size(path="."):
    total_size = 0
    try:
        with os.scandir(path) as it:
            for entry in it:
                if entry.is_file():
                    # entry.stat() is cached on some systems, making this very fast
                    total_size += entry.stat().st_size
                elif entry.is_dir():
                    # Recursively call the function for subdirectories
                    total_size += get_dir_size(entry.path)
    except PermissionError:
        # Handle folders you don't have access to
        return 0
    return total_size


def format_size(bytes):
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if bytes < 1024:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024


def _copy_single_file(src_file, root_src, dst_dir):
    """Helper function to copy a single file. (Runs inside the thread)"""
    # Calculate the relative path to maintain folder structure
    relative_path = src_file.relative_to(root_src)
    dest_item = dst_dir / relative_path

    # Ensure the destination subdirectory exists
    dest_item.parent.mkdir(parents=True, exist_ok=True)

    # Copy the file along with its metadata
    shutil.copy2(src_file, dest_item)
    return True


def copy_dir_with_progress(src, dst, max_workers=16):
    """
    Copies a directory recursively using multiple threads with a tqdm progress bar.
    """
    src_path = Path(src)
    dst_path = Path(dst)

    if not src_path.exists():
        print(f"Error: Source directory '{src}' does not exist.")
        return

    # 1. Scan and count all files
    print(f"Scanning '{src}' for files...")
    all_files = [f for f in src_path.rglob("*") if f.is_file()]
    total_files = len(all_files)

    if total_files == 0:
        print("No files found to copy.")
        return

    print(
        f"Found {total_files} files. Starting multithreaded copy with {max_workers} workers..."
    )

    # 2. Set up the Thread Pool
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all the copy tasks to the thread pool
        futures = {
            executor.submit(_copy_single_file, item, src_path, dst_path): item
            for item in all_files
        }

        # 3. Update the tqdm progress bar as each thread completes its task
        for future in tqdm(
            concurrent.futures.as_completed(futures),
            total=total_files,
            desc="Copying Data",
            unit="file",
        ):
            try:
                future.result()  # This will raise an exception if the thread failed
            except Exception as e:
                print(f"Error copying {futures[future].name}: {e}")

In [ ]:
# # # NOTE: Move dir from Drive to Colab for faster execution

# # copy_dir_with_progress(
# #     src=str(FEATURES_DIR),
# #     dst="/content"
# # )


# %timeit

# print("Copy feature dir")


# !cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/features.zip /content/

# !unzip /content/features.zip

# print("Copy pre-processed dir")

# !cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/preprocessed /content/

### Test Dataset

In [ ]:
config_path = PREPROCESSED_DIR / "train_config_clotho.yml"
opt = read_yaml(config_path)
opt

In [ ]:
!cp -r /content/drive/MyDrive/Dataset/DCASE2026/CLOTHO-MOMENT/preprocessed /content/

In [ ]:
opt["train_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_train.jsonl"
)
opt["val_path"] = str(
    Path("/content") / "preprocssed" / "clotho_moment_valid_val.jsonl"
)
opt["test_path"] = str(
    Path("/content") / "preprocssed" / "clotho_moment_valid_test.jsonl"
)
opt["a_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap")
opt["t_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap_text")

In [ ]:
dataset_config = EasyDict(
    data_path=opt.get("train_path"),
    ctx_mode=opt.get("ctx_mode"),
    a_feat_dir=opt.get("a_feat_dir"),
    q_feat_dir=opt.get("t_feat_dir"),
    q_feat_type="last_hidden_state",
    a_feat_type=opt.get("a_feat_type"),
    max_q_l=opt.get("max_q_l"),
    max_a_l=opt.get("max_a_l"),
    clip_len=opt.get("clip_length"),
    max_windows=opt.get("max_windows"),
    span_loss_type=opt.get("span_loss_type"),
    load_labels=True,
)

In [ ]:
train_dataset = StartEndDataset(
    **dataset_config,
)

In [ ]:
dataset_config

In [ ]:
train_sample = train_dataset[0]

train_sample

### Core Model

#### Positional Encoding

In [ ]:
# Ref: https://arxiv.org/abs/2108.12409

def get_alibi_slopes(n_heads: int):
    """Generate slopes for each head (geometric progression)"""
    start = 2 ** (-8.0 / n_heads)
    return torch.tensor(
        [start * (start**i) for i in range(n_heads)], dtype=torch.float32
    )


def create_alibi_bias(n_heads: int, seq_len: int, device: torch.device):
    """Create the ALiBi bias matrix: [n_heads, seq_len, seq_len]"""
    slopes = get_alibi_slopes(n_heads).to(device).view(n_heads, 1, 1)

    # Create distance matrix (negative for recency bias)
    pos = torch.arange(seq_len, device=device)
    distances = pos.unsqueeze(0) - pos.unsqueeze(1)  # [seq_len, seq_len]

    # Bias = -slope * distance  (only penalize past, but usually full matrix)
    alibi = distances.unsqueeze(0) * slopes  # [n_heads, seq_len, seq_len]
    return alibi  # You add this to attention scores (before softmax)


# Example integration in attention
class AttentionWithALiBi(nn.Module):
    def __init__(self, n_heads: int, head_dim: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = head_dim
        self.alibi_slopes = get_alibi_slopes(n_heads)

    def forward(self, q: torch.Tensor, k: torch.Tensor, v: torch.Tensor, mask=None):
        # q, k, v: [batch, heads, seq_len, head_dim]
        batch, heads, seq_len, _ = q.shape

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Add ALiBi bias
        alibi = create_alibi_bias(heads, seq_len, q.device)
        scores = scores + alibi

        if mask is not None:
            scores = scores + mask

        attn = F.softmax(scores, dim=-1)
        output = torch.matmul(attn, v)
        return output

#### Attention Module


1. Multi-head Attention


Consider

- Mamba-based Cross Attention

In [ ]:
# ==================== Saliency Guidance (from SG-DETR) ====================
import torch
import torch.nn as nn
import torch.nn.functional as F


class SaliencyGuidedCrossAttention(nn.Module):
    """Saliency-Guided Cross Attention (SGCA)"""

    def __init__(self, d_model=512, nhead=8, dropout=0.1):
        super().__init__()
        self.cross_attn = nn.MultiheadAttention(
            d_model, nhead, dropout=dropout, batch_first=True
        )
        self.saliency_proj = nn.Linear(d_model, 1)  # Predict local saliency
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, audio_feat, text_feat, audio_mask=None, text_mask=None):
        """
        audio_feat: (B, L_a, D)
        text_feat:  (B, L_t, D)
        """
        # 1. Compute local saliency (how relevant each audio clip is to the text)
        # Use mean-pooled text as query
        text_pooled = text_feat.mean(dim=1, keepdim=True)  # (B, 1, D)
        local_saliency = self.saliency_proj(audio_feat)  # (B, L_a, 1)
        local_saliency = torch.sigmoid(local_saliency)  # [0, 1]

        # 2. Standard cross-attention (Audio attends to Text)
        attn_output, _ = self.cross_attn(
            query=audio_feat,
            key=text_feat,
            value=text_feat,
            key_padding_mask=~text_mask if text_mask is not None else None,
        )

        # 3. Apply saliency guidance (soft weighting)
        attn_output = attn_output * local_saliency

        # Residual + Norm
        audio_feat = self.norm(audio_feat + self.dropout(attn_output))

        return audio_feat, local_saliency.squeeze(-1)  # (B, L_a)


class SaliencyAmplifier(nn.Module):
    """Refine local saliency using global context (SG-DETR style)"""

    def __init__(self, d_model=512):
        super().__init__()
        self.global_proj = nn.Sequential(
            nn.Linear(d_model, d_model), nn.ReLU(), nn.Linear(d_model, 1)
        )

    def forward(self, audio_feat, local_saliency):
        global_context = audio_feat.mean(dim=1)  # (B, D)
        global_score = torch.sigmoid(self.global_proj(global_context)).squeeze(-1)
        refined_saliency = local_saliency * global_score.unsqueeze(1)
        return refined_saliency

#### Encoder

#### Decoder

In [ ]:
def _get_activation_fn(activation):
    """Return an activation function given a string"""
    if activation == "relu":
        return F.relu
    if activation == "gelu":
        return F.gelu
    if activation == "glu":
        return F.glu
    if activation == "prelu":
        return nn.PReLU()
    if activation == "selu":
        return F.selu
    raise RuntimeError(f"activation should be relu/gelu, not {activation}.")


def _get_clones(module, N):
    """Creates N clones of the module.

    Args:
        module: Module to clone.
        N (int): Number of clones.

    Returns:
        nn.ModuleList: List of cloned modules.
    """
    return nn.ModuleList([copy.deepcopy(module) for i in range(N)])


def build_transformer(args):
    """Builds the transformer model.

    Args:
        args: Configuration arguments.

    Returns:
        Transformer: The transformer model.
    """
    return Transformer(
        d_model=args.hidden_dim,
        dropout=args.dropout,
        nhead=args.nheads,
        dim_feedforward=args.dim_feedforward,
        num_encoder_layers=args.enc_layers,
        num_decoder_layers=args.dec_layers,
        normalize_before=False,
        return_intermediate_dec=True,
        activation="prelu",
    )

#### Main


#### Criterion

In [ ]:
@torch.no_grad()
def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k
    output: (#items, #classes)
    target: int,
    """
    maxk = max(topk)
    num_items = output.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target)

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / num_items))
    return res

### Training and Validation


#### Computation 

#### Training pipeline


In [ ]:
train_config_dir = "/content/preprocessed/train_config_clotho.yml"
option_manager = BaseOptions(train_config_dir)
option_manager.parse()
train_opt = option_manager.option
train_opt.num_workers = 8

# --- Toggle between 'baseline' and 'enhanced' variant ---
train_opt.model_variant = "enhanced"  # Change to "baseline" to run original code

if train_opt.model_variant == "enhanced":
    train_opt.use_amp = True
    train_opt.use_focal_loss = True
    train_opt.use_flash_attention = True
    train_opt.use_compile = False  # Set to True if PyTorch 2.0+ and CUDA are available and compiled
else:
    train_opt.use_amp = False
    train_opt.use_focal_loss = False
    train_opt.use_flash_attention = False
    train_opt.use_compile = False

print(f"Selected variant: {train_opt.model_variant.upper()}")
print(f"  AMP (Mixed Precision): {train_opt.use_amp}")
print(f"  Focal Loss: {train_opt.use_focal_loss}")
print(f"  Flash Attention: {train_opt.use_flash_attention}")
print(f"  Compile: {train_opt.use_compile}")

os.makedirs("/content/results_pretraining", exist_ok=True)

train_opt.ckpt_filepath = "/content/results_pretraining/best_checkpoint.pth"
train_opt.train_log_filepath = "/content/results_pretraining/train.log"
train_opt.eval_log_filepath = "/content/results_pretraining/val.log"


In [ ]:
train_opt["train_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_train.jsonl"
)
train_opt["val_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_val.jsonl"
)
train_opt["test_path"] = str(
    Path("/content") / "preprocessed" / "clotho_moment_valid_test.jsonl"
)
train_opt["a_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap")
train_opt["t_feat_dir"] = str(Path("/content") / "clotho-moment" / "clap_text")

In [ ]:
"""
# Updated for A100 and Long Audio
lr_drop: 100           # Drop LR halfway through 200 epochs
bsz: 2048              # Scaled for 80GB VRAM
enc_layers: 3          # Increased depth
dec_layers: 3          # Increased depth
num_queries: 20        # Better coverage for multiple events
model_ema: True        # Enabled for stability
max_a_l: 1500          # Increased to support DCASE long audio (1500s)
saliency_margin: 0.4   # Slightly wider margin for contrastive loss
"""

# train_opt.lr_drop = 100
train_opt.bsz = 3000
# train_opt.enc_layers = 3
# train_opt.dec_layers = 3
# train_opt.num_queries = 20
# train_opt.model_ema = True
# train_opt.max_a_l = 1500
# train_opt.saliency_margin = 0.4

In [ ]:
# clear_gpu_cache()

In [ ]:
# --- Inspect the model's active components ---
model, criterion, optimizer, lr_scheduler = setup_model(train_opt)
print("Focal Loss Active on Criterion:", getattr(criterion, "use_focal_loss", False))

print("
Model components verification:")
attn_types = {}
for name, module in model.named_modules():
    cls_name = module.__class__.__name__
    if "attention" in cls_name.lower() or "attn" in name or "self_attn" in name or "cross_attn" in name:
        attn_types[cls_name] = attn_types.get(cls_name, 0) + 1

for cls_name, count in attn_types.items():
    print(f"  {cls_name}: {count} occurrences")


In [ ]:
train_pipeline(train_opt, is_wandb=True)

In [ ]:
# import torch
# import gc
# from numba import cuda

# def total_gpu_cleanup():
#     # 1. Clear PyTorch references
#     gc.collect()
#     torch.cuda.empty_cache()

#     # 2. Hard reset the hardware context via Numba
#     try:
#         device = cuda.get_current_device()
#         device.reset()
#         print("Hardware context reset successful.")
#     except Exception as e:
#         print(f"Numba reset failed: {e}")

#     # 3. CRITICAL: Re-initialize PyTorch's connection to the GPU
#     # This prevents the cudaErrorInvalidValue you're seeing.
#     if torch.cuda.is_available():
#         torch.cuda.init()
#         # Create a dummy tensor to 'wake up' the driver correctly
#         _ = torch.tensor([1.0]).cuda()

#     print("PyTorch context re-initialized. Memory is fresh.")

# # Run this when you hit OOM
# total_gpu_cleanup()

#### Evaluating

In [ ]:
train_opt.model_path = "/content/results_pretraining/best_checkpoint.pth"

In [ ]:
evaluation_pipeline(opt=train_opt)

In [ ]:
LOCAL_DIR

In [ ]:
ROOT_DIR = Path("/content")
filename = "results_pretraining_11May_002"
shutil.make_archive(
    base_name=ROOT_DIR / filename,
    format="zip",
    root_dir=ROOT_DIR / "results_pretraining",
)

print(f"Archive created at {ROOT_DIR}/{filename}.zip")

In [ ]:
source = f"/content/{filename}.zip"
destination = LOCAL_DIR / "CLOTHO-MOMENT" / "pfmc" / "11-May-2026"
destination.mkdir(parents=True, exist_ok=True)

destination = destination / filename
try:
    shutil.move(source, str(destination))
    print("Move successful!")
except FileNotFoundError:
    print("Source file not found.")
except PermissionError:
    print("Permission denied.")
except Exception as e:
    print(f"An error occurred: {e}")